# 2.3c — Régression en grande dimension : quand p > n (ridge, PCR, PLS)

**Navigation** : [<< 2.3b Naive Bayes](2.3b-Naive-Bayes-Generatif.ipynb) | [2.3d-Modele-Gaussien →](2.3d-Modele-Gaussien-LDA-QDA.ipynb) | Version Python : [2.3c-Regression-Grande-Dimension](2.3c-Regression-Grande-Dimension.ipynb) | [Index](../README.md)

**Kernel** : Rust (evcxr)

> **Jumeau Rust.** Ce carnet est le portage du carnet Python [2.3c-Regression-Grande-Dimension](2.3c-Regression-Grande-Dimension.ipynb) : même fil pédagogique, même régime p > n, même découpage train/test piloté par la graine 42. Quatre différences de forme, à connaître avant de lire :
>
> - **aucune crate** : l'algèbre linéaire est écrite à la main — Gram des lignes, spectre d'une matrice symétrique par rotations de Jacobi, inversion par Gauss-Jordan, NIPALS pour la PLS. Il n'y a donc ni scikit-learn, ni `lstsq`, ni `RidgeCV` ;
> - le générateur aléatoire est **maison** (xorshift64* et Box-Muller) : il ne tire pas la même suite que NumPy, les données diffèrent donc de la source, et **toutes les valeurs chiffrées de ce carnet sont celles du tirage Rust** ;
> - les figures deviennent des **tracés ASCII** : une marque par point pour la courbe RMSEP (`.` PCR, `x` PLS, `-` ligne du ridge optimal, `*` un point de courbe posé sur cette ligne, `:` colonne de la composante la plus chargée en bloc B), des barres horizontales pour les loadings ;
> - les lignes affichées par le code sont reprises **sans accents, à l'identique de la source** : ce sont des témoins de comparaison entre les deux carnets.

## Introduction

Le [2.3](2.3-Regression-lineaire-logistique.ipynb) diagnostiquait l'instabilité d'OLS sous colinéarité (VIF 52) et renvoyait en une phrase à la régularisation ridge. Le [2.6](2.6-Clustering-KMeans-PCA.ipynb) enseignait l'ACP comme réduction **non supervisée**. Ce notebook ouvre le régime que ni l'un ni l'autre n'exécute : **p > n**, où OLS devient indéterminée, et où la question n'est plus seulement *pénaliser* (ridge) mais *réduire* — sur des composantes **ignorant la cible** (PCR) ou **l'utilisant** (PLS).

**Concept central** : une direction $u$ de $X$ vaut pour la prédiction dans la mesure où $\mathrm{Cov}(Y, Xu)$ est grande — pas où $\mathrm{Var}(Xu)$ est grande. L'ACP trie par $\mathrm{Var}$, la PLS trie par $\mathrm{Cov}$ : sur des données où la cible vit dans une direction de *faible* variance, les deux méthodes divergent frontalement.

## 1. Données — p > n, colinéarité par blocs, split reproductible

Construction synthétique bornée et contrôlée : $n = 160$ observations ($120$ train / $40$ test), $p = 205$ variables en **deux blocs** —

- **bloc A** : 200 variables organisées en 40 groupes de 5, corrélées à $\rho = 0.8$ à l'intérieur de chaque groupe, **porteuses d'aucun signal** pour $Y$. Chaque groupe possède une direction de variance propre (son facteur commun, valeur propre $\approx 1 + 4 \times 0.8 = 4.2$) : le bloc A occupe ainsi ~40 directions dominantes de l'espace des variables ;
- **bloc B** : 5 variables corrélées ($\rho = 0.6$) partageant un facteur commun, dont une combinaison linéaire **de même signe** (alignée sur ce facteur) **engendre** $Y$, plus un bruit $\varepsilon$.

C'est le régime discriminant : la structure dominante de $X$ (les 40 facteurs du bloc A) est un leurre — le signal (bloc B) vit sur une seule direction de variance plus modeste, loin dans l'ordre des variances décroissantes. Split train/test et génération entièrement pilotés par `GRAINE = 42`.

Ici `p/n=1,71` : on étudie `p>n`, pas un rapport de dimensions extrême. Les covariances et rangs observés sont ceux d'un tirage fini.

> **Côté Rust.** Le générateur du préambule (xorshift64* puis Box-Muller) remplace `np.random.default_rng(SEED)` ; il produit des normales centrées réduites, comme la source, mais sur une autre suite. La standardisation est ajustée **sur le train seul**, comme dans la source.

In [2]:
// Cellule autonome : prelude (RNG, donnees, algebre) redefini ici, puis le calcul de la section.
// ---- Generateur pseudo-alatoire maison : xorshift64* + Box-Muller (aucun crate).
// La suite n'est pas celle de NumPy : les tirages different, les valeurs chiffrees sont celles de ce carnet.
#[allow(dead_code)]
struct Rng {
    etat: u64,
}

#[allow(dead_code)]
impl Rng {
    fn new(graine: u64) -> Rng {
        let mut etat = graine ^ 0x9E3779B97F4A7C15;
        if etat == 0 {
            etat = 0x2545F4914F6CDD1D;
        }
        Rng { etat }
    }

    fn brut(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x >> 12;
        x ^= x << 25;
        x ^= x >> 27;
        self.etat = x;
        x.wrapping_mul(0x2545F4914F6CDD1D)
    }

    // Uniforme sur [0, 1), 53 bits de mantisse.
    fn uniforme(&mut self) -> f64 {
        ((self.brut() >> 11) as f64 + 0.5) / 9007199254740992.0
    }

    // Normale centree reduite, methode de Box-Muller.
    fn normale(&mut self) -> f64 {
        const TAU: f64 = 6.283185307179586;
        loop {
            let u1 = self.uniforme();
            let u2 = self.uniforme();
            if u1 > 1e-300 {
                return (-2.0 * u1.ln()).sqrt() * (TAU * u2).cos();
            }
        }
    }

    // Entier uniforme dans 0..n, par rejet (pas le biais du modulo).
    fn indice(&mut self, n: usize) -> usize {
        let n = n as u64;
        let limite = u64::MAX - (u64::MAX % n);
        loop {
            let t = self.brut();
            if t < limite {
                return (t % n) as usize;
            }
        }
    }
}

// Permutation aleatoire (Fisher-Yates).
#[allow(dead_code)]
fn permutation(rng: &mut Rng, n: usize) -> Vec<usize> {
    let mut idx: Vec<usize> = (0..n).collect();
    for i in (1..n).rev() {
        let j = rng.indice(i + 1);
        idx.swap(i, j);
    }
    idx
}

// ---- Statistiques de base (ecarts-types au denominateur n, comme NumPy). ----
#[allow(dead_code)]
fn moyenne(v: &[f64]) -> f64 {
    v.iter().sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn variance(v: &[f64]) -> f64 {
    let m = moyenne(v);
    v.iter().map(|x| (x - m) * (x - m)).sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn ecart_type(v: &[f64]) -> f64 {
    variance(v).sqrt()
}

// Matrices stockees ligne par ligne : x[i][j].
#[allow(dead_code)]
fn mat_vec_ligne(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    x.iter()
        .map(|l| l.iter().zip(b).map(|(a, c)| a * c).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn colonnes_moyennes(x: &[Vec<f64>]) -> Vec<f64> {
    let n = x.len();
    (0..x[0].len())
        .map(|j| x.iter().map(|l| l[j]).sum::<f64>() / n as f64)
        .collect()
}

#[allow(dead_code)]
fn colonnes_ecarts(x: &[Vec<f64>]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| ecart_type(&x.iter().map(|l| l[j]).collect::<Vec<f64>>()))
        .collect()
}

#[allow(dead_code)]
fn standardiser(x: &[Vec<f64>], mu: &[f64], sigma: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            l.iter()
                .zip(mu.iter().zip(sigma))
                .map(|(a, (m, s))| (a - m) / s)
                .collect()
        })
        .collect()
}

#[allow(dead_code)]
fn standardiser_vec(v: &[f64], m: f64, s: f64) -> Vec<f64> {
    v.iter().map(|x| (x - m) / s).collect()
}

#[allow(dead_code)]
fn centrer(v: &[f64]) -> Vec<f64> {
    let m = moyenne(v);
    v.iter().map(|x| x - m).collect()
}

#[allow(dead_code)]
fn centrer_colonnes(x: &[Vec<f64>], mu: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| l.iter().zip(mu).map(|(a, m)| a - m).collect())
        .collect()
}

#[allow(dead_code)]
fn correlation(a: &[f64], b: &[f64]) -> f64 {
    let (ca, cb) = (centrer(a), centrer(b));
    let num: f64 = ca.iter().zip(&cb).map(|(x, y)| x * y).sum();
    let den: f64 = ca.iter().map(|x| x * x).sum::<f64>().sqrt()
        * cb.iter().map(|y| y * y).sum::<f64>().sqrt();
    num / den
}

#[allow(dead_code)]
fn rmse(a: &[f64], b: &[f64]) -> f64 {
    (a.iter().zip(b).map(|(x, y)| (x - y) * (x - y)).sum::<f64>() / a.len() as f64).sqrt()
}

// Indice du minimum (le premier en cas d'egalite), comme np.argmin.
#[allow(dead_code)]
fn argmin(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] < v[m] {
            m = k;
        }
    }
    m
}

// Indice du maximum (le premier en cas d'egalite), comme np.argmax.
#[allow(dead_code)]
fn argmax(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] > v[m] {
            m = k;
        }
    }
    m
}

#[allow(dead_code)]
fn logspace(a: f64, b: f64, n: usize) -> Vec<f64> {
    (0..n)
        .map(|i| 10f64.powf(a + (b - a) * i as f64 / (n as f64 - 1.0)))
        .collect()
}

// Colonne j d'une matrice stockee ligne par ligne.
#[allow(dead_code)]
fn colonne(x: &[Vec<f64>], j: usize) -> Vec<f64> {
    x.iter().map(|l| l[j]).collect()
}

// Somme des valeurs absolues sur deux blocs de variables : (0..debut_b, debut_b..fin).
#[allow(dead_code)]
fn masses_blocs(v: &[f64], debut_b: usize) -> (f64, f64) {
    let a: f64 = v[..debut_b].iter().map(|x| x.abs()).sum();
    let b: f64 = v[debut_b..].iter().map(|x| x.abs()).sum();
    (a, b)
}

// ---- Donnees : meme construction que la source (blocs A et B, cible, decoupage, standardisation). ----
#[allow(dead_code)]
const N: usize = 160;
#[allow(dead_code)]
const G_A: usize = 40;
#[allow(dead_code)]
const TAILLE_G: usize = 5;
#[allow(dead_code)]
const P_A: usize = 200;
#[allow(dead_code)]
const P_B: usize = 5;
#[allow(dead_code)]
const P: usize = 205;
#[allow(dead_code)]
const N_TRAIN: usize = 120;
#[allow(dead_code)]
const GRAINE: u64 = 42;

// Variables en groupes ; dans chaque groupe, corr(x_i, x_j) = rho_pair via facteur commun.
#[allow(dead_code)]
fn bloc_facteurs(rng: &mut Rng, n_vars: usize, n_groupes: usize, rho_pair: f64) -> Vec<Vec<f64>> {
    let mut x = vec![vec![0.0; n_vars]; N];
    for g in 0..n_groupes {
        let f: Vec<f64> = (0..N).map(|_| rho_pair.sqrt() * rng.normale()).collect();
        for j in 0..TAILLE_G {
            for i in 0..N {
                x[i][g * TAILLE_G + j] = f[i] + (1.0 - rho_pair).sqrt() * rng.normale();
            }
        }
    }
    x
}

// Un tirage complet depuis la graine : les deux blocs, le bruit, le decoupage et les poids du regime aligne.
#[allow(dead_code)]
struct Jeu {
    x: Vec<Vec<f64>>,
    bruit: Vec<f64>,
    idx: Vec<usize>,
    poids_fact: Vec<f64>,
}

// 40 directions de variance dominante sans signal (bloc A), 1 direction plus modeste generatrice de Y (bloc B).
#[allow(dead_code)]
fn jeu(graine: u64) -> Jeu {
    let mut rng = Rng::new(graine);
    let mut x = bloc_facteurs(&mut rng, P_A, G_A, 0.8);
    let xb = bloc_facteurs(&mut rng, P_B, 1, 0.6);
    for i in 0..N {
        x[i].extend(xb[i].iter().copied());
    }
    let bruit: Vec<f64> = (0..N).map(|_| rng.normale()).collect();
    let idx = permutation(&mut rng, N);
    let poids_fact: Vec<f64> = (0..G_A).map(|_| rng.normale() / (G_A as f64).sqrt()).collect();
    Jeu { x, bruit, idx, poids_fact }
}

// Betas de meme signe, alignes sur le facteur commun du bloc B ; Y = X beta + bruit.
#[allow(dead_code)]
fn cible(j: &Jeu) -> Vec<f64> {
    let formes = [1.3, 1.0, 1.2, 0.9, 1.1];
    let mut beta = vec![0.0; P];
    for k in 0..P_B {
        beta[P_A + k] = formes[k] * 1.2;
    }
    let mut y = mat_vec_ligne(&j.x, &beta);
    for i in 0..N {
        y[i] += j.bruit[i];
    }
    y
}

// Regime aligne : Y2 = facteurs_A . poids_fact * 3.0 + bruit. Les facteurs_A sont des proxis construits
// a partir de la premiere variable observee de chaque groupe du bloc A, pas les facteurs latents exacts.
#[allow(dead_code)]
fn cible_alignee(j: &Jeu) -> Vec<f64> {
    let mut y = vec![0.0; N];
    for i in 0..N {
        let mut s = 0.0;
        for g in 0..G_A {
            s += 0.8f64.sqrt() * j.x[i][g * TAILLE_G] * j.poids_fact[g];
        }
        y[i] = s * 3.0 + j.bruit[i];
    }
    y
}

// Decoupage 120 train / 40 test, standardisation ajustee sur le train uniquement (le geste du 2.1).
#[allow(dead_code)]
struct Prep {
    x_train_s: Vec<Vec<f64>>,
    x_test_s: Vec<Vec<f64>>,
    y_train_z: Vec<f64>,
    y_test_z: Vec<f64>,
    y_ecart: f64,
}

#[allow(dead_code)]
fn preparer(j: &Jeu, y: &[f64]) -> Prep {
    let x_train: Vec<Vec<f64>> = (0..N_TRAIN).map(|k| j.x[j.idx[k]].clone()).collect();
    let x_test: Vec<Vec<f64>> = (N_TRAIN..N).map(|k| j.x[j.idx[k]].clone()).collect();
    let y_train: Vec<f64> = (0..N_TRAIN).map(|k| y[j.idx[k]]).collect();
    let y_test: Vec<f64> = (N_TRAIN..N).map(|k| y[j.idx[k]]).collect();
    let (mu, sigma) = (colonnes_moyennes(&x_train), colonnes_ecarts(&x_train));
    let (y_m, y_s) = (moyenne(&y_train), ecart_type(&y_train));
    Prep {
        x_train_s: standardiser(&x_train, &mu, &sigma),
        x_test_s: standardiser(&x_test, &mu, &sigma),
        y_train_z: standardiser_vec(&y_train, y_m, y_s),
        y_test_z: standardiser_vec(&y_test, y_m, y_s),
        y_ecart: y_s,
    }
}

// ---- 1. Donnees : 40 directions de variance dominante sans signal, 1 direction qui engendre Y ----
let j = jeu(GRAINE);
let y = cible(&j);
let p = preparer(&j, &y);

// Qualite du jeu : part de variance du modele vrai sur le train, et correlations intra-bloc mesurees.
let r2_vrai = 1.0
    - (0..N_TRAIN)
        .map(|k| j.bruit[j.idx[k]] * j.bruit[j.idx[k]])
        .sum::<f64>()
        / N_TRAIN as f64
        / (p.y_ecart * p.y_ecart);
println!(
    "n_train={}, n_test={}, p={}  (p/n = {:.2})",
    p.x_train_s.len(),
    p.x_test_s.len(),
    P,
    P as f64 / N_TRAIN as f64
);
println!("R2 du modele vrai (bloc B seul) sur train : {:.3}", r2_vrai);
println!(
    "correlation intra-groupe mesuree (groupe 0) : {:.3} (visee 0.8)",
    correlation(&colonne(&p.x_train_s, 0), &colonne(&p.x_train_s, 1))
);
println!(
    "correlation intra-B mesuree (2 premieres)  : {:.3} (visee 0.6)",
    correlation(&colonne(&p.x_train_s, P_A), &colonne(&p.x_train_s, P_A + 1))
);

n_train=120, n_test=40, p=205  (p/n = 1.71)


R2 du modele vrai (bloc B seul) sur train : 0.973


correlation intra-groupe mesuree (groupe 0) : 0.851 (visee 0.8)


correlation intra-B mesuree (2 premieres)  : 0.669 (visee 0.6)


### Lecture du résultat

205 variables pour 120 observations d'entraînement (p/n = 1.71) : le régime est bien p > n. Le modèle vrai (bloc B seul) explique 97.3% de la variance du train — le signal existe et il est fort. Les corrélations intra-bloc mesurées (0.851 pour la paire du groupe 0, 0.669 pour les deux premières variables du bloc B, visées 0.8 et 0.6) confirment la structure en groupes ; chaque mesure porte sur une seule paire d'un même tirage, l'écart aux valeurs visées est celui d'un échantillon fini, pas un défaut de construction. Le point décisif pour la suite : chaque groupe du bloc A possède une direction de variance propre (valeur propre $\approx 4.2$ en population), le bloc B une seule direction génératrice plus modeste — le signal est donc **minoritaire dans l'ordre des variances**.

## 2. OLS en p > n — l'indétermination, montrée pas affirmée

Avec $p = 205 > n = 120$, la matrice $X^\top X$ (205 × 205) est **singulière** : son rang est au plus $n$. La solution OLS n'est pas unique. La solution de **norme minimale** passe par le Gram des lignes $K = X X^\top$ (120 × 120, de rang 119) : on diagonalise $K = U \Lambda U^\top$, on pose $\alpha = K^{+} y$ (somme des $u_i^\top y / \lambda_i \cdot u_i$ sur les valeurs propres retenues), puis $\beta = X^\top \alpha$. C'est exactement ce que rend `lstsq` côté Python, et c'est ce que calcule le préambule. La prédiction se projette ainsi sur *toutes* les directions de l'espace des lignes — les ~40 directions de bruit du bloc A comprises. On mesure : rang, écart train/test au plancher irréductible, et sensibilité.

In [3]:
// Cellule autonome : prelude (RNG, donnees, algebre) redefini ici, puis le calcul de la section.
// ---- Generateur pseudo-alatoire maison : xorshift64* + Box-Muller (aucun crate).
// La suite n'est pas celle de NumPy : les tirages different, les valeurs chiffrees sont celles de ce carnet.
#[allow(dead_code)]
struct Rng {
    etat: u64,
}

#[allow(dead_code)]
impl Rng {
    fn new(graine: u64) -> Rng {
        let mut etat = graine ^ 0x9E3779B97F4A7C15;
        if etat == 0 {
            etat = 0x2545F4914F6CDD1D;
        }
        Rng { etat }
    }

    fn brut(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x >> 12;
        x ^= x << 25;
        x ^= x >> 27;
        self.etat = x;
        x.wrapping_mul(0x2545F4914F6CDD1D)
    }

    // Uniforme sur [0, 1), 53 bits de mantisse.
    fn uniforme(&mut self) -> f64 {
        ((self.brut() >> 11) as f64 + 0.5) / 9007199254740992.0
    }

    // Normale centree reduite, methode de Box-Muller.
    fn normale(&mut self) -> f64 {
        const TAU: f64 = 6.283185307179586;
        loop {
            let u1 = self.uniforme();
            let u2 = self.uniforme();
            if u1 > 1e-300 {
                return (-2.0 * u1.ln()).sqrt() * (TAU * u2).cos();
            }
        }
    }

    // Entier uniforme dans 0..n, par rejet (pas le biais du modulo).
    fn indice(&mut self, n: usize) -> usize {
        let n = n as u64;
        let limite = u64::MAX - (u64::MAX % n);
        loop {
            let t = self.brut();
            if t < limite {
                return (t % n) as usize;
            }
        }
    }
}

// Permutation aleatoire (Fisher-Yates).
#[allow(dead_code)]
fn permutation(rng: &mut Rng, n: usize) -> Vec<usize> {
    let mut idx: Vec<usize> = (0..n).collect();
    for i in (1..n).rev() {
        let j = rng.indice(i + 1);
        idx.swap(i, j);
    }
    idx
}

// ---- Statistiques de base (ecarts-types au denominateur n, comme NumPy). ----
#[allow(dead_code)]
fn moyenne(v: &[f64]) -> f64 {
    v.iter().sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn variance(v: &[f64]) -> f64 {
    let m = moyenne(v);
    v.iter().map(|x| (x - m) * (x - m)).sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn ecart_type(v: &[f64]) -> f64 {
    variance(v).sqrt()
}

// Matrices stockees ligne par ligne : x[i][j].
#[allow(dead_code)]
fn mat_vec_ligne(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    x.iter()
        .map(|l| l.iter().zip(b).map(|(a, c)| a * c).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn colonnes_moyennes(x: &[Vec<f64>]) -> Vec<f64> {
    let n = x.len();
    (0..x[0].len())
        .map(|j| x.iter().map(|l| l[j]).sum::<f64>() / n as f64)
        .collect()
}

#[allow(dead_code)]
fn colonnes_ecarts(x: &[Vec<f64>]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| ecart_type(&x.iter().map(|l| l[j]).collect::<Vec<f64>>()))
        .collect()
}

#[allow(dead_code)]
fn standardiser(x: &[Vec<f64>], mu: &[f64], sigma: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            l.iter()
                .zip(mu.iter().zip(sigma))
                .map(|(a, (m, s))| (a - m) / s)
                .collect()
        })
        .collect()
}

#[allow(dead_code)]
fn standardiser_vec(v: &[f64], m: f64, s: f64) -> Vec<f64> {
    v.iter().map(|x| (x - m) / s).collect()
}

#[allow(dead_code)]
fn centrer(v: &[f64]) -> Vec<f64> {
    let m = moyenne(v);
    v.iter().map(|x| x - m).collect()
}

#[allow(dead_code)]
fn centrer_colonnes(x: &[Vec<f64>], mu: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| l.iter().zip(mu).map(|(a, m)| a - m).collect())
        .collect()
}

#[allow(dead_code)]
fn correlation(a: &[f64], b: &[f64]) -> f64 {
    let (ca, cb) = (centrer(a), centrer(b));
    let num: f64 = ca.iter().zip(&cb).map(|(x, y)| x * y).sum();
    let den: f64 = ca.iter().map(|x| x * x).sum::<f64>().sqrt()
        * cb.iter().map(|y| y * y).sum::<f64>().sqrt();
    num / den
}

#[allow(dead_code)]
fn rmse(a: &[f64], b: &[f64]) -> f64 {
    (a.iter().zip(b).map(|(x, y)| (x - y) * (x - y)).sum::<f64>() / a.len() as f64).sqrt()
}

// Indice du minimum (le premier en cas d'egalite), comme np.argmin.
#[allow(dead_code)]
fn argmin(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] < v[m] {
            m = k;
        }
    }
    m
}

// Indice du maximum (le premier en cas d'egalite), comme np.argmax.
#[allow(dead_code)]
fn argmax(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] > v[m] {
            m = k;
        }
    }
    m
}

#[allow(dead_code)]
fn logspace(a: f64, b: f64, n: usize) -> Vec<f64> {
    (0..n)
        .map(|i| 10f64.powf(a + (b - a) * i as f64 / (n as f64 - 1.0)))
        .collect()
}

// Colonne j d'une matrice stockee ligne par ligne.
#[allow(dead_code)]
fn colonne(x: &[Vec<f64>], j: usize) -> Vec<f64> {
    x.iter().map(|l| l[j]).collect()
}

// Somme des valeurs absolues sur deux blocs de variables : (0..debut_b, debut_b..fin).
#[allow(dead_code)]
fn masses_blocs(v: &[f64], debut_b: usize) -> (f64, f64) {
    let a: f64 = v[..debut_b].iter().map(|x| x.abs()).sum();
    let b: f64 = v[debut_b..].iter().map(|x| x.abs()).sum();
    (a, b)
}

// ---- Donnees : meme construction que la source (blocs A et B, cible, decoupage, standardisation). ----
#[allow(dead_code)]
const N: usize = 160;
#[allow(dead_code)]
const G_A: usize = 40;
#[allow(dead_code)]
const TAILLE_G: usize = 5;
#[allow(dead_code)]
const P_A: usize = 200;
#[allow(dead_code)]
const P_B: usize = 5;
#[allow(dead_code)]
const P: usize = 205;
#[allow(dead_code)]
const N_TRAIN: usize = 120;
#[allow(dead_code)]
const GRAINE: u64 = 42;

// Variables en groupes ; dans chaque groupe, corr(x_i, x_j) = rho_pair via facteur commun.
#[allow(dead_code)]
fn bloc_facteurs(rng: &mut Rng, n_vars: usize, n_groupes: usize, rho_pair: f64) -> Vec<Vec<f64>> {
    let mut x = vec![vec![0.0; n_vars]; N];
    for g in 0..n_groupes {
        let f: Vec<f64> = (0..N).map(|_| rho_pair.sqrt() * rng.normale()).collect();
        for j in 0..TAILLE_G {
            for i in 0..N {
                x[i][g * TAILLE_G + j] = f[i] + (1.0 - rho_pair).sqrt() * rng.normale();
            }
        }
    }
    x
}

// Un tirage complet depuis la graine : les deux blocs, le bruit, le decoupage et les poids du regime aligne.
#[allow(dead_code)]
struct Jeu {
    x: Vec<Vec<f64>>,
    bruit: Vec<f64>,
    idx: Vec<usize>,
    poids_fact: Vec<f64>,
}

// 40 directions de variance dominante sans signal (bloc A), 1 direction plus modeste generatrice de Y (bloc B).
#[allow(dead_code)]
fn jeu(graine: u64) -> Jeu {
    let mut rng = Rng::new(graine);
    let mut x = bloc_facteurs(&mut rng, P_A, G_A, 0.8);
    let xb = bloc_facteurs(&mut rng, P_B, 1, 0.6);
    for i in 0..N {
        x[i].extend(xb[i].iter().copied());
    }
    let bruit: Vec<f64> = (0..N).map(|_| rng.normale()).collect();
    let idx = permutation(&mut rng, N);
    let poids_fact: Vec<f64> = (0..G_A).map(|_| rng.normale() / (G_A as f64).sqrt()).collect();
    Jeu { x, bruit, idx, poids_fact }
}

// Betas de meme signe, alignes sur le facteur commun du bloc B ; Y = X beta + bruit.
#[allow(dead_code)]
fn cible(j: &Jeu) -> Vec<f64> {
    let formes = [1.3, 1.0, 1.2, 0.9, 1.1];
    let mut beta = vec![0.0; P];
    for k in 0..P_B {
        beta[P_A + k] = formes[k] * 1.2;
    }
    let mut y = mat_vec_ligne(&j.x, &beta);
    for i in 0..N {
        y[i] += j.bruit[i];
    }
    y
}

// Regime aligne : Y2 = facteurs_A . poids_fact * 3.0 + bruit. Les facteurs_A sont des proxis construits
// a partir de la premiere variable observee de chaque groupe du bloc A, pas les facteurs latents exacts.
#[allow(dead_code)]
fn cible_alignee(j: &Jeu) -> Vec<f64> {
    let mut y = vec![0.0; N];
    for i in 0..N {
        let mut s = 0.0;
        for g in 0..G_A {
            s += 0.8f64.sqrt() * j.x[i][g * TAILLE_G] * j.poids_fact[g];
        }
        y[i] = s * 3.0 + j.bruit[i];
    }
    y
}

// Decoupage 120 train / 40 test, standardisation ajustee sur le train uniquement (le geste du 2.1).
#[allow(dead_code)]
struct Prep {
    x_train_s: Vec<Vec<f64>>,
    x_test_s: Vec<Vec<f64>>,
    y_train_z: Vec<f64>,
    y_test_z: Vec<f64>,
    y_ecart: f64,
}

#[allow(dead_code)]
fn preparer(j: &Jeu, y: &[f64]) -> Prep {
    let x_train: Vec<Vec<f64>> = (0..N_TRAIN).map(|k| j.x[j.idx[k]].clone()).collect();
    let x_test: Vec<Vec<f64>> = (N_TRAIN..N).map(|k| j.x[j.idx[k]].clone()).collect();
    let y_train: Vec<f64> = (0..N_TRAIN).map(|k| y[j.idx[k]]).collect();
    let y_test: Vec<f64> = (N_TRAIN..N).map(|k| y[j.idx[k]]).collect();
    let (mu, sigma) = (colonnes_moyennes(&x_train), colonnes_ecarts(&x_train));
    let (y_m, y_s) = (moyenne(&y_train), ecart_type(&y_train));
    Prep {
        x_train_s: standardiser(&x_train, &mu, &sigma),
        x_test_s: standardiser(&x_test, &mu, &sigma),
        y_train_z: standardiser_vec(&y_train, y_m, y_s),
        y_test_z: standardiser_vec(&y_test, y_m, y_s),
        y_ecart: y_s,
    }
}

// ---- Algebre lineaire : tout passe par la matrice de Gram des lignes K = X X^T (n x n, ici 120 x 120),
// jamais par X^T X (p x p, singuliere des que p > n). ----
#[allow(dead_code)]
fn transpose_mat_vec(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| x.iter().zip(b).map(|(l, &v)| l[j] * v).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn gram(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let n = x.len();
    (0..n)
        .map(|i| {
            (0..n)
                .map(|k| x[i].iter().zip(&x[k]).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// Valeurs propres (decroissantes) et vecteurs propres d'une matrice symetrique par la methode de
// Jacobi cyclique. vecteurs[k][i] est la i-eme composante du k-eme vecteur propre ; le signe de chaque
// vecteur est choisi pour que sa plus grande composante en valeur absolue soit positive.
#[allow(dead_code)]
fn propres_sym(m: &[Vec<f64>]) -> (Vec<f64>, Vec<Vec<f64>>) {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut v: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|k| if i == k { 1.0 } else { 0.0 }).collect())
        .collect();
    for _balayage in 0..100 {
        let diag: f64 = (0..n).map(|i| a[i][i].abs()).fold(0.0f64, f64::max);
        let mut carres = 0.0f64;
        for i in 0..n {
            for k in (i + 1)..n {
                carres += a[i][k] * a[i][k];
            }
        }
        if carres.sqrt() <= 1e-13 * (1.0 + diag) {
            break;
        }
        let mut rotations = 0usize;
        for p in 0..n {
            for q in (p + 1)..n {
                let apq = a[p][q];
                // Pivot negligeable devant la diagonale : on l'annule sans tourner.
                if apq.abs() <= 1e-18 * (a[p][p].abs() + a[q][q].abs()) {
                    a[p][q] = 0.0;
                    a[q][p] = 0.0;
                    continue;
                }
                rotations += 1;
                let theta = (a[q][q] - a[p][p]) / (2.0 * apq);
                let t = if theta.abs() > 1e150 {
                    1.0 / (2.0 * theta)
                } else if theta >= 0.0 {
                    1.0 / (theta + (theta * theta + 1.0).sqrt())
                } else {
                    -1.0 / (-theta + (theta * theta + 1.0).sqrt())
                };
                let c = 1.0 / (t * t + 1.0).sqrt();
                let s = t * c;
                for k in 0..n {
                    if k != p && k != q {
                        let akp = a[k][p];
                        let akq = a[k][q];
                        a[k][p] = c * akp - s * akq;
                        a[p][k] = a[k][p];
                        a[k][q] = s * akp + c * akq;
                        a[q][k] = a[k][q];
                    }
                    let vkp = v[k][p];
                    let vkq = v[k][q];
                    v[k][p] = c * vkp - s * vkq;
                    v[k][q] = s * vkp + c * vkq;
                }
                a[p][p] -= t * apq;
                a[q][q] += t * apq;
                a[p][q] = 0.0;
                a[q][p] = 0.0;
            }
        }
        if rotations == 0 {
            break;
        }
    }
    let mut ordre: Vec<usize> = (0..n).collect();
    ordre.sort_by(|&i, &k| a[k][k].total_cmp(&a[i][i]));
    let valeurs: Vec<f64> = ordre.iter().map(|&i| a[i][i]).collect();
    let vecteurs: Vec<Vec<f64>> = ordre
        .iter()
        .map(|&k| {
            let colonne: Vec<f64> = v.iter().map(|l| l[k]).collect();
            let pivot = argmax(&colonne.iter().map(|x| x.abs()).collect::<Vec<f64>>());
            if colonne[pivot] < 0.0 {
                colonne.iter().map(|x| -x).collect()
            } else {
                colonne
            }
        })
        .collect();
    (valeurs, vecteurs)
}

// Rang numerique : nombre de valeurs propres au-dessus de 1e-8 fois la plus grande.
#[allow(dead_code)]
fn rang(valeurs: &[f64]) -> usize {
    let lmax = valeurs[0];
    valeurs.iter().filter(|v| **v > 1e-8 * lmax).count()
}

// OLS de norme minimale sans ordonnee, par K = X X^T : alpha = U_r L_r^{-1} U_r^T y, puis beta = X^T alpha.
// Les r valeurs propres non nulles fixent la prediction ; le noyau (p - r directions) ne la modifie pas.
#[allow(dead_code)]
fn ols_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64]) -> Vec<f64> {
    let lmax = valeurs[0];
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        if valeurs[k] > 1e-8 * lmax {
            let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
            for i in 0..n {
                alpha[i] += u[k][i] * d / valeurs[k];
            }
        }
    }
    alpha
}

// Coefficients duaux du ridge sans ordonnee : alpha = (K + lambda I)^{-1} y, dans la base des vecteurs propres.
#[allow(dead_code)]
fn ridge_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64], lambda: f64) -> Vec<f64> {
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
        for i in 0..n {
            alpha[i] += u[k][i] * d / (valeurs[k] + lambda);
        }
    }
    alpha
}

// beta = X^T alpha ; renvoie les previsions X beta sur train et test, et la norme L2 de beta.
#[allow(dead_code)]
fn predire(xtr: &[Vec<f64>], xte: &[Vec<f64>], alpha: &[f64]) -> (Vec<f64>, Vec<f64>, f64) {
    let beta = transpose_mat_vec(xtr, alpha);
    let norme = beta.iter().map(|b| b * b).sum::<f64>().sqrt();
    (mat_vec_ligne(xtr, &beta), mat_vec_ligne(xte, &beta), norme)
}

// Bornes du pli f d'une validation croisee a plis contigus (KFold sans melange).
#[allow(dead_code)]
fn bornes_pli(n: usize, plis: usize, f: usize) -> (usize, usize) {
    let (base, reste) = (n / plis, n % plis);
    let debut = f * base + reste.min(f);
    (debut, debut + base + if f < reste { 1 } else { 0 })
}

// Un pli contigu, avec la decomposition spectrale du Gram de son train : elle sert a tous les alpha du ridge.
#[allow(dead_code)]
struct PliContigu {
    xtr: Vec<Vec<f64>>,
    xva: Vec<Vec<f64>>,
    positions_val: Vec<usize>,
    valeurs: Vec<f64>,
    u: Vec<Vec<f64>>,
}

#[allow(dead_code)]
fn plis_contigus(x: &[Vec<f64>], plis: usize) -> Vec<PliContigu> {
    let n = x.len();
    (0..plis)
        .map(|f| {
            let (debut, fin) = bornes_pli(n, plis, f);
            let xtr: Vec<Vec<f64>> = (0..n)
                .filter(|&i| i < debut || i >= fin)
                .map(|i| x[i].clone())
                .collect();
            let xva: Vec<Vec<f64>> = x[debut..fin].to_vec();
            let (valeurs, u) = propres_sym(&gram(&xtr));
            PliContigu {
                xtr,
                xva,
                positions_val: (debut..fin).collect(),
                valeurs,
                u,
            }
        })
        .collect()
}

// RidgeCV(alphas, cv=plis, fit_intercept=False) : l'alpha retenu est celui qui maximise la moyenne
// des R2 de pli (le score par defaut de la validation croisee de scikit-learn). Renvoie son indice.
#[allow(dead_code)]
fn ridge_cv_sur_plis(plis_c: &[PliContigu], y: &[f64], alphas: &[f64]) -> usize {
    let mut score = vec![0.0; alphas.len()];
    for pli in plis_c {
        let ytr: Vec<f64> = (0..y.len())
            .filter(|i| !pli.positions_val.contains(i))
            .map(|i| y[i])
            .collect();
        let yva: Vec<f64> = pli.positions_val.iter().map(|&i| y[i]).collect();
        let sst: f64 = centrer(&yva).iter().map(|x| x * x).sum();
        for (j, &lam) in alphas.iter().enumerate() {
            let (_, p_va, _) = predire(
                &pli.xtr,
                &pli.xva,
                &ridge_alpha(&pli.valeurs, &pli.u, &ytr, lam),
            );
            let sse: f64 = yva.iter().zip(&p_va).map(|(t, p)| (t - p) * (t - p)).sum();
            score[j] += 1.0 - sse / sst;
        }
    }
    argmax(&score)
}

#[allow(dead_code)]
fn ridge_cv_retenu(x: &[Vec<f64>], y: &[f64], alphas: &[f64], plis: usize) -> usize {
    ridge_cv_sur_plis(&plis_contigus(x, plis), y, alphas)
}

// Composantes principales depuis la decomposition de K = X X^T : composante_k = X^T u_k / sqrt(lambda_k).
#[allow(dead_code)]
fn composantes(x: &[Vec<f64>], valeurs: &[f64], u: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let mut comps: Vec<Vec<f64>> = Vec::new();
    for k in 0..rang(valeurs) {
        let mut c = transpose_mat_vec(x, &u[k]);
        let s = valeurs[k].sqrt();
        for v in c.iter_mut() {
            *v /= s;
        }
        comps.push(c);
    }
    comps
}

// Centrage d'un echantillon sur ses propres colonnes : ce que fait PCA a l'ajustement.
#[allow(dead_code)]
fn centrer_propre(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    centrer_colonnes(x, &colonnes_moyennes(x))
}

// ACP, comme sklearn.decomposition.PCA : X est CENTRE sur ses colonnes, covariance (denominateur n),
// valeurs propres, part de variance, composantes. Le centrage ne change rien sur une matrice deja
// standardisee sur elle-meme (train complet) ; il est indispensable sur le train d'un pli, dont les
// colonnes ne sont pas de moyenne nulle.
#[allow(dead_code)]
fn acp(x: &[Vec<f64>]) -> (Vec<f64>, Vec<f64>, Vec<Vec<f64>>) {
    let xc = centrer_propre(x);
    let (valeurs, u) = propres_sym(&gram(&xc));
    let total: f64 = valeurs.iter().sum();
    let comps = composantes(&xc, &valeurs, &u);
    let ratio: Vec<f64> = valeurs.iter().map(|l| l / total).collect();
    (valeurs, ratio, comps)
}

// Variance cumulee jusqu'a la composante k comprise.
#[allow(dead_code)]
fn variance_cumulee(ratio: &[f64], k: usize) -> f64 {
    ratio[..=k].iter().sum()
}

// Scores t[i][k] = x_i . composante_k.
#[allow(dead_code)]
fn scores_composantes(x: &[Vec<f64>], comps: &[Vec<f64>]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            comps
                .iter()
                .map(|c| l.iter().zip(c).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// PCR a k composantes avec ordonnee : y = ybar + sum_{j<k} b_j t_j.
// Les scores d'une ACP sont orthogonaux deux a deux, donc b_j = (t_j . (y - ybar)) / (t_j . t_j).
#[allow(dead_code)]
fn regression_scores(
    t_tr: &[Vec<f64>],
    t_te: &[Vec<f64>],
    y: &[f64],
    k: usize,
) -> (Vec<f64>, Vec<f64>) {
    let ym = moyenne(y);
    let yc: Vec<f64> = y.iter().map(|v| v - ym).collect();
    let mut b = vec![0.0; k];
    for j in 0..k {
        let num: f64 = t_tr.iter().zip(&yc).map(|(t, &v)| t[j] * v).sum();
        let den: f64 = t_tr.iter().map(|t| t[j] * t[j]).sum();
        b[j] = num / den;
    }
    let projet = |tt: &[Vec<f64>]| -> Vec<f64> {
        tt.iter()
            .map(|t| ym + (0..k).map(|j| b[j] * t[j]).sum::<f64>())
            .collect()
    };
    (projet(t_tr), projet(t_te))
}

// RMSE test de la PCR pour k = 1..kmax (une seule ACP, une regression par k).
#[allow(dead_code)]
fn balayage_pcr(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (_, _, mut comps) = acp(xtr);
    comps.truncate(kmax);
    // PCA().fit(xtr).transform(...) : les deux echantillons sont centres sur la moyenne du train.
    let xmu = colonnes_moyennes(xtr);
    let (t_tr, t_te) = (
        scores_composantes(&centrer_colonnes(xtr, &xmu), &comps),
        scores_composantes(&centrer_colonnes(xte, &xmu), &comps),
    );
    (1..=kmax)
        .map(|k| rmse(yte, &regression_scores(&t_tr, &t_te, ytr, k).1))
        .collect()
}

// ---- 2. OLS en p > n : l'indetermination ----
// Le rang se lit sur la matrice de Gram des lignes K = X X^T (120 x 120), jamais sur X^T X (205 x 205).
let j = jeu(GRAINE);
let p = preparer(&j, &cible(&j));
let (valeurs, u) = propres_sym(&gram(&p.x_train_s));
let rang_x = rang(&valeurs);
println!(
    "rang(X_train) = {} sur p = {} colonnes -> {} directions du noyau",
    rang_x,
    P,
    P - rang_x
);
println!("(le centrage des colonnes retire aussi 1 : les lignes centrees sont liees)");
println!(
    "valeurs propres de K : plus grande {:.1} ; plus petite retenue {:.4} ; plus grande ecartee {:.1e}",
    valeurs[0],
    valeurs[rang_x - 1],
    if rang_x < valeurs.len() {
        valeurs[rang_x]
    } else {
        0.0
    }
);

// OLS de norme minimale, sans ordonnee : train parfait, test mauvais.
let alpha_ols = ols_alpha(&valeurs, &u, &p.y_train_z);
let beta_ols = transpose_mat_vec(&p.x_train_s, &alpha_ols);
let (ols_tr, ols_te, ols_norme) = predire(&p.x_train_s, &p.x_test_s, &alpha_ols);
let rmse_ols_test = rmse(&p.y_test_z, &ols_te);
println!(
    "\nOLS (min-norme) : RMSE train = {:.4}   RMSE test = {:.4}",
    rmse(&p.y_train_z, &ols_tr),
    rmse_ols_test
);
println!(
    "plancher irreductible (bruit en unites z) ~ {:.3}  -> OLS est a {:.1} x le bruit",
    1.0 / p.y_ecart,
    rmse_ols_test * p.y_ecart
);

// Sensibilite : une seule ligne de train perturbee decale la solution.
let mut rng_p = Rng::new(GRAINE + 1);
let mut ecarts: Vec<f64> = Vec::new();
for _b in 0..5 {
    let mut x_pert = p.x_train_s.clone();
    for v in 0..P {
        x_pert[0][v] += rng_p.normale() * 0.05;
    }
    let (val_p, u_p) = propres_sym(&gram(&x_pert));
    let beta_p = transpose_mat_vec(&x_pert, &ols_alpha(&val_p, &u_p, &p.y_train_z));
    let ecart: f64 = beta_p
        .iter()
        .zip(&beta_ols)
        .map(|(a, b)| (a - b) * (a - b))
        .sum::<f64>()
        .sqrt();
    ecarts.push(ecart / ols_norme);
}
println!(
    "norme des coefficients : {:.2} ; variation relative sur 5 perturbations d'une ligne : {:.2} a {:.2}",
    ols_norme,
    ecarts.iter().cloned().fold(f64::INFINITY, f64::min),
    ecarts.iter().cloned().fold(0.0, f64::max)
);

rang(X_train) = 119 sur p = 205 colonnes -> 86 directions du noyau


(le centrage des colonnes retire aussi 1 : les lignes centrees sont liees)


valeurs propres de K : plus grande 1257.9 ; plus petite retenue 3.6183 ; plus grande ecartee 4.3e-14


OLS (min-norme) : RMSE train = 0.0000   RMSE test = 0.2206


plancher irreductible (bruit en unites z) ~ 0.169  -> OLS est a 1.3 x le bruit


norme des coefficients : 0.62 ; variation relative sur 5 perturbations d'une ligne : 0.00 a 0.20


### Lecture du résultat

Le rang est 119 = n − 1 (le centrage des colonnes lie les lignes entre elles) : le noyau de la matrice possède 86 **directions** : cela ne signifie pas que 86 colonnes individuelles ne portent aucune information — $X^\top X$ est singulière, la solution OLS n'est pas unique. Le spectre de $K$ le montre net : la plus grande valeur propre vaut 1257.9, la plus petite retenue 3.6183, la plus grande écartée 4.3e-14 — il n'y a pas de continuum, le seuil sépare franchement. La solution de norme minimale interpole parfaitement le train (RMSE 0.0000) et généralise à 0.2206, soit **1.3× le plancher irréductible** (0.169). Ce n'est pas un effondrement — le signal est fort — mais une dégradation systématique : la prédiction s'est projetée sur *toutes* les directions de l'espace des lignes, y compris les 40 directions de bruit du bloc A. La variation relative de la solution sous 5 perturbations d'une seule ligne (0.00 à 0.20 pour 5% de bruit) montre la même instabilité, avec une amplitude irrégulière d'un tirage de perturbation à l'autre.

## 3. Ridge — la pénalité qui rend le problème déterminé

Ridge remplace $X^\top X$ par $X^\top X + \lambda I$ : la matrice devient définie positive **pour tout $\lambda > 0$**, le problème a une solution unique, et les coefficients sont raccourcis. C'est le remède que le 2.3 annonçait. Le $\lambda$ se choisit par validation croisée **à 5 plis contigus** dérivés de la graine (moyenne des $R^2$ de pli), sur une grille logarithmique de 25 valeurs de $10^{-3}$ à $10^{3}$ : c'est le rôle de `RidgeCV` côté Python.

In [4]:
// Cellule autonome : prelude (RNG, donnees, algebre) redefini ici, puis le calcul de la section.
// ---- Generateur pseudo-alatoire maison : xorshift64* + Box-Muller (aucun crate).
// La suite n'est pas celle de NumPy : les tirages different, les valeurs chiffrees sont celles de ce carnet.
#[allow(dead_code)]
struct Rng {
    etat: u64,
}

#[allow(dead_code)]
impl Rng {
    fn new(graine: u64) -> Rng {
        let mut etat = graine ^ 0x9E3779B97F4A7C15;
        if etat == 0 {
            etat = 0x2545F4914F6CDD1D;
        }
        Rng { etat }
    }

    fn brut(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x >> 12;
        x ^= x << 25;
        x ^= x >> 27;
        self.etat = x;
        x.wrapping_mul(0x2545F4914F6CDD1D)
    }

    // Uniforme sur [0, 1), 53 bits de mantisse.
    fn uniforme(&mut self) -> f64 {
        ((self.brut() >> 11) as f64 + 0.5) / 9007199254740992.0
    }

    // Normale centree reduite, methode de Box-Muller.
    fn normale(&mut self) -> f64 {
        const TAU: f64 = 6.283185307179586;
        loop {
            let u1 = self.uniforme();
            let u2 = self.uniforme();
            if u1 > 1e-300 {
                return (-2.0 * u1.ln()).sqrt() * (TAU * u2).cos();
            }
        }
    }

    // Entier uniforme dans 0..n, par rejet (pas le biais du modulo).
    fn indice(&mut self, n: usize) -> usize {
        let n = n as u64;
        let limite = u64::MAX - (u64::MAX % n);
        loop {
            let t = self.brut();
            if t < limite {
                return (t % n) as usize;
            }
        }
    }
}

// Permutation aleatoire (Fisher-Yates).
#[allow(dead_code)]
fn permutation(rng: &mut Rng, n: usize) -> Vec<usize> {
    let mut idx: Vec<usize> = (0..n).collect();
    for i in (1..n).rev() {
        let j = rng.indice(i + 1);
        idx.swap(i, j);
    }
    idx
}

// ---- Statistiques de base (ecarts-types au denominateur n, comme NumPy). ----
#[allow(dead_code)]
fn moyenne(v: &[f64]) -> f64 {
    v.iter().sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn variance(v: &[f64]) -> f64 {
    let m = moyenne(v);
    v.iter().map(|x| (x - m) * (x - m)).sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn ecart_type(v: &[f64]) -> f64 {
    variance(v).sqrt()
}

// Matrices stockees ligne par ligne : x[i][j].
#[allow(dead_code)]
fn mat_vec_ligne(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    x.iter()
        .map(|l| l.iter().zip(b).map(|(a, c)| a * c).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn colonnes_moyennes(x: &[Vec<f64>]) -> Vec<f64> {
    let n = x.len();
    (0..x[0].len())
        .map(|j| x.iter().map(|l| l[j]).sum::<f64>() / n as f64)
        .collect()
}

#[allow(dead_code)]
fn colonnes_ecarts(x: &[Vec<f64>]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| ecart_type(&x.iter().map(|l| l[j]).collect::<Vec<f64>>()))
        .collect()
}

#[allow(dead_code)]
fn standardiser(x: &[Vec<f64>], mu: &[f64], sigma: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            l.iter()
                .zip(mu.iter().zip(sigma))
                .map(|(a, (m, s))| (a - m) / s)
                .collect()
        })
        .collect()
}

#[allow(dead_code)]
fn standardiser_vec(v: &[f64], m: f64, s: f64) -> Vec<f64> {
    v.iter().map(|x| (x - m) / s).collect()
}

#[allow(dead_code)]
fn centrer(v: &[f64]) -> Vec<f64> {
    let m = moyenne(v);
    v.iter().map(|x| x - m).collect()
}

#[allow(dead_code)]
fn centrer_colonnes(x: &[Vec<f64>], mu: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| l.iter().zip(mu).map(|(a, m)| a - m).collect())
        .collect()
}

#[allow(dead_code)]
fn correlation(a: &[f64], b: &[f64]) -> f64 {
    let (ca, cb) = (centrer(a), centrer(b));
    let num: f64 = ca.iter().zip(&cb).map(|(x, y)| x * y).sum();
    let den: f64 = ca.iter().map(|x| x * x).sum::<f64>().sqrt()
        * cb.iter().map(|y| y * y).sum::<f64>().sqrt();
    num / den
}

#[allow(dead_code)]
fn rmse(a: &[f64], b: &[f64]) -> f64 {
    (a.iter().zip(b).map(|(x, y)| (x - y) * (x - y)).sum::<f64>() / a.len() as f64).sqrt()
}

// Indice du minimum (le premier en cas d'egalite), comme np.argmin.
#[allow(dead_code)]
fn argmin(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] < v[m] {
            m = k;
        }
    }
    m
}

// Indice du maximum (le premier en cas d'egalite), comme np.argmax.
#[allow(dead_code)]
fn argmax(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] > v[m] {
            m = k;
        }
    }
    m
}

#[allow(dead_code)]
fn logspace(a: f64, b: f64, n: usize) -> Vec<f64> {
    (0..n)
        .map(|i| 10f64.powf(a + (b - a) * i as f64 / (n as f64 - 1.0)))
        .collect()
}

// Colonne j d'une matrice stockee ligne par ligne.
#[allow(dead_code)]
fn colonne(x: &[Vec<f64>], j: usize) -> Vec<f64> {
    x.iter().map(|l| l[j]).collect()
}

// Somme des valeurs absolues sur deux blocs de variables : (0..debut_b, debut_b..fin).
#[allow(dead_code)]
fn masses_blocs(v: &[f64], debut_b: usize) -> (f64, f64) {
    let a: f64 = v[..debut_b].iter().map(|x| x.abs()).sum();
    let b: f64 = v[debut_b..].iter().map(|x| x.abs()).sum();
    (a, b)
}

// ---- Donnees : meme construction que la source (blocs A et B, cible, decoupage, standardisation). ----
#[allow(dead_code)]
const N: usize = 160;
#[allow(dead_code)]
const G_A: usize = 40;
#[allow(dead_code)]
const TAILLE_G: usize = 5;
#[allow(dead_code)]
const P_A: usize = 200;
#[allow(dead_code)]
const P_B: usize = 5;
#[allow(dead_code)]
const P: usize = 205;
#[allow(dead_code)]
const N_TRAIN: usize = 120;
#[allow(dead_code)]
const GRAINE: u64 = 42;

// Variables en groupes ; dans chaque groupe, corr(x_i, x_j) = rho_pair via facteur commun.
#[allow(dead_code)]
fn bloc_facteurs(rng: &mut Rng, n_vars: usize, n_groupes: usize, rho_pair: f64) -> Vec<Vec<f64>> {
    let mut x = vec![vec![0.0; n_vars]; N];
    for g in 0..n_groupes {
        let f: Vec<f64> = (0..N).map(|_| rho_pair.sqrt() * rng.normale()).collect();
        for j in 0..TAILLE_G {
            for i in 0..N {
                x[i][g * TAILLE_G + j] = f[i] + (1.0 - rho_pair).sqrt() * rng.normale();
            }
        }
    }
    x
}

// Un tirage complet depuis la graine : les deux blocs, le bruit, le decoupage et les poids du regime aligne.
#[allow(dead_code)]
struct Jeu {
    x: Vec<Vec<f64>>,
    bruit: Vec<f64>,
    idx: Vec<usize>,
    poids_fact: Vec<f64>,
}

// 40 directions de variance dominante sans signal (bloc A), 1 direction plus modeste generatrice de Y (bloc B).
#[allow(dead_code)]
fn jeu(graine: u64) -> Jeu {
    let mut rng = Rng::new(graine);
    let mut x = bloc_facteurs(&mut rng, P_A, G_A, 0.8);
    let xb = bloc_facteurs(&mut rng, P_B, 1, 0.6);
    for i in 0..N {
        x[i].extend(xb[i].iter().copied());
    }
    let bruit: Vec<f64> = (0..N).map(|_| rng.normale()).collect();
    let idx = permutation(&mut rng, N);
    let poids_fact: Vec<f64> = (0..G_A).map(|_| rng.normale() / (G_A as f64).sqrt()).collect();
    Jeu { x, bruit, idx, poids_fact }
}

// Betas de meme signe, alignes sur le facteur commun du bloc B ; Y = X beta + bruit.
#[allow(dead_code)]
fn cible(j: &Jeu) -> Vec<f64> {
    let formes = [1.3, 1.0, 1.2, 0.9, 1.1];
    let mut beta = vec![0.0; P];
    for k in 0..P_B {
        beta[P_A + k] = formes[k] * 1.2;
    }
    let mut y = mat_vec_ligne(&j.x, &beta);
    for i in 0..N {
        y[i] += j.bruit[i];
    }
    y
}

// Regime aligne : Y2 = facteurs_A . poids_fact * 3.0 + bruit. Les facteurs_A sont des proxis construits
// a partir de la premiere variable observee de chaque groupe du bloc A, pas les facteurs latents exacts.
#[allow(dead_code)]
fn cible_alignee(j: &Jeu) -> Vec<f64> {
    let mut y = vec![0.0; N];
    for i in 0..N {
        let mut s = 0.0;
        for g in 0..G_A {
            s += 0.8f64.sqrt() * j.x[i][g * TAILLE_G] * j.poids_fact[g];
        }
        y[i] = s * 3.0 + j.bruit[i];
    }
    y
}

// Decoupage 120 train / 40 test, standardisation ajustee sur le train uniquement (le geste du 2.1).
#[allow(dead_code)]
struct Prep {
    x_train_s: Vec<Vec<f64>>,
    x_test_s: Vec<Vec<f64>>,
    y_train_z: Vec<f64>,
    y_test_z: Vec<f64>,
    y_ecart: f64,
}

#[allow(dead_code)]
fn preparer(j: &Jeu, y: &[f64]) -> Prep {
    let x_train: Vec<Vec<f64>> = (0..N_TRAIN).map(|k| j.x[j.idx[k]].clone()).collect();
    let x_test: Vec<Vec<f64>> = (N_TRAIN..N).map(|k| j.x[j.idx[k]].clone()).collect();
    let y_train: Vec<f64> = (0..N_TRAIN).map(|k| y[j.idx[k]]).collect();
    let y_test: Vec<f64> = (N_TRAIN..N).map(|k| y[j.idx[k]]).collect();
    let (mu, sigma) = (colonnes_moyennes(&x_train), colonnes_ecarts(&x_train));
    let (y_m, y_s) = (moyenne(&y_train), ecart_type(&y_train));
    Prep {
        x_train_s: standardiser(&x_train, &mu, &sigma),
        x_test_s: standardiser(&x_test, &mu, &sigma),
        y_train_z: standardiser_vec(&y_train, y_m, y_s),
        y_test_z: standardiser_vec(&y_test, y_m, y_s),
        y_ecart: y_s,
    }
}

// ---- Algebre lineaire : tout passe par la matrice de Gram des lignes K = X X^T (n x n, ici 120 x 120),
// jamais par X^T X (p x p, singuliere des que p > n). ----
#[allow(dead_code)]
fn transpose_mat_vec(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| x.iter().zip(b).map(|(l, &v)| l[j] * v).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn gram(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let n = x.len();
    (0..n)
        .map(|i| {
            (0..n)
                .map(|k| x[i].iter().zip(&x[k]).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// Valeurs propres (decroissantes) et vecteurs propres d'une matrice symetrique par la methode de
// Jacobi cyclique. vecteurs[k][i] est la i-eme composante du k-eme vecteur propre ; le signe de chaque
// vecteur est choisi pour que sa plus grande composante en valeur absolue soit positive.
#[allow(dead_code)]
fn propres_sym(m: &[Vec<f64>]) -> (Vec<f64>, Vec<Vec<f64>>) {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut v: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|k| if i == k { 1.0 } else { 0.0 }).collect())
        .collect();
    for _balayage in 0..100 {
        let diag: f64 = (0..n).map(|i| a[i][i].abs()).fold(0.0f64, f64::max);
        let mut carres = 0.0f64;
        for i in 0..n {
            for k in (i + 1)..n {
                carres += a[i][k] * a[i][k];
            }
        }
        if carres.sqrt() <= 1e-13 * (1.0 + diag) {
            break;
        }
        let mut rotations = 0usize;
        for p in 0..n {
            for q in (p + 1)..n {
                let apq = a[p][q];
                // Pivot negligeable devant la diagonale : on l'annule sans tourner.
                if apq.abs() <= 1e-18 * (a[p][p].abs() + a[q][q].abs()) {
                    a[p][q] = 0.0;
                    a[q][p] = 0.0;
                    continue;
                }
                rotations += 1;
                let theta = (a[q][q] - a[p][p]) / (2.0 * apq);
                let t = if theta.abs() > 1e150 {
                    1.0 / (2.0 * theta)
                } else if theta >= 0.0 {
                    1.0 / (theta + (theta * theta + 1.0).sqrt())
                } else {
                    -1.0 / (-theta + (theta * theta + 1.0).sqrt())
                };
                let c = 1.0 / (t * t + 1.0).sqrt();
                let s = t * c;
                for k in 0..n {
                    if k != p && k != q {
                        let akp = a[k][p];
                        let akq = a[k][q];
                        a[k][p] = c * akp - s * akq;
                        a[p][k] = a[k][p];
                        a[k][q] = s * akp + c * akq;
                        a[q][k] = a[k][q];
                    }
                    let vkp = v[k][p];
                    let vkq = v[k][q];
                    v[k][p] = c * vkp - s * vkq;
                    v[k][q] = s * vkp + c * vkq;
                }
                a[p][p] -= t * apq;
                a[q][q] += t * apq;
                a[p][q] = 0.0;
                a[q][p] = 0.0;
            }
        }
        if rotations == 0 {
            break;
        }
    }
    let mut ordre: Vec<usize> = (0..n).collect();
    ordre.sort_by(|&i, &k| a[k][k].total_cmp(&a[i][i]));
    let valeurs: Vec<f64> = ordre.iter().map(|&i| a[i][i]).collect();
    let vecteurs: Vec<Vec<f64>> = ordre
        .iter()
        .map(|&k| {
            let colonne: Vec<f64> = v.iter().map(|l| l[k]).collect();
            let pivot = argmax(&colonne.iter().map(|x| x.abs()).collect::<Vec<f64>>());
            if colonne[pivot] < 0.0 {
                colonne.iter().map(|x| -x).collect()
            } else {
                colonne
            }
        })
        .collect();
    (valeurs, vecteurs)
}

// Rang numerique : nombre de valeurs propres au-dessus de 1e-8 fois la plus grande.
#[allow(dead_code)]
fn rang(valeurs: &[f64]) -> usize {
    let lmax = valeurs[0];
    valeurs.iter().filter(|v| **v > 1e-8 * lmax).count()
}

// OLS de norme minimale sans ordonnee, par K = X X^T : alpha = U_r L_r^{-1} U_r^T y, puis beta = X^T alpha.
// Les r valeurs propres non nulles fixent la prediction ; le noyau (p - r directions) ne la modifie pas.
#[allow(dead_code)]
fn ols_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64]) -> Vec<f64> {
    let lmax = valeurs[0];
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        if valeurs[k] > 1e-8 * lmax {
            let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
            for i in 0..n {
                alpha[i] += u[k][i] * d / valeurs[k];
            }
        }
    }
    alpha
}

// Coefficients duaux du ridge sans ordonnee : alpha = (K + lambda I)^{-1} y, dans la base des vecteurs propres.
#[allow(dead_code)]
fn ridge_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64], lambda: f64) -> Vec<f64> {
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
        for i in 0..n {
            alpha[i] += u[k][i] * d / (valeurs[k] + lambda);
        }
    }
    alpha
}

// beta = X^T alpha ; renvoie les previsions X beta sur train et test, et la norme L2 de beta.
#[allow(dead_code)]
fn predire(xtr: &[Vec<f64>], xte: &[Vec<f64>], alpha: &[f64]) -> (Vec<f64>, Vec<f64>, f64) {
    let beta = transpose_mat_vec(xtr, alpha);
    let norme = beta.iter().map(|b| b * b).sum::<f64>().sqrt();
    (mat_vec_ligne(xtr, &beta), mat_vec_ligne(xte, &beta), norme)
}

// Bornes du pli f d'une validation croisee a plis contigus (KFold sans melange).
#[allow(dead_code)]
fn bornes_pli(n: usize, plis: usize, f: usize) -> (usize, usize) {
    let (base, reste) = (n / plis, n % plis);
    let debut = f * base + reste.min(f);
    (debut, debut + base + if f < reste { 1 } else { 0 })
}

// Un pli contigu, avec la decomposition spectrale du Gram de son train : elle sert a tous les alpha du ridge.
#[allow(dead_code)]
struct PliContigu {
    xtr: Vec<Vec<f64>>,
    xva: Vec<Vec<f64>>,
    positions_val: Vec<usize>,
    valeurs: Vec<f64>,
    u: Vec<Vec<f64>>,
}

#[allow(dead_code)]
fn plis_contigus(x: &[Vec<f64>], plis: usize) -> Vec<PliContigu> {
    let n = x.len();
    (0..plis)
        .map(|f| {
            let (debut, fin) = bornes_pli(n, plis, f);
            let xtr: Vec<Vec<f64>> = (0..n)
                .filter(|&i| i < debut || i >= fin)
                .map(|i| x[i].clone())
                .collect();
            let xva: Vec<Vec<f64>> = x[debut..fin].to_vec();
            let (valeurs, u) = propres_sym(&gram(&xtr));
            PliContigu {
                xtr,
                xva,
                positions_val: (debut..fin).collect(),
                valeurs,
                u,
            }
        })
        .collect()
}

// RidgeCV(alphas, cv=plis, fit_intercept=False) : l'alpha retenu est celui qui maximise la moyenne
// des R2 de pli (le score par defaut de la validation croisee de scikit-learn). Renvoie son indice.
#[allow(dead_code)]
fn ridge_cv_sur_plis(plis_c: &[PliContigu], y: &[f64], alphas: &[f64]) -> usize {
    let mut score = vec![0.0; alphas.len()];
    for pli in plis_c {
        let ytr: Vec<f64> = (0..y.len())
            .filter(|i| !pli.positions_val.contains(i))
            .map(|i| y[i])
            .collect();
        let yva: Vec<f64> = pli.positions_val.iter().map(|&i| y[i]).collect();
        let sst: f64 = centrer(&yva).iter().map(|x| x * x).sum();
        for (j, &lam) in alphas.iter().enumerate() {
            let (_, p_va, _) = predire(
                &pli.xtr,
                &pli.xva,
                &ridge_alpha(&pli.valeurs, &pli.u, &ytr, lam),
            );
            let sse: f64 = yva.iter().zip(&p_va).map(|(t, p)| (t - p) * (t - p)).sum();
            score[j] += 1.0 - sse / sst;
        }
    }
    argmax(&score)
}

#[allow(dead_code)]
fn ridge_cv_retenu(x: &[Vec<f64>], y: &[f64], alphas: &[f64], plis: usize) -> usize {
    ridge_cv_sur_plis(&plis_contigus(x, plis), y, alphas)
}

// Composantes principales depuis la decomposition de K = X X^T : composante_k = X^T u_k / sqrt(lambda_k).
#[allow(dead_code)]
fn composantes(x: &[Vec<f64>], valeurs: &[f64], u: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let mut comps: Vec<Vec<f64>> = Vec::new();
    for k in 0..rang(valeurs) {
        let mut c = transpose_mat_vec(x, &u[k]);
        let s = valeurs[k].sqrt();
        for v in c.iter_mut() {
            *v /= s;
        }
        comps.push(c);
    }
    comps
}

// Centrage d'un echantillon sur ses propres colonnes : ce que fait PCA a l'ajustement.
#[allow(dead_code)]
fn centrer_propre(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    centrer_colonnes(x, &colonnes_moyennes(x))
}

// ACP, comme sklearn.decomposition.PCA : X est CENTRE sur ses colonnes, covariance (denominateur n),
// valeurs propres, part de variance, composantes. Le centrage ne change rien sur une matrice deja
// standardisee sur elle-meme (train complet) ; il est indispensable sur le train d'un pli, dont les
// colonnes ne sont pas de moyenne nulle.
#[allow(dead_code)]
fn acp(x: &[Vec<f64>]) -> (Vec<f64>, Vec<f64>, Vec<Vec<f64>>) {
    let xc = centrer_propre(x);
    let (valeurs, u) = propres_sym(&gram(&xc));
    let total: f64 = valeurs.iter().sum();
    let comps = composantes(&xc, &valeurs, &u);
    let ratio: Vec<f64> = valeurs.iter().map(|l| l / total).collect();
    (valeurs, ratio, comps)
}

// Variance cumulee jusqu'a la composante k comprise.
#[allow(dead_code)]
fn variance_cumulee(ratio: &[f64], k: usize) -> f64 {
    ratio[..=k].iter().sum()
}

// Scores t[i][k] = x_i . composante_k.
#[allow(dead_code)]
fn scores_composantes(x: &[Vec<f64>], comps: &[Vec<f64>]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            comps
                .iter()
                .map(|c| l.iter().zip(c).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// PCR a k composantes avec ordonnee : y = ybar + sum_{j<k} b_j t_j.
// Les scores d'une ACP sont orthogonaux deux a deux, donc b_j = (t_j . (y - ybar)) / (t_j . t_j).
#[allow(dead_code)]
fn regression_scores(
    t_tr: &[Vec<f64>],
    t_te: &[Vec<f64>],
    y: &[f64],
    k: usize,
) -> (Vec<f64>, Vec<f64>) {
    let ym = moyenne(y);
    let yc: Vec<f64> = y.iter().map(|v| v - ym).collect();
    let mut b = vec![0.0; k];
    for j in 0..k {
        let num: f64 = t_tr.iter().zip(&yc).map(|(t, &v)| t[j] * v).sum();
        let den: f64 = t_tr.iter().map(|t| t[j] * t[j]).sum();
        b[j] = num / den;
    }
    let projet = |tt: &[Vec<f64>]| -> Vec<f64> {
        tt.iter()
            .map(|t| ym + (0..k).map(|j| b[j] * t[j]).sum::<f64>())
            .collect()
    };
    (projet(t_tr), projet(t_te))
}

// RMSE test de la PCR pour k = 1..kmax (une seule ACP, une regression par k).
#[allow(dead_code)]
fn balayage_pcr(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (_, _, mut comps) = acp(xtr);
    comps.truncate(kmax);
    // PCA().fit(xtr).transform(...) : les deux echantillons sont centres sur la moyenne du train.
    let xmu = colonnes_moyennes(xtr);
    let (t_tr, t_te) = (
        scores_composantes(&centrer_colonnes(xtr, &xmu), &comps),
        scores_composantes(&centrer_colonnes(xte, &xmu), &comps),
    );
    (1..=kmax)
        .map(|k| rmse(yte, &regression_scores(&t_tr, &t_te, ytr, k).1))
        .collect()
}

// ---- 3. Ridge : la penalite qui rend le probleme determine ----
let j = jeu(GRAINE);
let p = preparer(&j, &cible(&j));
// Une seule decomposition de K = X X^T : elle sert a l'OLS, au ridge retenu et au balayage.
let (valeurs, u) = propres_sym(&gram(&p.x_train_s));
let (_, _, ols_norme) = predire(&p.x_train_s, &p.x_test_s, &ols_alpha(&valeurs, &u, &p.y_train_z));

// lambda choisi par validation croisee a 5 plis contigus, sur une grille logarithmique de 25 valeurs.
let alphas = logspace(-3.0, 3.0, 25);
let retenu = ridge_cv_retenu(&p.x_train_s, &p.y_train_z, &alphas, 5);
let (ridge_tr, ridge_te, ridge_norme) = predire(
    &p.x_train_s,
    &p.x_test_s,
    &ridge_alpha(&valeurs, &u, &p.y_train_z, alphas[retenu]),
);
println!("lambda CV = {:.4}", alphas[retenu]);
println!(
    "RMSE train = {:.4}   RMSE test = {:.4}",
    rmse(&p.y_train_z, &ridge_tr),
    rmse(&p.y_test_z, &ridge_te)
);
println!(
    "norme L2 des coefficients : {:.2}   (OLS min-norme : {:.2})",
    ridge_norme, ols_norme
);

// Meme test sur 40 valeurs de lambda : un jeu de coefficients duaux par valeur, pas une decomposition.
let grille = logspace(-3.0, 3.0, 40);
let mut rmse_test_ridge: Vec<f64> = Vec::new();
for lam in &grille {
    let (_, p_te, _) = predire(
        &p.x_train_s,
        &p.x_test_s,
        &ridge_alpha(&valeurs, &u, &p.y_train_z, *lam),
    );
    rmse_test_ridge.push(rmse(&p.y_test_z, &p_te));
}
let i_opt = argmin(&rmse_test_ridge);
println!(
    "\nRMSE test ridge : lambda=1e-3 {:.3} | optimum {:.3} (log10 lambda = {:.2}) | lambda=1e3 {:.3}",
    rmse_test_ridge[0],
    rmse_test_ridge[i_opt],
    grille[i_opt].log10(),
    rmse_test_ridge[grille.len() - 1]
);

lambda CV = 1.0000


RMSE train = 0.0090   RMSE test = 0.2094


norme L2 des coefficients : 0.60   (OLS min-norme : 0.62)


RMSE test ridge : lambda=1e-3 0.221 | optimum 0.179 (log10 lambda = 1.15) | lambda=1e3 0.779


### Lecture du résultat

La CV retient λ = 1.0000 : RMSE test 0.2094 contre 0.2206 pour OLS — un gain **modeste** (5%). L'explication est honnête : ici le signal est fort et concentré sur une seule direction, régime où la solution min-norme est déjà raisonnable. Le minimum observé sur les 40 valeurs de la grille test (log10 λ = 1.15, soit λ ≈ 14, RMSE 0.179) décrit cette grille et ce test, sans borner les autres choix — et il se place juste au-dessus de ce que la PLS atteindra au §5. Aux extrêmes : λ = 10⁻³ rend OLS (0.221), λ = 10³ écrase tout (0.779). Ridge rend le problème déterminé et raccourcit les coefficients (0.60 contre 0.62 pour la min-norme) — c'est le remède promis par le [2.3](2.3-Regression-lineaire-logistique.ipynb), pas une baguette magique.

### Exercice 1 — l'alpha qui écrase tout

Que se passe-t-il quand $\lambda \to \infty$ ? Complétez `ridge_extreme` pour évaluer ridge à $\lambda = 10^{6}$ et $10^{9}$ et renvoyer, pour chaque $\lambda$, la RMSE train, la RMSE test et la norme des coefficients, plus la RMSE de la meilleure prédiction constante. *Indice : tous les coefficients tendent vers 0 — que vaut alors la meilleure prédiction constante ?*

La garde compare les mesures à la voie spectrale du préambule (tolérance relative $10^{-8}$, absolue $10^{-10}$) et vérifie la RMSE constante à $10^{-10}$ ; une seconde fixture, minuscule et bien conditionnée, contrôle que la fonction ne dépend pas des données du §1.

In [5]:
// Cellule autonome : prelude (RNG, donnees, algebre) redefini ici, puis le calcul de la section.
// ---- Generateur pseudo-alatoire maison : xorshift64* + Box-Muller (aucun crate).
// La suite n'est pas celle de NumPy : les tirages different, les valeurs chiffrees sont celles de ce carnet.
#[allow(dead_code)]
struct Rng {
    etat: u64,
}

#[allow(dead_code)]
impl Rng {
    fn new(graine: u64) -> Rng {
        let mut etat = graine ^ 0x9E3779B97F4A7C15;
        if etat == 0 {
            etat = 0x2545F4914F6CDD1D;
        }
        Rng { etat }
    }

    fn brut(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x >> 12;
        x ^= x << 25;
        x ^= x >> 27;
        self.etat = x;
        x.wrapping_mul(0x2545F4914F6CDD1D)
    }

    // Uniforme sur [0, 1), 53 bits de mantisse.
    fn uniforme(&mut self) -> f64 {
        ((self.brut() >> 11) as f64 + 0.5) / 9007199254740992.0
    }

    // Normale centree reduite, methode de Box-Muller.
    fn normale(&mut self) -> f64 {
        const TAU: f64 = 6.283185307179586;
        loop {
            let u1 = self.uniforme();
            let u2 = self.uniforme();
            if u1 > 1e-300 {
                return (-2.0 * u1.ln()).sqrt() * (TAU * u2).cos();
            }
        }
    }

    // Entier uniforme dans 0..n, par rejet (pas le biais du modulo).
    fn indice(&mut self, n: usize) -> usize {
        let n = n as u64;
        let limite = u64::MAX - (u64::MAX % n);
        loop {
            let t = self.brut();
            if t < limite {
                return (t % n) as usize;
            }
        }
    }
}

// Permutation aleatoire (Fisher-Yates).
#[allow(dead_code)]
fn permutation(rng: &mut Rng, n: usize) -> Vec<usize> {
    let mut idx: Vec<usize> = (0..n).collect();
    for i in (1..n).rev() {
        let j = rng.indice(i + 1);
        idx.swap(i, j);
    }
    idx
}

// ---- Statistiques de base (ecarts-types au denominateur n, comme NumPy). ----
#[allow(dead_code)]
fn moyenne(v: &[f64]) -> f64 {
    v.iter().sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn variance(v: &[f64]) -> f64 {
    let m = moyenne(v);
    v.iter().map(|x| (x - m) * (x - m)).sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn ecart_type(v: &[f64]) -> f64 {
    variance(v).sqrt()
}

// Matrices stockees ligne par ligne : x[i][j].
#[allow(dead_code)]
fn mat_vec_ligne(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    x.iter()
        .map(|l| l.iter().zip(b).map(|(a, c)| a * c).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn colonnes_moyennes(x: &[Vec<f64>]) -> Vec<f64> {
    let n = x.len();
    (0..x[0].len())
        .map(|j| x.iter().map(|l| l[j]).sum::<f64>() / n as f64)
        .collect()
}

#[allow(dead_code)]
fn colonnes_ecarts(x: &[Vec<f64>]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| ecart_type(&x.iter().map(|l| l[j]).collect::<Vec<f64>>()))
        .collect()
}

#[allow(dead_code)]
fn standardiser(x: &[Vec<f64>], mu: &[f64], sigma: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            l.iter()
                .zip(mu.iter().zip(sigma))
                .map(|(a, (m, s))| (a - m) / s)
                .collect()
        })
        .collect()
}

#[allow(dead_code)]
fn standardiser_vec(v: &[f64], m: f64, s: f64) -> Vec<f64> {
    v.iter().map(|x| (x - m) / s).collect()
}

#[allow(dead_code)]
fn centrer(v: &[f64]) -> Vec<f64> {
    let m = moyenne(v);
    v.iter().map(|x| x - m).collect()
}

#[allow(dead_code)]
fn centrer_colonnes(x: &[Vec<f64>], mu: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| l.iter().zip(mu).map(|(a, m)| a - m).collect())
        .collect()
}

#[allow(dead_code)]
fn correlation(a: &[f64], b: &[f64]) -> f64 {
    let (ca, cb) = (centrer(a), centrer(b));
    let num: f64 = ca.iter().zip(&cb).map(|(x, y)| x * y).sum();
    let den: f64 = ca.iter().map(|x| x * x).sum::<f64>().sqrt()
        * cb.iter().map(|y| y * y).sum::<f64>().sqrt();
    num / den
}

#[allow(dead_code)]
fn rmse(a: &[f64], b: &[f64]) -> f64 {
    (a.iter().zip(b).map(|(x, y)| (x - y) * (x - y)).sum::<f64>() / a.len() as f64).sqrt()
}

// Indice du minimum (le premier en cas d'egalite), comme np.argmin.
#[allow(dead_code)]
fn argmin(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] < v[m] {
            m = k;
        }
    }
    m
}

// Indice du maximum (le premier en cas d'egalite), comme np.argmax.
#[allow(dead_code)]
fn argmax(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] > v[m] {
            m = k;
        }
    }
    m
}

#[allow(dead_code)]
fn logspace(a: f64, b: f64, n: usize) -> Vec<f64> {
    (0..n)
        .map(|i| 10f64.powf(a + (b - a) * i as f64 / (n as f64 - 1.0)))
        .collect()
}

// Colonne j d'une matrice stockee ligne par ligne.
#[allow(dead_code)]
fn colonne(x: &[Vec<f64>], j: usize) -> Vec<f64> {
    x.iter().map(|l| l[j]).collect()
}

// Somme des valeurs absolues sur deux blocs de variables : (0..debut_b, debut_b..fin).
#[allow(dead_code)]
fn masses_blocs(v: &[f64], debut_b: usize) -> (f64, f64) {
    let a: f64 = v[..debut_b].iter().map(|x| x.abs()).sum();
    let b: f64 = v[debut_b..].iter().map(|x| x.abs()).sum();
    (a, b)
}

// ---- Donnees : meme construction que la source (blocs A et B, cible, decoupage, standardisation). ----
#[allow(dead_code)]
const N: usize = 160;
#[allow(dead_code)]
const G_A: usize = 40;
#[allow(dead_code)]
const TAILLE_G: usize = 5;
#[allow(dead_code)]
const P_A: usize = 200;
#[allow(dead_code)]
const P_B: usize = 5;
#[allow(dead_code)]
const P: usize = 205;
#[allow(dead_code)]
const N_TRAIN: usize = 120;
#[allow(dead_code)]
const GRAINE: u64 = 42;

// Variables en groupes ; dans chaque groupe, corr(x_i, x_j) = rho_pair via facteur commun.
#[allow(dead_code)]
fn bloc_facteurs(rng: &mut Rng, n_vars: usize, n_groupes: usize, rho_pair: f64) -> Vec<Vec<f64>> {
    let mut x = vec![vec![0.0; n_vars]; N];
    for g in 0..n_groupes {
        let f: Vec<f64> = (0..N).map(|_| rho_pair.sqrt() * rng.normale()).collect();
        for j in 0..TAILLE_G {
            for i in 0..N {
                x[i][g * TAILLE_G + j] = f[i] + (1.0 - rho_pair).sqrt() * rng.normale();
            }
        }
    }
    x
}

// Un tirage complet depuis la graine : les deux blocs, le bruit, le decoupage et les poids du regime aligne.
#[allow(dead_code)]
struct Jeu {
    x: Vec<Vec<f64>>,
    bruit: Vec<f64>,
    idx: Vec<usize>,
    poids_fact: Vec<f64>,
}

// 40 directions de variance dominante sans signal (bloc A), 1 direction plus modeste generatrice de Y (bloc B).
#[allow(dead_code)]
fn jeu(graine: u64) -> Jeu {
    let mut rng = Rng::new(graine);
    let mut x = bloc_facteurs(&mut rng, P_A, G_A, 0.8);
    let xb = bloc_facteurs(&mut rng, P_B, 1, 0.6);
    for i in 0..N {
        x[i].extend(xb[i].iter().copied());
    }
    let bruit: Vec<f64> = (0..N).map(|_| rng.normale()).collect();
    let idx = permutation(&mut rng, N);
    let poids_fact: Vec<f64> = (0..G_A).map(|_| rng.normale() / (G_A as f64).sqrt()).collect();
    Jeu { x, bruit, idx, poids_fact }
}

// Betas de meme signe, alignes sur le facteur commun du bloc B ; Y = X beta + bruit.
#[allow(dead_code)]
fn cible(j: &Jeu) -> Vec<f64> {
    let formes = [1.3, 1.0, 1.2, 0.9, 1.1];
    let mut beta = vec![0.0; P];
    for k in 0..P_B {
        beta[P_A + k] = formes[k] * 1.2;
    }
    let mut y = mat_vec_ligne(&j.x, &beta);
    for i in 0..N {
        y[i] += j.bruit[i];
    }
    y
}

// Regime aligne : Y2 = facteurs_A . poids_fact * 3.0 + bruit. Les facteurs_A sont des proxis construits
// a partir de la premiere variable observee de chaque groupe du bloc A, pas les facteurs latents exacts.
#[allow(dead_code)]
fn cible_alignee(j: &Jeu) -> Vec<f64> {
    let mut y = vec![0.0; N];
    for i in 0..N {
        let mut s = 0.0;
        for g in 0..G_A {
            s += 0.8f64.sqrt() * j.x[i][g * TAILLE_G] * j.poids_fact[g];
        }
        y[i] = s * 3.0 + j.bruit[i];
    }
    y
}

// Decoupage 120 train / 40 test, standardisation ajustee sur le train uniquement (le geste du 2.1).
#[allow(dead_code)]
struct Prep {
    x_train_s: Vec<Vec<f64>>,
    x_test_s: Vec<Vec<f64>>,
    y_train_z: Vec<f64>,
    y_test_z: Vec<f64>,
    y_ecart: f64,
}

#[allow(dead_code)]
fn preparer(j: &Jeu, y: &[f64]) -> Prep {
    let x_train: Vec<Vec<f64>> = (0..N_TRAIN).map(|k| j.x[j.idx[k]].clone()).collect();
    let x_test: Vec<Vec<f64>> = (N_TRAIN..N).map(|k| j.x[j.idx[k]].clone()).collect();
    let y_train: Vec<f64> = (0..N_TRAIN).map(|k| y[j.idx[k]]).collect();
    let y_test: Vec<f64> = (N_TRAIN..N).map(|k| y[j.idx[k]]).collect();
    let (mu, sigma) = (colonnes_moyennes(&x_train), colonnes_ecarts(&x_train));
    let (y_m, y_s) = (moyenne(&y_train), ecart_type(&y_train));
    Prep {
        x_train_s: standardiser(&x_train, &mu, &sigma),
        x_test_s: standardiser(&x_test, &mu, &sigma),
        y_train_z: standardiser_vec(&y_train, y_m, y_s),
        y_test_z: standardiser_vec(&y_test, y_m, y_s),
        y_ecart: y_s,
    }
}

// ---- Algebre lineaire : tout passe par la matrice de Gram des lignes K = X X^T (n x n, ici 120 x 120),
// jamais par X^T X (p x p, singuliere des que p > n). ----
#[allow(dead_code)]
fn transpose_mat_vec(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| x.iter().zip(b).map(|(l, &v)| l[j] * v).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn gram(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let n = x.len();
    (0..n)
        .map(|i| {
            (0..n)
                .map(|k| x[i].iter().zip(&x[k]).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// Valeurs propres (decroissantes) et vecteurs propres d'une matrice symetrique par la methode de
// Jacobi cyclique. vecteurs[k][i] est la i-eme composante du k-eme vecteur propre ; le signe de chaque
// vecteur est choisi pour que sa plus grande composante en valeur absolue soit positive.
#[allow(dead_code)]
fn propres_sym(m: &[Vec<f64>]) -> (Vec<f64>, Vec<Vec<f64>>) {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut v: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|k| if i == k { 1.0 } else { 0.0 }).collect())
        .collect();
    for _balayage in 0..100 {
        let diag: f64 = (0..n).map(|i| a[i][i].abs()).fold(0.0f64, f64::max);
        let mut carres = 0.0f64;
        for i in 0..n {
            for k in (i + 1)..n {
                carres += a[i][k] * a[i][k];
            }
        }
        if carres.sqrt() <= 1e-13 * (1.0 + diag) {
            break;
        }
        let mut rotations = 0usize;
        for p in 0..n {
            for q in (p + 1)..n {
                let apq = a[p][q];
                // Pivot negligeable devant la diagonale : on l'annule sans tourner.
                if apq.abs() <= 1e-18 * (a[p][p].abs() + a[q][q].abs()) {
                    a[p][q] = 0.0;
                    a[q][p] = 0.0;
                    continue;
                }
                rotations += 1;
                let theta = (a[q][q] - a[p][p]) / (2.0 * apq);
                let t = if theta.abs() > 1e150 {
                    1.0 / (2.0 * theta)
                } else if theta >= 0.0 {
                    1.0 / (theta + (theta * theta + 1.0).sqrt())
                } else {
                    -1.0 / (-theta + (theta * theta + 1.0).sqrt())
                };
                let c = 1.0 / (t * t + 1.0).sqrt();
                let s = t * c;
                for k in 0..n {
                    if k != p && k != q {
                        let akp = a[k][p];
                        let akq = a[k][q];
                        a[k][p] = c * akp - s * akq;
                        a[p][k] = a[k][p];
                        a[k][q] = s * akp + c * akq;
                        a[q][k] = a[k][q];
                    }
                    let vkp = v[k][p];
                    let vkq = v[k][q];
                    v[k][p] = c * vkp - s * vkq;
                    v[k][q] = s * vkp + c * vkq;
                }
                a[p][p] -= t * apq;
                a[q][q] += t * apq;
                a[p][q] = 0.0;
                a[q][p] = 0.0;
            }
        }
        if rotations == 0 {
            break;
        }
    }
    let mut ordre: Vec<usize> = (0..n).collect();
    ordre.sort_by(|&i, &k| a[k][k].total_cmp(&a[i][i]));
    let valeurs: Vec<f64> = ordre.iter().map(|&i| a[i][i]).collect();
    let vecteurs: Vec<Vec<f64>> = ordre
        .iter()
        .map(|&k| {
            let colonne: Vec<f64> = v.iter().map(|l| l[k]).collect();
            let pivot = argmax(&colonne.iter().map(|x| x.abs()).collect::<Vec<f64>>());
            if colonne[pivot] < 0.0 {
                colonne.iter().map(|x| -x).collect()
            } else {
                colonne
            }
        })
        .collect();
    (valeurs, vecteurs)
}

// Rang numerique : nombre de valeurs propres au-dessus de 1e-8 fois la plus grande.
#[allow(dead_code)]
fn rang(valeurs: &[f64]) -> usize {
    let lmax = valeurs[0];
    valeurs.iter().filter(|v| **v > 1e-8 * lmax).count()
}

// OLS de norme minimale sans ordonnee, par K = X X^T : alpha = U_r L_r^{-1} U_r^T y, puis beta = X^T alpha.
// Les r valeurs propres non nulles fixent la prediction ; le noyau (p - r directions) ne la modifie pas.
#[allow(dead_code)]
fn ols_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64]) -> Vec<f64> {
    let lmax = valeurs[0];
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        if valeurs[k] > 1e-8 * lmax {
            let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
            for i in 0..n {
                alpha[i] += u[k][i] * d / valeurs[k];
            }
        }
    }
    alpha
}

// Coefficients duaux du ridge sans ordonnee : alpha = (K + lambda I)^{-1} y, dans la base des vecteurs propres.
#[allow(dead_code)]
fn ridge_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64], lambda: f64) -> Vec<f64> {
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
        for i in 0..n {
            alpha[i] += u[k][i] * d / (valeurs[k] + lambda);
        }
    }
    alpha
}

// beta = X^T alpha ; renvoie les previsions X beta sur train et test, et la norme L2 de beta.
#[allow(dead_code)]
fn predire(xtr: &[Vec<f64>], xte: &[Vec<f64>], alpha: &[f64]) -> (Vec<f64>, Vec<f64>, f64) {
    let beta = transpose_mat_vec(xtr, alpha);
    let norme = beta.iter().map(|b| b * b).sum::<f64>().sqrt();
    (mat_vec_ligne(xtr, &beta), mat_vec_ligne(xte, &beta), norme)
}

// Bornes du pli f d'une validation croisee a plis contigus (KFold sans melange).
#[allow(dead_code)]
fn bornes_pli(n: usize, plis: usize, f: usize) -> (usize, usize) {
    let (base, reste) = (n / plis, n % plis);
    let debut = f * base + reste.min(f);
    (debut, debut + base + if f < reste { 1 } else { 0 })
}

// Un pli contigu, avec la decomposition spectrale du Gram de son train : elle sert a tous les alpha du ridge.
#[allow(dead_code)]
struct PliContigu {
    xtr: Vec<Vec<f64>>,
    xva: Vec<Vec<f64>>,
    positions_val: Vec<usize>,
    valeurs: Vec<f64>,
    u: Vec<Vec<f64>>,
}

#[allow(dead_code)]
fn plis_contigus(x: &[Vec<f64>], plis: usize) -> Vec<PliContigu> {
    let n = x.len();
    (0..plis)
        .map(|f| {
            let (debut, fin) = bornes_pli(n, plis, f);
            let xtr: Vec<Vec<f64>> = (0..n)
                .filter(|&i| i < debut || i >= fin)
                .map(|i| x[i].clone())
                .collect();
            let xva: Vec<Vec<f64>> = x[debut..fin].to_vec();
            let (valeurs, u) = propres_sym(&gram(&xtr));
            PliContigu {
                xtr,
                xva,
                positions_val: (debut..fin).collect(),
                valeurs,
                u,
            }
        })
        .collect()
}

// RidgeCV(alphas, cv=plis, fit_intercept=False) : l'alpha retenu est celui qui maximise la moyenne
// des R2 de pli (le score par defaut de la validation croisee de scikit-learn). Renvoie son indice.
#[allow(dead_code)]
fn ridge_cv_sur_plis(plis_c: &[PliContigu], y: &[f64], alphas: &[f64]) -> usize {
    let mut score = vec![0.0; alphas.len()];
    for pli in plis_c {
        let ytr: Vec<f64> = (0..y.len())
            .filter(|i| !pli.positions_val.contains(i))
            .map(|i| y[i])
            .collect();
        let yva: Vec<f64> = pli.positions_val.iter().map(|&i| y[i]).collect();
        let sst: f64 = centrer(&yva).iter().map(|x| x * x).sum();
        for (j, &lam) in alphas.iter().enumerate() {
            let (_, p_va, _) = predire(
                &pli.xtr,
                &pli.xva,
                &ridge_alpha(&pli.valeurs, &pli.u, &ytr, lam),
            );
            let sse: f64 = yva.iter().zip(&p_va).map(|(t, p)| (t - p) * (t - p)).sum();
            score[j] += 1.0 - sse / sst;
        }
    }
    argmax(&score)
}

#[allow(dead_code)]
fn ridge_cv_retenu(x: &[Vec<f64>], y: &[f64], alphas: &[f64], plis: usize) -> usize {
    ridge_cv_sur_plis(&plis_contigus(x, plis), y, alphas)
}

// Composantes principales depuis la decomposition de K = X X^T : composante_k = X^T u_k / sqrt(lambda_k).
#[allow(dead_code)]
fn composantes(x: &[Vec<f64>], valeurs: &[f64], u: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let mut comps: Vec<Vec<f64>> = Vec::new();
    for k in 0..rang(valeurs) {
        let mut c = transpose_mat_vec(x, &u[k]);
        let s = valeurs[k].sqrt();
        for v in c.iter_mut() {
            *v /= s;
        }
        comps.push(c);
    }
    comps
}

// Centrage d'un echantillon sur ses propres colonnes : ce que fait PCA a l'ajustement.
#[allow(dead_code)]
fn centrer_propre(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    centrer_colonnes(x, &colonnes_moyennes(x))
}

// ACP, comme sklearn.decomposition.PCA : X est CENTRE sur ses colonnes, covariance (denominateur n),
// valeurs propres, part de variance, composantes. Le centrage ne change rien sur une matrice deja
// standardisee sur elle-meme (train complet) ; il est indispensable sur le train d'un pli, dont les
// colonnes ne sont pas de moyenne nulle.
#[allow(dead_code)]
fn acp(x: &[Vec<f64>]) -> (Vec<f64>, Vec<f64>, Vec<Vec<f64>>) {
    let xc = centrer_propre(x);
    let (valeurs, u) = propres_sym(&gram(&xc));
    let total: f64 = valeurs.iter().sum();
    let comps = composantes(&xc, &valeurs, &u);
    let ratio: Vec<f64> = valeurs.iter().map(|l| l / total).collect();
    (valeurs, ratio, comps)
}

// Variance cumulee jusqu'a la composante k comprise.
#[allow(dead_code)]
fn variance_cumulee(ratio: &[f64], k: usize) -> f64 {
    ratio[..=k].iter().sum()
}

// Scores t[i][k] = x_i . composante_k.
#[allow(dead_code)]
fn scores_composantes(x: &[Vec<f64>], comps: &[Vec<f64>]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            comps
                .iter()
                .map(|c| l.iter().zip(c).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// PCR a k composantes avec ordonnee : y = ybar + sum_{j<k} b_j t_j.
// Les scores d'une ACP sont orthogonaux deux a deux, donc b_j = (t_j . (y - ybar)) / (t_j . t_j).
#[allow(dead_code)]
fn regression_scores(
    t_tr: &[Vec<f64>],
    t_te: &[Vec<f64>],
    y: &[f64],
    k: usize,
) -> (Vec<f64>, Vec<f64>) {
    let ym = moyenne(y);
    let yc: Vec<f64> = y.iter().map(|v| v - ym).collect();
    let mut b = vec![0.0; k];
    for j in 0..k {
        let num: f64 = t_tr.iter().zip(&yc).map(|(t, &v)| t[j] * v).sum();
        let den: f64 = t_tr.iter().map(|t| t[j] * t[j]).sum();
        b[j] = num / den;
    }
    let projet = |tt: &[Vec<f64>]| -> Vec<f64> {
        tt.iter()
            .map(|t| ym + (0..k).map(|j| b[j] * t[j]).sum::<f64>())
            .collect()
    };
    (projet(t_tr), projet(t_te))
}

// RMSE test de la PCR pour k = 1..kmax (une seule ACP, une regression par k).
#[allow(dead_code)]
fn balayage_pcr(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (_, _, mut comps) = acp(xtr);
    comps.truncate(kmax);
    // PCA().fit(xtr).transform(...) : les deux echantillons sont centres sur la moyenne du train.
    let xmu = colonnes_moyennes(xtr);
    let (t_tr, t_te) = (
        scores_composantes(&centrer_colonnes(xtr, &xmu), &comps),
        scores_composantes(&centrer_colonnes(xte, &xmu), &comps),
    );
    (1..=kmax)
        .map(|k| rmse(yte, &regression_scores(&t_tr, &t_te, ytr, k).1))
        .collect()
}

// Exercice 1 : observer deux penalites ridge extremes.
#[allow(unused_variables)]
fn ridge_extreme(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
) -> Option<(Vec<(f64, f64, f64)>, f64)> {
    // TODO : pour alpha=1e6 puis 1e9, renvoyer (RMSE train, test, norme des coefficients), puis la RMSE constante.
    // Indice : tous les coefficients tendent vers 0 -- que vaut alors la meilleure prevision constante ?
    None
}

// Garde : ridge sans ordonnee, K + alpha I, sur la decomposition spectrale du Gram des lignes.
#[allow(dead_code)]
fn mesure_ridge(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    alpha: f64,
) -> (f64, f64, f64) {
    let (valeurs, u) = propres_sym(&gram(xtr));
    let (p_tr, p_te, norme) = predire(xtr, xte, &ridge_alpha(&valeurs, &u, ytr, alpha));
    (rmse(ytr, &p_tr), rmse(yte, &p_te), norme)
}

let tiny_x: Vec<Vec<f64>> = vec![
    vec![1.0, 0.0],
    vec![0.0, 1.0],
    vec![1.0, 1.0],
    vec![2.0, -1.0],
    vec![1.0, 2.0],
    vec![2.0, 2.0],
];
let tiny_y: Vec<f64> = vec![1.0, -1.0, 0.0, 3.0, -1.0, 1.0];
let j = jeu(GRAINE);
let p = preparer(&j, &cible(&j));
// Une fixture : (X train, y train, X test, y test).
let fixtures: Vec<(Vec<Vec<f64>>, Vec<f64>, Vec<Vec<f64>>, Vec<f64>)> = vec![
    (
        p.x_train_s.clone(),
        p.y_train_z.clone(),
        p.x_test_s.clone(),
        p.y_test_z.clone(),
    ),
    (
        tiny_x.clone(),
        tiny_y.clone(),
        tiny_x[..3].to_vec(),
        tiny_y[..3].to_vec(),
    ),
];
let reponses: Vec<Option<(Vec<(f64, f64, f64)>, f64)>> = fixtures
    .iter()
    .map(|(xtr, ytr, xte, yte)| ridge_extreme(xtr, ytr, xte, yte))
    .collect();
if reponses.iter().all(|r| r.is_none()) {
    println!("Exercice 1 a completer.");
} else if reponses
    .iter()
    .any(|r| match r {
        Some((mesures, _)) => mesures.len() != 2,
        None => true,
    })
{
    println!("Exercice 1 : a revoir");
} else {
    let mut ok = true;
    for ((xtr, ytr, xte, yte), reponse) in fixtures.iter().zip(&reponses) {
        if let Some((mesures, constante)) = reponse {
            for (k, alpha) in [1e6f64, 1e9].iter().enumerate() {
                let (a1, a2, a3) = mesures[k];
                let (b1, b2, b3) = mesure_ridge(xtr, ytr, xte, yte, *alpha);
                let proche = |a: f64, b: f64| (a - b).abs() <= 1e-10 + 1e-8 * b.abs();
                ok = ok
                    && a1.is_finite()
                    && a2.is_finite()
                    && a3.is_finite()
                    && proche(a1, b1)
                    && proche(a2, b2)
                    && proche(a3, b3);
            }
            let constante_attendue = rmse(yte, &vec![moyenne(ytr); yte.len()]);
            ok = ok && constante.is_finite() && (*constante - constante_attendue).abs() < 1e-10;
        }
    }
    println!(
        "{}",
        if ok {
            "Exercice 1 : mesures correctes ; interpreter la limite"
        } else {
            "Exercice 1 : a revoir"
        }
    );
};

Exercice 1 a completer.


## 4. PCR — régression sur composantes principales

PCR (*principal component regression*) projette $X$ sur ses $k$ premières composantes principales — celles du [2.6](2.6-Clustering-KMeans-PCA.ipynb), non réenseignées ici : le spectre du Gram des lignes les redonne — puis régresse $Y$ dessus par une régression à intercept sur des scores orthogonaux. L'ACP **ignore $Y$** : elle trie les directions par $\mathrm{Var}(Xu)$ décroissante. Où les directions du bloc B tombent-elles dans cet ordre ?

In [6]:
// Cellule autonome : prelude (RNG, donnees, algebre) redefini ici, puis le calcul de la section.
// ---- Generateur pseudo-alatoire maison : xorshift64* + Box-Muller (aucun crate).
// La suite n'est pas celle de NumPy : les tirages different, les valeurs chiffrees sont celles de ce carnet.
#[allow(dead_code)]
struct Rng {
    etat: u64,
}

#[allow(dead_code)]
impl Rng {
    fn new(graine: u64) -> Rng {
        let mut etat = graine ^ 0x9E3779B97F4A7C15;
        if etat == 0 {
            etat = 0x2545F4914F6CDD1D;
        }
        Rng { etat }
    }

    fn brut(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x >> 12;
        x ^= x << 25;
        x ^= x >> 27;
        self.etat = x;
        x.wrapping_mul(0x2545F4914F6CDD1D)
    }

    // Uniforme sur [0, 1), 53 bits de mantisse.
    fn uniforme(&mut self) -> f64 {
        ((self.brut() >> 11) as f64 + 0.5) / 9007199254740992.0
    }

    // Normale centree reduite, methode de Box-Muller.
    fn normale(&mut self) -> f64 {
        const TAU: f64 = 6.283185307179586;
        loop {
            let u1 = self.uniforme();
            let u2 = self.uniforme();
            if u1 > 1e-300 {
                return (-2.0 * u1.ln()).sqrt() * (TAU * u2).cos();
            }
        }
    }

    // Entier uniforme dans 0..n, par rejet (pas le biais du modulo).
    fn indice(&mut self, n: usize) -> usize {
        let n = n as u64;
        let limite = u64::MAX - (u64::MAX % n);
        loop {
            let t = self.brut();
            if t < limite {
                return (t % n) as usize;
            }
        }
    }
}

// Permutation aleatoire (Fisher-Yates).
#[allow(dead_code)]
fn permutation(rng: &mut Rng, n: usize) -> Vec<usize> {
    let mut idx: Vec<usize> = (0..n).collect();
    for i in (1..n).rev() {
        let j = rng.indice(i + 1);
        idx.swap(i, j);
    }
    idx
}

// ---- Statistiques de base (ecarts-types au denominateur n, comme NumPy). ----
#[allow(dead_code)]
fn moyenne(v: &[f64]) -> f64 {
    v.iter().sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn variance(v: &[f64]) -> f64 {
    let m = moyenne(v);
    v.iter().map(|x| (x - m) * (x - m)).sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn ecart_type(v: &[f64]) -> f64 {
    variance(v).sqrt()
}

// Matrices stockees ligne par ligne : x[i][j].
#[allow(dead_code)]
fn mat_vec_ligne(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    x.iter()
        .map(|l| l.iter().zip(b).map(|(a, c)| a * c).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn colonnes_moyennes(x: &[Vec<f64>]) -> Vec<f64> {
    let n = x.len();
    (0..x[0].len())
        .map(|j| x.iter().map(|l| l[j]).sum::<f64>() / n as f64)
        .collect()
}

#[allow(dead_code)]
fn colonnes_ecarts(x: &[Vec<f64>]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| ecart_type(&x.iter().map(|l| l[j]).collect::<Vec<f64>>()))
        .collect()
}

#[allow(dead_code)]
fn standardiser(x: &[Vec<f64>], mu: &[f64], sigma: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            l.iter()
                .zip(mu.iter().zip(sigma))
                .map(|(a, (m, s))| (a - m) / s)
                .collect()
        })
        .collect()
}

#[allow(dead_code)]
fn standardiser_vec(v: &[f64], m: f64, s: f64) -> Vec<f64> {
    v.iter().map(|x| (x - m) / s).collect()
}

#[allow(dead_code)]
fn centrer(v: &[f64]) -> Vec<f64> {
    let m = moyenne(v);
    v.iter().map(|x| x - m).collect()
}

#[allow(dead_code)]
fn centrer_colonnes(x: &[Vec<f64>], mu: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| l.iter().zip(mu).map(|(a, m)| a - m).collect())
        .collect()
}

#[allow(dead_code)]
fn correlation(a: &[f64], b: &[f64]) -> f64 {
    let (ca, cb) = (centrer(a), centrer(b));
    let num: f64 = ca.iter().zip(&cb).map(|(x, y)| x * y).sum();
    let den: f64 = ca.iter().map(|x| x * x).sum::<f64>().sqrt()
        * cb.iter().map(|y| y * y).sum::<f64>().sqrt();
    num / den
}

#[allow(dead_code)]
fn rmse(a: &[f64], b: &[f64]) -> f64 {
    (a.iter().zip(b).map(|(x, y)| (x - y) * (x - y)).sum::<f64>() / a.len() as f64).sqrt()
}

// Indice du minimum (le premier en cas d'egalite), comme np.argmin.
#[allow(dead_code)]
fn argmin(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] < v[m] {
            m = k;
        }
    }
    m
}

// Indice du maximum (le premier en cas d'egalite), comme np.argmax.
#[allow(dead_code)]
fn argmax(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] > v[m] {
            m = k;
        }
    }
    m
}

#[allow(dead_code)]
fn logspace(a: f64, b: f64, n: usize) -> Vec<f64> {
    (0..n)
        .map(|i| 10f64.powf(a + (b - a) * i as f64 / (n as f64 - 1.0)))
        .collect()
}

// Colonne j d'une matrice stockee ligne par ligne.
#[allow(dead_code)]
fn colonne(x: &[Vec<f64>], j: usize) -> Vec<f64> {
    x.iter().map(|l| l[j]).collect()
}

// Somme des valeurs absolues sur deux blocs de variables : (0..debut_b, debut_b..fin).
#[allow(dead_code)]
fn masses_blocs(v: &[f64], debut_b: usize) -> (f64, f64) {
    let a: f64 = v[..debut_b].iter().map(|x| x.abs()).sum();
    let b: f64 = v[debut_b..].iter().map(|x| x.abs()).sum();
    (a, b)
}

// ---- Donnees : meme construction que la source (blocs A et B, cible, decoupage, standardisation). ----
#[allow(dead_code)]
const N: usize = 160;
#[allow(dead_code)]
const G_A: usize = 40;
#[allow(dead_code)]
const TAILLE_G: usize = 5;
#[allow(dead_code)]
const P_A: usize = 200;
#[allow(dead_code)]
const P_B: usize = 5;
#[allow(dead_code)]
const P: usize = 205;
#[allow(dead_code)]
const N_TRAIN: usize = 120;
#[allow(dead_code)]
const GRAINE: u64 = 42;

// Variables en groupes ; dans chaque groupe, corr(x_i, x_j) = rho_pair via facteur commun.
#[allow(dead_code)]
fn bloc_facteurs(rng: &mut Rng, n_vars: usize, n_groupes: usize, rho_pair: f64) -> Vec<Vec<f64>> {
    let mut x = vec![vec![0.0; n_vars]; N];
    for g in 0..n_groupes {
        let f: Vec<f64> = (0..N).map(|_| rho_pair.sqrt() * rng.normale()).collect();
        for j in 0..TAILLE_G {
            for i in 0..N {
                x[i][g * TAILLE_G + j] = f[i] + (1.0 - rho_pair).sqrt() * rng.normale();
            }
        }
    }
    x
}

// Un tirage complet depuis la graine : les deux blocs, le bruit, le decoupage et les poids du regime aligne.
#[allow(dead_code)]
struct Jeu {
    x: Vec<Vec<f64>>,
    bruit: Vec<f64>,
    idx: Vec<usize>,
    poids_fact: Vec<f64>,
}

// 40 directions de variance dominante sans signal (bloc A), 1 direction plus modeste generatrice de Y (bloc B).
#[allow(dead_code)]
fn jeu(graine: u64) -> Jeu {
    let mut rng = Rng::new(graine);
    let mut x = bloc_facteurs(&mut rng, P_A, G_A, 0.8);
    let xb = bloc_facteurs(&mut rng, P_B, 1, 0.6);
    for i in 0..N {
        x[i].extend(xb[i].iter().copied());
    }
    let bruit: Vec<f64> = (0..N).map(|_| rng.normale()).collect();
    let idx = permutation(&mut rng, N);
    let poids_fact: Vec<f64> = (0..G_A).map(|_| rng.normale() / (G_A as f64).sqrt()).collect();
    Jeu { x, bruit, idx, poids_fact }
}

// Betas de meme signe, alignes sur le facteur commun du bloc B ; Y = X beta + bruit.
#[allow(dead_code)]
fn cible(j: &Jeu) -> Vec<f64> {
    let formes = [1.3, 1.0, 1.2, 0.9, 1.1];
    let mut beta = vec![0.0; P];
    for k in 0..P_B {
        beta[P_A + k] = formes[k] * 1.2;
    }
    let mut y = mat_vec_ligne(&j.x, &beta);
    for i in 0..N {
        y[i] += j.bruit[i];
    }
    y
}

// Regime aligne : Y2 = facteurs_A . poids_fact * 3.0 + bruit. Les facteurs_A sont des proxis construits
// a partir de la premiere variable observee de chaque groupe du bloc A, pas les facteurs latents exacts.
#[allow(dead_code)]
fn cible_alignee(j: &Jeu) -> Vec<f64> {
    let mut y = vec![0.0; N];
    for i in 0..N {
        let mut s = 0.0;
        for g in 0..G_A {
            s += 0.8f64.sqrt() * j.x[i][g * TAILLE_G] * j.poids_fact[g];
        }
        y[i] = s * 3.0 + j.bruit[i];
    }
    y
}

// Decoupage 120 train / 40 test, standardisation ajustee sur le train uniquement (le geste du 2.1).
#[allow(dead_code)]
struct Prep {
    x_train_s: Vec<Vec<f64>>,
    x_test_s: Vec<Vec<f64>>,
    y_train_z: Vec<f64>,
    y_test_z: Vec<f64>,
    y_ecart: f64,
}

#[allow(dead_code)]
fn preparer(j: &Jeu, y: &[f64]) -> Prep {
    let x_train: Vec<Vec<f64>> = (0..N_TRAIN).map(|k| j.x[j.idx[k]].clone()).collect();
    let x_test: Vec<Vec<f64>> = (N_TRAIN..N).map(|k| j.x[j.idx[k]].clone()).collect();
    let y_train: Vec<f64> = (0..N_TRAIN).map(|k| y[j.idx[k]]).collect();
    let y_test: Vec<f64> = (N_TRAIN..N).map(|k| y[j.idx[k]]).collect();
    let (mu, sigma) = (colonnes_moyennes(&x_train), colonnes_ecarts(&x_train));
    let (y_m, y_s) = (moyenne(&y_train), ecart_type(&y_train));
    Prep {
        x_train_s: standardiser(&x_train, &mu, &sigma),
        x_test_s: standardiser(&x_test, &mu, &sigma),
        y_train_z: standardiser_vec(&y_train, y_m, y_s),
        y_test_z: standardiser_vec(&y_test, y_m, y_s),
        y_ecart: y_s,
    }
}

// ---- Algebre lineaire : tout passe par la matrice de Gram des lignes K = X X^T (n x n, ici 120 x 120),
// jamais par X^T X (p x p, singuliere des que p > n). ----
#[allow(dead_code)]
fn transpose_mat_vec(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| x.iter().zip(b).map(|(l, &v)| l[j] * v).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn gram(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let n = x.len();
    (0..n)
        .map(|i| {
            (0..n)
                .map(|k| x[i].iter().zip(&x[k]).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// Valeurs propres (decroissantes) et vecteurs propres d'une matrice symetrique par la methode de
// Jacobi cyclique. vecteurs[k][i] est la i-eme composante du k-eme vecteur propre ; le signe de chaque
// vecteur est choisi pour que sa plus grande composante en valeur absolue soit positive.
#[allow(dead_code)]
fn propres_sym(m: &[Vec<f64>]) -> (Vec<f64>, Vec<Vec<f64>>) {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut v: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|k| if i == k { 1.0 } else { 0.0 }).collect())
        .collect();
    for _balayage in 0..100 {
        let diag: f64 = (0..n).map(|i| a[i][i].abs()).fold(0.0f64, f64::max);
        let mut carres = 0.0f64;
        for i in 0..n {
            for k in (i + 1)..n {
                carres += a[i][k] * a[i][k];
            }
        }
        if carres.sqrt() <= 1e-13 * (1.0 + diag) {
            break;
        }
        let mut rotations = 0usize;
        for p in 0..n {
            for q in (p + 1)..n {
                let apq = a[p][q];
                // Pivot negligeable devant la diagonale : on l'annule sans tourner.
                if apq.abs() <= 1e-18 * (a[p][p].abs() + a[q][q].abs()) {
                    a[p][q] = 0.0;
                    a[q][p] = 0.0;
                    continue;
                }
                rotations += 1;
                let theta = (a[q][q] - a[p][p]) / (2.0 * apq);
                let t = if theta.abs() > 1e150 {
                    1.0 / (2.0 * theta)
                } else if theta >= 0.0 {
                    1.0 / (theta + (theta * theta + 1.0).sqrt())
                } else {
                    -1.0 / (-theta + (theta * theta + 1.0).sqrt())
                };
                let c = 1.0 / (t * t + 1.0).sqrt();
                let s = t * c;
                for k in 0..n {
                    if k != p && k != q {
                        let akp = a[k][p];
                        let akq = a[k][q];
                        a[k][p] = c * akp - s * akq;
                        a[p][k] = a[k][p];
                        a[k][q] = s * akp + c * akq;
                        a[q][k] = a[k][q];
                    }
                    let vkp = v[k][p];
                    let vkq = v[k][q];
                    v[k][p] = c * vkp - s * vkq;
                    v[k][q] = s * vkp + c * vkq;
                }
                a[p][p] -= t * apq;
                a[q][q] += t * apq;
                a[p][q] = 0.0;
                a[q][p] = 0.0;
            }
        }
        if rotations == 0 {
            break;
        }
    }
    let mut ordre: Vec<usize> = (0..n).collect();
    ordre.sort_by(|&i, &k| a[k][k].total_cmp(&a[i][i]));
    let valeurs: Vec<f64> = ordre.iter().map(|&i| a[i][i]).collect();
    let vecteurs: Vec<Vec<f64>> = ordre
        .iter()
        .map(|&k| {
            let colonne: Vec<f64> = v.iter().map(|l| l[k]).collect();
            let pivot = argmax(&colonne.iter().map(|x| x.abs()).collect::<Vec<f64>>());
            if colonne[pivot] < 0.0 {
                colonne.iter().map(|x| -x).collect()
            } else {
                colonne
            }
        })
        .collect();
    (valeurs, vecteurs)
}

// Rang numerique : nombre de valeurs propres au-dessus de 1e-8 fois la plus grande.
#[allow(dead_code)]
fn rang(valeurs: &[f64]) -> usize {
    let lmax = valeurs[0];
    valeurs.iter().filter(|v| **v > 1e-8 * lmax).count()
}

// OLS de norme minimale sans ordonnee, par K = X X^T : alpha = U_r L_r^{-1} U_r^T y, puis beta = X^T alpha.
// Les r valeurs propres non nulles fixent la prediction ; le noyau (p - r directions) ne la modifie pas.
#[allow(dead_code)]
fn ols_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64]) -> Vec<f64> {
    let lmax = valeurs[0];
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        if valeurs[k] > 1e-8 * lmax {
            let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
            for i in 0..n {
                alpha[i] += u[k][i] * d / valeurs[k];
            }
        }
    }
    alpha
}

// Coefficients duaux du ridge sans ordonnee : alpha = (K + lambda I)^{-1} y, dans la base des vecteurs propres.
#[allow(dead_code)]
fn ridge_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64], lambda: f64) -> Vec<f64> {
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
        for i in 0..n {
            alpha[i] += u[k][i] * d / (valeurs[k] + lambda);
        }
    }
    alpha
}

// beta = X^T alpha ; renvoie les previsions X beta sur train et test, et la norme L2 de beta.
#[allow(dead_code)]
fn predire(xtr: &[Vec<f64>], xte: &[Vec<f64>], alpha: &[f64]) -> (Vec<f64>, Vec<f64>, f64) {
    let beta = transpose_mat_vec(xtr, alpha);
    let norme = beta.iter().map(|b| b * b).sum::<f64>().sqrt();
    (mat_vec_ligne(xtr, &beta), mat_vec_ligne(xte, &beta), norme)
}

// Bornes du pli f d'une validation croisee a plis contigus (KFold sans melange).
#[allow(dead_code)]
fn bornes_pli(n: usize, plis: usize, f: usize) -> (usize, usize) {
    let (base, reste) = (n / plis, n % plis);
    let debut = f * base + reste.min(f);
    (debut, debut + base + if f < reste { 1 } else { 0 })
}

// Un pli contigu, avec la decomposition spectrale du Gram de son train : elle sert a tous les alpha du ridge.
#[allow(dead_code)]
struct PliContigu {
    xtr: Vec<Vec<f64>>,
    xva: Vec<Vec<f64>>,
    positions_val: Vec<usize>,
    valeurs: Vec<f64>,
    u: Vec<Vec<f64>>,
}

#[allow(dead_code)]
fn plis_contigus(x: &[Vec<f64>], plis: usize) -> Vec<PliContigu> {
    let n = x.len();
    (0..plis)
        .map(|f| {
            let (debut, fin) = bornes_pli(n, plis, f);
            let xtr: Vec<Vec<f64>> = (0..n)
                .filter(|&i| i < debut || i >= fin)
                .map(|i| x[i].clone())
                .collect();
            let xva: Vec<Vec<f64>> = x[debut..fin].to_vec();
            let (valeurs, u) = propres_sym(&gram(&xtr));
            PliContigu {
                xtr,
                xva,
                positions_val: (debut..fin).collect(),
                valeurs,
                u,
            }
        })
        .collect()
}

// RidgeCV(alphas, cv=plis, fit_intercept=False) : l'alpha retenu est celui qui maximise la moyenne
// des R2 de pli (le score par defaut de la validation croisee de scikit-learn). Renvoie son indice.
#[allow(dead_code)]
fn ridge_cv_sur_plis(plis_c: &[PliContigu], y: &[f64], alphas: &[f64]) -> usize {
    let mut score = vec![0.0; alphas.len()];
    for pli in plis_c {
        let ytr: Vec<f64> = (0..y.len())
            .filter(|i| !pli.positions_val.contains(i))
            .map(|i| y[i])
            .collect();
        let yva: Vec<f64> = pli.positions_val.iter().map(|&i| y[i]).collect();
        let sst: f64 = centrer(&yva).iter().map(|x| x * x).sum();
        for (j, &lam) in alphas.iter().enumerate() {
            let (_, p_va, _) = predire(
                &pli.xtr,
                &pli.xva,
                &ridge_alpha(&pli.valeurs, &pli.u, &ytr, lam),
            );
            let sse: f64 = yva.iter().zip(&p_va).map(|(t, p)| (t - p) * (t - p)).sum();
            score[j] += 1.0 - sse / sst;
        }
    }
    argmax(&score)
}

#[allow(dead_code)]
fn ridge_cv_retenu(x: &[Vec<f64>], y: &[f64], alphas: &[f64], plis: usize) -> usize {
    ridge_cv_sur_plis(&plis_contigus(x, plis), y, alphas)
}

// Composantes principales depuis la decomposition de K = X X^T : composante_k = X^T u_k / sqrt(lambda_k).
#[allow(dead_code)]
fn composantes(x: &[Vec<f64>], valeurs: &[f64], u: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let mut comps: Vec<Vec<f64>> = Vec::new();
    for k in 0..rang(valeurs) {
        let mut c = transpose_mat_vec(x, &u[k]);
        let s = valeurs[k].sqrt();
        for v in c.iter_mut() {
            *v /= s;
        }
        comps.push(c);
    }
    comps
}

// Centrage d'un echantillon sur ses propres colonnes : ce que fait PCA a l'ajustement.
#[allow(dead_code)]
fn centrer_propre(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    centrer_colonnes(x, &colonnes_moyennes(x))
}

// ACP, comme sklearn.decomposition.PCA : X est CENTRE sur ses colonnes, covariance (denominateur n),
// valeurs propres, part de variance, composantes. Le centrage ne change rien sur une matrice deja
// standardisee sur elle-meme (train complet) ; il est indispensable sur le train d'un pli, dont les
// colonnes ne sont pas de moyenne nulle.
#[allow(dead_code)]
fn acp(x: &[Vec<f64>]) -> (Vec<f64>, Vec<f64>, Vec<Vec<f64>>) {
    let xc = centrer_propre(x);
    let (valeurs, u) = propres_sym(&gram(&xc));
    let total: f64 = valeurs.iter().sum();
    let comps = composantes(&xc, &valeurs, &u);
    let ratio: Vec<f64> = valeurs.iter().map(|l| l / total).collect();
    (valeurs, ratio, comps)
}

// Variance cumulee jusqu'a la composante k comprise.
#[allow(dead_code)]
fn variance_cumulee(ratio: &[f64], k: usize) -> f64 {
    ratio[..=k].iter().sum()
}

// Scores t[i][k] = x_i . composante_k.
#[allow(dead_code)]
fn scores_composantes(x: &[Vec<f64>], comps: &[Vec<f64>]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            comps
                .iter()
                .map(|c| l.iter().zip(c).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// PCR a k composantes avec ordonnee : y = ybar + sum_{j<k} b_j t_j.
// Les scores d'une ACP sont orthogonaux deux a deux, donc b_j = (t_j . (y - ybar)) / (t_j . t_j).
#[allow(dead_code)]
fn regression_scores(
    t_tr: &[Vec<f64>],
    t_te: &[Vec<f64>],
    y: &[f64],
    k: usize,
) -> (Vec<f64>, Vec<f64>) {
    let ym = moyenne(y);
    let yc: Vec<f64> = y.iter().map(|v| v - ym).collect();
    let mut b = vec![0.0; k];
    for j in 0..k {
        let num: f64 = t_tr.iter().zip(&yc).map(|(t, &v)| t[j] * v).sum();
        let den: f64 = t_tr.iter().map(|t| t[j] * t[j]).sum();
        b[j] = num / den;
    }
    let projet = |tt: &[Vec<f64>]| -> Vec<f64> {
        tt.iter()
            .map(|t| ym + (0..k).map(|j| b[j] * t[j]).sum::<f64>())
            .collect()
    };
    (projet(t_tr), projet(t_te))
}

// RMSE test de la PCR pour k = 1..kmax (une seule ACP, une regression par k).
#[allow(dead_code)]
fn balayage_pcr(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (_, _, mut comps) = acp(xtr);
    comps.truncate(kmax);
    // PCA().fit(xtr).transform(...) : les deux echantillons sont centres sur la moyenne du train.
    let xmu = colonnes_moyennes(xtr);
    let (t_tr, t_te) = (
        scores_composantes(&centrer_colonnes(xtr, &xmu), &comps),
        scores_composantes(&centrer_colonnes(xte, &xmu), &comps),
    );
    (1..=kmax)
        .map(|k| rmse(yte, &regression_scores(&t_tr, &t_te, ytr, k).1))
        .collect()
}

// ---- 4. PCR : ou tombent les directions du bloc B dans l'ordre des variances ? ----
let j = jeu(GRAINE);
let p = preparer(&j, &cible(&j));
let (_, ratio, comps) = acp(&p.x_train_s);
let imp_b: Vec<f64> = comps
    .iter()
    .map(|c| (P_A..P).map(|v| c[v] * c[v]).sum())
    .collect(); // part des 5 variables B par composante
let i_max_b = argmax(&imp_b);
let premier_hit = (0..imp_b.len()).find(|&k| imp_b[k] > 0.15);
println!(
    "variance expliquee des 40 premieres composantes : {:.3}",
    variance_cumulee(&ratio, 39)
);
println!(
    "la 1re composante porte {:.3} de la variance -- et {:.4} de la contribution du bloc B",
    ratio[0], imp_b[0]
);
println!(
    "composante la plus chargee en bloc B : index {} (part {:.3}, variance cumulee {:.3})",
    i_max_b,
    imp_b[i_max_b],
    variance_cumulee(&ratio, i_max_b)
);
if let Some(h) = premier_hit {
    println!(
        "premiere composante ou le bloc B pese > 15% : index {} (variance cumulee {:.3})",
        h,
        variance_cumulee(&ratio, h)
    );
}

// PCR : les scores d'une ACP sont orthogonaux deux a deux, la regression a k composantes s'ecrit
// composante par composante -- une seule ACP pour les 45 valeurs de k.
let rmse_pcr = balayage_pcr(&p.x_train_s, &p.y_train_z, &p.x_test_s, &p.y_test_z, 45);
let best_pcr_k = argmin(&rmse_pcr) + 1;
println!(
    "\nPCR : meilleur k = {}, RMSE test = {:.4}",
    best_pcr_k,
    rmse_pcr[best_pcr_k - 1]
);
for k in [1usize, 5, 20, 35, 40, 45] {
    println!("RMSE PCR a k={:2} : {:.4}", k, rmse_pcr[k - 1]);
};

variance expliquee des 40 premieres composantes : 0.888


la 1re composante porte 0.051 de la variance -- et 0.0244 de la contribution du bloc B


composante la plus chargee en bloc B : index 43 (part 0.104, variance cumulee 0.904)


PCR : meilleur k = 45, RMSE test = 0.2019


RMSE PCR a k= 1 : 1.0449


RMSE PCR a k= 5 : 0.9673


RMSE PCR a k=20 : 0.7784


RMSE PCR a k=35 : 0.4472


RMSE PCR a k=40 : 0.3599


RMSE PCR a k=45 : 0.2019


### Lecture du résultat

Les 40 premières composantes portent 88.8% de la variance — mais la variance expliquée seule ne démontre pas une absence d'information sur Y : la première composante (5.1% de variance) ne doit que 2.44% de ses loadings au bloc B. La composante la plus chargée en bloc B est l'**indice 43** (44ᵉ composante, part 10.4%, variance cumulée 90.4%) — et aucune composante n'atteint le seuil de 15% : dans ce tirage, le signal ne se concentre sur aucune direction de l'ACP, il reste dilué. La courbe PCR le paie intégralement : 1.0449 à k=1, encore 0.9673 à k=5, 0.7784 à k=20, 0.4472 à k=35, 0.3599 à k=40, puis 0.2019 à k=45. Deux conséquences à lire honnêtement : **PCR finit par trouver le signal**, mais après avoir inclus des dizaines de composantes de bruit ; et le meilleur point de la grille est son **plafond** (k=45, courbe encore décroissante), donc l'expérience établit que le signal entre au-delà de la 40ᵉ composante, pas qu'il entre exactement à la 45ᵉ.

### Exercice 2 — l'ordre des composantes PCR sous vos yeux

Complétez `rang_signal` pour retourner l'index de la composante principale la plus chargée en **bloc B** et la variance cumulée à cet index. *Indice : reprendre la part du bloc B calculée au §4 et la somme cumulée des ratios de variance.*

La garde accepte toute implémentation qui retrouve l'index du maximum et la variance cumulée à $10^{-10}$ près, sur les composantes du §4 comme sur une matrice 3 × 3 de synthèse.

In [7]:
// Cellule autonome : prelude (RNG, donnees, algebre) redefini ici, puis le calcul de la section.
// ---- Generateur pseudo-alatoire maison : xorshift64* + Box-Muller (aucun crate).
// La suite n'est pas celle de NumPy : les tirages different, les valeurs chiffrees sont celles de ce carnet.
#[allow(dead_code)]
struct Rng {
    etat: u64,
}

#[allow(dead_code)]
impl Rng {
    fn new(graine: u64) -> Rng {
        let mut etat = graine ^ 0x9E3779B97F4A7C15;
        if etat == 0 {
            etat = 0x2545F4914F6CDD1D;
        }
        Rng { etat }
    }

    fn brut(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x >> 12;
        x ^= x << 25;
        x ^= x >> 27;
        self.etat = x;
        x.wrapping_mul(0x2545F4914F6CDD1D)
    }

    // Uniforme sur [0, 1), 53 bits de mantisse.
    fn uniforme(&mut self) -> f64 {
        ((self.brut() >> 11) as f64 + 0.5) / 9007199254740992.0
    }

    // Normale centree reduite, methode de Box-Muller.
    fn normale(&mut self) -> f64 {
        const TAU: f64 = 6.283185307179586;
        loop {
            let u1 = self.uniforme();
            let u2 = self.uniforme();
            if u1 > 1e-300 {
                return (-2.0 * u1.ln()).sqrt() * (TAU * u2).cos();
            }
        }
    }

    // Entier uniforme dans 0..n, par rejet (pas le biais du modulo).
    fn indice(&mut self, n: usize) -> usize {
        let n = n as u64;
        let limite = u64::MAX - (u64::MAX % n);
        loop {
            let t = self.brut();
            if t < limite {
                return (t % n) as usize;
            }
        }
    }
}

// Permutation aleatoire (Fisher-Yates).
#[allow(dead_code)]
fn permutation(rng: &mut Rng, n: usize) -> Vec<usize> {
    let mut idx: Vec<usize> = (0..n).collect();
    for i in (1..n).rev() {
        let j = rng.indice(i + 1);
        idx.swap(i, j);
    }
    idx
}

// ---- Statistiques de base (ecarts-types au denominateur n, comme NumPy). ----
#[allow(dead_code)]
fn moyenne(v: &[f64]) -> f64 {
    v.iter().sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn variance(v: &[f64]) -> f64 {
    let m = moyenne(v);
    v.iter().map(|x| (x - m) * (x - m)).sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn ecart_type(v: &[f64]) -> f64 {
    variance(v).sqrt()
}

// Matrices stockees ligne par ligne : x[i][j].
#[allow(dead_code)]
fn mat_vec_ligne(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    x.iter()
        .map(|l| l.iter().zip(b).map(|(a, c)| a * c).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn colonnes_moyennes(x: &[Vec<f64>]) -> Vec<f64> {
    let n = x.len();
    (0..x[0].len())
        .map(|j| x.iter().map(|l| l[j]).sum::<f64>() / n as f64)
        .collect()
}

#[allow(dead_code)]
fn colonnes_ecarts(x: &[Vec<f64>]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| ecart_type(&x.iter().map(|l| l[j]).collect::<Vec<f64>>()))
        .collect()
}

#[allow(dead_code)]
fn standardiser(x: &[Vec<f64>], mu: &[f64], sigma: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            l.iter()
                .zip(mu.iter().zip(sigma))
                .map(|(a, (m, s))| (a - m) / s)
                .collect()
        })
        .collect()
}

#[allow(dead_code)]
fn standardiser_vec(v: &[f64], m: f64, s: f64) -> Vec<f64> {
    v.iter().map(|x| (x - m) / s).collect()
}

#[allow(dead_code)]
fn centrer(v: &[f64]) -> Vec<f64> {
    let m = moyenne(v);
    v.iter().map(|x| x - m).collect()
}

#[allow(dead_code)]
fn centrer_colonnes(x: &[Vec<f64>], mu: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| l.iter().zip(mu).map(|(a, m)| a - m).collect())
        .collect()
}

#[allow(dead_code)]
fn correlation(a: &[f64], b: &[f64]) -> f64 {
    let (ca, cb) = (centrer(a), centrer(b));
    let num: f64 = ca.iter().zip(&cb).map(|(x, y)| x * y).sum();
    let den: f64 = ca.iter().map(|x| x * x).sum::<f64>().sqrt()
        * cb.iter().map(|y| y * y).sum::<f64>().sqrt();
    num / den
}

#[allow(dead_code)]
fn rmse(a: &[f64], b: &[f64]) -> f64 {
    (a.iter().zip(b).map(|(x, y)| (x - y) * (x - y)).sum::<f64>() / a.len() as f64).sqrt()
}

// Indice du minimum (le premier en cas d'egalite), comme np.argmin.
#[allow(dead_code)]
fn argmin(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] < v[m] {
            m = k;
        }
    }
    m
}

// Indice du maximum (le premier en cas d'egalite), comme np.argmax.
#[allow(dead_code)]
fn argmax(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] > v[m] {
            m = k;
        }
    }
    m
}

#[allow(dead_code)]
fn logspace(a: f64, b: f64, n: usize) -> Vec<f64> {
    (0..n)
        .map(|i| 10f64.powf(a + (b - a) * i as f64 / (n as f64 - 1.0)))
        .collect()
}

// Colonne j d'une matrice stockee ligne par ligne.
#[allow(dead_code)]
fn colonne(x: &[Vec<f64>], j: usize) -> Vec<f64> {
    x.iter().map(|l| l[j]).collect()
}

// Somme des valeurs absolues sur deux blocs de variables : (0..debut_b, debut_b..fin).
#[allow(dead_code)]
fn masses_blocs(v: &[f64], debut_b: usize) -> (f64, f64) {
    let a: f64 = v[..debut_b].iter().map(|x| x.abs()).sum();
    let b: f64 = v[debut_b..].iter().map(|x| x.abs()).sum();
    (a, b)
}

// ---- Donnees : meme construction que la source (blocs A et B, cible, decoupage, standardisation). ----
#[allow(dead_code)]
const N: usize = 160;
#[allow(dead_code)]
const G_A: usize = 40;
#[allow(dead_code)]
const TAILLE_G: usize = 5;
#[allow(dead_code)]
const P_A: usize = 200;
#[allow(dead_code)]
const P_B: usize = 5;
#[allow(dead_code)]
const P: usize = 205;
#[allow(dead_code)]
const N_TRAIN: usize = 120;
#[allow(dead_code)]
const GRAINE: u64 = 42;

// Variables en groupes ; dans chaque groupe, corr(x_i, x_j) = rho_pair via facteur commun.
#[allow(dead_code)]
fn bloc_facteurs(rng: &mut Rng, n_vars: usize, n_groupes: usize, rho_pair: f64) -> Vec<Vec<f64>> {
    let mut x = vec![vec![0.0; n_vars]; N];
    for g in 0..n_groupes {
        let f: Vec<f64> = (0..N).map(|_| rho_pair.sqrt() * rng.normale()).collect();
        for j in 0..TAILLE_G {
            for i in 0..N {
                x[i][g * TAILLE_G + j] = f[i] + (1.0 - rho_pair).sqrt() * rng.normale();
            }
        }
    }
    x
}

// Un tirage complet depuis la graine : les deux blocs, le bruit, le decoupage et les poids du regime aligne.
#[allow(dead_code)]
struct Jeu {
    x: Vec<Vec<f64>>,
    bruit: Vec<f64>,
    idx: Vec<usize>,
    poids_fact: Vec<f64>,
}

// 40 directions de variance dominante sans signal (bloc A), 1 direction plus modeste generatrice de Y (bloc B).
#[allow(dead_code)]
fn jeu(graine: u64) -> Jeu {
    let mut rng = Rng::new(graine);
    let mut x = bloc_facteurs(&mut rng, P_A, G_A, 0.8);
    let xb = bloc_facteurs(&mut rng, P_B, 1, 0.6);
    for i in 0..N {
        x[i].extend(xb[i].iter().copied());
    }
    let bruit: Vec<f64> = (0..N).map(|_| rng.normale()).collect();
    let idx = permutation(&mut rng, N);
    let poids_fact: Vec<f64> = (0..G_A).map(|_| rng.normale() / (G_A as f64).sqrt()).collect();
    Jeu { x, bruit, idx, poids_fact }
}

// Betas de meme signe, alignes sur le facteur commun du bloc B ; Y = X beta + bruit.
#[allow(dead_code)]
fn cible(j: &Jeu) -> Vec<f64> {
    let formes = [1.3, 1.0, 1.2, 0.9, 1.1];
    let mut beta = vec![0.0; P];
    for k in 0..P_B {
        beta[P_A + k] = formes[k] * 1.2;
    }
    let mut y = mat_vec_ligne(&j.x, &beta);
    for i in 0..N {
        y[i] += j.bruit[i];
    }
    y
}

// Regime aligne : Y2 = facteurs_A . poids_fact * 3.0 + bruit. Les facteurs_A sont des proxis construits
// a partir de la premiere variable observee de chaque groupe du bloc A, pas les facteurs latents exacts.
#[allow(dead_code)]
fn cible_alignee(j: &Jeu) -> Vec<f64> {
    let mut y = vec![0.0; N];
    for i in 0..N {
        let mut s = 0.0;
        for g in 0..G_A {
            s += 0.8f64.sqrt() * j.x[i][g * TAILLE_G] * j.poids_fact[g];
        }
        y[i] = s * 3.0 + j.bruit[i];
    }
    y
}

// Decoupage 120 train / 40 test, standardisation ajustee sur le train uniquement (le geste du 2.1).
#[allow(dead_code)]
struct Prep {
    x_train_s: Vec<Vec<f64>>,
    x_test_s: Vec<Vec<f64>>,
    y_train_z: Vec<f64>,
    y_test_z: Vec<f64>,
    y_ecart: f64,
}

#[allow(dead_code)]
fn preparer(j: &Jeu, y: &[f64]) -> Prep {
    let x_train: Vec<Vec<f64>> = (0..N_TRAIN).map(|k| j.x[j.idx[k]].clone()).collect();
    let x_test: Vec<Vec<f64>> = (N_TRAIN..N).map(|k| j.x[j.idx[k]].clone()).collect();
    let y_train: Vec<f64> = (0..N_TRAIN).map(|k| y[j.idx[k]]).collect();
    let y_test: Vec<f64> = (N_TRAIN..N).map(|k| y[j.idx[k]]).collect();
    let (mu, sigma) = (colonnes_moyennes(&x_train), colonnes_ecarts(&x_train));
    let (y_m, y_s) = (moyenne(&y_train), ecart_type(&y_train));
    Prep {
        x_train_s: standardiser(&x_train, &mu, &sigma),
        x_test_s: standardiser(&x_test, &mu, &sigma),
        y_train_z: standardiser_vec(&y_train, y_m, y_s),
        y_test_z: standardiser_vec(&y_test, y_m, y_s),
        y_ecart: y_s,
    }
}

// ---- Algebre lineaire : tout passe par la matrice de Gram des lignes K = X X^T (n x n, ici 120 x 120),
// jamais par X^T X (p x p, singuliere des que p > n). ----
#[allow(dead_code)]
fn transpose_mat_vec(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| x.iter().zip(b).map(|(l, &v)| l[j] * v).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn gram(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let n = x.len();
    (0..n)
        .map(|i| {
            (0..n)
                .map(|k| x[i].iter().zip(&x[k]).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// Valeurs propres (decroissantes) et vecteurs propres d'une matrice symetrique par la methode de
// Jacobi cyclique. vecteurs[k][i] est la i-eme composante du k-eme vecteur propre ; le signe de chaque
// vecteur est choisi pour que sa plus grande composante en valeur absolue soit positive.
#[allow(dead_code)]
fn propres_sym(m: &[Vec<f64>]) -> (Vec<f64>, Vec<Vec<f64>>) {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut v: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|k| if i == k { 1.0 } else { 0.0 }).collect())
        .collect();
    for _balayage in 0..100 {
        let diag: f64 = (0..n).map(|i| a[i][i].abs()).fold(0.0f64, f64::max);
        let mut carres = 0.0f64;
        for i in 0..n {
            for k in (i + 1)..n {
                carres += a[i][k] * a[i][k];
            }
        }
        if carres.sqrt() <= 1e-13 * (1.0 + diag) {
            break;
        }
        let mut rotations = 0usize;
        for p in 0..n {
            for q in (p + 1)..n {
                let apq = a[p][q];
                // Pivot negligeable devant la diagonale : on l'annule sans tourner.
                if apq.abs() <= 1e-18 * (a[p][p].abs() + a[q][q].abs()) {
                    a[p][q] = 0.0;
                    a[q][p] = 0.0;
                    continue;
                }
                rotations += 1;
                let theta = (a[q][q] - a[p][p]) / (2.0 * apq);
                let t = if theta.abs() > 1e150 {
                    1.0 / (2.0 * theta)
                } else if theta >= 0.0 {
                    1.0 / (theta + (theta * theta + 1.0).sqrt())
                } else {
                    -1.0 / (-theta + (theta * theta + 1.0).sqrt())
                };
                let c = 1.0 / (t * t + 1.0).sqrt();
                let s = t * c;
                for k in 0..n {
                    if k != p && k != q {
                        let akp = a[k][p];
                        let akq = a[k][q];
                        a[k][p] = c * akp - s * akq;
                        a[p][k] = a[k][p];
                        a[k][q] = s * akp + c * akq;
                        a[q][k] = a[k][q];
                    }
                    let vkp = v[k][p];
                    let vkq = v[k][q];
                    v[k][p] = c * vkp - s * vkq;
                    v[k][q] = s * vkp + c * vkq;
                }
                a[p][p] -= t * apq;
                a[q][q] += t * apq;
                a[p][q] = 0.0;
                a[q][p] = 0.0;
            }
        }
        if rotations == 0 {
            break;
        }
    }
    let mut ordre: Vec<usize> = (0..n).collect();
    ordre.sort_by(|&i, &k| a[k][k].total_cmp(&a[i][i]));
    let valeurs: Vec<f64> = ordre.iter().map(|&i| a[i][i]).collect();
    let vecteurs: Vec<Vec<f64>> = ordre
        .iter()
        .map(|&k| {
            let colonne: Vec<f64> = v.iter().map(|l| l[k]).collect();
            let pivot = argmax(&colonne.iter().map(|x| x.abs()).collect::<Vec<f64>>());
            if colonne[pivot] < 0.0 {
                colonne.iter().map(|x| -x).collect()
            } else {
                colonne
            }
        })
        .collect();
    (valeurs, vecteurs)
}

// Rang numerique : nombre de valeurs propres au-dessus de 1e-8 fois la plus grande.
#[allow(dead_code)]
fn rang(valeurs: &[f64]) -> usize {
    let lmax = valeurs[0];
    valeurs.iter().filter(|v| **v > 1e-8 * lmax).count()
}

// OLS de norme minimale sans ordonnee, par K = X X^T : alpha = U_r L_r^{-1} U_r^T y, puis beta = X^T alpha.
// Les r valeurs propres non nulles fixent la prediction ; le noyau (p - r directions) ne la modifie pas.
#[allow(dead_code)]
fn ols_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64]) -> Vec<f64> {
    let lmax = valeurs[0];
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        if valeurs[k] > 1e-8 * lmax {
            let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
            for i in 0..n {
                alpha[i] += u[k][i] * d / valeurs[k];
            }
        }
    }
    alpha
}

// Coefficients duaux du ridge sans ordonnee : alpha = (K + lambda I)^{-1} y, dans la base des vecteurs propres.
#[allow(dead_code)]
fn ridge_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64], lambda: f64) -> Vec<f64> {
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
        for i in 0..n {
            alpha[i] += u[k][i] * d / (valeurs[k] + lambda);
        }
    }
    alpha
}

// beta = X^T alpha ; renvoie les previsions X beta sur train et test, et la norme L2 de beta.
#[allow(dead_code)]
fn predire(xtr: &[Vec<f64>], xte: &[Vec<f64>], alpha: &[f64]) -> (Vec<f64>, Vec<f64>, f64) {
    let beta = transpose_mat_vec(xtr, alpha);
    let norme = beta.iter().map(|b| b * b).sum::<f64>().sqrt();
    (mat_vec_ligne(xtr, &beta), mat_vec_ligne(xte, &beta), norme)
}

// Bornes du pli f d'une validation croisee a plis contigus (KFold sans melange).
#[allow(dead_code)]
fn bornes_pli(n: usize, plis: usize, f: usize) -> (usize, usize) {
    let (base, reste) = (n / plis, n % plis);
    let debut = f * base + reste.min(f);
    (debut, debut + base + if f < reste { 1 } else { 0 })
}

// Un pli contigu, avec la decomposition spectrale du Gram de son train : elle sert a tous les alpha du ridge.
#[allow(dead_code)]
struct PliContigu {
    xtr: Vec<Vec<f64>>,
    xva: Vec<Vec<f64>>,
    positions_val: Vec<usize>,
    valeurs: Vec<f64>,
    u: Vec<Vec<f64>>,
}

#[allow(dead_code)]
fn plis_contigus(x: &[Vec<f64>], plis: usize) -> Vec<PliContigu> {
    let n = x.len();
    (0..plis)
        .map(|f| {
            let (debut, fin) = bornes_pli(n, plis, f);
            let xtr: Vec<Vec<f64>> = (0..n)
                .filter(|&i| i < debut || i >= fin)
                .map(|i| x[i].clone())
                .collect();
            let xva: Vec<Vec<f64>> = x[debut..fin].to_vec();
            let (valeurs, u) = propres_sym(&gram(&xtr));
            PliContigu {
                xtr,
                xva,
                positions_val: (debut..fin).collect(),
                valeurs,
                u,
            }
        })
        .collect()
}

// RidgeCV(alphas, cv=plis, fit_intercept=False) : l'alpha retenu est celui qui maximise la moyenne
// des R2 de pli (le score par defaut de la validation croisee de scikit-learn). Renvoie son indice.
#[allow(dead_code)]
fn ridge_cv_sur_plis(plis_c: &[PliContigu], y: &[f64], alphas: &[f64]) -> usize {
    let mut score = vec![0.0; alphas.len()];
    for pli in plis_c {
        let ytr: Vec<f64> = (0..y.len())
            .filter(|i| !pli.positions_val.contains(i))
            .map(|i| y[i])
            .collect();
        let yva: Vec<f64> = pli.positions_val.iter().map(|&i| y[i]).collect();
        let sst: f64 = centrer(&yva).iter().map(|x| x * x).sum();
        for (j, &lam) in alphas.iter().enumerate() {
            let (_, p_va, _) = predire(
                &pli.xtr,
                &pli.xva,
                &ridge_alpha(&pli.valeurs, &pli.u, &ytr, lam),
            );
            let sse: f64 = yva.iter().zip(&p_va).map(|(t, p)| (t - p) * (t - p)).sum();
            score[j] += 1.0 - sse / sst;
        }
    }
    argmax(&score)
}

#[allow(dead_code)]
fn ridge_cv_retenu(x: &[Vec<f64>], y: &[f64], alphas: &[f64], plis: usize) -> usize {
    ridge_cv_sur_plis(&plis_contigus(x, plis), y, alphas)
}

// Composantes principales depuis la decomposition de K = X X^T : composante_k = X^T u_k / sqrt(lambda_k).
#[allow(dead_code)]
fn composantes(x: &[Vec<f64>], valeurs: &[f64], u: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let mut comps: Vec<Vec<f64>> = Vec::new();
    for k in 0..rang(valeurs) {
        let mut c = transpose_mat_vec(x, &u[k]);
        let s = valeurs[k].sqrt();
        for v in c.iter_mut() {
            *v /= s;
        }
        comps.push(c);
    }
    comps
}

// Centrage d'un echantillon sur ses propres colonnes : ce que fait PCA a l'ajustement.
#[allow(dead_code)]
fn centrer_propre(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    centrer_colonnes(x, &colonnes_moyennes(x))
}

// ACP, comme sklearn.decomposition.PCA : X est CENTRE sur ses colonnes, covariance (denominateur n),
// valeurs propres, part de variance, composantes. Le centrage ne change rien sur une matrice deja
// standardisee sur elle-meme (train complet) ; il est indispensable sur le train d'un pli, dont les
// colonnes ne sont pas de moyenne nulle.
#[allow(dead_code)]
fn acp(x: &[Vec<f64>]) -> (Vec<f64>, Vec<f64>, Vec<Vec<f64>>) {
    let xc = centrer_propre(x);
    let (valeurs, u) = propres_sym(&gram(&xc));
    let total: f64 = valeurs.iter().sum();
    let comps = composantes(&xc, &valeurs, &u);
    let ratio: Vec<f64> = valeurs.iter().map(|l| l / total).collect();
    (valeurs, ratio, comps)
}

// Variance cumulee jusqu'a la composante k comprise.
#[allow(dead_code)]
fn variance_cumulee(ratio: &[f64], k: usize) -> f64 {
    ratio[..=k].iter().sum()
}

// Scores t[i][k] = x_i . composante_k.
#[allow(dead_code)]
fn scores_composantes(x: &[Vec<f64>], comps: &[Vec<f64>]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            comps
                .iter()
                .map(|c| l.iter().zip(c).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// PCR a k composantes avec ordonnee : y = ybar + sum_{j<k} b_j t_j.
// Les scores d'une ACP sont orthogonaux deux a deux, donc b_j = (t_j . (y - ybar)) / (t_j . t_j).
#[allow(dead_code)]
fn regression_scores(
    t_tr: &[Vec<f64>],
    t_te: &[Vec<f64>],
    y: &[f64],
    k: usize,
) -> (Vec<f64>, Vec<f64>) {
    let ym = moyenne(y);
    let yc: Vec<f64> = y.iter().map(|v| v - ym).collect();
    let mut b = vec![0.0; k];
    for j in 0..k {
        let num: f64 = t_tr.iter().zip(&yc).map(|(t, &v)| t[j] * v).sum();
        let den: f64 = t_tr.iter().map(|t| t[j] * t[j]).sum();
        b[j] = num / den;
    }
    let projet = |tt: &[Vec<f64>]| -> Vec<f64> {
        tt.iter()
            .map(|t| ym + (0..k).map(|j| b[j] * t[j]).sum::<f64>())
            .collect()
    };
    (projet(t_tr), projet(t_te))
}

// RMSE test de la PCR pour k = 1..kmax (une seule ACP, une regression par k).
#[allow(dead_code)]
fn balayage_pcr(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (_, _, mut comps) = acp(xtr);
    comps.truncate(kmax);
    // PCA().fit(xtr).transform(...) : les deux echantillons sont centres sur la moyenne du train.
    let xmu = colonnes_moyennes(xtr);
    let (t_tr, t_te) = (
        scores_composantes(&centrer_colonnes(xtr, &xmu), &comps),
        scores_composantes(&centrer_colonnes(xte, &xmu), &comps),
    );
    (1..=kmax)
        .map(|k| rmse(yte, &regression_scores(&t_tr, &t_te, ytr, k).1))
        .collect()
}

// Exercice 2 : reperer la composante la plus chargee en bloc B.
#[allow(unused_variables)]
fn rang_signal(composantes: &[Vec<f64>], debut_b: usize, variance: &[f64]) -> Option<(usize, f64)> {
    // TODO : retourner (index du maximum de la somme des poids carres du bloc B, variance cumulee).
    None
}

let j = jeu(GRAINE);
let p = preparer(&j, &cible(&j));
let (_, ratio, comps) = acp(&p.x_train_s);
// Une fixture : (poids des composantes, indice de debut du bloc B, part de variance par composante).
let fixtures: Vec<(Vec<Vec<f64>>, usize, Vec<f64>)> = vec![
    (comps, P_A, ratio),
    (
        vec![vec![0.9, 0.1, 0.2], vec![0.1, 0.8, 0.7], vec![0.2, 0.3, 0.1]],
        1,
        vec![0.5, 0.3, 0.2],
    ),
];
let reponses: Vec<Option<(usize, f64)>> = fixtures
    .iter()
    .map(|(w, d, v)| rang_signal(w, *d, v))
    .collect();
if reponses.iter().all(|r| r.is_none()) {
    println!("Exercice 2 a completer.");
} else if reponses.iter().any(|r| r.is_none()) {
    println!("Exercice 2 : a revoir");
} else {
    let mut ok = true;
    for ((poids, debut, variance), reponse) in fixtures.iter().zip(&reponses) {
        let (index, cumulee) = match reponse {
            Some(r) => *r,
            None => continue,
        };
        let parts: Vec<f64> = poids
            .iter()
            .map(|c| (*debut..c.len()).map(|v| c[v] * c[v]).sum())
            .collect();
        let attendu = argmax(&parts);
        ok = ok
            && index == attendu
            && cumulee.is_finite()
            && (cumulee - variance_cumulee(variance, attendu)).abs() < 1e-10;
    }
    println!(
        "{}",
        if ok { "Exercice 2 : correct" } else { "Exercice 2 : a revoir" }
    );
};

Exercice 2 a completer.


## 5. PLS — des composantes qui regardent la cible

La PLS (*partial least squares*, régression sur composantes supervisées) construit ses composantes $u$ en maximisant $\mathrm{Cov}(Y, Xu)^2$ : chaque composante est choisie pour **à la fois** expliquer $X$ et prédire $Y$. C'est le tri par $\mathrm{Cov}$ contre le tri par $\mathrm{Var}$ de l'ACP.

> **Côté Rust.** Les composantes sont extraites par **NIPALS** écrit dans le préambule (poids $w = X_c^\top y_c$ normalisé, score $t = X_c w$, chargement $p = X_c^\top t / t^\top t$, déflation), puis $\beta$ cumulé par régression de $y_c$ sur les scores retenus — la même construction que `PLSRegression(scale=False)`, sans son code.

In [8]:
// Cellule autonome : prelude (RNG, donnees, algebre) redefini ici, puis le calcul de la section.
// ---- Generateur pseudo-alatoire maison : xorshift64* + Box-Muller (aucun crate).
// La suite n'est pas celle de NumPy : les tirages different, les valeurs chiffrees sont celles de ce carnet.
#[allow(dead_code)]
struct Rng {
    etat: u64,
}

#[allow(dead_code)]
impl Rng {
    fn new(graine: u64) -> Rng {
        let mut etat = graine ^ 0x9E3779B97F4A7C15;
        if etat == 0 {
            etat = 0x2545F4914F6CDD1D;
        }
        Rng { etat }
    }

    fn brut(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x >> 12;
        x ^= x << 25;
        x ^= x >> 27;
        self.etat = x;
        x.wrapping_mul(0x2545F4914F6CDD1D)
    }

    // Uniforme sur [0, 1), 53 bits de mantisse.
    fn uniforme(&mut self) -> f64 {
        ((self.brut() >> 11) as f64 + 0.5) / 9007199254740992.0
    }

    // Normale centree reduite, methode de Box-Muller.
    fn normale(&mut self) -> f64 {
        const TAU: f64 = 6.283185307179586;
        loop {
            let u1 = self.uniforme();
            let u2 = self.uniforme();
            if u1 > 1e-300 {
                return (-2.0 * u1.ln()).sqrt() * (TAU * u2).cos();
            }
        }
    }

    // Entier uniforme dans 0..n, par rejet (pas le biais du modulo).
    fn indice(&mut self, n: usize) -> usize {
        let n = n as u64;
        let limite = u64::MAX - (u64::MAX % n);
        loop {
            let t = self.brut();
            if t < limite {
                return (t % n) as usize;
            }
        }
    }
}

// Permutation aleatoire (Fisher-Yates).
#[allow(dead_code)]
fn permutation(rng: &mut Rng, n: usize) -> Vec<usize> {
    let mut idx: Vec<usize> = (0..n).collect();
    for i in (1..n).rev() {
        let j = rng.indice(i + 1);
        idx.swap(i, j);
    }
    idx
}

// ---- Statistiques de base (ecarts-types au denominateur n, comme NumPy). ----
#[allow(dead_code)]
fn moyenne(v: &[f64]) -> f64 {
    v.iter().sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn variance(v: &[f64]) -> f64 {
    let m = moyenne(v);
    v.iter().map(|x| (x - m) * (x - m)).sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn ecart_type(v: &[f64]) -> f64 {
    variance(v).sqrt()
}

// Matrices stockees ligne par ligne : x[i][j].
#[allow(dead_code)]
fn mat_vec_ligne(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    x.iter()
        .map(|l| l.iter().zip(b).map(|(a, c)| a * c).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn colonnes_moyennes(x: &[Vec<f64>]) -> Vec<f64> {
    let n = x.len();
    (0..x[0].len())
        .map(|j| x.iter().map(|l| l[j]).sum::<f64>() / n as f64)
        .collect()
}

#[allow(dead_code)]
fn colonnes_ecarts(x: &[Vec<f64>]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| ecart_type(&x.iter().map(|l| l[j]).collect::<Vec<f64>>()))
        .collect()
}

#[allow(dead_code)]
fn standardiser(x: &[Vec<f64>], mu: &[f64], sigma: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            l.iter()
                .zip(mu.iter().zip(sigma))
                .map(|(a, (m, s))| (a - m) / s)
                .collect()
        })
        .collect()
}

#[allow(dead_code)]
fn standardiser_vec(v: &[f64], m: f64, s: f64) -> Vec<f64> {
    v.iter().map(|x| (x - m) / s).collect()
}

#[allow(dead_code)]
fn centrer(v: &[f64]) -> Vec<f64> {
    let m = moyenne(v);
    v.iter().map(|x| x - m).collect()
}

#[allow(dead_code)]
fn centrer_colonnes(x: &[Vec<f64>], mu: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| l.iter().zip(mu).map(|(a, m)| a - m).collect())
        .collect()
}

#[allow(dead_code)]
fn correlation(a: &[f64], b: &[f64]) -> f64 {
    let (ca, cb) = (centrer(a), centrer(b));
    let num: f64 = ca.iter().zip(&cb).map(|(x, y)| x * y).sum();
    let den: f64 = ca.iter().map(|x| x * x).sum::<f64>().sqrt()
        * cb.iter().map(|y| y * y).sum::<f64>().sqrt();
    num / den
}

#[allow(dead_code)]
fn rmse(a: &[f64], b: &[f64]) -> f64 {
    (a.iter().zip(b).map(|(x, y)| (x - y) * (x - y)).sum::<f64>() / a.len() as f64).sqrt()
}

// Indice du minimum (le premier en cas d'egalite), comme np.argmin.
#[allow(dead_code)]
fn argmin(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] < v[m] {
            m = k;
        }
    }
    m
}

// Indice du maximum (le premier en cas d'egalite), comme np.argmax.
#[allow(dead_code)]
fn argmax(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] > v[m] {
            m = k;
        }
    }
    m
}

#[allow(dead_code)]
fn logspace(a: f64, b: f64, n: usize) -> Vec<f64> {
    (0..n)
        .map(|i| 10f64.powf(a + (b - a) * i as f64 / (n as f64 - 1.0)))
        .collect()
}

// Colonne j d'une matrice stockee ligne par ligne.
#[allow(dead_code)]
fn colonne(x: &[Vec<f64>], j: usize) -> Vec<f64> {
    x.iter().map(|l| l[j]).collect()
}

// Somme des valeurs absolues sur deux blocs de variables : (0..debut_b, debut_b..fin).
#[allow(dead_code)]
fn masses_blocs(v: &[f64], debut_b: usize) -> (f64, f64) {
    let a: f64 = v[..debut_b].iter().map(|x| x.abs()).sum();
    let b: f64 = v[debut_b..].iter().map(|x| x.abs()).sum();
    (a, b)
}

// ---- Donnees : meme construction que la source (blocs A et B, cible, decoupage, standardisation). ----
#[allow(dead_code)]
const N: usize = 160;
#[allow(dead_code)]
const G_A: usize = 40;
#[allow(dead_code)]
const TAILLE_G: usize = 5;
#[allow(dead_code)]
const P_A: usize = 200;
#[allow(dead_code)]
const P_B: usize = 5;
#[allow(dead_code)]
const P: usize = 205;
#[allow(dead_code)]
const N_TRAIN: usize = 120;
#[allow(dead_code)]
const GRAINE: u64 = 42;

// Variables en groupes ; dans chaque groupe, corr(x_i, x_j) = rho_pair via facteur commun.
#[allow(dead_code)]
fn bloc_facteurs(rng: &mut Rng, n_vars: usize, n_groupes: usize, rho_pair: f64) -> Vec<Vec<f64>> {
    let mut x = vec![vec![0.0; n_vars]; N];
    for g in 0..n_groupes {
        let f: Vec<f64> = (0..N).map(|_| rho_pair.sqrt() * rng.normale()).collect();
        for j in 0..TAILLE_G {
            for i in 0..N {
                x[i][g * TAILLE_G + j] = f[i] + (1.0 - rho_pair).sqrt() * rng.normale();
            }
        }
    }
    x
}

// Un tirage complet depuis la graine : les deux blocs, le bruit, le decoupage et les poids du regime aligne.
#[allow(dead_code)]
struct Jeu {
    x: Vec<Vec<f64>>,
    bruit: Vec<f64>,
    idx: Vec<usize>,
    poids_fact: Vec<f64>,
}

// 40 directions de variance dominante sans signal (bloc A), 1 direction plus modeste generatrice de Y (bloc B).
#[allow(dead_code)]
fn jeu(graine: u64) -> Jeu {
    let mut rng = Rng::new(graine);
    let mut x = bloc_facteurs(&mut rng, P_A, G_A, 0.8);
    let xb = bloc_facteurs(&mut rng, P_B, 1, 0.6);
    for i in 0..N {
        x[i].extend(xb[i].iter().copied());
    }
    let bruit: Vec<f64> = (0..N).map(|_| rng.normale()).collect();
    let idx = permutation(&mut rng, N);
    let poids_fact: Vec<f64> = (0..G_A).map(|_| rng.normale() / (G_A as f64).sqrt()).collect();
    Jeu { x, bruit, idx, poids_fact }
}

// Betas de meme signe, alignes sur le facteur commun du bloc B ; Y = X beta + bruit.
#[allow(dead_code)]
fn cible(j: &Jeu) -> Vec<f64> {
    let formes = [1.3, 1.0, 1.2, 0.9, 1.1];
    let mut beta = vec![0.0; P];
    for k in 0..P_B {
        beta[P_A + k] = formes[k] * 1.2;
    }
    let mut y = mat_vec_ligne(&j.x, &beta);
    for i in 0..N {
        y[i] += j.bruit[i];
    }
    y
}

// Regime aligne : Y2 = facteurs_A . poids_fact * 3.0 + bruit. Les facteurs_A sont des proxis construits
// a partir de la premiere variable observee de chaque groupe du bloc A, pas les facteurs latents exacts.
#[allow(dead_code)]
fn cible_alignee(j: &Jeu) -> Vec<f64> {
    let mut y = vec![0.0; N];
    for i in 0..N {
        let mut s = 0.0;
        for g in 0..G_A {
            s += 0.8f64.sqrt() * j.x[i][g * TAILLE_G] * j.poids_fact[g];
        }
        y[i] = s * 3.0 + j.bruit[i];
    }
    y
}

// Decoupage 120 train / 40 test, standardisation ajustee sur le train uniquement (le geste du 2.1).
#[allow(dead_code)]
struct Prep {
    x_train_s: Vec<Vec<f64>>,
    x_test_s: Vec<Vec<f64>>,
    y_train_z: Vec<f64>,
    y_test_z: Vec<f64>,
    y_ecart: f64,
}

#[allow(dead_code)]
fn preparer(j: &Jeu, y: &[f64]) -> Prep {
    let x_train: Vec<Vec<f64>> = (0..N_TRAIN).map(|k| j.x[j.idx[k]].clone()).collect();
    let x_test: Vec<Vec<f64>> = (N_TRAIN..N).map(|k| j.x[j.idx[k]].clone()).collect();
    let y_train: Vec<f64> = (0..N_TRAIN).map(|k| y[j.idx[k]]).collect();
    let y_test: Vec<f64> = (N_TRAIN..N).map(|k| y[j.idx[k]]).collect();
    let (mu, sigma) = (colonnes_moyennes(&x_train), colonnes_ecarts(&x_train));
    let (y_m, y_s) = (moyenne(&y_train), ecart_type(&y_train));
    Prep {
        x_train_s: standardiser(&x_train, &mu, &sigma),
        x_test_s: standardiser(&x_test, &mu, &sigma),
        y_train_z: standardiser_vec(&y_train, y_m, y_s),
        y_test_z: standardiser_vec(&y_test, y_m, y_s),
        y_ecart: y_s,
    }
}

// ---- Algebre lineaire : tout passe par la matrice de Gram des lignes K = X X^T (n x n, ici 120 x 120),
// jamais par X^T X (p x p, singuliere des que p > n). ----
#[allow(dead_code)]
fn transpose_mat_vec(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| x.iter().zip(b).map(|(l, &v)| l[j] * v).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn gram(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let n = x.len();
    (0..n)
        .map(|i| {
            (0..n)
                .map(|k| x[i].iter().zip(&x[k]).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// Valeurs propres (decroissantes) et vecteurs propres d'une matrice symetrique par la methode de
// Jacobi cyclique. vecteurs[k][i] est la i-eme composante du k-eme vecteur propre ; le signe de chaque
// vecteur est choisi pour que sa plus grande composante en valeur absolue soit positive.
#[allow(dead_code)]
fn propres_sym(m: &[Vec<f64>]) -> (Vec<f64>, Vec<Vec<f64>>) {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut v: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|k| if i == k { 1.0 } else { 0.0 }).collect())
        .collect();
    for _balayage in 0..100 {
        let diag: f64 = (0..n).map(|i| a[i][i].abs()).fold(0.0f64, f64::max);
        let mut carres = 0.0f64;
        for i in 0..n {
            for k in (i + 1)..n {
                carres += a[i][k] * a[i][k];
            }
        }
        if carres.sqrt() <= 1e-13 * (1.0 + diag) {
            break;
        }
        let mut rotations = 0usize;
        for p in 0..n {
            for q in (p + 1)..n {
                let apq = a[p][q];
                // Pivot negligeable devant la diagonale : on l'annule sans tourner.
                if apq.abs() <= 1e-18 * (a[p][p].abs() + a[q][q].abs()) {
                    a[p][q] = 0.0;
                    a[q][p] = 0.0;
                    continue;
                }
                rotations += 1;
                let theta = (a[q][q] - a[p][p]) / (2.0 * apq);
                let t = if theta.abs() > 1e150 {
                    1.0 / (2.0 * theta)
                } else if theta >= 0.0 {
                    1.0 / (theta + (theta * theta + 1.0).sqrt())
                } else {
                    -1.0 / (-theta + (theta * theta + 1.0).sqrt())
                };
                let c = 1.0 / (t * t + 1.0).sqrt();
                let s = t * c;
                for k in 0..n {
                    if k != p && k != q {
                        let akp = a[k][p];
                        let akq = a[k][q];
                        a[k][p] = c * akp - s * akq;
                        a[p][k] = a[k][p];
                        a[k][q] = s * akp + c * akq;
                        a[q][k] = a[k][q];
                    }
                    let vkp = v[k][p];
                    let vkq = v[k][q];
                    v[k][p] = c * vkp - s * vkq;
                    v[k][q] = s * vkp + c * vkq;
                }
                a[p][p] -= t * apq;
                a[q][q] += t * apq;
                a[p][q] = 0.0;
                a[q][p] = 0.0;
            }
        }
        if rotations == 0 {
            break;
        }
    }
    let mut ordre: Vec<usize> = (0..n).collect();
    ordre.sort_by(|&i, &k| a[k][k].total_cmp(&a[i][i]));
    let valeurs: Vec<f64> = ordre.iter().map(|&i| a[i][i]).collect();
    let vecteurs: Vec<Vec<f64>> = ordre
        .iter()
        .map(|&k| {
            let colonne: Vec<f64> = v.iter().map(|l| l[k]).collect();
            let pivot = argmax(&colonne.iter().map(|x| x.abs()).collect::<Vec<f64>>());
            if colonne[pivot] < 0.0 {
                colonne.iter().map(|x| -x).collect()
            } else {
                colonne
            }
        })
        .collect();
    (valeurs, vecteurs)
}

// Rang numerique : nombre de valeurs propres au-dessus de 1e-8 fois la plus grande.
#[allow(dead_code)]
fn rang(valeurs: &[f64]) -> usize {
    let lmax = valeurs[0];
    valeurs.iter().filter(|v| **v > 1e-8 * lmax).count()
}

// OLS de norme minimale sans ordonnee, par K = X X^T : alpha = U_r L_r^{-1} U_r^T y, puis beta = X^T alpha.
// Les r valeurs propres non nulles fixent la prediction ; le noyau (p - r directions) ne la modifie pas.
#[allow(dead_code)]
fn ols_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64]) -> Vec<f64> {
    let lmax = valeurs[0];
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        if valeurs[k] > 1e-8 * lmax {
            let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
            for i in 0..n {
                alpha[i] += u[k][i] * d / valeurs[k];
            }
        }
    }
    alpha
}

// Coefficients duaux du ridge sans ordonnee : alpha = (K + lambda I)^{-1} y, dans la base des vecteurs propres.
#[allow(dead_code)]
fn ridge_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64], lambda: f64) -> Vec<f64> {
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
        for i in 0..n {
            alpha[i] += u[k][i] * d / (valeurs[k] + lambda);
        }
    }
    alpha
}

// beta = X^T alpha ; renvoie les previsions X beta sur train et test, et la norme L2 de beta.
#[allow(dead_code)]
fn predire(xtr: &[Vec<f64>], xte: &[Vec<f64>], alpha: &[f64]) -> (Vec<f64>, Vec<f64>, f64) {
    let beta = transpose_mat_vec(xtr, alpha);
    let norme = beta.iter().map(|b| b * b).sum::<f64>().sqrt();
    (mat_vec_ligne(xtr, &beta), mat_vec_ligne(xte, &beta), norme)
}

// Bornes du pli f d'une validation croisee a plis contigus (KFold sans melange).
#[allow(dead_code)]
fn bornes_pli(n: usize, plis: usize, f: usize) -> (usize, usize) {
    let (base, reste) = (n / plis, n % plis);
    let debut = f * base + reste.min(f);
    (debut, debut + base + if f < reste { 1 } else { 0 })
}

// Un pli contigu, avec la decomposition spectrale du Gram de son train : elle sert a tous les alpha du ridge.
#[allow(dead_code)]
struct PliContigu {
    xtr: Vec<Vec<f64>>,
    xva: Vec<Vec<f64>>,
    positions_val: Vec<usize>,
    valeurs: Vec<f64>,
    u: Vec<Vec<f64>>,
}

#[allow(dead_code)]
fn plis_contigus(x: &[Vec<f64>], plis: usize) -> Vec<PliContigu> {
    let n = x.len();
    (0..plis)
        .map(|f| {
            let (debut, fin) = bornes_pli(n, plis, f);
            let xtr: Vec<Vec<f64>> = (0..n)
                .filter(|&i| i < debut || i >= fin)
                .map(|i| x[i].clone())
                .collect();
            let xva: Vec<Vec<f64>> = x[debut..fin].to_vec();
            let (valeurs, u) = propres_sym(&gram(&xtr));
            PliContigu {
                xtr,
                xva,
                positions_val: (debut..fin).collect(),
                valeurs,
                u,
            }
        })
        .collect()
}

// RidgeCV(alphas, cv=plis, fit_intercept=False) : l'alpha retenu est celui qui maximise la moyenne
// des R2 de pli (le score par defaut de la validation croisee de scikit-learn). Renvoie son indice.
#[allow(dead_code)]
fn ridge_cv_sur_plis(plis_c: &[PliContigu], y: &[f64], alphas: &[f64]) -> usize {
    let mut score = vec![0.0; alphas.len()];
    for pli in plis_c {
        let ytr: Vec<f64> = (0..y.len())
            .filter(|i| !pli.positions_val.contains(i))
            .map(|i| y[i])
            .collect();
        let yva: Vec<f64> = pli.positions_val.iter().map(|&i| y[i]).collect();
        let sst: f64 = centrer(&yva).iter().map(|x| x * x).sum();
        for (j, &lam) in alphas.iter().enumerate() {
            let (_, p_va, _) = predire(
                &pli.xtr,
                &pli.xva,
                &ridge_alpha(&pli.valeurs, &pli.u, &ytr, lam),
            );
            let sse: f64 = yva.iter().zip(&p_va).map(|(t, p)| (t - p) * (t - p)).sum();
            score[j] += 1.0 - sse / sst;
        }
    }
    argmax(&score)
}

#[allow(dead_code)]
fn ridge_cv_retenu(x: &[Vec<f64>], y: &[f64], alphas: &[f64], plis: usize) -> usize {
    ridge_cv_sur_plis(&plis_contigus(x, plis), y, alphas)
}

// Composantes principales depuis la decomposition de K = X X^T : composante_k = X^T u_k / sqrt(lambda_k).
#[allow(dead_code)]
fn composantes(x: &[Vec<f64>], valeurs: &[f64], u: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let mut comps: Vec<Vec<f64>> = Vec::new();
    for k in 0..rang(valeurs) {
        let mut c = transpose_mat_vec(x, &u[k]);
        let s = valeurs[k].sqrt();
        for v in c.iter_mut() {
            *v /= s;
        }
        comps.push(c);
    }
    comps
}

// Centrage d'un echantillon sur ses propres colonnes : ce que fait PCA a l'ajustement.
#[allow(dead_code)]
fn centrer_propre(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    centrer_colonnes(x, &colonnes_moyennes(x))
}

// ACP, comme sklearn.decomposition.PCA : X est CENTRE sur ses colonnes, covariance (denominateur n),
// valeurs propres, part de variance, composantes. Le centrage ne change rien sur une matrice deja
// standardisee sur elle-meme (train complet) ; il est indispensable sur le train d'un pli, dont les
// colonnes ne sont pas de moyenne nulle.
#[allow(dead_code)]
fn acp(x: &[Vec<f64>]) -> (Vec<f64>, Vec<f64>, Vec<Vec<f64>>) {
    let xc = centrer_propre(x);
    let (valeurs, u) = propres_sym(&gram(&xc));
    let total: f64 = valeurs.iter().sum();
    let comps = composantes(&xc, &valeurs, &u);
    let ratio: Vec<f64> = valeurs.iter().map(|l| l / total).collect();
    (valeurs, ratio, comps)
}

// Variance cumulee jusqu'a la composante k comprise.
#[allow(dead_code)]
fn variance_cumulee(ratio: &[f64], k: usize) -> f64 {
    ratio[..=k].iter().sum()
}

// Scores t[i][k] = x_i . composante_k.
#[allow(dead_code)]
fn scores_composantes(x: &[Vec<f64>], comps: &[Vec<f64>]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            comps
                .iter()
                .map(|c| l.iter().zip(c).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// PCR a k composantes avec ordonnee : y = ybar + sum_{j<k} b_j t_j.
// Les scores d'une ACP sont orthogonaux deux a deux, donc b_j = (t_j . (y - ybar)) / (t_j . t_j).
#[allow(dead_code)]
fn regression_scores(
    t_tr: &[Vec<f64>],
    t_te: &[Vec<f64>],
    y: &[f64],
    k: usize,
) -> (Vec<f64>, Vec<f64>) {
    let ym = moyenne(y);
    let yc: Vec<f64> = y.iter().map(|v| v - ym).collect();
    let mut b = vec![0.0; k];
    for j in 0..k {
        let num: f64 = t_tr.iter().zip(&yc).map(|(t, &v)| t[j] * v).sum();
        let den: f64 = t_tr.iter().map(|t| t[j] * t[j]).sum();
        b[j] = num / den;
    }
    let projet = |tt: &[Vec<f64>]| -> Vec<f64> {
        tt.iter()
            .map(|t| ym + (0..k).map(|j| b[j] * t[j]).sum::<f64>())
            .collect()
    };
    (projet(t_tr), projet(t_te))
}

// RMSE test de la PCR pour k = 1..kmax (une seule ACP, une regression par k).
#[allow(dead_code)]
fn balayage_pcr(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (_, _, mut comps) = acp(xtr);
    comps.truncate(kmax);
    // PCA().fit(xtr).transform(...) : les deux echantillons sont centres sur la moyenne du train.
    let xmu = colonnes_moyennes(xtr);
    let (t_tr, t_te) = (
        scores_composantes(&centrer_colonnes(xtr, &xmu), &comps),
        scores_composantes(&centrer_colonnes(xte, &xmu), &comps),
    );
    (1..=kmax)
        .map(|k| rmse(yte, &regression_scores(&t_tr, &t_te, ytr, k).1))
        .collect()
}

// ---- PLS1 par NIPALS (l'algorithme de PLSRegression). x et y doivent etre centres sur le train.
// Renvoie (w, p, t) : poids x, loadings x et scores, ranges par composante : w[j][v], t[j][i]. ----
#[allow(dead_code)]
fn nipals(
    x: &[Vec<f64>],
    y_c: &[f64],
    kmax: usize,
) -> (Vec<Vec<f64>>, Vec<Vec<f64>>, Vec<Vec<f64>>) {
    let n = x.len();
    let nvar = x[0].len();
    let mut xk = x.to_vec();
    let mut yk = y_c.to_vec();
    let mut w_all: Vec<Vec<f64>> = Vec::new();
    let mut p_all: Vec<Vec<f64>> = Vec::new();
    let mut t_all: Vec<Vec<f64>> = Vec::new();
    for _j in 0..kmax {
        // w = Xk^T yk / ||Xk^T yk|| (direction de covariance maximale avec le reste de y), t = Xk w
        let mut w = transpose_mat_vec(&xk, &yk);
        let nrm = w.iter().map(|v| v * v).sum::<f64>().sqrt();
        for v in w.iter_mut() {
            *v /= nrm;
        }
        let t = mat_vec_ligne(&xk, &w);
        let ttt: f64 = t.iter().map(|v| v * v).sum();
        // p = Xk^T t / (t^T t) ; q = yk^T t / (t^T t)
        let mut pp = transpose_mat_vec(&xk, &t);
        for v in pp.iter_mut() {
            *v /= ttt;
        }
        let q: f64 = t.iter().zip(&yk).map(|(a, b)| a * b).sum::<f64>() / ttt;
        // Deflation : Xk <- Xk - t p^T ; yk <- yk - q t
        for i in 0..n {
            for j in 0..nvar {
                xk[i][j] -= t[i] * pp[j];
            }
            yk[i] -= q * t[i];
        }
        w_all.push(w);
        p_all.push(pp);
        t_all.push(t);
    }
    (w_all, p_all, t_all)
}

// Inverse d'une matrice carree (Gauss-Jordan, pivot partiel) ; None si un pivot est nul.
#[allow(dead_code)]
fn inverse(m: &[Vec<f64>]) -> Option<Vec<Vec<f64>>> {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut inv: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|j| if i == j { 1.0 } else { 0.0 }).collect())
        .collect();
    for col in 0..n {
        let mut piv = col;
        for r in (col + 1)..n {
            if a[r][col].abs() > a[piv][col].abs() {
                piv = r;
            }
        }
        if !a[piv][col].is_finite() || a[piv][col].abs() < 1e-14 {
            return None;
        }
        a.swap(col, piv);
        inv.swap(col, piv);
        let d = a[col][col];
        for j in 0..n {
            a[col][j] /= d;
            inv[col][j] /= d;
        }
        for r in 0..n {
            if r != col {
                let f = a[r][col];
                for j in 0..n {
                    a[r][j] -= f * a[col][j];
                    inv[r][j] -= f * inv[col][j];
                }
            }
        }
    }
    Some(inv)
}

// Rotations de la PLS, R = W (P^T W)^{-1} : elles appliquent le modele a des lignes nouvelles
// (t = X_c R). M = P^T W etant triangulaire superieure, les k premieres colonnes de R sont celles
// du modele a k composantes.
#[allow(dead_code)]
fn rotations(w: &[Vec<f64>], pmat: &[Vec<f64>], kmax: usize) -> Option<Vec<Vec<f64>>> {
    let nvar = w[0].len();
    let m: Vec<Vec<f64>> = (0..kmax)
        .map(|i| {
            (0..kmax)
                .map(|j| pmat[i].iter().zip(&w[j]).map(|(a, b)| a * b).sum())
                .collect()
        })
        .collect();
    let inv = inverse(&m)?;
    let r: Vec<Vec<f64>> = (0..nvar)
        .map(|v| {
            (0..kmax)
                .map(|j| (0..kmax).map(|i| w[i][v] * inv[i][j]).sum())
                .collect()
        })
        .collect();
    Some(r)
}

// Poids cotes variables du modele PLS a k = 1..kmax composantes : betas[k - 1] pour k composantes.
// b_j = (t_j . y_c) / (t_j . t_j), puis beta_k = sum_{j<k} b_j R[:, j].
#[allow(dead_code)]
fn pls_poids(
    w: &[Vec<f64>],
    pmat: &[Vec<f64>],
    t: &[Vec<f64>],
    y_c: &[f64],
) -> Option<Vec<Vec<f64>>> {
    let kmax = t.len();
    let n = t[0].len();
    let nvar = w[0].len();
    let r = rotations(w, pmat, kmax)?;
    let mut betas: Vec<Vec<f64>> = Vec::new();
    let mut beta = vec![0.0; nvar];
    for j in 0..kmax {
        let num: f64 = (0..n).map(|i| t[j][i] * y_c[i]).sum();
        let den: f64 = t[j].iter().map(|v| v * v).sum();
        let b = num / den;
        for v in 0..nvar {
            beta[v] += b * r[v][j];
        }
        betas.push(beta.clone());
    }
    Some(betas)
}

// Prevision PLS pour des lignes deja centrees : y = ybar + x_c . beta.
#[allow(dead_code)]
fn pls_predire(x_c: &[Vec<f64>], y_moy: f64, beta: &[f64]) -> Vec<f64> {
    x_c.iter()
        .map(|l| y_moy + l.iter().zip(beta).map(|(a, b)| a * b).sum::<f64>())
        .collect()
}

// RMSE test de la PLS pour k = 1..kmax (un seul NIPALS, un vecteur de poids par k) ;
// NaN si les rotations ne sont pas calculables.
#[allow(dead_code)]
fn balayage_pls(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (xmu, y_moy) = (colonnes_moyennes(xtr), moyenne(ytr));
    let yc = centrer(ytr);
    let (w, pmat, t) = nipals(&centrer_colonnes(xtr, &xmu), &yc, kmax);
    let betas = match pls_poids(&w, &pmat, &t, &yc) {
        Some(b) => b,
        None => return vec![f64::NAN; kmax],
    };
    let xte_c = centrer_colonnes(xte, &xmu);
    (1..=kmax)
        .map(|k| rmse(yte, &pls_predire(&xte_c, y_moy, &betas[k - 1])))
        .collect()
}

// RMSE test de la PLS ajustee avec exactement k composantes.
#[allow(dead_code)]
fn rmse_pls_k(xtr: &[Vec<f64>], ytr: &[f64], xte: &[Vec<f64>], yte: &[f64], k: usize) -> f64 {
    match balayage_pls(xtr, ytr, xte, yte, k).last() {
        Some(r) => *r,
        None => f64::NAN,
    }
}

// ---- 5. PLS : des composantes qui regardent la cible ----
let j = jeu(GRAINE);
let p = preparer(&j, &cible(&j));
// Un seul NIPALS a 45 composantes : les k premiers poids du modele a k composantes sont ceux du modele a 45.
let rmse_pls = balayage_pls(&p.x_train_s, &p.y_train_z, &p.x_test_s, &p.y_test_z, 45);
let best_pls_k = argmin(&rmse_pls) + 1;
println!(
    "PLS : meilleur k = {}, RMSE test = {:.4}",
    best_pls_k,
    rmse_pls[best_pls_k - 1]
);
for k in [1usize, 2, 3, 5, 10, 20] {
    println!("RMSE PLS a k={:2} : {:.4}", k, rmse_pls[k - 1]);
};

// La premiere direction ne depend pas du nombre de composantes ajustees : un seul pas de NIPALS suffit.
let xmu = colonnes_moyennes(&p.x_train_s);
let (w1, _, _) = nipals(
    &centrer_colonnes(&p.x_train_s, &xmu),
    &centrer(&p.y_train_z),
    1,
);
let (_, _, comps) = acp(&p.x_train_s);
let part_b_pls: f64 = (P_A..P).map(|v| w1[0][v] * w1[0][v]).sum();
let part_b_pca: f64 = (P_A..P).map(|v| comps[0][v] * comps[0][v]).sum();
println!(
    "\npart du bloc B dans la 1re direction : PLS = {:.3}   PCA = {:.4}",
    part_b_pls, part_b_pca
);

PLS : meilleur k = 7, RMSE test = 0.1800


RMSE PLS a k= 1 : 0.5412


RMSE PLS a k= 2 : 0.3731


RMSE PLS a k= 3 : 0.2747


RMSE PLS a k= 5 : 0.1909


RMSE PLS a k=10 : 0.1909


RMSE PLS a k=20 : 0.2145


part du bloc B dans la 1re direction : PLS = 0.682   PCA = 0.0244


### Lecture du résultat

La PLS atteint dès 5 composantes ce que la PCR n'atteint qu'à 45 : RMSE 0.1909 à k=5, déjà 0.3731 à k=2 et 0.2747 à k=3 ; son meilleur point est k=7 (0.1800). Au-delà, elle se dégrade lentement (0.2145 à k=20) : les composantes suivantes modélisent du bruit sur ce train. La raison se lit dans la première direction : la part du bloc B dans w₁ vaut **68.2%** pour la PLS contre **2.44%** pour l'ACP — la première composante PLS est majoritairement faite de signal, parce qu'elle a été choisie pour maximiser Cov(Y, Xu) et non Var(Xu). C'est tout le concept central du notebook, mesuré.

## 6. Le face-à-face — RMSEP vs nombre de composantes

Mêmes splits, mêmes graines : les courbes RMSEP de PCR et PLS en fonction de $k$ racontent la même histoire de deux façons — combien de composantes faut-il dépenser avant d'atteindre le signal ?

> **Côté Rust.** La figure est un tracé ASCII : `.` pour la PCR, `x` pour la PLS, une ligne `-` pour la RMSE du meilleur ridge sur cette grille test, `*` là où un point de courbe tombe sur cette ligne, et une colonne `:` à l'abscisse de la composante la plus chargée en bloc B. Le cadre est déduit des données (aucun point n'est perdu hors cadre) et la légende affichée sous la figure donne les deux échelles.

In [9]:
// Cellule autonome : prelude (RNG, donnees, algebre) redefini ici, puis le calcul de la section.
// ---- Generateur pseudo-alatoire maison : xorshift64* + Box-Muller (aucun crate).
// La suite n'est pas celle de NumPy : les tirages different, les valeurs chiffrees sont celles de ce carnet.
#[allow(dead_code)]
struct Rng {
    etat: u64,
}

#[allow(dead_code)]
impl Rng {
    fn new(graine: u64) -> Rng {
        let mut etat = graine ^ 0x9E3779B97F4A7C15;
        if etat == 0 {
            etat = 0x2545F4914F6CDD1D;
        }
        Rng { etat }
    }

    fn brut(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x >> 12;
        x ^= x << 25;
        x ^= x >> 27;
        self.etat = x;
        x.wrapping_mul(0x2545F4914F6CDD1D)
    }

    // Uniforme sur [0, 1), 53 bits de mantisse.
    fn uniforme(&mut self) -> f64 {
        ((self.brut() >> 11) as f64 + 0.5) / 9007199254740992.0
    }

    // Normale centree reduite, methode de Box-Muller.
    fn normale(&mut self) -> f64 {
        const TAU: f64 = 6.283185307179586;
        loop {
            let u1 = self.uniforme();
            let u2 = self.uniforme();
            if u1 > 1e-300 {
                return (-2.0 * u1.ln()).sqrt() * (TAU * u2).cos();
            }
        }
    }

    // Entier uniforme dans 0..n, par rejet (pas le biais du modulo).
    fn indice(&mut self, n: usize) -> usize {
        let n = n as u64;
        let limite = u64::MAX - (u64::MAX % n);
        loop {
            let t = self.brut();
            if t < limite {
                return (t % n) as usize;
            }
        }
    }
}

// Permutation aleatoire (Fisher-Yates).
#[allow(dead_code)]
fn permutation(rng: &mut Rng, n: usize) -> Vec<usize> {
    let mut idx: Vec<usize> = (0..n).collect();
    for i in (1..n).rev() {
        let j = rng.indice(i + 1);
        idx.swap(i, j);
    }
    idx
}

// ---- Statistiques de base (ecarts-types au denominateur n, comme NumPy). ----
#[allow(dead_code)]
fn moyenne(v: &[f64]) -> f64 {
    v.iter().sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn variance(v: &[f64]) -> f64 {
    let m = moyenne(v);
    v.iter().map(|x| (x - m) * (x - m)).sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn ecart_type(v: &[f64]) -> f64 {
    variance(v).sqrt()
}

// Matrices stockees ligne par ligne : x[i][j].
#[allow(dead_code)]
fn mat_vec_ligne(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    x.iter()
        .map(|l| l.iter().zip(b).map(|(a, c)| a * c).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn colonnes_moyennes(x: &[Vec<f64>]) -> Vec<f64> {
    let n = x.len();
    (0..x[0].len())
        .map(|j| x.iter().map(|l| l[j]).sum::<f64>() / n as f64)
        .collect()
}

#[allow(dead_code)]
fn colonnes_ecarts(x: &[Vec<f64>]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| ecart_type(&x.iter().map(|l| l[j]).collect::<Vec<f64>>()))
        .collect()
}

#[allow(dead_code)]
fn standardiser(x: &[Vec<f64>], mu: &[f64], sigma: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            l.iter()
                .zip(mu.iter().zip(sigma))
                .map(|(a, (m, s))| (a - m) / s)
                .collect()
        })
        .collect()
}

#[allow(dead_code)]
fn standardiser_vec(v: &[f64], m: f64, s: f64) -> Vec<f64> {
    v.iter().map(|x| (x - m) / s).collect()
}

#[allow(dead_code)]
fn centrer(v: &[f64]) -> Vec<f64> {
    let m = moyenne(v);
    v.iter().map(|x| x - m).collect()
}

#[allow(dead_code)]
fn centrer_colonnes(x: &[Vec<f64>], mu: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| l.iter().zip(mu).map(|(a, m)| a - m).collect())
        .collect()
}

#[allow(dead_code)]
fn correlation(a: &[f64], b: &[f64]) -> f64 {
    let (ca, cb) = (centrer(a), centrer(b));
    let num: f64 = ca.iter().zip(&cb).map(|(x, y)| x * y).sum();
    let den: f64 = ca.iter().map(|x| x * x).sum::<f64>().sqrt()
        * cb.iter().map(|y| y * y).sum::<f64>().sqrt();
    num / den
}

#[allow(dead_code)]
fn rmse(a: &[f64], b: &[f64]) -> f64 {
    (a.iter().zip(b).map(|(x, y)| (x - y) * (x - y)).sum::<f64>() / a.len() as f64).sqrt()
}

// Indice du minimum (le premier en cas d'egalite), comme np.argmin.
#[allow(dead_code)]
fn argmin(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] < v[m] {
            m = k;
        }
    }
    m
}

// Indice du maximum (le premier en cas d'egalite), comme np.argmax.
#[allow(dead_code)]
fn argmax(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] > v[m] {
            m = k;
        }
    }
    m
}

#[allow(dead_code)]
fn logspace(a: f64, b: f64, n: usize) -> Vec<f64> {
    (0..n)
        .map(|i| 10f64.powf(a + (b - a) * i as f64 / (n as f64 - 1.0)))
        .collect()
}

// Colonne j d'une matrice stockee ligne par ligne.
#[allow(dead_code)]
fn colonne(x: &[Vec<f64>], j: usize) -> Vec<f64> {
    x.iter().map(|l| l[j]).collect()
}

// Somme des valeurs absolues sur deux blocs de variables : (0..debut_b, debut_b..fin).
#[allow(dead_code)]
fn masses_blocs(v: &[f64], debut_b: usize) -> (f64, f64) {
    let a: f64 = v[..debut_b].iter().map(|x| x.abs()).sum();
    let b: f64 = v[debut_b..].iter().map(|x| x.abs()).sum();
    (a, b)
}

// ---- Donnees : meme construction que la source (blocs A et B, cible, decoupage, standardisation). ----
#[allow(dead_code)]
const N: usize = 160;
#[allow(dead_code)]
const G_A: usize = 40;
#[allow(dead_code)]
const TAILLE_G: usize = 5;
#[allow(dead_code)]
const P_A: usize = 200;
#[allow(dead_code)]
const P_B: usize = 5;
#[allow(dead_code)]
const P: usize = 205;
#[allow(dead_code)]
const N_TRAIN: usize = 120;
#[allow(dead_code)]
const GRAINE: u64 = 42;

// Variables en groupes ; dans chaque groupe, corr(x_i, x_j) = rho_pair via facteur commun.
#[allow(dead_code)]
fn bloc_facteurs(rng: &mut Rng, n_vars: usize, n_groupes: usize, rho_pair: f64) -> Vec<Vec<f64>> {
    let mut x = vec![vec![0.0; n_vars]; N];
    for g in 0..n_groupes {
        let f: Vec<f64> = (0..N).map(|_| rho_pair.sqrt() * rng.normale()).collect();
        for j in 0..TAILLE_G {
            for i in 0..N {
                x[i][g * TAILLE_G + j] = f[i] + (1.0 - rho_pair).sqrt() * rng.normale();
            }
        }
    }
    x
}

// Un tirage complet depuis la graine : les deux blocs, le bruit, le decoupage et les poids du regime aligne.
#[allow(dead_code)]
struct Jeu {
    x: Vec<Vec<f64>>,
    bruit: Vec<f64>,
    idx: Vec<usize>,
    poids_fact: Vec<f64>,
}

// 40 directions de variance dominante sans signal (bloc A), 1 direction plus modeste generatrice de Y (bloc B).
#[allow(dead_code)]
fn jeu(graine: u64) -> Jeu {
    let mut rng = Rng::new(graine);
    let mut x = bloc_facteurs(&mut rng, P_A, G_A, 0.8);
    let xb = bloc_facteurs(&mut rng, P_B, 1, 0.6);
    for i in 0..N {
        x[i].extend(xb[i].iter().copied());
    }
    let bruit: Vec<f64> = (0..N).map(|_| rng.normale()).collect();
    let idx = permutation(&mut rng, N);
    let poids_fact: Vec<f64> = (0..G_A).map(|_| rng.normale() / (G_A as f64).sqrt()).collect();
    Jeu { x, bruit, idx, poids_fact }
}

// Betas de meme signe, alignes sur le facteur commun du bloc B ; Y = X beta + bruit.
#[allow(dead_code)]
fn cible(j: &Jeu) -> Vec<f64> {
    let formes = [1.3, 1.0, 1.2, 0.9, 1.1];
    let mut beta = vec![0.0; P];
    for k in 0..P_B {
        beta[P_A + k] = formes[k] * 1.2;
    }
    let mut y = mat_vec_ligne(&j.x, &beta);
    for i in 0..N {
        y[i] += j.bruit[i];
    }
    y
}

// Regime aligne : Y2 = facteurs_A . poids_fact * 3.0 + bruit. Les facteurs_A sont des proxis construits
// a partir de la premiere variable observee de chaque groupe du bloc A, pas les facteurs latents exacts.
#[allow(dead_code)]
fn cible_alignee(j: &Jeu) -> Vec<f64> {
    let mut y = vec![0.0; N];
    for i in 0..N {
        let mut s = 0.0;
        for g in 0..G_A {
            s += 0.8f64.sqrt() * j.x[i][g * TAILLE_G] * j.poids_fact[g];
        }
        y[i] = s * 3.0 + j.bruit[i];
    }
    y
}

// Decoupage 120 train / 40 test, standardisation ajustee sur le train uniquement (le geste du 2.1).
#[allow(dead_code)]
struct Prep {
    x_train_s: Vec<Vec<f64>>,
    x_test_s: Vec<Vec<f64>>,
    y_train_z: Vec<f64>,
    y_test_z: Vec<f64>,
    y_ecart: f64,
}

#[allow(dead_code)]
fn preparer(j: &Jeu, y: &[f64]) -> Prep {
    let x_train: Vec<Vec<f64>> = (0..N_TRAIN).map(|k| j.x[j.idx[k]].clone()).collect();
    let x_test: Vec<Vec<f64>> = (N_TRAIN..N).map(|k| j.x[j.idx[k]].clone()).collect();
    let y_train: Vec<f64> = (0..N_TRAIN).map(|k| y[j.idx[k]]).collect();
    let y_test: Vec<f64> = (N_TRAIN..N).map(|k| y[j.idx[k]]).collect();
    let (mu, sigma) = (colonnes_moyennes(&x_train), colonnes_ecarts(&x_train));
    let (y_m, y_s) = (moyenne(&y_train), ecart_type(&y_train));
    Prep {
        x_train_s: standardiser(&x_train, &mu, &sigma),
        x_test_s: standardiser(&x_test, &mu, &sigma),
        y_train_z: standardiser_vec(&y_train, y_m, y_s),
        y_test_z: standardiser_vec(&y_test, y_m, y_s),
        y_ecart: y_s,
    }
}

// ---- Algebre lineaire : tout passe par la matrice de Gram des lignes K = X X^T (n x n, ici 120 x 120),
// jamais par X^T X (p x p, singuliere des que p > n). ----
#[allow(dead_code)]
fn transpose_mat_vec(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| x.iter().zip(b).map(|(l, &v)| l[j] * v).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn gram(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let n = x.len();
    (0..n)
        .map(|i| {
            (0..n)
                .map(|k| x[i].iter().zip(&x[k]).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// Valeurs propres (decroissantes) et vecteurs propres d'une matrice symetrique par la methode de
// Jacobi cyclique. vecteurs[k][i] est la i-eme composante du k-eme vecteur propre ; le signe de chaque
// vecteur est choisi pour que sa plus grande composante en valeur absolue soit positive.
#[allow(dead_code)]
fn propres_sym(m: &[Vec<f64>]) -> (Vec<f64>, Vec<Vec<f64>>) {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut v: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|k| if i == k { 1.0 } else { 0.0 }).collect())
        .collect();
    for _balayage in 0..100 {
        let diag: f64 = (0..n).map(|i| a[i][i].abs()).fold(0.0f64, f64::max);
        let mut carres = 0.0f64;
        for i in 0..n {
            for k in (i + 1)..n {
                carres += a[i][k] * a[i][k];
            }
        }
        if carres.sqrt() <= 1e-13 * (1.0 + diag) {
            break;
        }
        let mut rotations = 0usize;
        for p in 0..n {
            for q in (p + 1)..n {
                let apq = a[p][q];
                // Pivot negligeable devant la diagonale : on l'annule sans tourner.
                if apq.abs() <= 1e-18 * (a[p][p].abs() + a[q][q].abs()) {
                    a[p][q] = 0.0;
                    a[q][p] = 0.0;
                    continue;
                }
                rotations += 1;
                let theta = (a[q][q] - a[p][p]) / (2.0 * apq);
                let t = if theta.abs() > 1e150 {
                    1.0 / (2.0 * theta)
                } else if theta >= 0.0 {
                    1.0 / (theta + (theta * theta + 1.0).sqrt())
                } else {
                    -1.0 / (-theta + (theta * theta + 1.0).sqrt())
                };
                let c = 1.0 / (t * t + 1.0).sqrt();
                let s = t * c;
                for k in 0..n {
                    if k != p && k != q {
                        let akp = a[k][p];
                        let akq = a[k][q];
                        a[k][p] = c * akp - s * akq;
                        a[p][k] = a[k][p];
                        a[k][q] = s * akp + c * akq;
                        a[q][k] = a[k][q];
                    }
                    let vkp = v[k][p];
                    let vkq = v[k][q];
                    v[k][p] = c * vkp - s * vkq;
                    v[k][q] = s * vkp + c * vkq;
                }
                a[p][p] -= t * apq;
                a[q][q] += t * apq;
                a[p][q] = 0.0;
                a[q][p] = 0.0;
            }
        }
        if rotations == 0 {
            break;
        }
    }
    let mut ordre: Vec<usize> = (0..n).collect();
    ordre.sort_by(|&i, &k| a[k][k].total_cmp(&a[i][i]));
    let valeurs: Vec<f64> = ordre.iter().map(|&i| a[i][i]).collect();
    let vecteurs: Vec<Vec<f64>> = ordre
        .iter()
        .map(|&k| {
            let colonne: Vec<f64> = v.iter().map(|l| l[k]).collect();
            let pivot = argmax(&colonne.iter().map(|x| x.abs()).collect::<Vec<f64>>());
            if colonne[pivot] < 0.0 {
                colonne.iter().map(|x| -x).collect()
            } else {
                colonne
            }
        })
        .collect();
    (valeurs, vecteurs)
}

// Rang numerique : nombre de valeurs propres au-dessus de 1e-8 fois la plus grande.
#[allow(dead_code)]
fn rang(valeurs: &[f64]) -> usize {
    let lmax = valeurs[0];
    valeurs.iter().filter(|v| **v > 1e-8 * lmax).count()
}

// OLS de norme minimale sans ordonnee, par K = X X^T : alpha = U_r L_r^{-1} U_r^T y, puis beta = X^T alpha.
// Les r valeurs propres non nulles fixent la prediction ; le noyau (p - r directions) ne la modifie pas.
#[allow(dead_code)]
fn ols_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64]) -> Vec<f64> {
    let lmax = valeurs[0];
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        if valeurs[k] > 1e-8 * lmax {
            let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
            for i in 0..n {
                alpha[i] += u[k][i] * d / valeurs[k];
            }
        }
    }
    alpha
}

// Coefficients duaux du ridge sans ordonnee : alpha = (K + lambda I)^{-1} y, dans la base des vecteurs propres.
#[allow(dead_code)]
fn ridge_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64], lambda: f64) -> Vec<f64> {
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
        for i in 0..n {
            alpha[i] += u[k][i] * d / (valeurs[k] + lambda);
        }
    }
    alpha
}

// beta = X^T alpha ; renvoie les previsions X beta sur train et test, et la norme L2 de beta.
#[allow(dead_code)]
fn predire(xtr: &[Vec<f64>], xte: &[Vec<f64>], alpha: &[f64]) -> (Vec<f64>, Vec<f64>, f64) {
    let beta = transpose_mat_vec(xtr, alpha);
    let norme = beta.iter().map(|b| b * b).sum::<f64>().sqrt();
    (mat_vec_ligne(xtr, &beta), mat_vec_ligne(xte, &beta), norme)
}

// Bornes du pli f d'une validation croisee a plis contigus (KFold sans melange).
#[allow(dead_code)]
fn bornes_pli(n: usize, plis: usize, f: usize) -> (usize, usize) {
    let (base, reste) = (n / plis, n % plis);
    let debut = f * base + reste.min(f);
    (debut, debut + base + if f < reste { 1 } else { 0 })
}

// Un pli contigu, avec la decomposition spectrale du Gram de son train : elle sert a tous les alpha du ridge.
#[allow(dead_code)]
struct PliContigu {
    xtr: Vec<Vec<f64>>,
    xva: Vec<Vec<f64>>,
    positions_val: Vec<usize>,
    valeurs: Vec<f64>,
    u: Vec<Vec<f64>>,
}

#[allow(dead_code)]
fn plis_contigus(x: &[Vec<f64>], plis: usize) -> Vec<PliContigu> {
    let n = x.len();
    (0..plis)
        .map(|f| {
            let (debut, fin) = bornes_pli(n, plis, f);
            let xtr: Vec<Vec<f64>> = (0..n)
                .filter(|&i| i < debut || i >= fin)
                .map(|i| x[i].clone())
                .collect();
            let xva: Vec<Vec<f64>> = x[debut..fin].to_vec();
            let (valeurs, u) = propres_sym(&gram(&xtr));
            PliContigu {
                xtr,
                xva,
                positions_val: (debut..fin).collect(),
                valeurs,
                u,
            }
        })
        .collect()
}

// RidgeCV(alphas, cv=plis, fit_intercept=False) : l'alpha retenu est celui qui maximise la moyenne
// des R2 de pli (le score par defaut de la validation croisee de scikit-learn). Renvoie son indice.
#[allow(dead_code)]
fn ridge_cv_sur_plis(plis_c: &[PliContigu], y: &[f64], alphas: &[f64]) -> usize {
    let mut score = vec![0.0; alphas.len()];
    for pli in plis_c {
        let ytr: Vec<f64> = (0..y.len())
            .filter(|i| !pli.positions_val.contains(i))
            .map(|i| y[i])
            .collect();
        let yva: Vec<f64> = pli.positions_val.iter().map(|&i| y[i]).collect();
        let sst: f64 = centrer(&yva).iter().map(|x| x * x).sum();
        for (j, &lam) in alphas.iter().enumerate() {
            let (_, p_va, _) = predire(
                &pli.xtr,
                &pli.xva,
                &ridge_alpha(&pli.valeurs, &pli.u, &ytr, lam),
            );
            let sse: f64 = yva.iter().zip(&p_va).map(|(t, p)| (t - p) * (t - p)).sum();
            score[j] += 1.0 - sse / sst;
        }
    }
    argmax(&score)
}

#[allow(dead_code)]
fn ridge_cv_retenu(x: &[Vec<f64>], y: &[f64], alphas: &[f64], plis: usize) -> usize {
    ridge_cv_sur_plis(&plis_contigus(x, plis), y, alphas)
}

// Composantes principales depuis la decomposition de K = X X^T : composante_k = X^T u_k / sqrt(lambda_k).
#[allow(dead_code)]
fn composantes(x: &[Vec<f64>], valeurs: &[f64], u: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let mut comps: Vec<Vec<f64>> = Vec::new();
    for k in 0..rang(valeurs) {
        let mut c = transpose_mat_vec(x, &u[k]);
        let s = valeurs[k].sqrt();
        for v in c.iter_mut() {
            *v /= s;
        }
        comps.push(c);
    }
    comps
}

// Centrage d'un echantillon sur ses propres colonnes : ce que fait PCA a l'ajustement.
#[allow(dead_code)]
fn centrer_propre(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    centrer_colonnes(x, &colonnes_moyennes(x))
}

// ACP, comme sklearn.decomposition.PCA : X est CENTRE sur ses colonnes, covariance (denominateur n),
// valeurs propres, part de variance, composantes. Le centrage ne change rien sur une matrice deja
// standardisee sur elle-meme (train complet) ; il est indispensable sur le train d'un pli, dont les
// colonnes ne sont pas de moyenne nulle.
#[allow(dead_code)]
fn acp(x: &[Vec<f64>]) -> (Vec<f64>, Vec<f64>, Vec<Vec<f64>>) {
    let xc = centrer_propre(x);
    let (valeurs, u) = propres_sym(&gram(&xc));
    let total: f64 = valeurs.iter().sum();
    let comps = composantes(&xc, &valeurs, &u);
    let ratio: Vec<f64> = valeurs.iter().map(|l| l / total).collect();
    (valeurs, ratio, comps)
}

// Variance cumulee jusqu'a la composante k comprise.
#[allow(dead_code)]
fn variance_cumulee(ratio: &[f64], k: usize) -> f64 {
    ratio[..=k].iter().sum()
}

// Scores t[i][k] = x_i . composante_k.
#[allow(dead_code)]
fn scores_composantes(x: &[Vec<f64>], comps: &[Vec<f64>]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            comps
                .iter()
                .map(|c| l.iter().zip(c).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// PCR a k composantes avec ordonnee : y = ybar + sum_{j<k} b_j t_j.
// Les scores d'une ACP sont orthogonaux deux a deux, donc b_j = (t_j . (y - ybar)) / (t_j . t_j).
#[allow(dead_code)]
fn regression_scores(
    t_tr: &[Vec<f64>],
    t_te: &[Vec<f64>],
    y: &[f64],
    k: usize,
) -> (Vec<f64>, Vec<f64>) {
    let ym = moyenne(y);
    let yc: Vec<f64> = y.iter().map(|v| v - ym).collect();
    let mut b = vec![0.0; k];
    for j in 0..k {
        let num: f64 = t_tr.iter().zip(&yc).map(|(t, &v)| t[j] * v).sum();
        let den: f64 = t_tr.iter().map(|t| t[j] * t[j]).sum();
        b[j] = num / den;
    }
    let projet = |tt: &[Vec<f64>]| -> Vec<f64> {
        tt.iter()
            .map(|t| ym + (0..k).map(|j| b[j] * t[j]).sum::<f64>())
            .collect()
    };
    (projet(t_tr), projet(t_te))
}

// RMSE test de la PCR pour k = 1..kmax (une seule ACP, une regression par k).
#[allow(dead_code)]
fn balayage_pcr(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (_, _, mut comps) = acp(xtr);
    comps.truncate(kmax);
    // PCA().fit(xtr).transform(...) : les deux echantillons sont centres sur la moyenne du train.
    let xmu = colonnes_moyennes(xtr);
    let (t_tr, t_te) = (
        scores_composantes(&centrer_colonnes(xtr, &xmu), &comps),
        scores_composantes(&centrer_colonnes(xte, &xmu), &comps),
    );
    (1..=kmax)
        .map(|k| rmse(yte, &regression_scores(&t_tr, &t_te, ytr, k).1))
        .collect()
}

// ---- PLS1 par NIPALS (l'algorithme de PLSRegression). x et y doivent etre centres sur le train.
// Renvoie (w, p, t) : poids x, loadings x et scores, ranges par composante : w[j][v], t[j][i]. ----
#[allow(dead_code)]
fn nipals(
    x: &[Vec<f64>],
    y_c: &[f64],
    kmax: usize,
) -> (Vec<Vec<f64>>, Vec<Vec<f64>>, Vec<Vec<f64>>) {
    let n = x.len();
    let nvar = x[0].len();
    let mut xk = x.to_vec();
    let mut yk = y_c.to_vec();
    let mut w_all: Vec<Vec<f64>> = Vec::new();
    let mut p_all: Vec<Vec<f64>> = Vec::new();
    let mut t_all: Vec<Vec<f64>> = Vec::new();
    for _j in 0..kmax {
        // w = Xk^T yk / ||Xk^T yk|| (direction de covariance maximale avec le reste de y), t = Xk w
        let mut w = transpose_mat_vec(&xk, &yk);
        let nrm = w.iter().map(|v| v * v).sum::<f64>().sqrt();
        for v in w.iter_mut() {
            *v /= nrm;
        }
        let t = mat_vec_ligne(&xk, &w);
        let ttt: f64 = t.iter().map(|v| v * v).sum();
        // p = Xk^T t / (t^T t) ; q = yk^T t / (t^T t)
        let mut pp = transpose_mat_vec(&xk, &t);
        for v in pp.iter_mut() {
            *v /= ttt;
        }
        let q: f64 = t.iter().zip(&yk).map(|(a, b)| a * b).sum::<f64>() / ttt;
        // Deflation : Xk <- Xk - t p^T ; yk <- yk - q t
        for i in 0..n {
            for j in 0..nvar {
                xk[i][j] -= t[i] * pp[j];
            }
            yk[i] -= q * t[i];
        }
        w_all.push(w);
        p_all.push(pp);
        t_all.push(t);
    }
    (w_all, p_all, t_all)
}

// Inverse d'une matrice carree (Gauss-Jordan, pivot partiel) ; None si un pivot est nul.
#[allow(dead_code)]
fn inverse(m: &[Vec<f64>]) -> Option<Vec<Vec<f64>>> {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut inv: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|j| if i == j { 1.0 } else { 0.0 }).collect())
        .collect();
    for col in 0..n {
        let mut piv = col;
        for r in (col + 1)..n {
            if a[r][col].abs() > a[piv][col].abs() {
                piv = r;
            }
        }
        if !a[piv][col].is_finite() || a[piv][col].abs() < 1e-14 {
            return None;
        }
        a.swap(col, piv);
        inv.swap(col, piv);
        let d = a[col][col];
        for j in 0..n {
            a[col][j] /= d;
            inv[col][j] /= d;
        }
        for r in 0..n {
            if r != col {
                let f = a[r][col];
                for j in 0..n {
                    a[r][j] -= f * a[col][j];
                    inv[r][j] -= f * inv[col][j];
                }
            }
        }
    }
    Some(inv)
}

// Rotations de la PLS, R = W (P^T W)^{-1} : elles appliquent le modele a des lignes nouvelles
// (t = X_c R). M = P^T W etant triangulaire superieure, les k premieres colonnes de R sont celles
// du modele a k composantes.
#[allow(dead_code)]
fn rotations(w: &[Vec<f64>], pmat: &[Vec<f64>], kmax: usize) -> Option<Vec<Vec<f64>>> {
    let nvar = w[0].len();
    let m: Vec<Vec<f64>> = (0..kmax)
        .map(|i| {
            (0..kmax)
                .map(|j| pmat[i].iter().zip(&w[j]).map(|(a, b)| a * b).sum())
                .collect()
        })
        .collect();
    let inv = inverse(&m)?;
    let r: Vec<Vec<f64>> = (0..nvar)
        .map(|v| {
            (0..kmax)
                .map(|j| (0..kmax).map(|i| w[i][v] * inv[i][j]).sum())
                .collect()
        })
        .collect();
    Some(r)
}

// Poids cotes variables du modele PLS a k = 1..kmax composantes : betas[k - 1] pour k composantes.
// b_j = (t_j . y_c) / (t_j . t_j), puis beta_k = sum_{j<k} b_j R[:, j].
#[allow(dead_code)]
fn pls_poids(
    w: &[Vec<f64>],
    pmat: &[Vec<f64>],
    t: &[Vec<f64>],
    y_c: &[f64],
) -> Option<Vec<Vec<f64>>> {
    let kmax = t.len();
    let n = t[0].len();
    let nvar = w[0].len();
    let r = rotations(w, pmat, kmax)?;
    let mut betas: Vec<Vec<f64>> = Vec::new();
    let mut beta = vec![0.0; nvar];
    for j in 0..kmax {
        let num: f64 = (0..n).map(|i| t[j][i] * y_c[i]).sum();
        let den: f64 = t[j].iter().map(|v| v * v).sum();
        let b = num / den;
        for v in 0..nvar {
            beta[v] += b * r[v][j];
        }
        betas.push(beta.clone());
    }
    Some(betas)
}

// Prevision PLS pour des lignes deja centrees : y = ybar + x_c . beta.
#[allow(dead_code)]
fn pls_predire(x_c: &[Vec<f64>], y_moy: f64, beta: &[f64]) -> Vec<f64> {
    x_c.iter()
        .map(|l| y_moy + l.iter().zip(beta).map(|(a, b)| a * b).sum::<f64>())
        .collect()
}

// RMSE test de la PLS pour k = 1..kmax (un seul NIPALS, un vecteur de poids par k) ;
// NaN si les rotations ne sont pas calculables.
#[allow(dead_code)]
fn balayage_pls(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (xmu, y_moy) = (colonnes_moyennes(xtr), moyenne(ytr));
    let yc = centrer(ytr);
    let (w, pmat, t) = nipals(&centrer_colonnes(xtr, &xmu), &yc, kmax);
    let betas = match pls_poids(&w, &pmat, &t, &yc) {
        Some(b) => b,
        None => return vec![f64::NAN; kmax],
    };
    let xte_c = centrer_colonnes(xte, &xmu);
    (1..=kmax)
        .map(|k| rmse(yte, &pls_predire(&xte_c, y_moy, &betas[k - 1])))
        .collect()
}

// RMSE test de la PLS ajustee avec exactement k composantes.
#[allow(dead_code)]
fn rmse_pls_k(xtr: &[Vec<f64>], ytr: &[f64], xte: &[Vec<f64>], yte: &[f64], k: usize) -> f64 {
    match balayage_pls(xtr, ytr, xte, yte, k).last() {
        Some(r) => *r,
        None => f64::NAN,
    }
}

// ---- Figures en texte : le noyau evcxr n'affiche pas de graphique. ----

// Courbes RMSE test de la PCR et de la PLS selon k, ligne du meilleur ridge sur ce test, colonne de la
// composante la plus chargee en bloc B. Figure en texte : une marque par point. L'axe vertical part du
// multiple de 0,15 inferieur ou egal a la plus petite RMSE tracee et monte jusqu'au multiple du pas
// superieur ou egal a la plus grande : aucune valeur finie ne sort du cadre (un cadre fixe
// 1,00 -> 0,20 masquait ici le premier point de la PCR et confondait la ligne du ridge avec la PLS).
// Le pas (0,025, puis 0,05, puis 0,1) est le plus fin qui garde la trame sous 40 lignes.
// Renvoie la figure, la borne basse, la borne haute et le pas d'une ligne.
#[allow(dead_code)]
fn figure_courbes(pcr: &[f64], pls: &[f64], ridge: f64, i_max_b: usize) -> (String, f64, f64, f64) {
    let (lignes_max, droite) = (40usize, 8usize);
    // Chaque k occupe deux colonnes ; la premiere courbe commence a la colonne 4, et huit colonnes
    // restent libres a droite du dernier point pour que la colonne du bloc B se lise comme une marque.
    let colonnes = (4 + 2 * pcr.len().max(1) + droite).max(64);
    let colonne_k = |k: usize| (4 + 2 * k.saturating_sub(1)).min(colonnes - 1);
    let mut y_haut = f64::NEG_INFINITY;
    let mut y_bas = f64::INFINITY;
    for &r in pcr.iter().chain(pls.iter()).chain([ridge].iter()) {
        if r.is_finite() {
            y_haut = y_haut.max(r);
            y_bas = y_bas.min(r);
        }
    }
    if !(y_haut.is_finite() && y_bas.is_finite()) {
        y_haut = 1.0;
        y_bas = 0.0;
    }
    // Borne basse : multiple de 0,15, comme les graduations de l'axe.
    y_bas = (y_bas / 0.15).floor() * 0.15;
    let mut pas = 0.1f64;
    for candidat in [0.025f64, 0.05, 0.1] {
        if ((y_haut / candidat).ceil() * candidat - y_bas) / candidat <= (lignes_max - 1) as f64 {
            pas = candidat;
            break;
        }
    }
    y_haut = ((y_haut / pas).ceil() * pas).max(y_bas + pas);
    let lignes = (((y_haut - y_bas) / pas).round() as usize).min(lignes_max - 1) + 1;
    let ligne_rmse = |r: f64| -> usize {
        let brute = ((y_haut - r) / (y_haut - y_bas) * (lignes as f64 - 1.0)).round();
        if !brute.is_finite() {
            lignes - 1
        } else {
            (brute.max(0.0) as usize).min(lignes - 1)
        }
    };
    let mut trame: Vec<Vec<char>> = vec![vec![' '; colonnes]; lignes];
    // D'abord la ligne du ridge, puis les courbes par-dessus : un point qui tombe sur la ligne du
    // ridge s'ecrit '*', pour que la rencontre reste lisible.
    if ridge.is_finite() {
        let l = ligne_rmse(ridge);
        for c in colonne_k(1)..=colonne_k(pcr.len()) {
            trame[l][c] = '-';
        }
    }
    for (courbe, marque) in [(pcr, '.'), (pls, 'x')] {
        for (k, &r) in courbe.iter().enumerate() {
            if r.is_finite() {
                let c = colonne_k(k + 1);
                let l = ligne_rmse(r);
                trame[l][c] = if trame[l][c] == '-' { '*' } else { marque };
            }
        }
    }
    let c_b = colonne_k(i_max_b + 1);
    for l in 0..lignes {
        if trame[l][c_b] == ' ' {
            trame[l][c_b] = ':';
        }
    }
    let mut sortie = String::new();
    for (l, ligne) in trame.iter().enumerate() {
        let y = y_bas + (lignes - 1 - l) as f64 * pas;
        let graduation = (y / 0.15 - (y / 0.15).round()).abs() < 1e-9;
        if graduation {
            sortie.push_str(&format!("{:4.2} |{}\n", y, ligne.iter().collect::<String>()));
        } else {
            sortie.push_str(&format!("     |{}\n", ligne.iter().collect::<String>()));
        }
    }
    // Axe des k : un tiret sous chaque position, une graduation tous les 5 k.
    let mut graduation: Vec<char> = vec![' '; colonnes];
    for c in colonne_k(1)..=colonne_k(pcr.len()) {
        graduation[c] = '-';
    }
    for k in (5..=pcr.len()).step_by(5) {
        graduation[colonne_k(k)] = '+';
    }
    sortie.push_str(&format!("     +{}\n", graduation.iter().collect::<String>()));
    let mut legendek: Vec<char> = vec![' '; colonnes];
    for k in (5..=pcr.len()).step_by(5) {
        for (d, ch) in format!("{}", k).chars().enumerate() {
            let c = colonne_k(k) + d;
            if c < colonnes {
                legendek[c] = ch;
            }
        }
    }
    sortie.push_str(&format!("      {}\n", legendek.iter().collect::<String>()));
    (sortie, y_bas, y_haut, pas)
}

// Profil d'un vecteur de poids, une barre par lot : bloc A en lots de `taille_lot` variables (le plus
// fort |loading| du lot), bloc B variable par variable. `echelle` est la valeur qui occupe toute la
// largeur : la meme pour deux panneaux les rend comparables.
#[allow(dead_code)]
fn profil_loadings(poids: &[f64], debut_b: usize, taille_lot: usize, echelle: f64) -> String {
    const LARGEUR: usize = 40;
    let barre = |debut: usize, fin: usize| -> String {
        let m: f64 = (debut..fin).map(|v| poids[v].abs()).fold(0.0f64, f64::max);
        // Largeur ramenee a 0..LARGEUR ; une echelle nulle ou non finie donne une barre vide.
        let n = if echelle.is_finite() && echelle > 0.0 {
            (((m / echelle) * LARGEUR as f64).round().max(0.0) as usize).min(LARGEUR)
        } else {
            0
        };
        format!("{:<40} {:.4}", "#".repeat(n), m)
    };
    let mut sortie = String::new();
    for debut in (0..debut_b).step_by(taille_lot.max(1)) {
        let fin = (debut + taille_lot).min(debut_b);
        sortie.push_str(&format!("A {:3}-{:3} |{}\n", debut, fin - 1, barre(debut, fin)));
    }
    for v in debut_b..poids.len() {
        sortie.push_str(&format!("B {:3}     |{}\n", v, barre(v, v + 1)));
    }
    sortie
}

// ---- 6. Le face-a-face : meme split, meme graine, deux facons de choisir les composantes ----
let j = jeu(GRAINE);
let p = preparer(&j, &cible(&j));
let (valeurs, u) = propres_sym(&gram(&p.x_train_s));
let comps = composantes(&p.x_train_s, &valeurs, &u);
let imp_b: Vec<f64> = comps
    .iter()
    .map(|c| (P_A..P).map(|v| c[v] * c[v]).sum())
    .collect();
let i_max_b = argmax(&imp_b);
let rmse_pcr = balayage_pcr(&p.x_train_s, &p.y_train_z, &p.x_test_s, &p.y_test_z, 45);
let rmse_pls = balayage_pls(&p.x_train_s, &p.y_train_z, &p.x_test_s, &p.y_test_z, 45);
// La ligne ridge est un minimum retrospectif de la grille test a 40 valeurs.
let grille = logspace(-3.0, 3.0, 40);
let mut rmse_test_ridge: Vec<f64> = Vec::new();
for lam in &grille {
    let (_, p_te, _) = predire(
        &p.x_train_s,
        &p.x_test_s,
        &ridge_alpha(&valeurs, &u, &p.y_train_z, *lam),
    );
    rmse_test_ridge.push(rmse(&p.y_test_z, &p_te));
}
let ridge_min = rmse_test_ridge[argmin(&rmse_test_ridge)];
let (figure, y_bas, y_haut, pas) =
    figure_courbes(&rmse_pcr, &rmse_pls, ridge_min, i_max_b);
println!("{}", figure);
println!(
    "PCR : k=1 {:.3} -> meilleur {:.3} (k={})",
    rmse_pcr[0],
    rmse_pcr[argmin(&rmse_pcr)],
    argmin(&rmse_pcr) + 1
);
println!(
    "PLS : k=1 {:.3} -> meilleur {:.3} (k={})",
    rmse_pls[0],
    rmse_pls[argmin(&rmse_pls)],
    argmin(&rmse_pls) + 1
);
println!(
    "a k=5 : PCR {:.3} vs PLS {:.3}  (ratio {:.1}x)",
    rmse_pcr[4],
    rmse_pls[4],
    rmse_pcr[4] / rmse_pls[4]
);
// La legende tient lieu des etiquettes d'un traceur : marques, puis echelle des deux axes.
println!(
    "legende : . PCR   x PLS   - meilleur ridge de la grille test ({:.3})   * un point de courbe sur la ligne du ridge   : composante la plus chargee en bloc B (indice {}, rang {})",
    ridge_min, i_max_b, i_max_b + 1
);
println!(
    "axe vertical : RMSE test de {:.2} (bas) a {:.2} (haut), {:.3} par ligne ; axe horizontal : k de 1 a {}",
    y_bas,
    y_haut,
    pas,
    rmse_pcr.len()
);

     |                                                                                          :           


1.05 |    . .                                                                                   :           


     |                                                                                          :           


     |                                                                                          :           


     |        . . . .   .                                                                       :           


     |                .   . .                                                                   :           


... (35 outputs left out)

PLS : k=1 0.541 -> meilleur 0.180 (k=7)


a k=5 : PCR 0.967 vs PLS 0.191  (ratio 5.1x)


legende : . PCR   x PLS   - meilleur ridge de la grille test (0.179)   * un point de courbe sur la ligne du ridge   : composante la plus chargee en bloc B (indice 43, rang 44)


axe vertical : RMSE test de 0.15 (bas) a 1.07 (haut), 0.025 par ligne ; axe horizontal : k de 1 a 45


### Lecture du résultat

À k = 5 : PCR 0.967 contre PLS 0.191 — **5.1× d'écart** pour le même budget de composantes. À k = 45 les deux courbes se rejoignent (0.202 et 0.202), et la PLS s'était déjà posée à son optimum dès k = 7 : la PCR finit par engloutir le signal dans son budget, la PLS l'a capturé d'emblée. La colonne `:` (indice 43, rang 44) matérialise l'entrée du signal dans l'ordre des variances ; la ligne pointillée du ridge (0.179) rappelle que la pénalisation seule se place entre les deux régimes. Les marques `*` de la ligne basse sont les points de PLS qui touchent déjà cette ligne : c'est dire que la PLS rejoint ridge en quelques composantes, là où la PCR a besoin de tout son budget.

## 7. Sélection de k par validation croisée — la courbe qui choisit

En pratique on ne regarde pas le test pour choisir $k$ : on valide croisée **sur le train** (5 plis), et l'on trace la RMSEP de CV. Le $k$ retenu doit coïncider avec ce que la figure du §6 laissait espérer.

La standardisation de X et Y a été ajustée sur tout le train avant les plis : les estimations internes de CV comportent donc une fuite de prétraitement entre plis. Le test reste exclu. Pour une estimation sans cette fuite, placer les transformations dans chaque pipeline de validation.

> **Côté Rust.** Les plis sont **contigus** : une permutation du train dérivée de la graine (`graine ^ 0xA5A5A5A5`) est découpée en 5 blocs, chacun servant de validation à son tour ; la RMSE affichée est celle des prédictions hors-pli **mises en commun**, pas la moyenne des 5 RMSE.

In [10]:
// Cellule autonome : prelude (RNG, donnees, algebre) redefini ici, puis le calcul de la section.
// ---- Generateur pseudo-alatoire maison : xorshift64* + Box-Muller (aucun crate).
// La suite n'est pas celle de NumPy : les tirages different, les valeurs chiffrees sont celles de ce carnet.
#[allow(dead_code)]
struct Rng {
    etat: u64,
}

#[allow(dead_code)]
impl Rng {
    fn new(graine: u64) -> Rng {
        let mut etat = graine ^ 0x9E3779B97F4A7C15;
        if etat == 0 {
            etat = 0x2545F4914F6CDD1D;
        }
        Rng { etat }
    }

    fn brut(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x >> 12;
        x ^= x << 25;
        x ^= x >> 27;
        self.etat = x;
        x.wrapping_mul(0x2545F4914F6CDD1D)
    }

    // Uniforme sur [0, 1), 53 bits de mantisse.
    fn uniforme(&mut self) -> f64 {
        ((self.brut() >> 11) as f64 + 0.5) / 9007199254740992.0
    }

    // Normale centree reduite, methode de Box-Muller.
    fn normale(&mut self) -> f64 {
        const TAU: f64 = 6.283185307179586;
        loop {
            let u1 = self.uniforme();
            let u2 = self.uniforme();
            if u1 > 1e-300 {
                return (-2.0 * u1.ln()).sqrt() * (TAU * u2).cos();
            }
        }
    }

    // Entier uniforme dans 0..n, par rejet (pas le biais du modulo).
    fn indice(&mut self, n: usize) -> usize {
        let n = n as u64;
        let limite = u64::MAX - (u64::MAX % n);
        loop {
            let t = self.brut();
            if t < limite {
                return (t % n) as usize;
            }
        }
    }
}

// Permutation aleatoire (Fisher-Yates).
#[allow(dead_code)]
fn permutation(rng: &mut Rng, n: usize) -> Vec<usize> {
    let mut idx: Vec<usize> = (0..n).collect();
    for i in (1..n).rev() {
        let j = rng.indice(i + 1);
        idx.swap(i, j);
    }
    idx
}

// ---- Statistiques de base (ecarts-types au denominateur n, comme NumPy). ----
#[allow(dead_code)]
fn moyenne(v: &[f64]) -> f64 {
    v.iter().sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn variance(v: &[f64]) -> f64 {
    let m = moyenne(v);
    v.iter().map(|x| (x - m) * (x - m)).sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn ecart_type(v: &[f64]) -> f64 {
    variance(v).sqrt()
}

// Matrices stockees ligne par ligne : x[i][j].
#[allow(dead_code)]
fn mat_vec_ligne(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    x.iter()
        .map(|l| l.iter().zip(b).map(|(a, c)| a * c).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn colonnes_moyennes(x: &[Vec<f64>]) -> Vec<f64> {
    let n = x.len();
    (0..x[0].len())
        .map(|j| x.iter().map(|l| l[j]).sum::<f64>() / n as f64)
        .collect()
}

#[allow(dead_code)]
fn colonnes_ecarts(x: &[Vec<f64>]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| ecart_type(&x.iter().map(|l| l[j]).collect::<Vec<f64>>()))
        .collect()
}

#[allow(dead_code)]
fn standardiser(x: &[Vec<f64>], mu: &[f64], sigma: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            l.iter()
                .zip(mu.iter().zip(sigma))
                .map(|(a, (m, s))| (a - m) / s)
                .collect()
        })
        .collect()
}

#[allow(dead_code)]
fn standardiser_vec(v: &[f64], m: f64, s: f64) -> Vec<f64> {
    v.iter().map(|x| (x - m) / s).collect()
}

#[allow(dead_code)]
fn centrer(v: &[f64]) -> Vec<f64> {
    let m = moyenne(v);
    v.iter().map(|x| x - m).collect()
}

#[allow(dead_code)]
fn centrer_colonnes(x: &[Vec<f64>], mu: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| l.iter().zip(mu).map(|(a, m)| a - m).collect())
        .collect()
}

#[allow(dead_code)]
fn correlation(a: &[f64], b: &[f64]) -> f64 {
    let (ca, cb) = (centrer(a), centrer(b));
    let num: f64 = ca.iter().zip(&cb).map(|(x, y)| x * y).sum();
    let den: f64 = ca.iter().map(|x| x * x).sum::<f64>().sqrt()
        * cb.iter().map(|y| y * y).sum::<f64>().sqrt();
    num / den
}

#[allow(dead_code)]
fn rmse(a: &[f64], b: &[f64]) -> f64 {
    (a.iter().zip(b).map(|(x, y)| (x - y) * (x - y)).sum::<f64>() / a.len() as f64).sqrt()
}

// Indice du minimum (le premier en cas d'egalite), comme np.argmin.
#[allow(dead_code)]
fn argmin(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] < v[m] {
            m = k;
        }
    }
    m
}

// Indice du maximum (le premier en cas d'egalite), comme np.argmax.
#[allow(dead_code)]
fn argmax(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] > v[m] {
            m = k;
        }
    }
    m
}

#[allow(dead_code)]
fn logspace(a: f64, b: f64, n: usize) -> Vec<f64> {
    (0..n)
        .map(|i| 10f64.powf(a + (b - a) * i as f64 / (n as f64 - 1.0)))
        .collect()
}

// Colonne j d'une matrice stockee ligne par ligne.
#[allow(dead_code)]
fn colonne(x: &[Vec<f64>], j: usize) -> Vec<f64> {
    x.iter().map(|l| l[j]).collect()
}

// Somme des valeurs absolues sur deux blocs de variables : (0..debut_b, debut_b..fin).
#[allow(dead_code)]
fn masses_blocs(v: &[f64], debut_b: usize) -> (f64, f64) {
    let a: f64 = v[..debut_b].iter().map(|x| x.abs()).sum();
    let b: f64 = v[debut_b..].iter().map(|x| x.abs()).sum();
    (a, b)
}

// ---- Donnees : meme construction que la source (blocs A et B, cible, decoupage, standardisation). ----
#[allow(dead_code)]
const N: usize = 160;
#[allow(dead_code)]
const G_A: usize = 40;
#[allow(dead_code)]
const TAILLE_G: usize = 5;
#[allow(dead_code)]
const P_A: usize = 200;
#[allow(dead_code)]
const P_B: usize = 5;
#[allow(dead_code)]
const P: usize = 205;
#[allow(dead_code)]
const N_TRAIN: usize = 120;
#[allow(dead_code)]
const GRAINE: u64 = 42;

// Variables en groupes ; dans chaque groupe, corr(x_i, x_j) = rho_pair via facteur commun.
#[allow(dead_code)]
fn bloc_facteurs(rng: &mut Rng, n_vars: usize, n_groupes: usize, rho_pair: f64) -> Vec<Vec<f64>> {
    let mut x = vec![vec![0.0; n_vars]; N];
    for g in 0..n_groupes {
        let f: Vec<f64> = (0..N).map(|_| rho_pair.sqrt() * rng.normale()).collect();
        for j in 0..TAILLE_G {
            for i in 0..N {
                x[i][g * TAILLE_G + j] = f[i] + (1.0 - rho_pair).sqrt() * rng.normale();
            }
        }
    }
    x
}

// Un tirage complet depuis la graine : les deux blocs, le bruit, le decoupage et les poids du regime aligne.
#[allow(dead_code)]
struct Jeu {
    x: Vec<Vec<f64>>,
    bruit: Vec<f64>,
    idx: Vec<usize>,
    poids_fact: Vec<f64>,
}

// 40 directions de variance dominante sans signal (bloc A), 1 direction plus modeste generatrice de Y (bloc B).
#[allow(dead_code)]
fn jeu(graine: u64) -> Jeu {
    let mut rng = Rng::new(graine);
    let mut x = bloc_facteurs(&mut rng, P_A, G_A, 0.8);
    let xb = bloc_facteurs(&mut rng, P_B, 1, 0.6);
    for i in 0..N {
        x[i].extend(xb[i].iter().copied());
    }
    let bruit: Vec<f64> = (0..N).map(|_| rng.normale()).collect();
    let idx = permutation(&mut rng, N);
    let poids_fact: Vec<f64> = (0..G_A).map(|_| rng.normale() / (G_A as f64).sqrt()).collect();
    Jeu { x, bruit, idx, poids_fact }
}

// Betas de meme signe, alignes sur le facteur commun du bloc B ; Y = X beta + bruit.
#[allow(dead_code)]
fn cible(j: &Jeu) -> Vec<f64> {
    let formes = [1.3, 1.0, 1.2, 0.9, 1.1];
    let mut beta = vec![0.0; P];
    for k in 0..P_B {
        beta[P_A + k] = formes[k] * 1.2;
    }
    let mut y = mat_vec_ligne(&j.x, &beta);
    for i in 0..N {
        y[i] += j.bruit[i];
    }
    y
}

// Regime aligne : Y2 = facteurs_A . poids_fact * 3.0 + bruit. Les facteurs_A sont des proxis construits
// a partir de la premiere variable observee de chaque groupe du bloc A, pas les facteurs latents exacts.
#[allow(dead_code)]
fn cible_alignee(j: &Jeu) -> Vec<f64> {
    let mut y = vec![0.0; N];
    for i in 0..N {
        let mut s = 0.0;
        for g in 0..G_A {
            s += 0.8f64.sqrt() * j.x[i][g * TAILLE_G] * j.poids_fact[g];
        }
        y[i] = s * 3.0 + j.bruit[i];
    }
    y
}

// Decoupage 120 train / 40 test, standardisation ajustee sur le train uniquement (le geste du 2.1).
#[allow(dead_code)]
struct Prep {
    x_train_s: Vec<Vec<f64>>,
    x_test_s: Vec<Vec<f64>>,
    y_train_z: Vec<f64>,
    y_test_z: Vec<f64>,
    y_ecart: f64,
}

#[allow(dead_code)]
fn preparer(j: &Jeu, y: &[f64]) -> Prep {
    let x_train: Vec<Vec<f64>> = (0..N_TRAIN).map(|k| j.x[j.idx[k]].clone()).collect();
    let x_test: Vec<Vec<f64>> = (N_TRAIN..N).map(|k| j.x[j.idx[k]].clone()).collect();
    let y_train: Vec<f64> = (0..N_TRAIN).map(|k| y[j.idx[k]]).collect();
    let y_test: Vec<f64> = (N_TRAIN..N).map(|k| y[j.idx[k]]).collect();
    let (mu, sigma) = (colonnes_moyennes(&x_train), colonnes_ecarts(&x_train));
    let (y_m, y_s) = (moyenne(&y_train), ecart_type(&y_train));
    Prep {
        x_train_s: standardiser(&x_train, &mu, &sigma),
        x_test_s: standardiser(&x_test, &mu, &sigma),
        y_train_z: standardiser_vec(&y_train, y_m, y_s),
        y_test_z: standardiser_vec(&y_test, y_m, y_s),
        y_ecart: y_s,
    }
}

// ---- Algebre lineaire : tout passe par la matrice de Gram des lignes K = X X^T (n x n, ici 120 x 120),
// jamais par X^T X (p x p, singuliere des que p > n). ----
#[allow(dead_code)]
fn transpose_mat_vec(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| x.iter().zip(b).map(|(l, &v)| l[j] * v).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn gram(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let n = x.len();
    (0..n)
        .map(|i| {
            (0..n)
                .map(|k| x[i].iter().zip(&x[k]).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// Valeurs propres (decroissantes) et vecteurs propres d'une matrice symetrique par la methode de
// Jacobi cyclique. vecteurs[k][i] est la i-eme composante du k-eme vecteur propre ; le signe de chaque
// vecteur est choisi pour que sa plus grande composante en valeur absolue soit positive.
#[allow(dead_code)]
fn propres_sym(m: &[Vec<f64>]) -> (Vec<f64>, Vec<Vec<f64>>) {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut v: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|k| if i == k { 1.0 } else { 0.0 }).collect())
        .collect();
    for _balayage in 0..100 {
        let diag: f64 = (0..n).map(|i| a[i][i].abs()).fold(0.0f64, f64::max);
        let mut carres = 0.0f64;
        for i in 0..n {
            for k in (i + 1)..n {
                carres += a[i][k] * a[i][k];
            }
        }
        if carres.sqrt() <= 1e-13 * (1.0 + diag) {
            break;
        }
        let mut rotations = 0usize;
        for p in 0..n {
            for q in (p + 1)..n {
                let apq = a[p][q];
                // Pivot negligeable devant la diagonale : on l'annule sans tourner.
                if apq.abs() <= 1e-18 * (a[p][p].abs() + a[q][q].abs()) {
                    a[p][q] = 0.0;
                    a[q][p] = 0.0;
                    continue;
                }
                rotations += 1;
                let theta = (a[q][q] - a[p][p]) / (2.0 * apq);
                let t = if theta.abs() > 1e150 {
                    1.0 / (2.0 * theta)
                } else if theta >= 0.0 {
                    1.0 / (theta + (theta * theta + 1.0).sqrt())
                } else {
                    -1.0 / (-theta + (theta * theta + 1.0).sqrt())
                };
                let c = 1.0 / (t * t + 1.0).sqrt();
                let s = t * c;
                for k in 0..n {
                    if k != p && k != q {
                        let akp = a[k][p];
                        let akq = a[k][q];
                        a[k][p] = c * akp - s * akq;
                        a[p][k] = a[k][p];
                        a[k][q] = s * akp + c * akq;
                        a[q][k] = a[k][q];
                    }
                    let vkp = v[k][p];
                    let vkq = v[k][q];
                    v[k][p] = c * vkp - s * vkq;
                    v[k][q] = s * vkp + c * vkq;
                }
                a[p][p] -= t * apq;
                a[q][q] += t * apq;
                a[p][q] = 0.0;
                a[q][p] = 0.0;
            }
        }
        if rotations == 0 {
            break;
        }
    }
    let mut ordre: Vec<usize> = (0..n).collect();
    ordre.sort_by(|&i, &k| a[k][k].total_cmp(&a[i][i]));
    let valeurs: Vec<f64> = ordre.iter().map(|&i| a[i][i]).collect();
    let vecteurs: Vec<Vec<f64>> = ordre
        .iter()
        .map(|&k| {
            let colonne: Vec<f64> = v.iter().map(|l| l[k]).collect();
            let pivot = argmax(&colonne.iter().map(|x| x.abs()).collect::<Vec<f64>>());
            if colonne[pivot] < 0.0 {
                colonne.iter().map(|x| -x).collect()
            } else {
                colonne
            }
        })
        .collect();
    (valeurs, vecteurs)
}

// Rang numerique : nombre de valeurs propres au-dessus de 1e-8 fois la plus grande.
#[allow(dead_code)]
fn rang(valeurs: &[f64]) -> usize {
    let lmax = valeurs[0];
    valeurs.iter().filter(|v| **v > 1e-8 * lmax).count()
}

// OLS de norme minimale sans ordonnee, par K = X X^T : alpha = U_r L_r^{-1} U_r^T y, puis beta = X^T alpha.
// Les r valeurs propres non nulles fixent la prediction ; le noyau (p - r directions) ne la modifie pas.
#[allow(dead_code)]
fn ols_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64]) -> Vec<f64> {
    let lmax = valeurs[0];
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        if valeurs[k] > 1e-8 * lmax {
            let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
            for i in 0..n {
                alpha[i] += u[k][i] * d / valeurs[k];
            }
        }
    }
    alpha
}

// Coefficients duaux du ridge sans ordonnee : alpha = (K + lambda I)^{-1} y, dans la base des vecteurs propres.
#[allow(dead_code)]
fn ridge_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64], lambda: f64) -> Vec<f64> {
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
        for i in 0..n {
            alpha[i] += u[k][i] * d / (valeurs[k] + lambda);
        }
    }
    alpha
}

// beta = X^T alpha ; renvoie les previsions X beta sur train et test, et la norme L2 de beta.
#[allow(dead_code)]
fn predire(xtr: &[Vec<f64>], xte: &[Vec<f64>], alpha: &[f64]) -> (Vec<f64>, Vec<f64>, f64) {
    let beta = transpose_mat_vec(xtr, alpha);
    let norme = beta.iter().map(|b| b * b).sum::<f64>().sqrt();
    (mat_vec_ligne(xtr, &beta), mat_vec_ligne(xte, &beta), norme)
}

// Bornes du pli f d'une validation croisee a plis contigus (KFold sans melange).
#[allow(dead_code)]
fn bornes_pli(n: usize, plis: usize, f: usize) -> (usize, usize) {
    let (base, reste) = (n / plis, n % plis);
    let debut = f * base + reste.min(f);
    (debut, debut + base + if f < reste { 1 } else { 0 })
}

// Un pli contigu, avec la decomposition spectrale du Gram de son train : elle sert a tous les alpha du ridge.
#[allow(dead_code)]
struct PliContigu {
    xtr: Vec<Vec<f64>>,
    xva: Vec<Vec<f64>>,
    positions_val: Vec<usize>,
    valeurs: Vec<f64>,
    u: Vec<Vec<f64>>,
}

#[allow(dead_code)]
fn plis_contigus(x: &[Vec<f64>], plis: usize) -> Vec<PliContigu> {
    let n = x.len();
    (0..plis)
        .map(|f| {
            let (debut, fin) = bornes_pli(n, plis, f);
            let xtr: Vec<Vec<f64>> = (0..n)
                .filter(|&i| i < debut || i >= fin)
                .map(|i| x[i].clone())
                .collect();
            let xva: Vec<Vec<f64>> = x[debut..fin].to_vec();
            let (valeurs, u) = propres_sym(&gram(&xtr));
            PliContigu {
                xtr,
                xva,
                positions_val: (debut..fin).collect(),
                valeurs,
                u,
            }
        })
        .collect()
}

// RidgeCV(alphas, cv=plis, fit_intercept=False) : l'alpha retenu est celui qui maximise la moyenne
// des R2 de pli (le score par defaut de la validation croisee de scikit-learn). Renvoie son indice.
#[allow(dead_code)]
fn ridge_cv_sur_plis(plis_c: &[PliContigu], y: &[f64], alphas: &[f64]) -> usize {
    let mut score = vec![0.0; alphas.len()];
    for pli in plis_c {
        let ytr: Vec<f64> = (0..y.len())
            .filter(|i| !pli.positions_val.contains(i))
            .map(|i| y[i])
            .collect();
        let yva: Vec<f64> = pli.positions_val.iter().map(|&i| y[i]).collect();
        let sst: f64 = centrer(&yva).iter().map(|x| x * x).sum();
        for (j, &lam) in alphas.iter().enumerate() {
            let (_, p_va, _) = predire(
                &pli.xtr,
                &pli.xva,
                &ridge_alpha(&pli.valeurs, &pli.u, &ytr, lam),
            );
            let sse: f64 = yva.iter().zip(&p_va).map(|(t, p)| (t - p) * (t - p)).sum();
            score[j] += 1.0 - sse / sst;
        }
    }
    argmax(&score)
}

#[allow(dead_code)]
fn ridge_cv_retenu(x: &[Vec<f64>], y: &[f64], alphas: &[f64], plis: usize) -> usize {
    ridge_cv_sur_plis(&plis_contigus(x, plis), y, alphas)
}

// Composantes principales depuis la decomposition de K = X X^T : composante_k = X^T u_k / sqrt(lambda_k).
#[allow(dead_code)]
fn composantes(x: &[Vec<f64>], valeurs: &[f64], u: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let mut comps: Vec<Vec<f64>> = Vec::new();
    for k in 0..rang(valeurs) {
        let mut c = transpose_mat_vec(x, &u[k]);
        let s = valeurs[k].sqrt();
        for v in c.iter_mut() {
            *v /= s;
        }
        comps.push(c);
    }
    comps
}

// Centrage d'un echantillon sur ses propres colonnes : ce que fait PCA a l'ajustement.
#[allow(dead_code)]
fn centrer_propre(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    centrer_colonnes(x, &colonnes_moyennes(x))
}

// ACP, comme sklearn.decomposition.PCA : X est CENTRE sur ses colonnes, covariance (denominateur n),
// valeurs propres, part de variance, composantes. Le centrage ne change rien sur une matrice deja
// standardisee sur elle-meme (train complet) ; il est indispensable sur le train d'un pli, dont les
// colonnes ne sont pas de moyenne nulle.
#[allow(dead_code)]
fn acp(x: &[Vec<f64>]) -> (Vec<f64>, Vec<f64>, Vec<Vec<f64>>) {
    let xc = centrer_propre(x);
    let (valeurs, u) = propres_sym(&gram(&xc));
    let total: f64 = valeurs.iter().sum();
    let comps = composantes(&xc, &valeurs, &u);
    let ratio: Vec<f64> = valeurs.iter().map(|l| l / total).collect();
    (valeurs, ratio, comps)
}

// Variance cumulee jusqu'a la composante k comprise.
#[allow(dead_code)]
fn variance_cumulee(ratio: &[f64], k: usize) -> f64 {
    ratio[..=k].iter().sum()
}

// Scores t[i][k] = x_i . composante_k.
#[allow(dead_code)]
fn scores_composantes(x: &[Vec<f64>], comps: &[Vec<f64>]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            comps
                .iter()
                .map(|c| l.iter().zip(c).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// PCR a k composantes avec ordonnee : y = ybar + sum_{j<k} b_j t_j.
// Les scores d'une ACP sont orthogonaux deux a deux, donc b_j = (t_j . (y - ybar)) / (t_j . t_j).
#[allow(dead_code)]
fn regression_scores(
    t_tr: &[Vec<f64>],
    t_te: &[Vec<f64>],
    y: &[f64],
    k: usize,
) -> (Vec<f64>, Vec<f64>) {
    let ym = moyenne(y);
    let yc: Vec<f64> = y.iter().map(|v| v - ym).collect();
    let mut b = vec![0.0; k];
    for j in 0..k {
        let num: f64 = t_tr.iter().zip(&yc).map(|(t, &v)| t[j] * v).sum();
        let den: f64 = t_tr.iter().map(|t| t[j] * t[j]).sum();
        b[j] = num / den;
    }
    let projet = |tt: &[Vec<f64>]| -> Vec<f64> {
        tt.iter()
            .map(|t| ym + (0..k).map(|j| b[j] * t[j]).sum::<f64>())
            .collect()
    };
    (projet(t_tr), projet(t_te))
}

// RMSE test de la PCR pour k = 1..kmax (une seule ACP, une regression par k).
#[allow(dead_code)]
fn balayage_pcr(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (_, _, mut comps) = acp(xtr);
    comps.truncate(kmax);
    // PCA().fit(xtr).transform(...) : les deux echantillons sont centres sur la moyenne du train.
    let xmu = colonnes_moyennes(xtr);
    let (t_tr, t_te) = (
        scores_composantes(&centrer_colonnes(xtr, &xmu), &comps),
        scores_composantes(&centrer_colonnes(xte, &xmu), &comps),
    );
    (1..=kmax)
        .map(|k| rmse(yte, &regression_scores(&t_tr, &t_te, ytr, k).1))
        .collect()
}

// ---- PLS1 par NIPALS (l'algorithme de PLSRegression). x et y doivent etre centres sur le train.
// Renvoie (w, p, t) : poids x, loadings x et scores, ranges par composante : w[j][v], t[j][i]. ----
#[allow(dead_code)]
fn nipals(
    x: &[Vec<f64>],
    y_c: &[f64],
    kmax: usize,
) -> (Vec<Vec<f64>>, Vec<Vec<f64>>, Vec<Vec<f64>>) {
    let n = x.len();
    let nvar = x[0].len();
    let mut xk = x.to_vec();
    let mut yk = y_c.to_vec();
    let mut w_all: Vec<Vec<f64>> = Vec::new();
    let mut p_all: Vec<Vec<f64>> = Vec::new();
    let mut t_all: Vec<Vec<f64>> = Vec::new();
    for _j in 0..kmax {
        // w = Xk^T yk / ||Xk^T yk|| (direction de covariance maximale avec le reste de y), t = Xk w
        let mut w = transpose_mat_vec(&xk, &yk);
        let nrm = w.iter().map(|v| v * v).sum::<f64>().sqrt();
        for v in w.iter_mut() {
            *v /= nrm;
        }
        let t = mat_vec_ligne(&xk, &w);
        let ttt: f64 = t.iter().map(|v| v * v).sum();
        // p = Xk^T t / (t^T t) ; q = yk^T t / (t^T t)
        let mut pp = transpose_mat_vec(&xk, &t);
        for v in pp.iter_mut() {
            *v /= ttt;
        }
        let q: f64 = t.iter().zip(&yk).map(|(a, b)| a * b).sum::<f64>() / ttt;
        // Deflation : Xk <- Xk - t p^T ; yk <- yk - q t
        for i in 0..n {
            for j in 0..nvar {
                xk[i][j] -= t[i] * pp[j];
            }
            yk[i] -= q * t[i];
        }
        w_all.push(w);
        p_all.push(pp);
        t_all.push(t);
    }
    (w_all, p_all, t_all)
}

// Inverse d'une matrice carree (Gauss-Jordan, pivot partiel) ; None si un pivot est nul.
#[allow(dead_code)]
fn inverse(m: &[Vec<f64>]) -> Option<Vec<Vec<f64>>> {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut inv: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|j| if i == j { 1.0 } else { 0.0 }).collect())
        .collect();
    for col in 0..n {
        let mut piv = col;
        for r in (col + 1)..n {
            if a[r][col].abs() > a[piv][col].abs() {
                piv = r;
            }
        }
        if !a[piv][col].is_finite() || a[piv][col].abs() < 1e-14 {
            return None;
        }
        a.swap(col, piv);
        inv.swap(col, piv);
        let d = a[col][col];
        for j in 0..n {
            a[col][j] /= d;
            inv[col][j] /= d;
        }
        for r in 0..n {
            if r != col {
                let f = a[r][col];
                for j in 0..n {
                    a[r][j] -= f * a[col][j];
                    inv[r][j] -= f * inv[col][j];
                }
            }
        }
    }
    Some(inv)
}

// Rotations de la PLS, R = W (P^T W)^{-1} : elles appliquent le modele a des lignes nouvelles
// (t = X_c R). M = P^T W etant triangulaire superieure, les k premieres colonnes de R sont celles
// du modele a k composantes.
#[allow(dead_code)]
fn rotations(w: &[Vec<f64>], pmat: &[Vec<f64>], kmax: usize) -> Option<Vec<Vec<f64>>> {
    let nvar = w[0].len();
    let m: Vec<Vec<f64>> = (0..kmax)
        .map(|i| {
            (0..kmax)
                .map(|j| pmat[i].iter().zip(&w[j]).map(|(a, b)| a * b).sum())
                .collect()
        })
        .collect();
    let inv = inverse(&m)?;
    let r: Vec<Vec<f64>> = (0..nvar)
        .map(|v| {
            (0..kmax)
                .map(|j| (0..kmax).map(|i| w[i][v] * inv[i][j]).sum())
                .collect()
        })
        .collect();
    Some(r)
}

// Poids cotes variables du modele PLS a k = 1..kmax composantes : betas[k - 1] pour k composantes.
// b_j = (t_j . y_c) / (t_j . t_j), puis beta_k = sum_{j<k} b_j R[:, j].
#[allow(dead_code)]
fn pls_poids(
    w: &[Vec<f64>],
    pmat: &[Vec<f64>],
    t: &[Vec<f64>],
    y_c: &[f64],
) -> Option<Vec<Vec<f64>>> {
    let kmax = t.len();
    let n = t[0].len();
    let nvar = w[0].len();
    let r = rotations(w, pmat, kmax)?;
    let mut betas: Vec<Vec<f64>> = Vec::new();
    let mut beta = vec![0.0; nvar];
    for j in 0..kmax {
        let num: f64 = (0..n).map(|i| t[j][i] * y_c[i]).sum();
        let den: f64 = t[j].iter().map(|v| v * v).sum();
        let b = num / den;
        for v in 0..nvar {
            beta[v] += b * r[v][j];
        }
        betas.push(beta.clone());
    }
    Some(betas)
}

// Prevision PLS pour des lignes deja centrees : y = ybar + x_c . beta.
#[allow(dead_code)]
fn pls_predire(x_c: &[Vec<f64>], y_moy: f64, beta: &[f64]) -> Vec<f64> {
    x_c.iter()
        .map(|l| y_moy + l.iter().zip(beta).map(|(a, b)| a * b).sum::<f64>())
        .collect()
}

// RMSE test de la PLS pour k = 1..kmax (un seul NIPALS, un vecteur de poids par k) ;
// NaN si les rotations ne sont pas calculables.
#[allow(dead_code)]
fn balayage_pls(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (xmu, y_moy) = (colonnes_moyennes(xtr), moyenne(ytr));
    let yc = centrer(ytr);
    let (w, pmat, t) = nipals(&centrer_colonnes(xtr, &xmu), &yc, kmax);
    let betas = match pls_poids(&w, &pmat, &t, &yc) {
        Some(b) => b,
        None => return vec![f64::NAN; kmax],
    };
    let xte_c = centrer_colonnes(xte, &xmu);
    (1..=kmax)
        .map(|k| rmse(yte, &pls_predire(&xte_c, y_moy, &betas[k - 1])))
        .collect()
}

// RMSE test de la PLS ajustee avec exactement k composantes.
#[allow(dead_code)]
fn rmse_pls_k(xtr: &[Vec<f64>], ytr: &[f64], xte: &[Vec<f64>], yte: &[f64], k: usize) -> f64 {
    match balayage_pls(xtr, ytr, xte, yte, k).last() {
        Some(r) => *r,
        None => f64::NAN,
    }
}

// ---- Selection de k par validation croisee sur le train : 5 plis melanges, decoupage derive de la graine.
// L'ACP et la PLS sont refittees sur le train de chaque pli ; la standardisation globale du train n'est
// pas refaite (meme fuite de pretraitement que la source). L'erreur est pooled sur les 120 previsions
// hors pli. Renvoie (k PCR retenu, k PLS retenu, RMSE de CV par k).
#[allow(dead_code)]
fn cv_k(
    x: &[Vec<f64>],
    y: &[f64],
    kmax: usize,
    plis: usize,
    graine: u64,
) -> (usize, usize, Vec<f64>, Vec<f64>) {
    let n = x.len();
    let ordre = permutation(&mut Rng::new(graine ^ 0xA5A5A5A5), n);
    let mut carres_pcr = vec![0.0; kmax];
    let mut carres_pls = vec![0.0; kmax];
    for f in 0..plis {
        let (debut, fin) = bornes_pli(n, plis, f);
        let mut itr: Vec<usize> = ordre[..debut].to_vec();
        itr.extend_from_slice(&ordre[fin..]);
        let xtr: Vec<Vec<f64>> = itr.iter().map(|&i| x[i].clone()).collect();
        let ytr: Vec<f64> = itr.iter().map(|&i| y[i]).collect();
        let xva: Vec<Vec<f64>> = ordre[debut..fin].iter().map(|&i| x[i].clone()).collect();
        let yva: Vec<f64> = ordre[debut..fin].iter().map(|&i| y[i]).collect();
        // PCR : ACP refittee sur le train du pli, scores du train et du pli de validation centres sur la
        // moyenne du train du pli (contrat PCA().fit(train).transform(...)).
        let xmu = colonnes_moyennes(&xtr);
        let (_, _, mut comps) = acp(&xtr);
        comps.truncate(kmax);
        let (t_tr, t_va) = (
            scores_composantes(&centrer_colonnes(&xtr, &xmu), &comps),
            scores_composantes(&centrer_colonnes(&xva, &xmu), &comps),
        );
        // PLS : NIPALS ajuste sur le train du pli, X et y centres sur ce train
        let rmse_pls = balayage_pls(&xtr, &ytr, &xva, &yva, kmax);
        for k in 1..=kmax {
            let (_, p_pcr) = regression_scores(&t_tr, &t_va, &ytr, k);
            carres_pcr[k - 1] += rmse(&yva, &p_pcr) * rmse(&yva, &p_pcr) * yva.len() as f64 / n as f64;
            carres_pls[k - 1] += rmse_pls[k - 1] * rmse_pls[k - 1] * yva.len() as f64 / n as f64;
        }
    }
    let racines = |c: &[f64]| -> Vec<f64> { c.iter().map(|v| v.sqrt()).collect() };
    let (rp, rq) = (racines(&carres_pcr), racines(&carres_pls));
    (argmin(&rp) + 1, argmin(&rq) + 1, rp, rq)
}

// ---- 7. Selection de k par validation croisee sur le train ----
let j = jeu(GRAINE);
let p = preparer(&j, &cible(&j));
// 5 plis melanges, decoupage derive de la graine ; ACP et PLS refaites dans chaque pli.
let (k_pcr_cv, k_pls_cv, rmse_pcr_cv, rmse_pls_cv) =
    cv_k(&p.x_train_s, &p.y_train_z, 45, 5, GRAINE);
let rmse_pcr = balayage_pcr(&p.x_train_s, &p.y_train_z, &p.x_test_s, &p.y_test_z, 45);
let rmse_pls = balayage_pls(&p.x_train_s, &p.y_train_z, &p.x_test_s, &p.y_test_z, 45);
println!(
    "k CV retenu : PCR = {} (RMSE CV {:.4})   PLS = {} (RMSE CV {:.4})",
    k_pcr_cv,
    rmse_pcr_cv[k_pcr_cv - 1],
    k_pls_cv,
    rmse_pls_cv[k_pls_cv - 1]
);
println!(
    "RMSE test au k CV   : PCR {:.4}   PLS {:.4}",
    rmse_pcr[k_pcr_cv - 1],
    rmse_pls[k_pls_cv - 1]
);
println!(
    "k optimal test      : PCR {}   PLS {}",
    argmin(&rmse_pcr) + 1,
    argmin(&rmse_pls) + 1
);

k CV retenu : PCR = 45 (RMSE CV 0.3012)   PLS = 9 (RMSE CV 0.2586)


RMSE test au k CV   : PCR 0.2019   PLS 0.1895


k optimal test      : PCR 45   PLS 7


### Lecture du résultat

La validation croisée — qui ne voit jamais le test — retient k = 45 pour la PCR (RMSE CV 0.3012) et k = 9 pour la PLS (0.2586), contre des optima test de 45 et 7 : les RMSE test aux k retenus (0.2019 et 0.1895) sont à peine au-dessus des minima (0.2019 et 0.1800). La sélection par CV retrouve donc bien le régime de chaque méthode. Noter le choix CV de la PCR : 45 composantes, soit le **plafond de la grille** — la CV ne trouve pas de coude et prend tout ce qu'on lui donne, parce que les composantes de bruit ne nuisent presque plus une fois le signal inclus ; la PLS, elle, n'a besoin que de 9 composantes pour la même performance. Un $k$ choisi à la limite haute est un résultat à lire, pas un nombre à citer.

## 8. Scores, loadings — où chaque méthode met ses poids

Le diagnostic final compare les **loadings** (poids des variables) : la PLS concentre-t-elle sa première composante sur le bloc B, là où l'ACP la dépensait sur le bloc A ?

> **Côté Rust.** Les deux panneaux sont des barres ASCII : une ligne par groupe de 10 variables du bloc A (`A   0-  9`, …), une ligne par variable du bloc B (`B 200`, …), même échelle dans les deux panneaux pour que la comparaison soit directe.

In [11]:
// Cellule autonome : prelude (RNG, donnees, algebre) redefini ici, puis le calcul de la section.
// ---- Generateur pseudo-alatoire maison : xorshift64* + Box-Muller (aucun crate).
// La suite n'est pas celle de NumPy : les tirages different, les valeurs chiffrees sont celles de ce carnet.
#[allow(dead_code)]
struct Rng {
    etat: u64,
}

#[allow(dead_code)]
impl Rng {
    fn new(graine: u64) -> Rng {
        let mut etat = graine ^ 0x9E3779B97F4A7C15;
        if etat == 0 {
            etat = 0x2545F4914F6CDD1D;
        }
        Rng { etat }
    }

    fn brut(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x >> 12;
        x ^= x << 25;
        x ^= x >> 27;
        self.etat = x;
        x.wrapping_mul(0x2545F4914F6CDD1D)
    }

    // Uniforme sur [0, 1), 53 bits de mantisse.
    fn uniforme(&mut self) -> f64 {
        ((self.brut() >> 11) as f64 + 0.5) / 9007199254740992.0
    }

    // Normale centree reduite, methode de Box-Muller.
    fn normale(&mut self) -> f64 {
        const TAU: f64 = 6.283185307179586;
        loop {
            let u1 = self.uniforme();
            let u2 = self.uniforme();
            if u1 > 1e-300 {
                return (-2.0 * u1.ln()).sqrt() * (TAU * u2).cos();
            }
        }
    }

    // Entier uniforme dans 0..n, par rejet (pas le biais du modulo).
    fn indice(&mut self, n: usize) -> usize {
        let n = n as u64;
        let limite = u64::MAX - (u64::MAX % n);
        loop {
            let t = self.brut();
            if t < limite {
                return (t % n) as usize;
            }
        }
    }
}

// Permutation aleatoire (Fisher-Yates).
#[allow(dead_code)]
fn permutation(rng: &mut Rng, n: usize) -> Vec<usize> {
    let mut idx: Vec<usize> = (0..n).collect();
    for i in (1..n).rev() {
        let j = rng.indice(i + 1);
        idx.swap(i, j);
    }
    idx
}

// ---- Statistiques de base (ecarts-types au denominateur n, comme NumPy). ----
#[allow(dead_code)]
fn moyenne(v: &[f64]) -> f64 {
    v.iter().sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn variance(v: &[f64]) -> f64 {
    let m = moyenne(v);
    v.iter().map(|x| (x - m) * (x - m)).sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn ecart_type(v: &[f64]) -> f64 {
    variance(v).sqrt()
}

// Matrices stockees ligne par ligne : x[i][j].
#[allow(dead_code)]
fn mat_vec_ligne(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    x.iter()
        .map(|l| l.iter().zip(b).map(|(a, c)| a * c).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn colonnes_moyennes(x: &[Vec<f64>]) -> Vec<f64> {
    let n = x.len();
    (0..x[0].len())
        .map(|j| x.iter().map(|l| l[j]).sum::<f64>() / n as f64)
        .collect()
}

#[allow(dead_code)]
fn colonnes_ecarts(x: &[Vec<f64>]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| ecart_type(&x.iter().map(|l| l[j]).collect::<Vec<f64>>()))
        .collect()
}

#[allow(dead_code)]
fn standardiser(x: &[Vec<f64>], mu: &[f64], sigma: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            l.iter()
                .zip(mu.iter().zip(sigma))
                .map(|(a, (m, s))| (a - m) / s)
                .collect()
        })
        .collect()
}

#[allow(dead_code)]
fn standardiser_vec(v: &[f64], m: f64, s: f64) -> Vec<f64> {
    v.iter().map(|x| (x - m) / s).collect()
}

#[allow(dead_code)]
fn centrer(v: &[f64]) -> Vec<f64> {
    let m = moyenne(v);
    v.iter().map(|x| x - m).collect()
}

#[allow(dead_code)]
fn centrer_colonnes(x: &[Vec<f64>], mu: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| l.iter().zip(mu).map(|(a, m)| a - m).collect())
        .collect()
}

#[allow(dead_code)]
fn correlation(a: &[f64], b: &[f64]) -> f64 {
    let (ca, cb) = (centrer(a), centrer(b));
    let num: f64 = ca.iter().zip(&cb).map(|(x, y)| x * y).sum();
    let den: f64 = ca.iter().map(|x| x * x).sum::<f64>().sqrt()
        * cb.iter().map(|y| y * y).sum::<f64>().sqrt();
    num / den
}

#[allow(dead_code)]
fn rmse(a: &[f64], b: &[f64]) -> f64 {
    (a.iter().zip(b).map(|(x, y)| (x - y) * (x - y)).sum::<f64>() / a.len() as f64).sqrt()
}

// Indice du minimum (le premier en cas d'egalite), comme np.argmin.
#[allow(dead_code)]
fn argmin(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] < v[m] {
            m = k;
        }
    }
    m
}

// Indice du maximum (le premier en cas d'egalite), comme np.argmax.
#[allow(dead_code)]
fn argmax(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] > v[m] {
            m = k;
        }
    }
    m
}

#[allow(dead_code)]
fn logspace(a: f64, b: f64, n: usize) -> Vec<f64> {
    (0..n)
        .map(|i| 10f64.powf(a + (b - a) * i as f64 / (n as f64 - 1.0)))
        .collect()
}

// Colonne j d'une matrice stockee ligne par ligne.
#[allow(dead_code)]
fn colonne(x: &[Vec<f64>], j: usize) -> Vec<f64> {
    x.iter().map(|l| l[j]).collect()
}

// Somme des valeurs absolues sur deux blocs de variables : (0..debut_b, debut_b..fin).
#[allow(dead_code)]
fn masses_blocs(v: &[f64], debut_b: usize) -> (f64, f64) {
    let a: f64 = v[..debut_b].iter().map(|x| x.abs()).sum();
    let b: f64 = v[debut_b..].iter().map(|x| x.abs()).sum();
    (a, b)
}

// ---- Donnees : meme construction que la source (blocs A et B, cible, decoupage, standardisation). ----
#[allow(dead_code)]
const N: usize = 160;
#[allow(dead_code)]
const G_A: usize = 40;
#[allow(dead_code)]
const TAILLE_G: usize = 5;
#[allow(dead_code)]
const P_A: usize = 200;
#[allow(dead_code)]
const P_B: usize = 5;
#[allow(dead_code)]
const P: usize = 205;
#[allow(dead_code)]
const N_TRAIN: usize = 120;
#[allow(dead_code)]
const GRAINE: u64 = 42;

// Variables en groupes ; dans chaque groupe, corr(x_i, x_j) = rho_pair via facteur commun.
#[allow(dead_code)]
fn bloc_facteurs(rng: &mut Rng, n_vars: usize, n_groupes: usize, rho_pair: f64) -> Vec<Vec<f64>> {
    let mut x = vec![vec![0.0; n_vars]; N];
    for g in 0..n_groupes {
        let f: Vec<f64> = (0..N).map(|_| rho_pair.sqrt() * rng.normale()).collect();
        for j in 0..TAILLE_G {
            for i in 0..N {
                x[i][g * TAILLE_G + j] = f[i] + (1.0 - rho_pair).sqrt() * rng.normale();
            }
        }
    }
    x
}

// Un tirage complet depuis la graine : les deux blocs, le bruit, le decoupage et les poids du regime aligne.
#[allow(dead_code)]
struct Jeu {
    x: Vec<Vec<f64>>,
    bruit: Vec<f64>,
    idx: Vec<usize>,
    poids_fact: Vec<f64>,
}

// 40 directions de variance dominante sans signal (bloc A), 1 direction plus modeste generatrice de Y (bloc B).
#[allow(dead_code)]
fn jeu(graine: u64) -> Jeu {
    let mut rng = Rng::new(graine);
    let mut x = bloc_facteurs(&mut rng, P_A, G_A, 0.8);
    let xb = bloc_facteurs(&mut rng, P_B, 1, 0.6);
    for i in 0..N {
        x[i].extend(xb[i].iter().copied());
    }
    let bruit: Vec<f64> = (0..N).map(|_| rng.normale()).collect();
    let idx = permutation(&mut rng, N);
    let poids_fact: Vec<f64> = (0..G_A).map(|_| rng.normale() / (G_A as f64).sqrt()).collect();
    Jeu { x, bruit, idx, poids_fact }
}

// Betas de meme signe, alignes sur le facteur commun du bloc B ; Y = X beta + bruit.
#[allow(dead_code)]
fn cible(j: &Jeu) -> Vec<f64> {
    let formes = [1.3, 1.0, 1.2, 0.9, 1.1];
    let mut beta = vec![0.0; P];
    for k in 0..P_B {
        beta[P_A + k] = formes[k] * 1.2;
    }
    let mut y = mat_vec_ligne(&j.x, &beta);
    for i in 0..N {
        y[i] += j.bruit[i];
    }
    y
}

// Regime aligne : Y2 = facteurs_A . poids_fact * 3.0 + bruit. Les facteurs_A sont des proxis construits
// a partir de la premiere variable observee de chaque groupe du bloc A, pas les facteurs latents exacts.
#[allow(dead_code)]
fn cible_alignee(j: &Jeu) -> Vec<f64> {
    let mut y = vec![0.0; N];
    for i in 0..N {
        let mut s = 0.0;
        for g in 0..G_A {
            s += 0.8f64.sqrt() * j.x[i][g * TAILLE_G] * j.poids_fact[g];
        }
        y[i] = s * 3.0 + j.bruit[i];
    }
    y
}

// Decoupage 120 train / 40 test, standardisation ajustee sur le train uniquement (le geste du 2.1).
#[allow(dead_code)]
struct Prep {
    x_train_s: Vec<Vec<f64>>,
    x_test_s: Vec<Vec<f64>>,
    y_train_z: Vec<f64>,
    y_test_z: Vec<f64>,
    y_ecart: f64,
}

#[allow(dead_code)]
fn preparer(j: &Jeu, y: &[f64]) -> Prep {
    let x_train: Vec<Vec<f64>> = (0..N_TRAIN).map(|k| j.x[j.idx[k]].clone()).collect();
    let x_test: Vec<Vec<f64>> = (N_TRAIN..N).map(|k| j.x[j.idx[k]].clone()).collect();
    let y_train: Vec<f64> = (0..N_TRAIN).map(|k| y[j.idx[k]]).collect();
    let y_test: Vec<f64> = (N_TRAIN..N).map(|k| y[j.idx[k]]).collect();
    let (mu, sigma) = (colonnes_moyennes(&x_train), colonnes_ecarts(&x_train));
    let (y_m, y_s) = (moyenne(&y_train), ecart_type(&y_train));
    Prep {
        x_train_s: standardiser(&x_train, &mu, &sigma),
        x_test_s: standardiser(&x_test, &mu, &sigma),
        y_train_z: standardiser_vec(&y_train, y_m, y_s),
        y_test_z: standardiser_vec(&y_test, y_m, y_s),
        y_ecart: y_s,
    }
}

// ---- Algebre lineaire : tout passe par la matrice de Gram des lignes K = X X^T (n x n, ici 120 x 120),
// jamais par X^T X (p x p, singuliere des que p > n). ----
#[allow(dead_code)]
fn transpose_mat_vec(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| x.iter().zip(b).map(|(l, &v)| l[j] * v).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn gram(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let n = x.len();
    (0..n)
        .map(|i| {
            (0..n)
                .map(|k| x[i].iter().zip(&x[k]).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// Valeurs propres (decroissantes) et vecteurs propres d'une matrice symetrique par la methode de
// Jacobi cyclique. vecteurs[k][i] est la i-eme composante du k-eme vecteur propre ; le signe de chaque
// vecteur est choisi pour que sa plus grande composante en valeur absolue soit positive.
#[allow(dead_code)]
fn propres_sym(m: &[Vec<f64>]) -> (Vec<f64>, Vec<Vec<f64>>) {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut v: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|k| if i == k { 1.0 } else { 0.0 }).collect())
        .collect();
    for _balayage in 0..100 {
        let diag: f64 = (0..n).map(|i| a[i][i].abs()).fold(0.0f64, f64::max);
        let mut carres = 0.0f64;
        for i in 0..n {
            for k in (i + 1)..n {
                carres += a[i][k] * a[i][k];
            }
        }
        if carres.sqrt() <= 1e-13 * (1.0 + diag) {
            break;
        }
        let mut rotations = 0usize;
        for p in 0..n {
            for q in (p + 1)..n {
                let apq = a[p][q];
                // Pivot negligeable devant la diagonale : on l'annule sans tourner.
                if apq.abs() <= 1e-18 * (a[p][p].abs() + a[q][q].abs()) {
                    a[p][q] = 0.0;
                    a[q][p] = 0.0;
                    continue;
                }
                rotations += 1;
                let theta = (a[q][q] - a[p][p]) / (2.0 * apq);
                let t = if theta.abs() > 1e150 {
                    1.0 / (2.0 * theta)
                } else if theta >= 0.0 {
                    1.0 / (theta + (theta * theta + 1.0).sqrt())
                } else {
                    -1.0 / (-theta + (theta * theta + 1.0).sqrt())
                };
                let c = 1.0 / (t * t + 1.0).sqrt();
                let s = t * c;
                for k in 0..n {
                    if k != p && k != q {
                        let akp = a[k][p];
                        let akq = a[k][q];
                        a[k][p] = c * akp - s * akq;
                        a[p][k] = a[k][p];
                        a[k][q] = s * akp + c * akq;
                        a[q][k] = a[k][q];
                    }
                    let vkp = v[k][p];
                    let vkq = v[k][q];
                    v[k][p] = c * vkp - s * vkq;
                    v[k][q] = s * vkp + c * vkq;
                }
                a[p][p] -= t * apq;
                a[q][q] += t * apq;
                a[p][q] = 0.0;
                a[q][p] = 0.0;
            }
        }
        if rotations == 0 {
            break;
        }
    }
    let mut ordre: Vec<usize> = (0..n).collect();
    ordre.sort_by(|&i, &k| a[k][k].total_cmp(&a[i][i]));
    let valeurs: Vec<f64> = ordre.iter().map(|&i| a[i][i]).collect();
    let vecteurs: Vec<Vec<f64>> = ordre
        .iter()
        .map(|&k| {
            let colonne: Vec<f64> = v.iter().map(|l| l[k]).collect();
            let pivot = argmax(&colonne.iter().map(|x| x.abs()).collect::<Vec<f64>>());
            if colonne[pivot] < 0.0 {
                colonne.iter().map(|x| -x).collect()
            } else {
                colonne
            }
        })
        .collect();
    (valeurs, vecteurs)
}

// Rang numerique : nombre de valeurs propres au-dessus de 1e-8 fois la plus grande.
#[allow(dead_code)]
fn rang(valeurs: &[f64]) -> usize {
    let lmax = valeurs[0];
    valeurs.iter().filter(|v| **v > 1e-8 * lmax).count()
}

// OLS de norme minimale sans ordonnee, par K = X X^T : alpha = U_r L_r^{-1} U_r^T y, puis beta = X^T alpha.
// Les r valeurs propres non nulles fixent la prediction ; le noyau (p - r directions) ne la modifie pas.
#[allow(dead_code)]
fn ols_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64]) -> Vec<f64> {
    let lmax = valeurs[0];
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        if valeurs[k] > 1e-8 * lmax {
            let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
            for i in 0..n {
                alpha[i] += u[k][i] * d / valeurs[k];
            }
        }
    }
    alpha
}

// Coefficients duaux du ridge sans ordonnee : alpha = (K + lambda I)^{-1} y, dans la base des vecteurs propres.
#[allow(dead_code)]
fn ridge_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64], lambda: f64) -> Vec<f64> {
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
        for i in 0..n {
            alpha[i] += u[k][i] * d / (valeurs[k] + lambda);
        }
    }
    alpha
}

// beta = X^T alpha ; renvoie les previsions X beta sur train et test, et la norme L2 de beta.
#[allow(dead_code)]
fn predire(xtr: &[Vec<f64>], xte: &[Vec<f64>], alpha: &[f64]) -> (Vec<f64>, Vec<f64>, f64) {
    let beta = transpose_mat_vec(xtr, alpha);
    let norme = beta.iter().map(|b| b * b).sum::<f64>().sqrt();
    (mat_vec_ligne(xtr, &beta), mat_vec_ligne(xte, &beta), norme)
}

// Bornes du pli f d'une validation croisee a plis contigus (KFold sans melange).
#[allow(dead_code)]
fn bornes_pli(n: usize, plis: usize, f: usize) -> (usize, usize) {
    let (base, reste) = (n / plis, n % plis);
    let debut = f * base + reste.min(f);
    (debut, debut + base + if f < reste { 1 } else { 0 })
}

// Un pli contigu, avec la decomposition spectrale du Gram de son train : elle sert a tous les alpha du ridge.
#[allow(dead_code)]
struct PliContigu {
    xtr: Vec<Vec<f64>>,
    xva: Vec<Vec<f64>>,
    positions_val: Vec<usize>,
    valeurs: Vec<f64>,
    u: Vec<Vec<f64>>,
}

#[allow(dead_code)]
fn plis_contigus(x: &[Vec<f64>], plis: usize) -> Vec<PliContigu> {
    let n = x.len();
    (0..plis)
        .map(|f| {
            let (debut, fin) = bornes_pli(n, plis, f);
            let xtr: Vec<Vec<f64>> = (0..n)
                .filter(|&i| i < debut || i >= fin)
                .map(|i| x[i].clone())
                .collect();
            let xva: Vec<Vec<f64>> = x[debut..fin].to_vec();
            let (valeurs, u) = propres_sym(&gram(&xtr));
            PliContigu {
                xtr,
                xva,
                positions_val: (debut..fin).collect(),
                valeurs,
                u,
            }
        })
        .collect()
}

// RidgeCV(alphas, cv=plis, fit_intercept=False) : l'alpha retenu est celui qui maximise la moyenne
// des R2 de pli (le score par defaut de la validation croisee de scikit-learn). Renvoie son indice.
#[allow(dead_code)]
fn ridge_cv_sur_plis(plis_c: &[PliContigu], y: &[f64], alphas: &[f64]) -> usize {
    let mut score = vec![0.0; alphas.len()];
    for pli in plis_c {
        let ytr: Vec<f64> = (0..y.len())
            .filter(|i| !pli.positions_val.contains(i))
            .map(|i| y[i])
            .collect();
        let yva: Vec<f64> = pli.positions_val.iter().map(|&i| y[i]).collect();
        let sst: f64 = centrer(&yva).iter().map(|x| x * x).sum();
        for (j, &lam) in alphas.iter().enumerate() {
            let (_, p_va, _) = predire(
                &pli.xtr,
                &pli.xva,
                &ridge_alpha(&pli.valeurs, &pli.u, &ytr, lam),
            );
            let sse: f64 = yva.iter().zip(&p_va).map(|(t, p)| (t - p) * (t - p)).sum();
            score[j] += 1.0 - sse / sst;
        }
    }
    argmax(&score)
}

#[allow(dead_code)]
fn ridge_cv_retenu(x: &[Vec<f64>], y: &[f64], alphas: &[f64], plis: usize) -> usize {
    ridge_cv_sur_plis(&plis_contigus(x, plis), y, alphas)
}

// Composantes principales depuis la decomposition de K = X X^T : composante_k = X^T u_k / sqrt(lambda_k).
#[allow(dead_code)]
fn composantes(x: &[Vec<f64>], valeurs: &[f64], u: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let mut comps: Vec<Vec<f64>> = Vec::new();
    for k in 0..rang(valeurs) {
        let mut c = transpose_mat_vec(x, &u[k]);
        let s = valeurs[k].sqrt();
        for v in c.iter_mut() {
            *v /= s;
        }
        comps.push(c);
    }
    comps
}

// Centrage d'un echantillon sur ses propres colonnes : ce que fait PCA a l'ajustement.
#[allow(dead_code)]
fn centrer_propre(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    centrer_colonnes(x, &colonnes_moyennes(x))
}

// ACP, comme sklearn.decomposition.PCA : X est CENTRE sur ses colonnes, covariance (denominateur n),
// valeurs propres, part de variance, composantes. Le centrage ne change rien sur une matrice deja
// standardisee sur elle-meme (train complet) ; il est indispensable sur le train d'un pli, dont les
// colonnes ne sont pas de moyenne nulle.
#[allow(dead_code)]
fn acp(x: &[Vec<f64>]) -> (Vec<f64>, Vec<f64>, Vec<Vec<f64>>) {
    let xc = centrer_propre(x);
    let (valeurs, u) = propres_sym(&gram(&xc));
    let total: f64 = valeurs.iter().sum();
    let comps = composantes(&xc, &valeurs, &u);
    let ratio: Vec<f64> = valeurs.iter().map(|l| l / total).collect();
    (valeurs, ratio, comps)
}

// Variance cumulee jusqu'a la composante k comprise.
#[allow(dead_code)]
fn variance_cumulee(ratio: &[f64], k: usize) -> f64 {
    ratio[..=k].iter().sum()
}

// Scores t[i][k] = x_i . composante_k.
#[allow(dead_code)]
fn scores_composantes(x: &[Vec<f64>], comps: &[Vec<f64>]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            comps
                .iter()
                .map(|c| l.iter().zip(c).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// PCR a k composantes avec ordonnee : y = ybar + sum_{j<k} b_j t_j.
// Les scores d'une ACP sont orthogonaux deux a deux, donc b_j = (t_j . (y - ybar)) / (t_j . t_j).
#[allow(dead_code)]
fn regression_scores(
    t_tr: &[Vec<f64>],
    t_te: &[Vec<f64>],
    y: &[f64],
    k: usize,
) -> (Vec<f64>, Vec<f64>) {
    let ym = moyenne(y);
    let yc: Vec<f64> = y.iter().map(|v| v - ym).collect();
    let mut b = vec![0.0; k];
    for j in 0..k {
        let num: f64 = t_tr.iter().zip(&yc).map(|(t, &v)| t[j] * v).sum();
        let den: f64 = t_tr.iter().map(|t| t[j] * t[j]).sum();
        b[j] = num / den;
    }
    let projet = |tt: &[Vec<f64>]| -> Vec<f64> {
        tt.iter()
            .map(|t| ym + (0..k).map(|j| b[j] * t[j]).sum::<f64>())
            .collect()
    };
    (projet(t_tr), projet(t_te))
}

// RMSE test de la PCR pour k = 1..kmax (une seule ACP, une regression par k).
#[allow(dead_code)]
fn balayage_pcr(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (_, _, mut comps) = acp(xtr);
    comps.truncate(kmax);
    // PCA().fit(xtr).transform(...) : les deux echantillons sont centres sur la moyenne du train.
    let xmu = colonnes_moyennes(xtr);
    let (t_tr, t_te) = (
        scores_composantes(&centrer_colonnes(xtr, &xmu), &comps),
        scores_composantes(&centrer_colonnes(xte, &xmu), &comps),
    );
    (1..=kmax)
        .map(|k| rmse(yte, &regression_scores(&t_tr, &t_te, ytr, k).1))
        .collect()
}

// ---- PLS1 par NIPALS (l'algorithme de PLSRegression). x et y doivent etre centres sur le train.
// Renvoie (w, p, t) : poids x, loadings x et scores, ranges par composante : w[j][v], t[j][i]. ----
#[allow(dead_code)]
fn nipals(
    x: &[Vec<f64>],
    y_c: &[f64],
    kmax: usize,
) -> (Vec<Vec<f64>>, Vec<Vec<f64>>, Vec<Vec<f64>>) {
    let n = x.len();
    let nvar = x[0].len();
    let mut xk = x.to_vec();
    let mut yk = y_c.to_vec();
    let mut w_all: Vec<Vec<f64>> = Vec::new();
    let mut p_all: Vec<Vec<f64>> = Vec::new();
    let mut t_all: Vec<Vec<f64>> = Vec::new();
    for _j in 0..kmax {
        // w = Xk^T yk / ||Xk^T yk|| (direction de covariance maximale avec le reste de y), t = Xk w
        let mut w = transpose_mat_vec(&xk, &yk);
        let nrm = w.iter().map(|v| v * v).sum::<f64>().sqrt();
        for v in w.iter_mut() {
            *v /= nrm;
        }
        let t = mat_vec_ligne(&xk, &w);
        let ttt: f64 = t.iter().map(|v| v * v).sum();
        // p = Xk^T t / (t^T t) ; q = yk^T t / (t^T t)
        let mut pp = transpose_mat_vec(&xk, &t);
        for v in pp.iter_mut() {
            *v /= ttt;
        }
        let q: f64 = t.iter().zip(&yk).map(|(a, b)| a * b).sum::<f64>() / ttt;
        // Deflation : Xk <- Xk - t p^T ; yk <- yk - q t
        for i in 0..n {
            for j in 0..nvar {
                xk[i][j] -= t[i] * pp[j];
            }
            yk[i] -= q * t[i];
        }
        w_all.push(w);
        p_all.push(pp);
        t_all.push(t);
    }
    (w_all, p_all, t_all)
}

// Inverse d'une matrice carree (Gauss-Jordan, pivot partiel) ; None si un pivot est nul.
#[allow(dead_code)]
fn inverse(m: &[Vec<f64>]) -> Option<Vec<Vec<f64>>> {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut inv: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|j| if i == j { 1.0 } else { 0.0 }).collect())
        .collect();
    for col in 0..n {
        let mut piv = col;
        for r in (col + 1)..n {
            if a[r][col].abs() > a[piv][col].abs() {
                piv = r;
            }
        }
        if !a[piv][col].is_finite() || a[piv][col].abs() < 1e-14 {
            return None;
        }
        a.swap(col, piv);
        inv.swap(col, piv);
        let d = a[col][col];
        for j in 0..n {
            a[col][j] /= d;
            inv[col][j] /= d;
        }
        for r in 0..n {
            if r != col {
                let f = a[r][col];
                for j in 0..n {
                    a[r][j] -= f * a[col][j];
                    inv[r][j] -= f * inv[col][j];
                }
            }
        }
    }
    Some(inv)
}

// Rotations de la PLS, R = W (P^T W)^{-1} : elles appliquent le modele a des lignes nouvelles
// (t = X_c R). M = P^T W etant triangulaire superieure, les k premieres colonnes de R sont celles
// du modele a k composantes.
#[allow(dead_code)]
fn rotations(w: &[Vec<f64>], pmat: &[Vec<f64>], kmax: usize) -> Option<Vec<Vec<f64>>> {
    let nvar = w[0].len();
    let m: Vec<Vec<f64>> = (0..kmax)
        .map(|i| {
            (0..kmax)
                .map(|j| pmat[i].iter().zip(&w[j]).map(|(a, b)| a * b).sum())
                .collect()
        })
        .collect();
    let inv = inverse(&m)?;
    let r: Vec<Vec<f64>> = (0..nvar)
        .map(|v| {
            (0..kmax)
                .map(|j| (0..kmax).map(|i| w[i][v] * inv[i][j]).sum())
                .collect()
        })
        .collect();
    Some(r)
}

// Poids cotes variables du modele PLS a k = 1..kmax composantes : betas[k - 1] pour k composantes.
// b_j = (t_j . y_c) / (t_j . t_j), puis beta_k = sum_{j<k} b_j R[:, j].
#[allow(dead_code)]
fn pls_poids(
    w: &[Vec<f64>],
    pmat: &[Vec<f64>],
    t: &[Vec<f64>],
    y_c: &[f64],
) -> Option<Vec<Vec<f64>>> {
    let kmax = t.len();
    let n = t[0].len();
    let nvar = w[0].len();
    let r = rotations(w, pmat, kmax)?;
    let mut betas: Vec<Vec<f64>> = Vec::new();
    let mut beta = vec![0.0; nvar];
    for j in 0..kmax {
        let num: f64 = (0..n).map(|i| t[j][i] * y_c[i]).sum();
        let den: f64 = t[j].iter().map(|v| v * v).sum();
        let b = num / den;
        for v in 0..nvar {
            beta[v] += b * r[v][j];
        }
        betas.push(beta.clone());
    }
    Some(betas)
}

// Prevision PLS pour des lignes deja centrees : y = ybar + x_c . beta.
#[allow(dead_code)]
fn pls_predire(x_c: &[Vec<f64>], y_moy: f64, beta: &[f64]) -> Vec<f64> {
    x_c.iter()
        .map(|l| y_moy + l.iter().zip(beta).map(|(a, b)| a * b).sum::<f64>())
        .collect()
}

// RMSE test de la PLS pour k = 1..kmax (un seul NIPALS, un vecteur de poids par k) ;
// NaN si les rotations ne sont pas calculables.
#[allow(dead_code)]
fn balayage_pls(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (xmu, y_moy) = (colonnes_moyennes(xtr), moyenne(ytr));
    let yc = centrer(ytr);
    let (w, pmat, t) = nipals(&centrer_colonnes(xtr, &xmu), &yc, kmax);
    let betas = match pls_poids(&w, &pmat, &t, &yc) {
        Some(b) => b,
        None => return vec![f64::NAN; kmax],
    };
    let xte_c = centrer_colonnes(xte, &xmu);
    (1..=kmax)
        .map(|k| rmse(yte, &pls_predire(&xte_c, y_moy, &betas[k - 1])))
        .collect()
}

// RMSE test de la PLS ajustee avec exactement k composantes.
#[allow(dead_code)]
fn rmse_pls_k(xtr: &[Vec<f64>], ytr: &[f64], xte: &[Vec<f64>], yte: &[f64], k: usize) -> f64 {
    match balayage_pls(xtr, ytr, xte, yte, k).last() {
        Some(r) => *r,
        None => f64::NAN,
    }
}

// ---- Figures en texte : le noyau evcxr n'affiche pas de graphique. ----

// Courbes RMSE test de la PCR et de la PLS selon k, ligne du meilleur ridge sur ce test, colonne de la
// composante la plus chargee en bloc B. Figure en texte : une marque par point. L'axe vertical part du
// multiple de 0,15 inferieur ou egal a la plus petite RMSE tracee et monte jusqu'au multiple du pas
// superieur ou egal a la plus grande : aucune valeur finie ne sort du cadre (un cadre fixe
// 1,00 -> 0,20 masquait ici le premier point de la PCR et confondait la ligne du ridge avec la PLS).
// Le pas (0,025, puis 0,05, puis 0,1) est le plus fin qui garde la trame sous 40 lignes.
// Renvoie la figure, la borne basse, la borne haute et le pas d'une ligne.
#[allow(dead_code)]
fn figure_courbes(pcr: &[f64], pls: &[f64], ridge: f64, i_max_b: usize) -> (String, f64, f64, f64) {
    let (lignes_max, droite) = (40usize, 8usize);
    // Chaque k occupe deux colonnes ; la premiere courbe commence a la colonne 4, et huit colonnes
    // restent libres a droite du dernier point pour que la colonne du bloc B se lise comme une marque.
    let colonnes = (4 + 2 * pcr.len().max(1) + droite).max(64);
    let colonne_k = |k: usize| (4 + 2 * k.saturating_sub(1)).min(colonnes - 1);
    let mut y_haut = f64::NEG_INFINITY;
    let mut y_bas = f64::INFINITY;
    for &r in pcr.iter().chain(pls.iter()).chain([ridge].iter()) {
        if r.is_finite() {
            y_haut = y_haut.max(r);
            y_bas = y_bas.min(r);
        }
    }
    if !(y_haut.is_finite() && y_bas.is_finite()) {
        y_haut = 1.0;
        y_bas = 0.0;
    }
    // Borne basse : multiple de 0,15, comme les graduations de l'axe.
    y_bas = (y_bas / 0.15).floor() * 0.15;
    let mut pas = 0.1f64;
    for candidat in [0.025f64, 0.05, 0.1] {
        if ((y_haut / candidat).ceil() * candidat - y_bas) / candidat <= (lignes_max - 1) as f64 {
            pas = candidat;
            break;
        }
    }
    y_haut = ((y_haut / pas).ceil() * pas).max(y_bas + pas);
    let lignes = (((y_haut - y_bas) / pas).round() as usize).min(lignes_max - 1) + 1;
    let ligne_rmse = |r: f64| -> usize {
        let brute = ((y_haut - r) / (y_haut - y_bas) * (lignes as f64 - 1.0)).round();
        if !brute.is_finite() {
            lignes - 1
        } else {
            (brute.max(0.0) as usize).min(lignes - 1)
        }
    };
    let mut trame: Vec<Vec<char>> = vec![vec![' '; colonnes]; lignes];
    // D'abord la ligne du ridge, puis les courbes par-dessus : un point qui tombe sur la ligne du
    // ridge s'ecrit '*', pour que la rencontre reste lisible.
    if ridge.is_finite() {
        let l = ligne_rmse(ridge);
        for c in colonne_k(1)..=colonne_k(pcr.len()) {
            trame[l][c] = '-';
        }
    }
    for (courbe, marque) in [(pcr, '.'), (pls, 'x')] {
        for (k, &r) in courbe.iter().enumerate() {
            if r.is_finite() {
                let c = colonne_k(k + 1);
                let l = ligne_rmse(r);
                trame[l][c] = if trame[l][c] == '-' { '*' } else { marque };
            }
        }
    }
    let c_b = colonne_k(i_max_b + 1);
    for l in 0..lignes {
        if trame[l][c_b] == ' ' {
            trame[l][c_b] = ':';
        }
    }
    let mut sortie = String::new();
    for (l, ligne) in trame.iter().enumerate() {
        let y = y_bas + (lignes - 1 - l) as f64 * pas;
        let graduation = (y / 0.15 - (y / 0.15).round()).abs() < 1e-9;
        if graduation {
            sortie.push_str(&format!("{:4.2} |{}\n", y, ligne.iter().collect::<String>()));
        } else {
            sortie.push_str(&format!("     |{}\n", ligne.iter().collect::<String>()));
        }
    }
    // Axe des k : un tiret sous chaque position, une graduation tous les 5 k.
    let mut graduation: Vec<char> = vec![' '; colonnes];
    for c in colonne_k(1)..=colonne_k(pcr.len()) {
        graduation[c] = '-';
    }
    for k in (5..=pcr.len()).step_by(5) {
        graduation[colonne_k(k)] = '+';
    }
    sortie.push_str(&format!("     +{}\n", graduation.iter().collect::<String>()));
    let mut legendek: Vec<char> = vec![' '; colonnes];
    for k in (5..=pcr.len()).step_by(5) {
        for (d, ch) in format!("{}", k).chars().enumerate() {
            let c = colonne_k(k) + d;
            if c < colonnes {
                legendek[c] = ch;
            }
        }
    }
    sortie.push_str(&format!("      {}\n", legendek.iter().collect::<String>()));
    (sortie, y_bas, y_haut, pas)
}

// Profil d'un vecteur de poids, une barre par lot : bloc A en lots de `taille_lot` variables (le plus
// fort |loading| du lot), bloc B variable par variable. `echelle` est la valeur qui occupe toute la
// largeur : la meme pour deux panneaux les rend comparables.
#[allow(dead_code)]
fn profil_loadings(poids: &[f64], debut_b: usize, taille_lot: usize, echelle: f64) -> String {
    const LARGEUR: usize = 40;
    let barre = |debut: usize, fin: usize| -> String {
        let m: f64 = (debut..fin).map(|v| poids[v].abs()).fold(0.0f64, f64::max);
        // Largeur ramenee a 0..LARGEUR ; une echelle nulle ou non finie donne une barre vide.
        let n = if echelle.is_finite() && echelle > 0.0 {
            (((m / echelle) * LARGEUR as f64).round().max(0.0) as usize).min(LARGEUR)
        } else {
            0
        };
        format!("{:<40} {:.4}", "#".repeat(n), m)
    };
    let mut sortie = String::new();
    for debut in (0..debut_b).step_by(taille_lot.max(1)) {
        let fin = (debut + taille_lot).min(debut_b);
        sortie.push_str(&format!("A {:3}-{:3} |{}\n", debut, fin - 1, barre(debut, fin)));
    }
    for v in debut_b..poids.len() {
        sortie.push_str(&format!("B {:3}     |{}\n", v, barre(v, v + 1)));
    }
    sortie
}

// ---- 8. Loadings : ou chaque methode met ses poids ----
let j = jeu(GRAINE);
let p = preparer(&j, &cible(&j));
let (_, _, comps) = acp(&p.x_train_s);
// x_weights_[:, 0] ne depend pas du nombre de composantes ajustees : un seul pas de NIPALS suffit.
let xmu = colonnes_moyennes(&p.x_train_s);
let (w1, _, _) = nipals(
    &centrer_colonnes(&p.x_train_s, &xmu),
    &centrer(&p.y_train_z),
    1,
);
let echelle = comps[0]
    .iter()
    .chain(w1[0].iter())
    .map(|v| v.abs())
    .fold(0.0f64, f64::max);
println!("1re composante PCA : |loading| par variable");
println!("{}", profil_loadings(&comps[0], P_A, 10, echelle));
println!("1re composante PLS : |loading| par variable");
println!("{}", profil_loadings(&w1[0], P_A, 10, echelle));
println!(
    "variables 0-199 : bloc A sans signal | 200-204 : bloc B signal ; meme echelle sur les deux panneaux ({:.3} = 40 caracteres)",
    echelle
);
let (poids_a_pca, poids_b_pca) = masses_blocs(&comps[0], P_A);
let (poids_a_pls, poids_b_pls) = masses_blocs(&w1[0], P_A);
println!(
    "PCA u1 : {:.2} de masse |loading| sur bloc A vs {:.2} sur bloc B",
    poids_a_pca, poids_b_pca
);
println!(
    "PLS w1 : {:.2} de masse |loading| sur bloc A vs {:.2} sur bloc B",
    poids_a_pls, poids_b_pls
);
println!(
    "part de la masse PLS w1 sur les 5 variables du signal : {:.0}%",
    100.0 * poids_b_pls / (poids_a_pls + poids_b_pls)
);

1re composante PCA : |loading| par variable


A   0-  9 |#########                                0.0813


A  10- 19 |##########                               0.0974


A  20- 29 |#########                                0.0896


A  30- 39 |#############                            0.1250


A  40- 49 |######                                   0.0572


... (46 outputs left out)

variables 0-199 : bloc A sans signal | 200-204 : bloc B signal ; meme echelle sur les deux panneaux (0.381 = 40 caracteres)


PCA u1 : 11.20 de masse |loading| sur bloc A vs 0.34 sur bloc B


PLS w1 : 6.59 de masse |loading| sur bloc A vs 1.85 sur bloc B


part de la masse PLS w1 sur les 5 variables du signal : 22%


### Lecture du résultat

La première composante ACP répartit sa masse presque uniformément sur le bloc A (11.20 contre 0.34 sur le bloc B — les 200 variables du leurre). La première composante PLS concentre 1.85 de masse sur les 5 variables du signal, soit **22% de sa masse sur 2.4% des variables** — environ 9× leur part uniforme ; en norme carrée (la direction elle-même), la part du bloc B monte à 68.2%. Le w₁ garde néanmoins une composante sur le bloc A : la PLS maximise la *covariance*, qui récompense aussi les directions de forte variance de X — le tri par Cov n'est pas un tri par corrélation pure.

## 9. Comparaison finale — verdict par régime

Deux régimes, mêmes split et graine : le régime **leurre** (notre bloc A dominant en variance, signal en variance mineure) et un régime **aligné** où la cible vit cette fois dans les directions de variance dominante (une combinaison des facteurs du bloc A). La conclusion n'est pas « PLS gagne toujours » mais un verdict conditionné.

Les quantités `facteurs_A` du code sont des proxies construits à partir de variables observées, et non les facteurs latents exacts du générateur. Les valeurs `k` obtenues sur le régime leurre sont réutilisées pour le régime aligné : le second régime ne bénéficie pas d'une sélection indépendante de `k`.

In [12]:
// Cellule autonome : prelude (RNG, donnees, algebre) redefini ici, puis le calcul de la section.
// ---- Generateur pseudo-alatoire maison : xorshift64* + Box-Muller (aucun crate).
// La suite n'est pas celle de NumPy : les tirages different, les valeurs chiffrees sont celles de ce carnet.
#[allow(dead_code)]
struct Rng {
    etat: u64,
}

#[allow(dead_code)]
impl Rng {
    fn new(graine: u64) -> Rng {
        let mut etat = graine ^ 0x9E3779B97F4A7C15;
        if etat == 0 {
            etat = 0x2545F4914F6CDD1D;
        }
        Rng { etat }
    }

    fn brut(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x >> 12;
        x ^= x << 25;
        x ^= x >> 27;
        self.etat = x;
        x.wrapping_mul(0x2545F4914F6CDD1D)
    }

    // Uniforme sur [0, 1), 53 bits de mantisse.
    fn uniforme(&mut self) -> f64 {
        ((self.brut() >> 11) as f64 + 0.5) / 9007199254740992.0
    }

    // Normale centree reduite, methode de Box-Muller.
    fn normale(&mut self) -> f64 {
        const TAU: f64 = 6.283185307179586;
        loop {
            let u1 = self.uniforme();
            let u2 = self.uniforme();
            if u1 > 1e-300 {
                return (-2.0 * u1.ln()).sqrt() * (TAU * u2).cos();
            }
        }
    }

    // Entier uniforme dans 0..n, par rejet (pas le biais du modulo).
    fn indice(&mut self, n: usize) -> usize {
        let n = n as u64;
        let limite = u64::MAX - (u64::MAX % n);
        loop {
            let t = self.brut();
            if t < limite {
                return (t % n) as usize;
            }
        }
    }
}

// Permutation aleatoire (Fisher-Yates).
#[allow(dead_code)]
fn permutation(rng: &mut Rng, n: usize) -> Vec<usize> {
    let mut idx: Vec<usize> = (0..n).collect();
    for i in (1..n).rev() {
        let j = rng.indice(i + 1);
        idx.swap(i, j);
    }
    idx
}

// ---- Statistiques de base (ecarts-types au denominateur n, comme NumPy). ----
#[allow(dead_code)]
fn moyenne(v: &[f64]) -> f64 {
    v.iter().sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn variance(v: &[f64]) -> f64 {
    let m = moyenne(v);
    v.iter().map(|x| (x - m) * (x - m)).sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn ecart_type(v: &[f64]) -> f64 {
    variance(v).sqrt()
}

// Matrices stockees ligne par ligne : x[i][j].
#[allow(dead_code)]
fn mat_vec_ligne(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    x.iter()
        .map(|l| l.iter().zip(b).map(|(a, c)| a * c).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn colonnes_moyennes(x: &[Vec<f64>]) -> Vec<f64> {
    let n = x.len();
    (0..x[0].len())
        .map(|j| x.iter().map(|l| l[j]).sum::<f64>() / n as f64)
        .collect()
}

#[allow(dead_code)]
fn colonnes_ecarts(x: &[Vec<f64>]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| ecart_type(&x.iter().map(|l| l[j]).collect::<Vec<f64>>()))
        .collect()
}

#[allow(dead_code)]
fn standardiser(x: &[Vec<f64>], mu: &[f64], sigma: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            l.iter()
                .zip(mu.iter().zip(sigma))
                .map(|(a, (m, s))| (a - m) / s)
                .collect()
        })
        .collect()
}

#[allow(dead_code)]
fn standardiser_vec(v: &[f64], m: f64, s: f64) -> Vec<f64> {
    v.iter().map(|x| (x - m) / s).collect()
}

#[allow(dead_code)]
fn centrer(v: &[f64]) -> Vec<f64> {
    let m = moyenne(v);
    v.iter().map(|x| x - m).collect()
}

#[allow(dead_code)]
fn centrer_colonnes(x: &[Vec<f64>], mu: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| l.iter().zip(mu).map(|(a, m)| a - m).collect())
        .collect()
}

#[allow(dead_code)]
fn correlation(a: &[f64], b: &[f64]) -> f64 {
    let (ca, cb) = (centrer(a), centrer(b));
    let num: f64 = ca.iter().zip(&cb).map(|(x, y)| x * y).sum();
    let den: f64 = ca.iter().map(|x| x * x).sum::<f64>().sqrt()
        * cb.iter().map(|y| y * y).sum::<f64>().sqrt();
    num / den
}

#[allow(dead_code)]
fn rmse(a: &[f64], b: &[f64]) -> f64 {
    (a.iter().zip(b).map(|(x, y)| (x - y) * (x - y)).sum::<f64>() / a.len() as f64).sqrt()
}

// Indice du minimum (le premier en cas d'egalite), comme np.argmin.
#[allow(dead_code)]
fn argmin(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] < v[m] {
            m = k;
        }
    }
    m
}

// Indice du maximum (le premier en cas d'egalite), comme np.argmax.
#[allow(dead_code)]
fn argmax(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] > v[m] {
            m = k;
        }
    }
    m
}

#[allow(dead_code)]
fn logspace(a: f64, b: f64, n: usize) -> Vec<f64> {
    (0..n)
        .map(|i| 10f64.powf(a + (b - a) * i as f64 / (n as f64 - 1.0)))
        .collect()
}

// Colonne j d'une matrice stockee ligne par ligne.
#[allow(dead_code)]
fn colonne(x: &[Vec<f64>], j: usize) -> Vec<f64> {
    x.iter().map(|l| l[j]).collect()
}

// Somme des valeurs absolues sur deux blocs de variables : (0..debut_b, debut_b..fin).
#[allow(dead_code)]
fn masses_blocs(v: &[f64], debut_b: usize) -> (f64, f64) {
    let a: f64 = v[..debut_b].iter().map(|x| x.abs()).sum();
    let b: f64 = v[debut_b..].iter().map(|x| x.abs()).sum();
    (a, b)
}

// ---- Donnees : meme construction que la source (blocs A et B, cible, decoupage, standardisation). ----
#[allow(dead_code)]
const N: usize = 160;
#[allow(dead_code)]
const G_A: usize = 40;
#[allow(dead_code)]
const TAILLE_G: usize = 5;
#[allow(dead_code)]
const P_A: usize = 200;
#[allow(dead_code)]
const P_B: usize = 5;
#[allow(dead_code)]
const P: usize = 205;
#[allow(dead_code)]
const N_TRAIN: usize = 120;
#[allow(dead_code)]
const GRAINE: u64 = 42;

// Variables en groupes ; dans chaque groupe, corr(x_i, x_j) = rho_pair via facteur commun.
#[allow(dead_code)]
fn bloc_facteurs(rng: &mut Rng, n_vars: usize, n_groupes: usize, rho_pair: f64) -> Vec<Vec<f64>> {
    let mut x = vec![vec![0.0; n_vars]; N];
    for g in 0..n_groupes {
        let f: Vec<f64> = (0..N).map(|_| rho_pair.sqrt() * rng.normale()).collect();
        for j in 0..TAILLE_G {
            for i in 0..N {
                x[i][g * TAILLE_G + j] = f[i] + (1.0 - rho_pair).sqrt() * rng.normale();
            }
        }
    }
    x
}

// Un tirage complet depuis la graine : les deux blocs, le bruit, le decoupage et les poids du regime aligne.
#[allow(dead_code)]
struct Jeu {
    x: Vec<Vec<f64>>,
    bruit: Vec<f64>,
    idx: Vec<usize>,
    poids_fact: Vec<f64>,
}

// 40 directions de variance dominante sans signal (bloc A), 1 direction plus modeste generatrice de Y (bloc B).
#[allow(dead_code)]
fn jeu(graine: u64) -> Jeu {
    let mut rng = Rng::new(graine);
    let mut x = bloc_facteurs(&mut rng, P_A, G_A, 0.8);
    let xb = bloc_facteurs(&mut rng, P_B, 1, 0.6);
    for i in 0..N {
        x[i].extend(xb[i].iter().copied());
    }
    let bruit: Vec<f64> = (0..N).map(|_| rng.normale()).collect();
    let idx = permutation(&mut rng, N);
    let poids_fact: Vec<f64> = (0..G_A).map(|_| rng.normale() / (G_A as f64).sqrt()).collect();
    Jeu { x, bruit, idx, poids_fact }
}

// Betas de meme signe, alignes sur le facteur commun du bloc B ; Y = X beta + bruit.
#[allow(dead_code)]
fn cible(j: &Jeu) -> Vec<f64> {
    let formes = [1.3, 1.0, 1.2, 0.9, 1.1];
    let mut beta = vec![0.0; P];
    for k in 0..P_B {
        beta[P_A + k] = formes[k] * 1.2;
    }
    let mut y = mat_vec_ligne(&j.x, &beta);
    for i in 0..N {
        y[i] += j.bruit[i];
    }
    y
}

// Regime aligne : Y2 = facteurs_A . poids_fact * 3.0 + bruit. Les facteurs_A sont des proxis construits
// a partir de la premiere variable observee de chaque groupe du bloc A, pas les facteurs latents exacts.
#[allow(dead_code)]
fn cible_alignee(j: &Jeu) -> Vec<f64> {
    let mut y = vec![0.0; N];
    for i in 0..N {
        let mut s = 0.0;
        for g in 0..G_A {
            s += 0.8f64.sqrt() * j.x[i][g * TAILLE_G] * j.poids_fact[g];
        }
        y[i] = s * 3.0 + j.bruit[i];
    }
    y
}

// Decoupage 120 train / 40 test, standardisation ajustee sur le train uniquement (le geste du 2.1).
#[allow(dead_code)]
struct Prep {
    x_train_s: Vec<Vec<f64>>,
    x_test_s: Vec<Vec<f64>>,
    y_train_z: Vec<f64>,
    y_test_z: Vec<f64>,
    y_ecart: f64,
}

#[allow(dead_code)]
fn preparer(j: &Jeu, y: &[f64]) -> Prep {
    let x_train: Vec<Vec<f64>> = (0..N_TRAIN).map(|k| j.x[j.idx[k]].clone()).collect();
    let x_test: Vec<Vec<f64>> = (N_TRAIN..N).map(|k| j.x[j.idx[k]].clone()).collect();
    let y_train: Vec<f64> = (0..N_TRAIN).map(|k| y[j.idx[k]]).collect();
    let y_test: Vec<f64> = (N_TRAIN..N).map(|k| y[j.idx[k]]).collect();
    let (mu, sigma) = (colonnes_moyennes(&x_train), colonnes_ecarts(&x_train));
    let (y_m, y_s) = (moyenne(&y_train), ecart_type(&y_train));
    Prep {
        x_train_s: standardiser(&x_train, &mu, &sigma),
        x_test_s: standardiser(&x_test, &mu, &sigma),
        y_train_z: standardiser_vec(&y_train, y_m, y_s),
        y_test_z: standardiser_vec(&y_test, y_m, y_s),
        y_ecart: y_s,
    }
}

// ---- Algebre lineaire : tout passe par la matrice de Gram des lignes K = X X^T (n x n, ici 120 x 120),
// jamais par X^T X (p x p, singuliere des que p > n). ----
#[allow(dead_code)]
fn transpose_mat_vec(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| x.iter().zip(b).map(|(l, &v)| l[j] * v).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn gram(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let n = x.len();
    (0..n)
        .map(|i| {
            (0..n)
                .map(|k| x[i].iter().zip(&x[k]).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// Valeurs propres (decroissantes) et vecteurs propres d'une matrice symetrique par la methode de
// Jacobi cyclique. vecteurs[k][i] est la i-eme composante du k-eme vecteur propre ; le signe de chaque
// vecteur est choisi pour que sa plus grande composante en valeur absolue soit positive.
#[allow(dead_code)]
fn propres_sym(m: &[Vec<f64>]) -> (Vec<f64>, Vec<Vec<f64>>) {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut v: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|k| if i == k { 1.0 } else { 0.0 }).collect())
        .collect();
    for _balayage in 0..100 {
        let diag: f64 = (0..n).map(|i| a[i][i].abs()).fold(0.0f64, f64::max);
        let mut carres = 0.0f64;
        for i in 0..n {
            for k in (i + 1)..n {
                carres += a[i][k] * a[i][k];
            }
        }
        if carres.sqrt() <= 1e-13 * (1.0 + diag) {
            break;
        }
        let mut rotations = 0usize;
        for p in 0..n {
            for q in (p + 1)..n {
                let apq = a[p][q];
                // Pivot negligeable devant la diagonale : on l'annule sans tourner.
                if apq.abs() <= 1e-18 * (a[p][p].abs() + a[q][q].abs()) {
                    a[p][q] = 0.0;
                    a[q][p] = 0.0;
                    continue;
                }
                rotations += 1;
                let theta = (a[q][q] - a[p][p]) / (2.0 * apq);
                let t = if theta.abs() > 1e150 {
                    1.0 / (2.0 * theta)
                } else if theta >= 0.0 {
                    1.0 / (theta + (theta * theta + 1.0).sqrt())
                } else {
                    -1.0 / (-theta + (theta * theta + 1.0).sqrt())
                };
                let c = 1.0 / (t * t + 1.0).sqrt();
                let s = t * c;
                for k in 0..n {
                    if k != p && k != q {
                        let akp = a[k][p];
                        let akq = a[k][q];
                        a[k][p] = c * akp - s * akq;
                        a[p][k] = a[k][p];
                        a[k][q] = s * akp + c * akq;
                        a[q][k] = a[k][q];
                    }
                    let vkp = v[k][p];
                    let vkq = v[k][q];
                    v[k][p] = c * vkp - s * vkq;
                    v[k][q] = s * vkp + c * vkq;
                }
                a[p][p] -= t * apq;
                a[q][q] += t * apq;
                a[p][q] = 0.0;
                a[q][p] = 0.0;
            }
        }
        if rotations == 0 {
            break;
        }
    }
    let mut ordre: Vec<usize> = (0..n).collect();
    ordre.sort_by(|&i, &k| a[k][k].total_cmp(&a[i][i]));
    let valeurs: Vec<f64> = ordre.iter().map(|&i| a[i][i]).collect();
    let vecteurs: Vec<Vec<f64>> = ordre
        .iter()
        .map(|&k| {
            let colonne: Vec<f64> = v.iter().map(|l| l[k]).collect();
            let pivot = argmax(&colonne.iter().map(|x| x.abs()).collect::<Vec<f64>>());
            if colonne[pivot] < 0.0 {
                colonne.iter().map(|x| -x).collect()
            } else {
                colonne
            }
        })
        .collect();
    (valeurs, vecteurs)
}

// Rang numerique : nombre de valeurs propres au-dessus de 1e-8 fois la plus grande.
#[allow(dead_code)]
fn rang(valeurs: &[f64]) -> usize {
    let lmax = valeurs[0];
    valeurs.iter().filter(|v| **v > 1e-8 * lmax).count()
}

// OLS de norme minimale sans ordonnee, par K = X X^T : alpha = U_r L_r^{-1} U_r^T y, puis beta = X^T alpha.
// Les r valeurs propres non nulles fixent la prediction ; le noyau (p - r directions) ne la modifie pas.
#[allow(dead_code)]
fn ols_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64]) -> Vec<f64> {
    let lmax = valeurs[0];
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        if valeurs[k] > 1e-8 * lmax {
            let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
            for i in 0..n {
                alpha[i] += u[k][i] * d / valeurs[k];
            }
        }
    }
    alpha
}

// Coefficients duaux du ridge sans ordonnee : alpha = (K + lambda I)^{-1} y, dans la base des vecteurs propres.
#[allow(dead_code)]
fn ridge_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64], lambda: f64) -> Vec<f64> {
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
        for i in 0..n {
            alpha[i] += u[k][i] * d / (valeurs[k] + lambda);
        }
    }
    alpha
}

// beta = X^T alpha ; renvoie les previsions X beta sur train et test, et la norme L2 de beta.
#[allow(dead_code)]
fn predire(xtr: &[Vec<f64>], xte: &[Vec<f64>], alpha: &[f64]) -> (Vec<f64>, Vec<f64>, f64) {
    let beta = transpose_mat_vec(xtr, alpha);
    let norme = beta.iter().map(|b| b * b).sum::<f64>().sqrt();
    (mat_vec_ligne(xtr, &beta), mat_vec_ligne(xte, &beta), norme)
}

// Bornes du pli f d'une validation croisee a plis contigus (KFold sans melange).
#[allow(dead_code)]
fn bornes_pli(n: usize, plis: usize, f: usize) -> (usize, usize) {
    let (base, reste) = (n / plis, n % plis);
    let debut = f * base + reste.min(f);
    (debut, debut + base + if f < reste { 1 } else { 0 })
}

// Un pli contigu, avec la decomposition spectrale du Gram de son train : elle sert a tous les alpha du ridge.
#[allow(dead_code)]
struct PliContigu {
    xtr: Vec<Vec<f64>>,
    xva: Vec<Vec<f64>>,
    positions_val: Vec<usize>,
    valeurs: Vec<f64>,
    u: Vec<Vec<f64>>,
}

#[allow(dead_code)]
fn plis_contigus(x: &[Vec<f64>], plis: usize) -> Vec<PliContigu> {
    let n = x.len();
    (0..plis)
        .map(|f| {
            let (debut, fin) = bornes_pli(n, plis, f);
            let xtr: Vec<Vec<f64>> = (0..n)
                .filter(|&i| i < debut || i >= fin)
                .map(|i| x[i].clone())
                .collect();
            let xva: Vec<Vec<f64>> = x[debut..fin].to_vec();
            let (valeurs, u) = propres_sym(&gram(&xtr));
            PliContigu {
                xtr,
                xva,
                positions_val: (debut..fin).collect(),
                valeurs,
                u,
            }
        })
        .collect()
}

// RidgeCV(alphas, cv=plis, fit_intercept=False) : l'alpha retenu est celui qui maximise la moyenne
// des R2 de pli (le score par defaut de la validation croisee de scikit-learn). Renvoie son indice.
#[allow(dead_code)]
fn ridge_cv_sur_plis(plis_c: &[PliContigu], y: &[f64], alphas: &[f64]) -> usize {
    let mut score = vec![0.0; alphas.len()];
    for pli in plis_c {
        let ytr: Vec<f64> = (0..y.len())
            .filter(|i| !pli.positions_val.contains(i))
            .map(|i| y[i])
            .collect();
        let yva: Vec<f64> = pli.positions_val.iter().map(|&i| y[i]).collect();
        let sst: f64 = centrer(&yva).iter().map(|x| x * x).sum();
        for (j, &lam) in alphas.iter().enumerate() {
            let (_, p_va, _) = predire(
                &pli.xtr,
                &pli.xva,
                &ridge_alpha(&pli.valeurs, &pli.u, &ytr, lam),
            );
            let sse: f64 = yva.iter().zip(&p_va).map(|(t, p)| (t - p) * (t - p)).sum();
            score[j] += 1.0 - sse / sst;
        }
    }
    argmax(&score)
}

#[allow(dead_code)]
fn ridge_cv_retenu(x: &[Vec<f64>], y: &[f64], alphas: &[f64], plis: usize) -> usize {
    ridge_cv_sur_plis(&plis_contigus(x, plis), y, alphas)
}

// Composantes principales depuis la decomposition de K = X X^T : composante_k = X^T u_k / sqrt(lambda_k).
#[allow(dead_code)]
fn composantes(x: &[Vec<f64>], valeurs: &[f64], u: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let mut comps: Vec<Vec<f64>> = Vec::new();
    for k in 0..rang(valeurs) {
        let mut c = transpose_mat_vec(x, &u[k]);
        let s = valeurs[k].sqrt();
        for v in c.iter_mut() {
            *v /= s;
        }
        comps.push(c);
    }
    comps
}

// Centrage d'un echantillon sur ses propres colonnes : ce que fait PCA a l'ajustement.
#[allow(dead_code)]
fn centrer_propre(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    centrer_colonnes(x, &colonnes_moyennes(x))
}

// ACP, comme sklearn.decomposition.PCA : X est CENTRE sur ses colonnes, covariance (denominateur n),
// valeurs propres, part de variance, composantes. Le centrage ne change rien sur une matrice deja
// standardisee sur elle-meme (train complet) ; il est indispensable sur le train d'un pli, dont les
// colonnes ne sont pas de moyenne nulle.
#[allow(dead_code)]
fn acp(x: &[Vec<f64>]) -> (Vec<f64>, Vec<f64>, Vec<Vec<f64>>) {
    let xc = centrer_propre(x);
    let (valeurs, u) = propres_sym(&gram(&xc));
    let total: f64 = valeurs.iter().sum();
    let comps = composantes(&xc, &valeurs, &u);
    let ratio: Vec<f64> = valeurs.iter().map(|l| l / total).collect();
    (valeurs, ratio, comps)
}

// Variance cumulee jusqu'a la composante k comprise.
#[allow(dead_code)]
fn variance_cumulee(ratio: &[f64], k: usize) -> f64 {
    ratio[..=k].iter().sum()
}

// Scores t[i][k] = x_i . composante_k.
#[allow(dead_code)]
fn scores_composantes(x: &[Vec<f64>], comps: &[Vec<f64>]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            comps
                .iter()
                .map(|c| l.iter().zip(c).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// PCR a k composantes avec ordonnee : y = ybar + sum_{j<k} b_j t_j.
// Les scores d'une ACP sont orthogonaux deux a deux, donc b_j = (t_j . (y - ybar)) / (t_j . t_j).
#[allow(dead_code)]
fn regression_scores(
    t_tr: &[Vec<f64>],
    t_te: &[Vec<f64>],
    y: &[f64],
    k: usize,
) -> (Vec<f64>, Vec<f64>) {
    let ym = moyenne(y);
    let yc: Vec<f64> = y.iter().map(|v| v - ym).collect();
    let mut b = vec![0.0; k];
    for j in 0..k {
        let num: f64 = t_tr.iter().zip(&yc).map(|(t, &v)| t[j] * v).sum();
        let den: f64 = t_tr.iter().map(|t| t[j] * t[j]).sum();
        b[j] = num / den;
    }
    let projet = |tt: &[Vec<f64>]| -> Vec<f64> {
        tt.iter()
            .map(|t| ym + (0..k).map(|j| b[j] * t[j]).sum::<f64>())
            .collect()
    };
    (projet(t_tr), projet(t_te))
}

// RMSE test de la PCR pour k = 1..kmax (une seule ACP, une regression par k).
#[allow(dead_code)]
fn balayage_pcr(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (_, _, mut comps) = acp(xtr);
    comps.truncate(kmax);
    // PCA().fit(xtr).transform(...) : les deux echantillons sont centres sur la moyenne du train.
    let xmu = colonnes_moyennes(xtr);
    let (t_tr, t_te) = (
        scores_composantes(&centrer_colonnes(xtr, &xmu), &comps),
        scores_composantes(&centrer_colonnes(xte, &xmu), &comps),
    );
    (1..=kmax)
        .map(|k| rmse(yte, &regression_scores(&t_tr, &t_te, ytr, k).1))
        .collect()
}

// ---- PLS1 par NIPALS (l'algorithme de PLSRegression). x et y doivent etre centres sur le train.
// Renvoie (w, p, t) : poids x, loadings x et scores, ranges par composante : w[j][v], t[j][i]. ----
#[allow(dead_code)]
fn nipals(
    x: &[Vec<f64>],
    y_c: &[f64],
    kmax: usize,
) -> (Vec<Vec<f64>>, Vec<Vec<f64>>, Vec<Vec<f64>>) {
    let n = x.len();
    let nvar = x[0].len();
    let mut xk = x.to_vec();
    let mut yk = y_c.to_vec();
    let mut w_all: Vec<Vec<f64>> = Vec::new();
    let mut p_all: Vec<Vec<f64>> = Vec::new();
    let mut t_all: Vec<Vec<f64>> = Vec::new();
    for _j in 0..kmax {
        // w = Xk^T yk / ||Xk^T yk|| (direction de covariance maximale avec le reste de y), t = Xk w
        let mut w = transpose_mat_vec(&xk, &yk);
        let nrm = w.iter().map(|v| v * v).sum::<f64>().sqrt();
        for v in w.iter_mut() {
            *v /= nrm;
        }
        let t = mat_vec_ligne(&xk, &w);
        let ttt: f64 = t.iter().map(|v| v * v).sum();
        // p = Xk^T t / (t^T t) ; q = yk^T t / (t^T t)
        let mut pp = transpose_mat_vec(&xk, &t);
        for v in pp.iter_mut() {
            *v /= ttt;
        }
        let q: f64 = t.iter().zip(&yk).map(|(a, b)| a * b).sum::<f64>() / ttt;
        // Deflation : Xk <- Xk - t p^T ; yk <- yk - q t
        for i in 0..n {
            for j in 0..nvar {
                xk[i][j] -= t[i] * pp[j];
            }
            yk[i] -= q * t[i];
        }
        w_all.push(w);
        p_all.push(pp);
        t_all.push(t);
    }
    (w_all, p_all, t_all)
}

// Inverse d'une matrice carree (Gauss-Jordan, pivot partiel) ; None si un pivot est nul.
#[allow(dead_code)]
fn inverse(m: &[Vec<f64>]) -> Option<Vec<Vec<f64>>> {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut inv: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|j| if i == j { 1.0 } else { 0.0 }).collect())
        .collect();
    for col in 0..n {
        let mut piv = col;
        for r in (col + 1)..n {
            if a[r][col].abs() > a[piv][col].abs() {
                piv = r;
            }
        }
        if !a[piv][col].is_finite() || a[piv][col].abs() < 1e-14 {
            return None;
        }
        a.swap(col, piv);
        inv.swap(col, piv);
        let d = a[col][col];
        for j in 0..n {
            a[col][j] /= d;
            inv[col][j] /= d;
        }
        for r in 0..n {
            if r != col {
                let f = a[r][col];
                for j in 0..n {
                    a[r][j] -= f * a[col][j];
                    inv[r][j] -= f * inv[col][j];
                }
            }
        }
    }
    Some(inv)
}

// Rotations de la PLS, R = W (P^T W)^{-1} : elles appliquent le modele a des lignes nouvelles
// (t = X_c R). M = P^T W etant triangulaire superieure, les k premieres colonnes de R sont celles
// du modele a k composantes.
#[allow(dead_code)]
fn rotations(w: &[Vec<f64>], pmat: &[Vec<f64>], kmax: usize) -> Option<Vec<Vec<f64>>> {
    let nvar = w[0].len();
    let m: Vec<Vec<f64>> = (0..kmax)
        .map(|i| {
            (0..kmax)
                .map(|j| pmat[i].iter().zip(&w[j]).map(|(a, b)| a * b).sum())
                .collect()
        })
        .collect();
    let inv = inverse(&m)?;
    let r: Vec<Vec<f64>> = (0..nvar)
        .map(|v| {
            (0..kmax)
                .map(|j| (0..kmax).map(|i| w[i][v] * inv[i][j]).sum())
                .collect()
        })
        .collect();
    Some(r)
}

// Poids cotes variables du modele PLS a k = 1..kmax composantes : betas[k - 1] pour k composantes.
// b_j = (t_j . y_c) / (t_j . t_j), puis beta_k = sum_{j<k} b_j R[:, j].
#[allow(dead_code)]
fn pls_poids(
    w: &[Vec<f64>],
    pmat: &[Vec<f64>],
    t: &[Vec<f64>],
    y_c: &[f64],
) -> Option<Vec<Vec<f64>>> {
    let kmax = t.len();
    let n = t[0].len();
    let nvar = w[0].len();
    let r = rotations(w, pmat, kmax)?;
    let mut betas: Vec<Vec<f64>> = Vec::new();
    let mut beta = vec![0.0; nvar];
    for j in 0..kmax {
        let num: f64 = (0..n).map(|i| t[j][i] * y_c[i]).sum();
        let den: f64 = t[j].iter().map(|v| v * v).sum();
        let b = num / den;
        for v in 0..nvar {
            beta[v] += b * r[v][j];
        }
        betas.push(beta.clone());
    }
    Some(betas)
}

// Prevision PLS pour des lignes deja centrees : y = ybar + x_c . beta.
#[allow(dead_code)]
fn pls_predire(x_c: &[Vec<f64>], y_moy: f64, beta: &[f64]) -> Vec<f64> {
    x_c.iter()
        .map(|l| y_moy + l.iter().zip(beta).map(|(a, b)| a * b).sum::<f64>())
        .collect()
}

// RMSE test de la PLS pour k = 1..kmax (un seul NIPALS, un vecteur de poids par k) ;
// NaN si les rotations ne sont pas calculables.
#[allow(dead_code)]
fn balayage_pls(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (xmu, y_moy) = (colonnes_moyennes(xtr), moyenne(ytr));
    let yc = centrer(ytr);
    let (w, pmat, t) = nipals(&centrer_colonnes(xtr, &xmu), &yc, kmax);
    let betas = match pls_poids(&w, &pmat, &t, &yc) {
        Some(b) => b,
        None => return vec![f64::NAN; kmax],
    };
    let xte_c = centrer_colonnes(xte, &xmu);
    (1..=kmax)
        .map(|k| rmse(yte, &pls_predire(&xte_c, y_moy, &betas[k - 1])))
        .collect()
}

// RMSE test de la PLS ajustee avec exactement k composantes.
#[allow(dead_code)]
fn rmse_pls_k(xtr: &[Vec<f64>], ytr: &[f64], xte: &[Vec<f64>], yte: &[f64], k: usize) -> f64 {
    match balayage_pls(xtr, ytr, xte, yte, k).last() {
        Some(r) => *r,
        None => f64::NAN,
    }
}

// ---- Selection de k par validation croisee sur le train : 5 plis melanges, decoupage derive de la graine.
// L'ACP et la PLS sont refittees sur le train de chaque pli ; la standardisation globale du train n'est
// pas refaite (meme fuite de pretraitement que la source). L'erreur est pooled sur les 120 previsions
// hors pli. Renvoie (k PCR retenu, k PLS retenu, RMSE de CV par k).
#[allow(dead_code)]
fn cv_k(
    x: &[Vec<f64>],
    y: &[f64],
    kmax: usize,
    plis: usize,
    graine: u64,
) -> (usize, usize, Vec<f64>, Vec<f64>) {
    let n = x.len();
    let ordre = permutation(&mut Rng::new(graine ^ 0xA5A5A5A5), n);
    let mut carres_pcr = vec![0.0; kmax];
    let mut carres_pls = vec![0.0; kmax];
    for f in 0..plis {
        let (debut, fin) = bornes_pli(n, plis, f);
        let mut itr: Vec<usize> = ordre[..debut].to_vec();
        itr.extend_from_slice(&ordre[fin..]);
        let xtr: Vec<Vec<f64>> = itr.iter().map(|&i| x[i].clone()).collect();
        let ytr: Vec<f64> = itr.iter().map(|&i| y[i]).collect();
        let xva: Vec<Vec<f64>> = ordre[debut..fin].iter().map(|&i| x[i].clone()).collect();
        let yva: Vec<f64> = ordre[debut..fin].iter().map(|&i| y[i]).collect();
        // PCR : ACP refittee sur le train du pli, scores du train et du pli de validation centres sur la
        // moyenne du train du pli (contrat PCA().fit(train).transform(...)).
        let xmu = colonnes_moyennes(&xtr);
        let (_, _, mut comps) = acp(&xtr);
        comps.truncate(kmax);
        let (t_tr, t_va) = (
            scores_composantes(&centrer_colonnes(&xtr, &xmu), &comps),
            scores_composantes(&centrer_colonnes(&xva, &xmu), &comps),
        );
        // PLS : NIPALS ajuste sur le train du pli, X et y centres sur ce train
        let rmse_pls = balayage_pls(&xtr, &ytr, &xva, &yva, kmax);
        for k in 1..=kmax {
            let (_, p_pcr) = regression_scores(&t_tr, &t_va, &ytr, k);
            carres_pcr[k - 1] += rmse(&yva, &p_pcr) * rmse(&yva, &p_pcr) * yva.len() as f64 / n as f64;
            carres_pls[k - 1] += rmse_pls[k - 1] * rmse_pls[k - 1] * yva.len() as f64 / n as f64;
        }
    }
    let racines = |c: &[f64]| -> Vec<f64> { c.iter().map(|v| v.sqrt()).collect() };
    let (rp, rq) = (racines(&carres_pcr), racines(&carres_pls));
    (argmin(&rp) + 1, argmin(&rq) + 1, rp, rq)
}

// ---- 9. Comparaison finale : verdict par regime ----
// Regime aligne : Y est une combinaison des facteurs du bloc A (variance dominante). Meme X, meme bruit,
// meme decoupage que le regime leurre ; seules les cibles changent.
let j = jeu(GRAINE);
let p_leurre = preparer(&j, &cible(&j));
let p_aligne = preparer(&j, &cible_alignee(&j));
// Les k retenus par validation croisee sur le regime leurre sont reutilises tels quels pour le regime aligne.
let (k_pcr_cv, k_pls_cv, _, _) = cv_k(&p_leurre.x_train_s, &p_leurre.y_train_z, 45, 5, GRAINE);

// RMSE test des quatre methodes sur un jeu standardise ; une seule decomposition de K = X X^T
// pour l'OLS, le ridge et les scores de l'ACP.
#[allow(dead_code)]
fn bilan(p: &Prep, k_pcr: usize, k_pls: usize) -> Vec<f64> {
    let (valeurs, u) = propres_sym(&gram(&p.x_train_s));
    let ols = rmse(
        &p.y_test_z,
        &predire(&p.x_train_s, &p.x_test_s, &ols_alpha(&valeurs, &u, &p.y_train_z)).1,
    );
    let alphas = logspace(-3.0, 3.0, 25);
    let retenu = ridge_cv_retenu(&p.x_train_s, &p.y_train_z, &alphas, 5);
    let ridge = rmse(
        &p.y_test_z,
        &predire(
            &p.x_train_s,
            &p.x_test_s,
            &ridge_alpha(&valeurs, &u, &p.y_train_z, alphas[retenu]),
        )
        .1,
    );
    let mut comps = composantes(&p.x_train_s, &valeurs, &u);
    comps.truncate(k_pcr);
    let (t_tr, t_te) = (
        scores_composantes(&p.x_train_s, &comps),
        scores_composantes(&p.x_test_s, &comps),
    );
    let pcr = rmse(&p.y_test_z, &regression_scores(&t_tr, &t_te, &p.y_train_z, k_pcr).1);
    let pls = rmse_pls_k(&p.x_train_s, &p.y_train_z, &p.x_test_s, &p.y_test_z, k_pls);
    vec![ols, ridge, pcr, pls]
}

let leurre = bilan(&p_leurre, k_pcr_cv, k_pls_cv);
let aligne = bilan(&p_aligne, k_pcr_cv, k_pls_cv);
println!("{:22} {:>14} {:>14}", "methode", "regime leurre", "regime aligne");
let methodes = [
    String::from("OLS min-norme"),
    String::from("Ridge (lambda CV)"),
    format!("PCR (k={})", k_pcr_cv),
    format!("PLS (k={})", k_pls_cv),
];
for (m, (l, a)) in methodes.iter().zip(leurre.iter().zip(&aligne)) {
    println!("{:22} {:14.4} {:14.4}", m, l, a);
};

methode                 regime leurre  regime aligne


OLS min-norme                  0.2206         0.5260


Ridge (lambda CV)              0.2094         0.4824


PCR (k=45)                     0.2019         0.5814


PLS (k=9)                      0.1895         0.5470


### Lecture du résultat

**Régime leurre** (cible dans une direction de faible variance) : PLS k=9 0.1895 < PCR k=45 0.2019 < ridge 0.2094 < OLS 0.2206 — les réductions de dimension dominent, et la PLS est la plus économe (9 composantes contre 45). **Régime aligné** (cible dans les directions dominantes) : ridge 0.4824 < OLS 0.5260 < PLS 0.5470 < PCR 0.5814 — la pénalisation seule suffit et gagne.

Ce deuxième classement demande une lecture prudente, et c'est peut-être la leçon la plus utile du carnet : sur ce tirage, OLS passe devant les deux réductions. Ce n'est pas un résultat contradictoire, c'est une conséquence de protocole — les $k$ (45 et 9) ont été choisis sur le régime leurre et sont réutilisés tels quels, alors que la cible alignée se répartit sur les 40 proxies du bloc A que ces composantes ne couvrent pas spécialement. Le §9bis, qui re-choisit les hyperparamètres à chaque graine, rend l'ordre attendu sur les moyennes (ridge < PLS < PCR < OLS). Le verdict reste donc conditionné : *aucune méthode ne domine dans tous les régimes* ; c'est la position de la cible dans la hiérarchie des variances qui décide — et c'est précisément ce que la PLS découvre automatiquement en triant par Cov(Y, ·).

### Exercice 3 — votre propre régime

Construisez un régime où **PCR et PLS sont équivalents** : une cible alignée sur la première composante principale. Complétez `regime_equivalent` : standardiser $X$ sur le train, construire $Y = X_{\text{standardis\'{e}} \cdot \text{composante}_0$, standardiser $Y$ sur le train, ajuster PCR et PLS à $k=3$ et retourner les deux RMSE test. *Indice : la composante 0 est unitaire ; projetter les lignes standardisées dessus donne exactement la première composante — variance maximale et cible confondues.*

La garde reconstruit la cible attendue **comme l'énoncé** (sur $X$ standardisé) et compare à $10^{-6}$ près ; une seconde fixture, 40 × 5 avec composantes identité, vérifie que la fonction ne dépend pas des données du §1.

In [13]:
// Cellule autonome : prelude (RNG, donnees, algebre) redefini ici, puis le calcul de la section.
// ---- Generateur pseudo-alatoire maison : xorshift64* + Box-Muller (aucun crate).
// La suite n'est pas celle de NumPy : les tirages different, les valeurs chiffrees sont celles de ce carnet.
#[allow(dead_code)]
struct Rng {
    etat: u64,
}

#[allow(dead_code)]
impl Rng {
    fn new(graine: u64) -> Rng {
        let mut etat = graine ^ 0x9E3779B97F4A7C15;
        if etat == 0 {
            etat = 0x2545F4914F6CDD1D;
        }
        Rng { etat }
    }

    fn brut(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x >> 12;
        x ^= x << 25;
        x ^= x >> 27;
        self.etat = x;
        x.wrapping_mul(0x2545F4914F6CDD1D)
    }

    // Uniforme sur [0, 1), 53 bits de mantisse.
    fn uniforme(&mut self) -> f64 {
        ((self.brut() >> 11) as f64 + 0.5) / 9007199254740992.0
    }

    // Normale centree reduite, methode de Box-Muller.
    fn normale(&mut self) -> f64 {
        const TAU: f64 = 6.283185307179586;
        loop {
            let u1 = self.uniforme();
            let u2 = self.uniforme();
            if u1 > 1e-300 {
                return (-2.0 * u1.ln()).sqrt() * (TAU * u2).cos();
            }
        }
    }

    // Entier uniforme dans 0..n, par rejet (pas le biais du modulo).
    fn indice(&mut self, n: usize) -> usize {
        let n = n as u64;
        let limite = u64::MAX - (u64::MAX % n);
        loop {
            let t = self.brut();
            if t < limite {
                return (t % n) as usize;
            }
        }
    }
}

// Permutation aleatoire (Fisher-Yates).
#[allow(dead_code)]
fn permutation(rng: &mut Rng, n: usize) -> Vec<usize> {
    let mut idx: Vec<usize> = (0..n).collect();
    for i in (1..n).rev() {
        let j = rng.indice(i + 1);
        idx.swap(i, j);
    }
    idx
}

// ---- Statistiques de base (ecarts-types au denominateur n, comme NumPy). ----
#[allow(dead_code)]
fn moyenne(v: &[f64]) -> f64 {
    v.iter().sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn variance(v: &[f64]) -> f64 {
    let m = moyenne(v);
    v.iter().map(|x| (x - m) * (x - m)).sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn ecart_type(v: &[f64]) -> f64 {
    variance(v).sqrt()
}

// Matrices stockees ligne par ligne : x[i][j].
#[allow(dead_code)]
fn mat_vec_ligne(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    x.iter()
        .map(|l| l.iter().zip(b).map(|(a, c)| a * c).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn colonnes_moyennes(x: &[Vec<f64>]) -> Vec<f64> {
    let n = x.len();
    (0..x[0].len())
        .map(|j| x.iter().map(|l| l[j]).sum::<f64>() / n as f64)
        .collect()
}

#[allow(dead_code)]
fn colonnes_ecarts(x: &[Vec<f64>]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| ecart_type(&x.iter().map(|l| l[j]).collect::<Vec<f64>>()))
        .collect()
}

#[allow(dead_code)]
fn standardiser(x: &[Vec<f64>], mu: &[f64], sigma: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            l.iter()
                .zip(mu.iter().zip(sigma))
                .map(|(a, (m, s))| (a - m) / s)
                .collect()
        })
        .collect()
}

#[allow(dead_code)]
fn standardiser_vec(v: &[f64], m: f64, s: f64) -> Vec<f64> {
    v.iter().map(|x| (x - m) / s).collect()
}

#[allow(dead_code)]
fn centrer(v: &[f64]) -> Vec<f64> {
    let m = moyenne(v);
    v.iter().map(|x| x - m).collect()
}

#[allow(dead_code)]
fn centrer_colonnes(x: &[Vec<f64>], mu: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| l.iter().zip(mu).map(|(a, m)| a - m).collect())
        .collect()
}

#[allow(dead_code)]
fn correlation(a: &[f64], b: &[f64]) -> f64 {
    let (ca, cb) = (centrer(a), centrer(b));
    let num: f64 = ca.iter().zip(&cb).map(|(x, y)| x * y).sum();
    let den: f64 = ca.iter().map(|x| x * x).sum::<f64>().sqrt()
        * cb.iter().map(|y| y * y).sum::<f64>().sqrt();
    num / den
}

#[allow(dead_code)]
fn rmse(a: &[f64], b: &[f64]) -> f64 {
    (a.iter().zip(b).map(|(x, y)| (x - y) * (x - y)).sum::<f64>() / a.len() as f64).sqrt()
}

// Indice du minimum (le premier en cas d'egalite), comme np.argmin.
#[allow(dead_code)]
fn argmin(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] < v[m] {
            m = k;
        }
    }
    m
}

// Indice du maximum (le premier en cas d'egalite), comme np.argmax.
#[allow(dead_code)]
fn argmax(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] > v[m] {
            m = k;
        }
    }
    m
}

#[allow(dead_code)]
fn logspace(a: f64, b: f64, n: usize) -> Vec<f64> {
    (0..n)
        .map(|i| 10f64.powf(a + (b - a) * i as f64 / (n as f64 - 1.0)))
        .collect()
}

// Colonne j d'une matrice stockee ligne par ligne.
#[allow(dead_code)]
fn colonne(x: &[Vec<f64>], j: usize) -> Vec<f64> {
    x.iter().map(|l| l[j]).collect()
}

// Somme des valeurs absolues sur deux blocs de variables : (0..debut_b, debut_b..fin).
#[allow(dead_code)]
fn masses_blocs(v: &[f64], debut_b: usize) -> (f64, f64) {
    let a: f64 = v[..debut_b].iter().map(|x| x.abs()).sum();
    let b: f64 = v[debut_b..].iter().map(|x| x.abs()).sum();
    (a, b)
}

// ---- Donnees : meme construction que la source (blocs A et B, cible, decoupage, standardisation). ----
#[allow(dead_code)]
const N: usize = 160;
#[allow(dead_code)]
const G_A: usize = 40;
#[allow(dead_code)]
const TAILLE_G: usize = 5;
#[allow(dead_code)]
const P_A: usize = 200;
#[allow(dead_code)]
const P_B: usize = 5;
#[allow(dead_code)]
const P: usize = 205;
#[allow(dead_code)]
const N_TRAIN: usize = 120;
#[allow(dead_code)]
const GRAINE: u64 = 42;

// Variables en groupes ; dans chaque groupe, corr(x_i, x_j) = rho_pair via facteur commun.
#[allow(dead_code)]
fn bloc_facteurs(rng: &mut Rng, n_vars: usize, n_groupes: usize, rho_pair: f64) -> Vec<Vec<f64>> {
    let mut x = vec![vec![0.0; n_vars]; N];
    for g in 0..n_groupes {
        let f: Vec<f64> = (0..N).map(|_| rho_pair.sqrt() * rng.normale()).collect();
        for j in 0..TAILLE_G {
            for i in 0..N {
                x[i][g * TAILLE_G + j] = f[i] + (1.0 - rho_pair).sqrt() * rng.normale();
            }
        }
    }
    x
}

// Un tirage complet depuis la graine : les deux blocs, le bruit, le decoupage et les poids du regime aligne.
#[allow(dead_code)]
struct Jeu {
    x: Vec<Vec<f64>>,
    bruit: Vec<f64>,
    idx: Vec<usize>,
    poids_fact: Vec<f64>,
}

// 40 directions de variance dominante sans signal (bloc A), 1 direction plus modeste generatrice de Y (bloc B).
#[allow(dead_code)]
fn jeu(graine: u64) -> Jeu {
    let mut rng = Rng::new(graine);
    let mut x = bloc_facteurs(&mut rng, P_A, G_A, 0.8);
    let xb = bloc_facteurs(&mut rng, P_B, 1, 0.6);
    for i in 0..N {
        x[i].extend(xb[i].iter().copied());
    }
    let bruit: Vec<f64> = (0..N).map(|_| rng.normale()).collect();
    let idx = permutation(&mut rng, N);
    let poids_fact: Vec<f64> = (0..G_A).map(|_| rng.normale() / (G_A as f64).sqrt()).collect();
    Jeu { x, bruit, idx, poids_fact }
}

// Betas de meme signe, alignes sur le facteur commun du bloc B ; Y = X beta + bruit.
#[allow(dead_code)]
fn cible(j: &Jeu) -> Vec<f64> {
    let formes = [1.3, 1.0, 1.2, 0.9, 1.1];
    let mut beta = vec![0.0; P];
    for k in 0..P_B {
        beta[P_A + k] = formes[k] * 1.2;
    }
    let mut y = mat_vec_ligne(&j.x, &beta);
    for i in 0..N {
        y[i] += j.bruit[i];
    }
    y
}

// Regime aligne : Y2 = facteurs_A . poids_fact * 3.0 + bruit. Les facteurs_A sont des proxis construits
// a partir de la premiere variable observee de chaque groupe du bloc A, pas les facteurs latents exacts.
#[allow(dead_code)]
fn cible_alignee(j: &Jeu) -> Vec<f64> {
    let mut y = vec![0.0; N];
    for i in 0..N {
        let mut s = 0.0;
        for g in 0..G_A {
            s += 0.8f64.sqrt() * j.x[i][g * TAILLE_G] * j.poids_fact[g];
        }
        y[i] = s * 3.0 + j.bruit[i];
    }
    y
}

// Decoupage 120 train / 40 test, standardisation ajustee sur le train uniquement (le geste du 2.1).
#[allow(dead_code)]
struct Prep {
    x_train_s: Vec<Vec<f64>>,
    x_test_s: Vec<Vec<f64>>,
    y_train_z: Vec<f64>,
    y_test_z: Vec<f64>,
    y_ecart: f64,
}

#[allow(dead_code)]
fn preparer(j: &Jeu, y: &[f64]) -> Prep {
    let x_train: Vec<Vec<f64>> = (0..N_TRAIN).map(|k| j.x[j.idx[k]].clone()).collect();
    let x_test: Vec<Vec<f64>> = (N_TRAIN..N).map(|k| j.x[j.idx[k]].clone()).collect();
    let y_train: Vec<f64> = (0..N_TRAIN).map(|k| y[j.idx[k]]).collect();
    let y_test: Vec<f64> = (N_TRAIN..N).map(|k| y[j.idx[k]]).collect();
    let (mu, sigma) = (colonnes_moyennes(&x_train), colonnes_ecarts(&x_train));
    let (y_m, y_s) = (moyenne(&y_train), ecart_type(&y_train));
    Prep {
        x_train_s: standardiser(&x_train, &mu, &sigma),
        x_test_s: standardiser(&x_test, &mu, &sigma),
        y_train_z: standardiser_vec(&y_train, y_m, y_s),
        y_test_z: standardiser_vec(&y_test, y_m, y_s),
        y_ecart: y_s,
    }
}

// ---- Algebre lineaire : tout passe par la matrice de Gram des lignes K = X X^T (n x n, ici 120 x 120),
// jamais par X^T X (p x p, singuliere des que p > n). ----
#[allow(dead_code)]
fn transpose_mat_vec(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| x.iter().zip(b).map(|(l, &v)| l[j] * v).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn gram(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let n = x.len();
    (0..n)
        .map(|i| {
            (0..n)
                .map(|k| x[i].iter().zip(&x[k]).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// Valeurs propres (decroissantes) et vecteurs propres d'une matrice symetrique par la methode de
// Jacobi cyclique. vecteurs[k][i] est la i-eme composante du k-eme vecteur propre ; le signe de chaque
// vecteur est choisi pour que sa plus grande composante en valeur absolue soit positive.
#[allow(dead_code)]
fn propres_sym(m: &[Vec<f64>]) -> (Vec<f64>, Vec<Vec<f64>>) {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut v: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|k| if i == k { 1.0 } else { 0.0 }).collect())
        .collect();
    for _balayage in 0..100 {
        let diag: f64 = (0..n).map(|i| a[i][i].abs()).fold(0.0f64, f64::max);
        let mut carres = 0.0f64;
        for i in 0..n {
            for k in (i + 1)..n {
                carres += a[i][k] * a[i][k];
            }
        }
        if carres.sqrt() <= 1e-13 * (1.0 + diag) {
            break;
        }
        let mut rotations = 0usize;
        for p in 0..n {
            for q in (p + 1)..n {
                let apq = a[p][q];
                // Pivot negligeable devant la diagonale : on l'annule sans tourner.
                if apq.abs() <= 1e-18 * (a[p][p].abs() + a[q][q].abs()) {
                    a[p][q] = 0.0;
                    a[q][p] = 0.0;
                    continue;
                }
                rotations += 1;
                let theta = (a[q][q] - a[p][p]) / (2.0 * apq);
                let t = if theta.abs() > 1e150 {
                    1.0 / (2.0 * theta)
                } else if theta >= 0.0 {
                    1.0 / (theta + (theta * theta + 1.0).sqrt())
                } else {
                    -1.0 / (-theta + (theta * theta + 1.0).sqrt())
                };
                let c = 1.0 / (t * t + 1.0).sqrt();
                let s = t * c;
                for k in 0..n {
                    if k != p && k != q {
                        let akp = a[k][p];
                        let akq = a[k][q];
                        a[k][p] = c * akp - s * akq;
                        a[p][k] = a[k][p];
                        a[k][q] = s * akp + c * akq;
                        a[q][k] = a[k][q];
                    }
                    let vkp = v[k][p];
                    let vkq = v[k][q];
                    v[k][p] = c * vkp - s * vkq;
                    v[k][q] = s * vkp + c * vkq;
                }
                a[p][p] -= t * apq;
                a[q][q] += t * apq;
                a[p][q] = 0.0;
                a[q][p] = 0.0;
            }
        }
        if rotations == 0 {
            break;
        }
    }
    let mut ordre: Vec<usize> = (0..n).collect();
    ordre.sort_by(|&i, &k| a[k][k].total_cmp(&a[i][i]));
    let valeurs: Vec<f64> = ordre.iter().map(|&i| a[i][i]).collect();
    let vecteurs: Vec<Vec<f64>> = ordre
        .iter()
        .map(|&k| {
            let colonne: Vec<f64> = v.iter().map(|l| l[k]).collect();
            let pivot = argmax(&colonne.iter().map(|x| x.abs()).collect::<Vec<f64>>());
            if colonne[pivot] < 0.0 {
                colonne.iter().map(|x| -x).collect()
            } else {
                colonne
            }
        })
        .collect();
    (valeurs, vecteurs)
}

// Rang numerique : nombre de valeurs propres au-dessus de 1e-8 fois la plus grande.
#[allow(dead_code)]
fn rang(valeurs: &[f64]) -> usize {
    let lmax = valeurs[0];
    valeurs.iter().filter(|v| **v > 1e-8 * lmax).count()
}

// OLS de norme minimale sans ordonnee, par K = X X^T : alpha = U_r L_r^{-1} U_r^T y, puis beta = X^T alpha.
// Les r valeurs propres non nulles fixent la prediction ; le noyau (p - r directions) ne la modifie pas.
#[allow(dead_code)]
fn ols_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64]) -> Vec<f64> {
    let lmax = valeurs[0];
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        if valeurs[k] > 1e-8 * lmax {
            let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
            for i in 0..n {
                alpha[i] += u[k][i] * d / valeurs[k];
            }
        }
    }
    alpha
}

// Coefficients duaux du ridge sans ordonnee : alpha = (K + lambda I)^{-1} y, dans la base des vecteurs propres.
#[allow(dead_code)]
fn ridge_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64], lambda: f64) -> Vec<f64> {
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
        for i in 0..n {
            alpha[i] += u[k][i] * d / (valeurs[k] + lambda);
        }
    }
    alpha
}

// beta = X^T alpha ; renvoie les previsions X beta sur train et test, et la norme L2 de beta.
#[allow(dead_code)]
fn predire(xtr: &[Vec<f64>], xte: &[Vec<f64>], alpha: &[f64]) -> (Vec<f64>, Vec<f64>, f64) {
    let beta = transpose_mat_vec(xtr, alpha);
    let norme = beta.iter().map(|b| b * b).sum::<f64>().sqrt();
    (mat_vec_ligne(xtr, &beta), mat_vec_ligne(xte, &beta), norme)
}

// Bornes du pli f d'une validation croisee a plis contigus (KFold sans melange).
#[allow(dead_code)]
fn bornes_pli(n: usize, plis: usize, f: usize) -> (usize, usize) {
    let (base, reste) = (n / plis, n % plis);
    let debut = f * base + reste.min(f);
    (debut, debut + base + if f < reste { 1 } else { 0 })
}

// Un pli contigu, avec la decomposition spectrale du Gram de son train : elle sert a tous les alpha du ridge.
#[allow(dead_code)]
struct PliContigu {
    xtr: Vec<Vec<f64>>,
    xva: Vec<Vec<f64>>,
    positions_val: Vec<usize>,
    valeurs: Vec<f64>,
    u: Vec<Vec<f64>>,
}

#[allow(dead_code)]
fn plis_contigus(x: &[Vec<f64>], plis: usize) -> Vec<PliContigu> {
    let n = x.len();
    (0..plis)
        .map(|f| {
            let (debut, fin) = bornes_pli(n, plis, f);
            let xtr: Vec<Vec<f64>> = (0..n)
                .filter(|&i| i < debut || i >= fin)
                .map(|i| x[i].clone())
                .collect();
            let xva: Vec<Vec<f64>> = x[debut..fin].to_vec();
            let (valeurs, u) = propres_sym(&gram(&xtr));
            PliContigu {
                xtr,
                xva,
                positions_val: (debut..fin).collect(),
                valeurs,
                u,
            }
        })
        .collect()
}

// RidgeCV(alphas, cv=plis, fit_intercept=False) : l'alpha retenu est celui qui maximise la moyenne
// des R2 de pli (le score par defaut de la validation croisee de scikit-learn). Renvoie son indice.
#[allow(dead_code)]
fn ridge_cv_sur_plis(plis_c: &[PliContigu], y: &[f64], alphas: &[f64]) -> usize {
    let mut score = vec![0.0; alphas.len()];
    for pli in plis_c {
        let ytr: Vec<f64> = (0..y.len())
            .filter(|i| !pli.positions_val.contains(i))
            .map(|i| y[i])
            .collect();
        let yva: Vec<f64> = pli.positions_val.iter().map(|&i| y[i]).collect();
        let sst: f64 = centrer(&yva).iter().map(|x| x * x).sum();
        for (j, &lam) in alphas.iter().enumerate() {
            let (_, p_va, _) = predire(
                &pli.xtr,
                &pli.xva,
                &ridge_alpha(&pli.valeurs, &pli.u, &ytr, lam),
            );
            let sse: f64 = yva.iter().zip(&p_va).map(|(t, p)| (t - p) * (t - p)).sum();
            score[j] += 1.0 - sse / sst;
        }
    }
    argmax(&score)
}

#[allow(dead_code)]
fn ridge_cv_retenu(x: &[Vec<f64>], y: &[f64], alphas: &[f64], plis: usize) -> usize {
    ridge_cv_sur_plis(&plis_contigus(x, plis), y, alphas)
}

// Composantes principales depuis la decomposition de K = X X^T : composante_k = X^T u_k / sqrt(lambda_k).
#[allow(dead_code)]
fn composantes(x: &[Vec<f64>], valeurs: &[f64], u: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let mut comps: Vec<Vec<f64>> = Vec::new();
    for k in 0..rang(valeurs) {
        let mut c = transpose_mat_vec(x, &u[k]);
        let s = valeurs[k].sqrt();
        for v in c.iter_mut() {
            *v /= s;
        }
        comps.push(c);
    }
    comps
}

// Centrage d'un echantillon sur ses propres colonnes : ce que fait PCA a l'ajustement.
#[allow(dead_code)]
fn centrer_propre(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    centrer_colonnes(x, &colonnes_moyennes(x))
}

// ACP, comme sklearn.decomposition.PCA : X est CENTRE sur ses colonnes, covariance (denominateur n),
// valeurs propres, part de variance, composantes. Le centrage ne change rien sur une matrice deja
// standardisee sur elle-meme (train complet) ; il est indispensable sur le train d'un pli, dont les
// colonnes ne sont pas de moyenne nulle.
#[allow(dead_code)]
fn acp(x: &[Vec<f64>]) -> (Vec<f64>, Vec<f64>, Vec<Vec<f64>>) {
    let xc = centrer_propre(x);
    let (valeurs, u) = propres_sym(&gram(&xc));
    let total: f64 = valeurs.iter().sum();
    let comps = composantes(&xc, &valeurs, &u);
    let ratio: Vec<f64> = valeurs.iter().map(|l| l / total).collect();
    (valeurs, ratio, comps)
}

// Variance cumulee jusqu'a la composante k comprise.
#[allow(dead_code)]
fn variance_cumulee(ratio: &[f64], k: usize) -> f64 {
    ratio[..=k].iter().sum()
}

// Scores t[i][k] = x_i . composante_k.
#[allow(dead_code)]
fn scores_composantes(x: &[Vec<f64>], comps: &[Vec<f64>]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            comps
                .iter()
                .map(|c| l.iter().zip(c).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// PCR a k composantes avec ordonnee : y = ybar + sum_{j<k} b_j t_j.
// Les scores d'une ACP sont orthogonaux deux a deux, donc b_j = (t_j . (y - ybar)) / (t_j . t_j).
#[allow(dead_code)]
fn regression_scores(
    t_tr: &[Vec<f64>],
    t_te: &[Vec<f64>],
    y: &[f64],
    k: usize,
) -> (Vec<f64>, Vec<f64>) {
    let ym = moyenne(y);
    let yc: Vec<f64> = y.iter().map(|v| v - ym).collect();
    let mut b = vec![0.0; k];
    for j in 0..k {
        let num: f64 = t_tr.iter().zip(&yc).map(|(t, &v)| t[j] * v).sum();
        let den: f64 = t_tr.iter().map(|t| t[j] * t[j]).sum();
        b[j] = num / den;
    }
    let projet = |tt: &[Vec<f64>]| -> Vec<f64> {
        tt.iter()
            .map(|t| ym + (0..k).map(|j| b[j] * t[j]).sum::<f64>())
            .collect()
    };
    (projet(t_tr), projet(t_te))
}

// RMSE test de la PCR pour k = 1..kmax (une seule ACP, une regression par k).
#[allow(dead_code)]
fn balayage_pcr(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (_, _, mut comps) = acp(xtr);
    comps.truncate(kmax);
    // PCA().fit(xtr).transform(...) : les deux echantillons sont centres sur la moyenne du train.
    let xmu = colonnes_moyennes(xtr);
    let (t_tr, t_te) = (
        scores_composantes(&centrer_colonnes(xtr, &xmu), &comps),
        scores_composantes(&centrer_colonnes(xte, &xmu), &comps),
    );
    (1..=kmax)
        .map(|k| rmse(yte, &regression_scores(&t_tr, &t_te, ytr, k).1))
        .collect()
}

// ---- PLS1 par NIPALS (l'algorithme de PLSRegression). x et y doivent etre centres sur le train.
// Renvoie (w, p, t) : poids x, loadings x et scores, ranges par composante : w[j][v], t[j][i]. ----
#[allow(dead_code)]
fn nipals(
    x: &[Vec<f64>],
    y_c: &[f64],
    kmax: usize,
) -> (Vec<Vec<f64>>, Vec<Vec<f64>>, Vec<Vec<f64>>) {
    let n = x.len();
    let nvar = x[0].len();
    let mut xk = x.to_vec();
    let mut yk = y_c.to_vec();
    let mut w_all: Vec<Vec<f64>> = Vec::new();
    let mut p_all: Vec<Vec<f64>> = Vec::new();
    let mut t_all: Vec<Vec<f64>> = Vec::new();
    for _j in 0..kmax {
        // w = Xk^T yk / ||Xk^T yk|| (direction de covariance maximale avec le reste de y), t = Xk w
        let mut w = transpose_mat_vec(&xk, &yk);
        let nrm = w.iter().map(|v| v * v).sum::<f64>().sqrt();
        for v in w.iter_mut() {
            *v /= nrm;
        }
        let t = mat_vec_ligne(&xk, &w);
        let ttt: f64 = t.iter().map(|v| v * v).sum();
        // p = Xk^T t / (t^T t) ; q = yk^T t / (t^T t)
        let mut pp = transpose_mat_vec(&xk, &t);
        for v in pp.iter_mut() {
            *v /= ttt;
        }
        let q: f64 = t.iter().zip(&yk).map(|(a, b)| a * b).sum::<f64>() / ttt;
        // Deflation : Xk <- Xk - t p^T ; yk <- yk - q t
        for i in 0..n {
            for j in 0..nvar {
                xk[i][j] -= t[i] * pp[j];
            }
            yk[i] -= q * t[i];
        }
        w_all.push(w);
        p_all.push(pp);
        t_all.push(t);
    }
    (w_all, p_all, t_all)
}

// Inverse d'une matrice carree (Gauss-Jordan, pivot partiel) ; None si un pivot est nul.
#[allow(dead_code)]
fn inverse(m: &[Vec<f64>]) -> Option<Vec<Vec<f64>>> {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut inv: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|j| if i == j { 1.0 } else { 0.0 }).collect())
        .collect();
    for col in 0..n {
        let mut piv = col;
        for r in (col + 1)..n {
            if a[r][col].abs() > a[piv][col].abs() {
                piv = r;
            }
        }
        if !a[piv][col].is_finite() || a[piv][col].abs() < 1e-14 {
            return None;
        }
        a.swap(col, piv);
        inv.swap(col, piv);
        let d = a[col][col];
        for j in 0..n {
            a[col][j] /= d;
            inv[col][j] /= d;
        }
        for r in 0..n {
            if r != col {
                let f = a[r][col];
                for j in 0..n {
                    a[r][j] -= f * a[col][j];
                    inv[r][j] -= f * inv[col][j];
                }
            }
        }
    }
    Some(inv)
}

// Rotations de la PLS, R = W (P^T W)^{-1} : elles appliquent le modele a des lignes nouvelles
// (t = X_c R). M = P^T W etant triangulaire superieure, les k premieres colonnes de R sont celles
// du modele a k composantes.
#[allow(dead_code)]
fn rotations(w: &[Vec<f64>], pmat: &[Vec<f64>], kmax: usize) -> Option<Vec<Vec<f64>>> {
    let nvar = w[0].len();
    let m: Vec<Vec<f64>> = (0..kmax)
        .map(|i| {
            (0..kmax)
                .map(|j| pmat[i].iter().zip(&w[j]).map(|(a, b)| a * b).sum())
                .collect()
        })
        .collect();
    let inv = inverse(&m)?;
    let r: Vec<Vec<f64>> = (0..nvar)
        .map(|v| {
            (0..kmax)
                .map(|j| (0..kmax).map(|i| w[i][v] * inv[i][j]).sum())
                .collect()
        })
        .collect();
    Some(r)
}

// Poids cotes variables du modele PLS a k = 1..kmax composantes : betas[k - 1] pour k composantes.
// b_j = (t_j . y_c) / (t_j . t_j), puis beta_k = sum_{j<k} b_j R[:, j].
#[allow(dead_code)]
fn pls_poids(
    w: &[Vec<f64>],
    pmat: &[Vec<f64>],
    t: &[Vec<f64>],
    y_c: &[f64],
) -> Option<Vec<Vec<f64>>> {
    let kmax = t.len();
    let n = t[0].len();
    let nvar = w[0].len();
    let r = rotations(w, pmat, kmax)?;
    let mut betas: Vec<Vec<f64>> = Vec::new();
    let mut beta = vec![0.0; nvar];
    for j in 0..kmax {
        let num: f64 = (0..n).map(|i| t[j][i] * y_c[i]).sum();
        let den: f64 = t[j].iter().map(|v| v * v).sum();
        let b = num / den;
        for v in 0..nvar {
            beta[v] += b * r[v][j];
        }
        betas.push(beta.clone());
    }
    Some(betas)
}

// Prevision PLS pour des lignes deja centrees : y = ybar + x_c . beta.
#[allow(dead_code)]
fn pls_predire(x_c: &[Vec<f64>], y_moy: f64, beta: &[f64]) -> Vec<f64> {
    x_c.iter()
        .map(|l| y_moy + l.iter().zip(beta).map(|(a, b)| a * b).sum::<f64>())
        .collect()
}

// RMSE test de la PLS pour k = 1..kmax (un seul NIPALS, un vecteur de poids par k) ;
// NaN si les rotations ne sont pas calculables.
#[allow(dead_code)]
fn balayage_pls(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (xmu, y_moy) = (colonnes_moyennes(xtr), moyenne(ytr));
    let yc = centrer(ytr);
    let (w, pmat, t) = nipals(&centrer_colonnes(xtr, &xmu), &yc, kmax);
    let betas = match pls_poids(&w, &pmat, &t, &yc) {
        Some(b) => b,
        None => return vec![f64::NAN; kmax],
    };
    let xte_c = centrer_colonnes(xte, &xmu);
    (1..=kmax)
        .map(|k| rmse(yte, &pls_predire(&xte_c, y_moy, &betas[k - 1])))
        .collect()
}

// RMSE test de la PLS ajustee avec exactement k composantes.
#[allow(dead_code)]
fn rmse_pls_k(xtr: &[Vec<f64>], ytr: &[f64], xte: &[Vec<f64>], yte: &[f64], k: usize) -> f64 {
    match balayage_pls(xtr, ytr, xte, yte, k).last() {
        Some(r) => *r,
        None => f64::NAN,
    }
}

// Exercice 3 : aligner une cible sur une composante et comparer PCR et PLS.
#[allow(unused_variables)]
fn regime_equivalent(
    x: &[Vec<f64>],
    composantes: &[Vec<f64>],
    idx_train: &[usize],
    idx_test: &[usize],
) -> Option<(f64, f64)> {
    // TODO : standardiser X sur train, construire Y = X_standardise . composantes[0], standardiser Y sur train,
    // ajuster PCR et PLS a k=3 puis retourner les deux RMSE test.
    None
}

// Garde : exactement la procedure de l'enonce, ecrite avec les fonctions du preambule.
#[allow(dead_code)]
fn reference_regime(
    x: &[Vec<f64>],
    poids: &[Vec<f64>],
    train: &[usize],
    test: &[usize],
) -> (f64, f64) {
    let xtr: Vec<Vec<f64>> = train.iter().map(|&i| x[i].clone()).collect();
    let xte: Vec<Vec<f64>> = test.iter().map(|&i| x[i].clone()).collect();
    let (mu, sd) = (colonnes_moyennes(&xtr), colonnes_ecarts(&xtr));
    let (xs_tr, xs_te) = (standardiser(&xtr, &mu, &sd), standardiser(&xte, &mu, &sd));
    // La cible vient du X standardise (moyennes et ecarts-types du train, sur toutes les lignes).
    let cible: Vec<f64> = mat_vec_ligne(&standardiser(x, &mu, &sd), &poids[0]);
    let (ytr, yte): (Vec<f64>, Vec<f64>) = (
        train.iter().map(|&i| cible[i]).collect(),
        test.iter().map(|&i| cible[i]).collect(),
    );
    let (ym, ys) = (moyenne(&ytr), ecart_type(&ytr));
    let (ztr, zte) = (
        standardiser_vec(&ytr, ym, ys),
        standardiser_vec(&yte, ym, ys),
    );
    let (_, _, mut comps) = acp(&xs_tr);
    comps.truncate(3);
    let (t_tr, t_te) = (
        scores_composantes(&xs_tr, &comps),
        scores_composantes(&xs_te, &comps),
    );
    let pcr = rmse(&zte, &regression_scores(&t_tr, &t_te, &ztr, 3).1);
    let pls = rmse_pls_k(&xs_tr, &ztr, &xs_te, &zte, 3);
    (pcr, pls)
}

let j = jeu(GRAINE);
let p = preparer(&j, &cible(&j));
let (_, _, comps) = acp(&p.x_train_s);
// Une fixture : (X non standardise, poids des composantes, indices train, indices test).
let mut r_fixture = Rng::new(192);
let tiny_x: Vec<Vec<f64>> = (0..40)
    .map(|_| (0..5).map(|_| r_fixture.normale()).collect())
    .collect();
let identite: Vec<Vec<f64>> = (0..5)
    .map(|i| (0..5).map(|k| if i == k { 1.0 } else { 0.0 }).collect())
    .collect();
let tout: Vec<usize> = (0..40).collect();
let fixtures: Vec<(Vec<Vec<f64>>, Vec<Vec<f64>>, Vec<usize>, Vec<usize>)> = vec![
    (
        j.x.clone(),
        comps,
        j.idx[..N_TRAIN].to_vec(),
        j.idx[N_TRAIN..].to_vec(),
    ),
    (tiny_x, identite, tout[..30].to_vec(), tout[30..].to_vec()),
];
let reponses: Vec<Option<(f64, f64)>> = fixtures
    .iter()
    .map(|(d, w, tr, te)| regime_equivalent(d, w, tr, te))
    .collect();
if reponses.iter().all(|r| r.is_none()) {
    println!("Exercice 3 a completer.");
} else if reponses.iter().any(|r| r.is_none()) {
    println!("Exercice 3 : a revoir");
} else {
    let mut ok = true;
    for ((d, w, tr, te), reponse) in fixtures.iter().zip(&reponses) {
        let (a, b) = match reponse {
            Some(r) => *r,
            None => continue,
        };
        let attendu = reference_regime(d, w, tr, te);
        let proche = |u: f64, v: f64| u.is_finite() && (u - v).abs() < 1e-6;
        ok = ok && proche(a, attendu.0) && proche(b, attendu.1);
    }
    println!(
        "{}",
        if ok {
            "Exercice 3 : mesures coherentes ; interpreter le regime"
        } else {
            "Exercice 3 : a revoir"
        }
    );
};

Exercice 3 a completer.


## 9bis. Robustesse — le verdict sur dix graines

Un verdict sur un seul tirage peut flatter une méthode. La robustesse demande de **redessiner chaque graine** : données, bruit, découpage train/test, folds de CV, hyperparamètres — $\lambda$ et $k$ sont **re-choisis à l'intérieur de chaque graine**, sans réutiliser quoi que ce soit du tirage de la section 1. Dix graines, les deux régimes de la section 9, moyenne ± écart-type du RMSE test.

Pour chaque graine, les `k` sont sélectionnés sur le régime leurre puis réutilisés pour le régime aligné ; ils ne sont pas réajustés séparément pour ce dernier. Comme plus haut, la standardisation globale du train précède les plis internes.

In [14]:
// Cellule autonome : prelude (RNG, donnees, algebre) redefini ici, puis le calcul de la section.
// ---- Generateur pseudo-alatoire maison : xorshift64* + Box-Muller (aucun crate).
// La suite n'est pas celle de NumPy : les tirages different, les valeurs chiffrees sont celles de ce carnet.
#[allow(dead_code)]
struct Rng {
    etat: u64,
}

#[allow(dead_code)]
impl Rng {
    fn new(graine: u64) -> Rng {
        let mut etat = graine ^ 0x9E3779B97F4A7C15;
        if etat == 0 {
            etat = 0x2545F4914F6CDD1D;
        }
        Rng { etat }
    }

    fn brut(&mut self) -> u64 {
        let mut x = self.etat;
        x ^= x >> 12;
        x ^= x << 25;
        x ^= x >> 27;
        self.etat = x;
        x.wrapping_mul(0x2545F4914F6CDD1D)
    }

    // Uniforme sur [0, 1), 53 bits de mantisse.
    fn uniforme(&mut self) -> f64 {
        ((self.brut() >> 11) as f64 + 0.5) / 9007199254740992.0
    }

    // Normale centree reduite, methode de Box-Muller.
    fn normale(&mut self) -> f64 {
        const TAU: f64 = 6.283185307179586;
        loop {
            let u1 = self.uniforme();
            let u2 = self.uniforme();
            if u1 > 1e-300 {
                return (-2.0 * u1.ln()).sqrt() * (TAU * u2).cos();
            }
        }
    }

    // Entier uniforme dans 0..n, par rejet (pas le biais du modulo).
    fn indice(&mut self, n: usize) -> usize {
        let n = n as u64;
        let limite = u64::MAX - (u64::MAX % n);
        loop {
            let t = self.brut();
            if t < limite {
                return (t % n) as usize;
            }
        }
    }
}

// Permutation aleatoire (Fisher-Yates).
#[allow(dead_code)]
fn permutation(rng: &mut Rng, n: usize) -> Vec<usize> {
    let mut idx: Vec<usize> = (0..n).collect();
    for i in (1..n).rev() {
        let j = rng.indice(i + 1);
        idx.swap(i, j);
    }
    idx
}

// ---- Statistiques de base (ecarts-types au denominateur n, comme NumPy). ----
#[allow(dead_code)]
fn moyenne(v: &[f64]) -> f64 {
    v.iter().sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn variance(v: &[f64]) -> f64 {
    let m = moyenne(v);
    v.iter().map(|x| (x - m) * (x - m)).sum::<f64>() / v.len() as f64
}

#[allow(dead_code)]
fn ecart_type(v: &[f64]) -> f64 {
    variance(v).sqrt()
}

// Matrices stockees ligne par ligne : x[i][j].
#[allow(dead_code)]
fn mat_vec_ligne(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    x.iter()
        .map(|l| l.iter().zip(b).map(|(a, c)| a * c).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn colonnes_moyennes(x: &[Vec<f64>]) -> Vec<f64> {
    let n = x.len();
    (0..x[0].len())
        .map(|j| x.iter().map(|l| l[j]).sum::<f64>() / n as f64)
        .collect()
}

#[allow(dead_code)]
fn colonnes_ecarts(x: &[Vec<f64>]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| ecart_type(&x.iter().map(|l| l[j]).collect::<Vec<f64>>()))
        .collect()
}

#[allow(dead_code)]
fn standardiser(x: &[Vec<f64>], mu: &[f64], sigma: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            l.iter()
                .zip(mu.iter().zip(sigma))
                .map(|(a, (m, s))| (a - m) / s)
                .collect()
        })
        .collect()
}

#[allow(dead_code)]
fn standardiser_vec(v: &[f64], m: f64, s: f64) -> Vec<f64> {
    v.iter().map(|x| (x - m) / s).collect()
}

#[allow(dead_code)]
fn centrer(v: &[f64]) -> Vec<f64> {
    let m = moyenne(v);
    v.iter().map(|x| x - m).collect()
}

#[allow(dead_code)]
fn centrer_colonnes(x: &[Vec<f64>], mu: &[f64]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| l.iter().zip(mu).map(|(a, m)| a - m).collect())
        .collect()
}

#[allow(dead_code)]
fn correlation(a: &[f64], b: &[f64]) -> f64 {
    let (ca, cb) = (centrer(a), centrer(b));
    let num: f64 = ca.iter().zip(&cb).map(|(x, y)| x * y).sum();
    let den: f64 = ca.iter().map(|x| x * x).sum::<f64>().sqrt()
        * cb.iter().map(|y| y * y).sum::<f64>().sqrt();
    num / den
}

#[allow(dead_code)]
fn rmse(a: &[f64], b: &[f64]) -> f64 {
    (a.iter().zip(b).map(|(x, y)| (x - y) * (x - y)).sum::<f64>() / a.len() as f64).sqrt()
}

// Indice du minimum (le premier en cas d'egalite), comme np.argmin.
#[allow(dead_code)]
fn argmin(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] < v[m] {
            m = k;
        }
    }
    m
}

// Indice du maximum (le premier en cas d'egalite), comme np.argmax.
#[allow(dead_code)]
fn argmax(v: &[f64]) -> usize {
    let mut m = 0usize;
    for k in 1..v.len() {
        if v[k] > v[m] {
            m = k;
        }
    }
    m
}

#[allow(dead_code)]
fn logspace(a: f64, b: f64, n: usize) -> Vec<f64> {
    (0..n)
        .map(|i| 10f64.powf(a + (b - a) * i as f64 / (n as f64 - 1.0)))
        .collect()
}

// Colonne j d'une matrice stockee ligne par ligne.
#[allow(dead_code)]
fn colonne(x: &[Vec<f64>], j: usize) -> Vec<f64> {
    x.iter().map(|l| l[j]).collect()
}

// Somme des valeurs absolues sur deux blocs de variables : (0..debut_b, debut_b..fin).
#[allow(dead_code)]
fn masses_blocs(v: &[f64], debut_b: usize) -> (f64, f64) {
    let a: f64 = v[..debut_b].iter().map(|x| x.abs()).sum();
    let b: f64 = v[debut_b..].iter().map(|x| x.abs()).sum();
    (a, b)
}

// ---- Donnees : meme construction que la source (blocs A et B, cible, decoupage, standardisation). ----
#[allow(dead_code)]
const N: usize = 160;
#[allow(dead_code)]
const G_A: usize = 40;
#[allow(dead_code)]
const TAILLE_G: usize = 5;
#[allow(dead_code)]
const P_A: usize = 200;
#[allow(dead_code)]
const P_B: usize = 5;
#[allow(dead_code)]
const P: usize = 205;
#[allow(dead_code)]
const N_TRAIN: usize = 120;
#[allow(dead_code)]
const GRAINE: u64 = 42;

// Variables en groupes ; dans chaque groupe, corr(x_i, x_j) = rho_pair via facteur commun.
#[allow(dead_code)]
fn bloc_facteurs(rng: &mut Rng, n_vars: usize, n_groupes: usize, rho_pair: f64) -> Vec<Vec<f64>> {
    let mut x = vec![vec![0.0; n_vars]; N];
    for g in 0..n_groupes {
        let f: Vec<f64> = (0..N).map(|_| rho_pair.sqrt() * rng.normale()).collect();
        for j in 0..TAILLE_G {
            for i in 0..N {
                x[i][g * TAILLE_G + j] = f[i] + (1.0 - rho_pair).sqrt() * rng.normale();
            }
        }
    }
    x
}

// Un tirage complet depuis la graine : les deux blocs, le bruit, le decoupage et les poids du regime aligne.
#[allow(dead_code)]
struct Jeu {
    x: Vec<Vec<f64>>,
    bruit: Vec<f64>,
    idx: Vec<usize>,
    poids_fact: Vec<f64>,
}

// 40 directions de variance dominante sans signal (bloc A), 1 direction plus modeste generatrice de Y (bloc B).
#[allow(dead_code)]
fn jeu(graine: u64) -> Jeu {
    let mut rng = Rng::new(graine);
    let mut x = bloc_facteurs(&mut rng, P_A, G_A, 0.8);
    let xb = bloc_facteurs(&mut rng, P_B, 1, 0.6);
    for i in 0..N {
        x[i].extend(xb[i].iter().copied());
    }
    let bruit: Vec<f64> = (0..N).map(|_| rng.normale()).collect();
    let poids_fact: Vec<f64> = (0..G_A).map(|_| rng.normale() / (G_A as f64).sqrt()).collect();
    let idx = permutation(&mut rng, N);
    Jeu { x, bruit, idx, poids_fact }
}

// Betas de meme signe, alignes sur le facteur commun du bloc B ; Y = X beta + bruit.
#[allow(dead_code)]
fn cible(j: &Jeu) -> Vec<f64> {
    let formes = [1.3, 1.0, 1.2, 0.9, 1.1];
    let mut beta = vec![0.0; P];
    for k in 0..P_B {
        beta[P_A + k] = formes[k] * 1.2;
    }
    let mut y = mat_vec_ligne(&j.x, &beta);
    for i in 0..N {
        y[i] += j.bruit[i];
    }
    y
}

// Regime aligne : Y2 = facteurs_A . poids_fact * 3.0 + bruit. Les facteurs_A sont des proxis construits
// a partir de la premiere variable observee de chaque groupe du bloc A, pas les facteurs latents exacts.
#[allow(dead_code)]
fn cible_alignee(j: &Jeu) -> Vec<f64> {
    let mut y = vec![0.0; N];
    for i in 0..N {
        let mut s = 0.0;
        for g in 0..G_A {
            s += 0.8f64.sqrt() * j.x[i][g * TAILLE_G] * j.poids_fact[g];
        }
        y[i] = s * 3.0 + j.bruit[i];
    }
    y
}

// Decoupage 120 train / 40 test, standardisation ajustee sur le train uniquement (le geste du 2.1).
#[allow(dead_code)]
struct Prep {
    x_train_s: Vec<Vec<f64>>,
    x_test_s: Vec<Vec<f64>>,
    y_train_z: Vec<f64>,
    y_test_z: Vec<f64>,
    y_ecart: f64,
}

#[allow(dead_code)]
fn preparer(j: &Jeu, y: &[f64]) -> Prep {
    let x_train: Vec<Vec<f64>> = (0..N_TRAIN).map(|k| j.x[j.idx[k]].clone()).collect();
    let x_test: Vec<Vec<f64>> = (N_TRAIN..N).map(|k| j.x[j.idx[k]].clone()).collect();
    let y_train: Vec<f64> = (0..N_TRAIN).map(|k| y[j.idx[k]]).collect();
    let y_test: Vec<f64> = (N_TRAIN..N).map(|k| y[j.idx[k]]).collect();
    let (mu, sigma) = (colonnes_moyennes(&x_train), colonnes_ecarts(&x_train));
    let (y_m, y_s) = (moyenne(&y_train), ecart_type(&y_train));
    Prep {
        x_train_s: standardiser(&x_train, &mu, &sigma),
        x_test_s: standardiser(&x_test, &mu, &sigma),
        y_train_z: standardiser_vec(&y_train, y_m, y_s),
        y_test_z: standardiser_vec(&y_test, y_m, y_s),
        y_ecart: y_s,
    }
}

// ---- Algebre lineaire : tout passe par la matrice de Gram des lignes K = X X^T (n x n, ici 120 x 120),
// jamais par X^T X (p x p, singuliere des que p > n). ----
#[allow(dead_code)]
fn transpose_mat_vec(x: &[Vec<f64>], b: &[f64]) -> Vec<f64> {
    (0..x[0].len())
        .map(|j| x.iter().zip(b).map(|(l, &v)| l[j] * v).sum::<f64>())
        .collect()
}

#[allow(dead_code)]
fn gram(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let n = x.len();
    (0..n)
        .map(|i| {
            (0..n)
                .map(|k| x[i].iter().zip(&x[k]).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// Valeurs propres (decroissantes) et vecteurs propres d'une matrice symetrique par la methode de
// Jacobi cyclique. vecteurs[k][i] est la i-eme composante du k-eme vecteur propre ; le signe de chaque
// vecteur est choisi pour que sa plus grande composante en valeur absolue soit positive.
#[allow(dead_code)]
fn propres_sym(m: &[Vec<f64>]) -> (Vec<f64>, Vec<Vec<f64>>) {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut v: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|k| if i == k { 1.0 } else { 0.0 }).collect())
        .collect();
    for _balayage in 0..100 {
        let diag: f64 = (0..n).map(|i| a[i][i].abs()).fold(0.0f64, f64::max);
        let mut carres = 0.0f64;
        for i in 0..n {
            for k in (i + 1)..n {
                carres += a[i][k] * a[i][k];
            }
        }
        if carres.sqrt() <= 1e-13 * (1.0 + diag) {
            break;
        }
        let mut rotations = 0usize;
        for p in 0..n {
            for q in (p + 1)..n {
                let apq = a[p][q];
                // Pivot negligeable devant la diagonale : on l'annule sans tourner.
                if apq.abs() <= 1e-18 * (a[p][p].abs() + a[q][q].abs()) {
                    a[p][q] = 0.0;
                    a[q][p] = 0.0;
                    continue;
                }
                rotations += 1;
                let theta = (a[q][q] - a[p][p]) / (2.0 * apq);
                let t = if theta.abs() > 1e150 {
                    1.0 / (2.0 * theta)
                } else if theta >= 0.0 {
                    1.0 / (theta + (theta * theta + 1.0).sqrt())
                } else {
                    -1.0 / (-theta + (theta * theta + 1.0).sqrt())
                };
                let c = 1.0 / (t * t + 1.0).sqrt();
                let s = t * c;
                for k in 0..n {
                    if k != p && k != q {
                        let akp = a[k][p];
                        let akq = a[k][q];
                        a[k][p] = c * akp - s * akq;
                        a[p][k] = a[k][p];
                        a[k][q] = s * akp + c * akq;
                        a[q][k] = a[k][q];
                    }
                    let vkp = v[k][p];
                    let vkq = v[k][q];
                    v[k][p] = c * vkp - s * vkq;
                    v[k][q] = s * vkp + c * vkq;
                }
                a[p][p] -= t * apq;
                a[q][q] += t * apq;
                a[p][q] = 0.0;
                a[q][p] = 0.0;
            }
        }
        if rotations == 0 {
            break;
        }
    }
    let mut ordre: Vec<usize> = (0..n).collect();
    ordre.sort_by(|&i, &k| a[k][k].total_cmp(&a[i][i]));
    let valeurs: Vec<f64> = ordre.iter().map(|&i| a[i][i]).collect();
    let vecteurs: Vec<Vec<f64>> = ordre
        .iter()
        .map(|&k| {
            let colonne: Vec<f64> = v.iter().map(|l| l[k]).collect();
            let pivot = argmax(&colonne.iter().map(|x| x.abs()).collect::<Vec<f64>>());
            if colonne[pivot] < 0.0 {
                colonne.iter().map(|x| -x).collect()
            } else {
                colonne
            }
        })
        .collect();
    (valeurs, vecteurs)
}

// Rang numerique : nombre de valeurs propres au-dessus de 1e-8 fois la plus grande.
#[allow(dead_code)]
fn rang(valeurs: &[f64]) -> usize {
    let lmax = valeurs[0];
    valeurs.iter().filter(|v| **v > 1e-8 * lmax).count()
}

// OLS de norme minimale sans ordonnee, par K = X X^T : alpha = U_r L_r^{-1} U_r^T y, puis beta = X^T alpha.
// Les r valeurs propres non nulles fixent la prediction ; le noyau (p - r directions) ne la modifie pas.
#[allow(dead_code)]
fn ols_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64]) -> Vec<f64> {
    let lmax = valeurs[0];
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        if valeurs[k] > 1e-8 * lmax {
            let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
            for i in 0..n {
                alpha[i] += u[k][i] * d / valeurs[k];
            }
        }
    }
    alpha
}

// Coefficients duaux du ridge sans ordonnee : alpha = (K + lambda I)^{-1} y, dans la base des vecteurs propres.
#[allow(dead_code)]
fn ridge_alpha(valeurs: &[f64], u: &[Vec<f64>], y: &[f64], lambda: f64) -> Vec<f64> {
    let n = y.len();
    let mut alpha = vec![0.0; n];
    for k in 0..valeurs.len() {
        let d: f64 = (0..n).map(|i| u[k][i] * y[i]).sum();
        for i in 0..n {
            alpha[i] += u[k][i] * d / (valeurs[k] + lambda);
        }
    }
    alpha
}

// beta = X^T alpha ; renvoie les previsions X beta sur train et test, et la norme L2 de beta.
#[allow(dead_code)]
fn predire(xtr: &[Vec<f64>], xte: &[Vec<f64>], alpha: &[f64]) -> (Vec<f64>, Vec<f64>, f64) {
    let beta = transpose_mat_vec(xtr, alpha);
    let norme = beta.iter().map(|b| b * b).sum::<f64>().sqrt();
    (mat_vec_ligne(xtr, &beta), mat_vec_ligne(xte, &beta), norme)
}

// Bornes du pli f d'une validation croisee a plis contigus (KFold sans melange).
#[allow(dead_code)]
fn bornes_pli(n: usize, plis: usize, f: usize) -> (usize, usize) {
    let (base, reste) = (n / plis, n % plis);
    let debut = f * base + reste.min(f);
    (debut, debut + base + if f < reste { 1 } else { 0 })
}

// Un pli contigu, avec la decomposition spectrale du Gram de son train : elle sert a tous les alpha du ridge.
#[allow(dead_code)]
struct PliContigu {
    xtr: Vec<Vec<f64>>,
    xva: Vec<Vec<f64>>,
    positions_val: Vec<usize>,
    valeurs: Vec<f64>,
    u: Vec<Vec<f64>>,
}

#[allow(dead_code)]
fn plis_contigus(x: &[Vec<f64>], plis: usize) -> Vec<PliContigu> {
    let n = x.len();
    (0..plis)
        .map(|f| {
            let (debut, fin) = bornes_pli(n, plis, f);
            let xtr: Vec<Vec<f64>> = (0..n)
                .filter(|&i| i < debut || i >= fin)
                .map(|i| x[i].clone())
                .collect();
            let xva: Vec<Vec<f64>> = x[debut..fin].to_vec();
            let (valeurs, u) = propres_sym(&gram(&xtr));
            PliContigu {
                xtr,
                xva,
                positions_val: (debut..fin).collect(),
                valeurs,
                u,
            }
        })
        .collect()
}

// RidgeCV(alphas, cv=plis, fit_intercept=False) : l'alpha retenu est celui qui maximise la moyenne
// des R2 de pli (le score par defaut de la validation croisee de scikit-learn). Renvoie son indice.
#[allow(dead_code)]
fn ridge_cv_sur_plis(plis_c: &[PliContigu], y: &[f64], alphas: &[f64]) -> usize {
    let mut score = vec![0.0; alphas.len()];
    for pli in plis_c {
        let ytr: Vec<f64> = (0..y.len())
            .filter(|i| !pli.positions_val.contains(i))
            .map(|i| y[i])
            .collect();
        let yva: Vec<f64> = pli.positions_val.iter().map(|&i| y[i]).collect();
        let sst: f64 = centrer(&yva).iter().map(|x| x * x).sum();
        for (j, &lam) in alphas.iter().enumerate() {
            let (_, p_va, _) = predire(
                &pli.xtr,
                &pli.xva,
                &ridge_alpha(&pli.valeurs, &pli.u, &ytr, lam),
            );
            let sse: f64 = yva.iter().zip(&p_va).map(|(t, p)| (t - p) * (t - p)).sum();
            score[j] += 1.0 - sse / sst;
        }
    }
    argmax(&score)
}

#[allow(dead_code)]
fn ridge_cv_retenu(x: &[Vec<f64>], y: &[f64], alphas: &[f64], plis: usize) -> usize {
    ridge_cv_sur_plis(&plis_contigus(x, plis), y, alphas)
}

// Composantes principales depuis la decomposition de K = X X^T : composante_k = X^T u_k / sqrt(lambda_k).
#[allow(dead_code)]
fn composantes(x: &[Vec<f64>], valeurs: &[f64], u: &[Vec<f64>]) -> Vec<Vec<f64>> {
    let mut comps: Vec<Vec<f64>> = Vec::new();
    for k in 0..rang(valeurs) {
        let mut c = transpose_mat_vec(x, &u[k]);
        let s = valeurs[k].sqrt();
        for v in c.iter_mut() {
            *v /= s;
        }
        comps.push(c);
    }
    comps
}

// Centrage d'un echantillon sur ses propres colonnes : ce que fait PCA a l'ajustement.
#[allow(dead_code)]
fn centrer_propre(x: &[Vec<f64>]) -> Vec<Vec<f64>> {
    centrer_colonnes(x, &colonnes_moyennes(x))
}

// ACP, comme sklearn.decomposition.PCA : X est CENTRE sur ses colonnes, covariance (denominateur n),
// valeurs propres, part de variance, composantes. Le centrage ne change rien sur une matrice deja
// standardisee sur elle-meme (train complet) ; il est indispensable sur le train d'un pli, dont les
// colonnes ne sont pas de moyenne nulle.
#[allow(dead_code)]
fn acp(x: &[Vec<f64>]) -> (Vec<f64>, Vec<f64>, Vec<Vec<f64>>) {
    let xc = centrer_propre(x);
    let (valeurs, u) = propres_sym(&gram(&xc));
    let total: f64 = valeurs.iter().sum();
    let comps = composantes(&xc, &valeurs, &u);
    let ratio: Vec<f64> = valeurs.iter().map(|l| l / total).collect();
    (valeurs, ratio, comps)
}

// Variance cumulee jusqu'a la composante k comprise.
#[allow(dead_code)]
fn variance_cumulee(ratio: &[f64], k: usize) -> f64 {
    ratio[..=k].iter().sum()
}

// Scores t[i][k] = x_i . composante_k.
#[allow(dead_code)]
fn scores_composantes(x: &[Vec<f64>], comps: &[Vec<f64>]) -> Vec<Vec<f64>> {
    x.iter()
        .map(|l| {
            comps
                .iter()
                .map(|c| l.iter().zip(c).map(|(a, b)| a * b).sum::<f64>())
                .collect()
        })
        .collect()
}

// PCR a k composantes avec ordonnee : y = ybar + sum_{j<k} b_j t_j.
// Les scores d'une ACP sont orthogonaux deux a deux, donc b_j = (t_j . (y - ybar)) / (t_j . t_j).
#[allow(dead_code)]
fn regression_scores(
    t_tr: &[Vec<f64>],
    t_te: &[Vec<f64>],
    y: &[f64],
    k: usize,
) -> (Vec<f64>, Vec<f64>) {
    let ym = moyenne(y);
    let yc: Vec<f64> = y.iter().map(|v| v - ym).collect();
    let mut b = vec![0.0; k];
    for j in 0..k {
        let num: f64 = t_tr.iter().zip(&yc).map(|(t, &v)| t[j] * v).sum();
        let den: f64 = t_tr.iter().map(|t| t[j] * t[j]).sum();
        b[j] = num / den;
    }
    let projet = |tt: &[Vec<f64>]| -> Vec<f64> {
        tt.iter()
            .map(|t| ym + (0..k).map(|j| b[j] * t[j]).sum::<f64>())
            .collect()
    };
    (projet(t_tr), projet(t_te))
}

// RMSE test de la PCR pour k = 1..kmax (une seule ACP, une regression par k).
#[allow(dead_code)]
fn balayage_pcr(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (_, _, mut comps) = acp(xtr);
    comps.truncate(kmax);
    // PCA().fit(xtr).transform(...) : les deux echantillons sont centres sur la moyenne du train.
    let xmu = colonnes_moyennes(xtr);
    let (t_tr, t_te) = (
        scores_composantes(&centrer_colonnes(xtr, &xmu), &comps),
        scores_composantes(&centrer_colonnes(xte, &xmu), &comps),
    );
    (1..=kmax)
        .map(|k| rmse(yte, &regression_scores(&t_tr, &t_te, ytr, k).1))
        .collect()
}

// ---- PLS1 par NIPALS (l'algorithme de PLSRegression). x et y doivent etre centres sur le train.
// Renvoie (w, p, t) : poids x, loadings x et scores, ranges par composante : w[j][v], t[j][i]. ----
#[allow(dead_code)]
fn nipals(
    x: &[Vec<f64>],
    y_c: &[f64],
    kmax: usize,
) -> (Vec<Vec<f64>>, Vec<Vec<f64>>, Vec<Vec<f64>>) {
    let n = x.len();
    let nvar = x[0].len();
    let mut xk = x.to_vec();
    let mut yk = y_c.to_vec();
    let mut w_all: Vec<Vec<f64>> = Vec::new();
    let mut p_all: Vec<Vec<f64>> = Vec::new();
    let mut t_all: Vec<Vec<f64>> = Vec::new();
    for _j in 0..kmax {
        // w = Xk^T yk / ||Xk^T yk|| (direction de covariance maximale avec le reste de y), t = Xk w
        let mut w = transpose_mat_vec(&xk, &yk);
        let nrm = w.iter().map(|v| v * v).sum::<f64>().sqrt();
        for v in w.iter_mut() {
            *v /= nrm;
        }
        let t = mat_vec_ligne(&xk, &w);
        let ttt: f64 = t.iter().map(|v| v * v).sum();
        // p = Xk^T t / (t^T t) ; q = yk^T t / (t^T t)
        let mut pp = transpose_mat_vec(&xk, &t);
        for v in pp.iter_mut() {
            *v /= ttt;
        }
        let q: f64 = t.iter().zip(&yk).map(|(a, b)| a * b).sum::<f64>() / ttt;
        // Deflation : Xk <- Xk - t p^T ; yk <- yk - q t
        for i in 0..n {
            for j in 0..nvar {
                xk[i][j] -= t[i] * pp[j];
            }
            yk[i] -= q * t[i];
        }
        w_all.push(w);
        p_all.push(pp);
        t_all.push(t);
    }
    (w_all, p_all, t_all)
}

// Inverse d'une matrice carree (Gauss-Jordan, pivot partiel) ; None si un pivot est nul.
#[allow(dead_code)]
fn inverse(m: &[Vec<f64>]) -> Option<Vec<Vec<f64>>> {
    let n = m.len();
    let mut a: Vec<Vec<f64>> = m.to_vec();
    let mut inv: Vec<Vec<f64>> = (0..n)
        .map(|i| (0..n).map(|j| if i == j { 1.0 } else { 0.0 }).collect())
        .collect();
    for col in 0..n {
        let mut piv = col;
        for r in (col + 1)..n {
            if a[r][col].abs() > a[piv][col].abs() {
                piv = r;
            }
        }
        if !a[piv][col].is_finite() || a[piv][col].abs() < 1e-14 {
            return None;
        }
        a.swap(col, piv);
        inv.swap(col, piv);
        let d = a[col][col];
        for j in 0..n {
            a[col][j] /= d;
            inv[col][j] /= d;
        }
        for r in 0..n {
            if r != col {
                let f = a[r][col];
                for j in 0..n {
                    a[r][j] -= f * a[col][j];
                    inv[r][j] -= f * inv[col][j];
                }
            }
        }
    }
    Some(inv)
}

// Rotations de la PLS, R = W (P^T W)^{-1} : elles appliquent le modele a des lignes nouvelles
// (t = X_c R). M = P^T W etant triangulaire superieure, les k premieres colonnes de R sont celles
// du modele a k composantes.
#[allow(dead_code)]
fn rotations(w: &[Vec<f64>], pmat: &[Vec<f64>], kmax: usize) -> Option<Vec<Vec<f64>>> {
    let nvar = w[0].len();
    let m: Vec<Vec<f64>> = (0..kmax)
        .map(|i| {
            (0..kmax)
                .map(|j| pmat[i].iter().zip(&w[j]).map(|(a, b)| a * b).sum())
                .collect()
        })
        .collect();
    let inv = inverse(&m)?;
    let r: Vec<Vec<f64>> = (0..nvar)
        .map(|v| {
            (0..kmax)
                .map(|j| (0..kmax).map(|i| w[i][v] * inv[i][j]).sum())
                .collect()
        })
        .collect();
    Some(r)
}

// Poids cotes variables du modele PLS a k = 1..kmax composantes : betas[k - 1] pour k composantes.
// b_j = (t_j . y_c) / (t_j . t_j), puis beta_k = sum_{j<k} b_j R[:, j].
#[allow(dead_code)]
fn pls_poids(
    w: &[Vec<f64>],
    pmat: &[Vec<f64>],
    t: &[Vec<f64>],
    y_c: &[f64],
) -> Option<Vec<Vec<f64>>> {
    let kmax = t.len();
    let n = t[0].len();
    let nvar = w[0].len();
    let r = rotations(w, pmat, kmax)?;
    let mut betas: Vec<Vec<f64>> = Vec::new();
    let mut beta = vec![0.0; nvar];
    for j in 0..kmax {
        let num: f64 = (0..n).map(|i| t[j][i] * y_c[i]).sum();
        let den: f64 = t[j].iter().map(|v| v * v).sum();
        let b = num / den;
        for v in 0..nvar {
            beta[v] += b * r[v][j];
        }
        betas.push(beta.clone());
    }
    Some(betas)
}

// Prevision PLS pour des lignes deja centrees : y = ybar + x_c . beta.
#[allow(dead_code)]
fn pls_predire(x_c: &[Vec<f64>], y_moy: f64, beta: &[f64]) -> Vec<f64> {
    x_c.iter()
        .map(|l| y_moy + l.iter().zip(beta).map(|(a, b)| a * b).sum::<f64>())
        .collect()
}

// RMSE test de la PLS pour k = 1..kmax (un seul NIPALS, un vecteur de poids par k) ;
// NaN si les rotations ne sont pas calculables.
#[allow(dead_code)]
fn balayage_pls(
    xtr: &[Vec<f64>],
    ytr: &[f64],
    xte: &[Vec<f64>],
    yte: &[f64],
    kmax: usize,
) -> Vec<f64> {
    let (xmu, y_moy) = (colonnes_moyennes(xtr), moyenne(ytr));
    let yc = centrer(ytr);
    let (w, pmat, t) = nipals(&centrer_colonnes(xtr, &xmu), &yc, kmax);
    let betas = match pls_poids(&w, &pmat, &t, &yc) {
        Some(b) => b,
        None => return vec![f64::NAN; kmax],
    };
    let xte_c = centrer_colonnes(xte, &xmu);
    (1..=kmax)
        .map(|k| rmse(yte, &pls_predire(&xte_c, y_moy, &betas[k - 1])))
        .collect()
}

// RMSE test de la PLS ajustee avec exactement k composantes.
#[allow(dead_code)]
fn rmse_pls_k(xtr: &[Vec<f64>], ytr: &[f64], xte: &[Vec<f64>], yte: &[f64], k: usize) -> f64 {
    match balayage_pls(xtr, ytr, xte, yte, k).last() {
        Some(r) => *r,
        None => f64::NAN,
    }
}

// ---- Selection de k par validation croisee sur le train : 5 plis melanges, decoupage derive de la graine.
// L'ACP et la PLS sont refittees sur le train de chaque pli ; la standardisation globale du train n'est
// pas refaite (meme fuite de pretraitement que la source). L'erreur est pooled sur les 120 previsions
// hors pli. Renvoie (k PCR retenu, k PLS retenu, RMSE de CV par k).
#[allow(dead_code)]
fn cv_k(
    x: &[Vec<f64>],
    y: &[f64],
    kmax: usize,
    plis: usize,
    graine: u64,
) -> (usize, usize, Vec<f64>, Vec<f64>) {
    let n = x.len();
    let ordre = permutation(&mut Rng::new(graine ^ 0xA5A5A5A5), n);
    let mut carres_pcr = vec![0.0; kmax];
    let mut carres_pls = vec![0.0; kmax];
    for f in 0..plis {
        let (debut, fin) = bornes_pli(n, plis, f);
        let mut itr: Vec<usize> = ordre[..debut].to_vec();
        itr.extend_from_slice(&ordre[fin..]);
        let xtr: Vec<Vec<f64>> = itr.iter().map(|&i| x[i].clone()).collect();
        let ytr: Vec<f64> = itr.iter().map(|&i| y[i]).collect();
        let xva: Vec<Vec<f64>> = ordre[debut..fin].iter().map(|&i| x[i].clone()).collect();
        let yva: Vec<f64> = ordre[debut..fin].iter().map(|&i| y[i]).collect();
        // PCR : ACP refittee sur le train du pli, scores du train et du pli de validation centres sur la
        // moyenne du train du pli (contrat PCA().fit(train).transform(...)).
        let xmu = colonnes_moyennes(&xtr);
        let (_, _, mut comps) = acp(&xtr);
        comps.truncate(kmax);
        let (t_tr, t_va) = (
            scores_composantes(&centrer_colonnes(&xtr, &xmu), &comps),
            scores_composantes(&centrer_colonnes(&xva, &xmu), &comps),
        );
        // PLS : NIPALS ajuste sur le train du pli, X et y centres sur ce train
        let rmse_pls = balayage_pls(&xtr, &ytr, &xva, &yva, kmax);
        for k in 1..=kmax {
            let (_, p_pcr) = regression_scores(&t_tr, &t_va, &ytr, k);
            carres_pcr[k - 1] += rmse(&yva, &p_pcr) * rmse(&yva, &p_pcr) * yva.len() as f64 / n as f64;
            carres_pls[k - 1] += rmse_pls[k - 1] * rmse_pls[k - 1] * yva.len() as f64 / n as f64;
        }
    }
    let racines = |c: &[f64]| -> Vec<f64> { c.iter().map(|v| v.sqrt()).collect() };
    let (rp, rq) = (racines(&carres_pcr), racines(&carres_pls));
    (argmin(&rp) + 1, argmin(&rq) + 1, rp, rq)
}

// ---- 9bis. Robustesse : le verdict sur dix graines ----
// Une graine complete : blocs, bruit, poids, decoupage, standardisation, validation croisee de lambda et
// de k, puis les huit RMSE test. Les deux regimes partagent X : plis et decomposition spectrale servent deux fois.
#[allow(dead_code)]
fn experience_complete(graine: u64, alphas: &[f64], kmax: usize) -> (Vec<f64>, Vec<f64>, usize, usize) {
    let j = jeu(graine);
    let pl = preparer(&j, &cible(&j));
    let pa = preparer(&j, &cible_alignee(&j));
    // Les k sont selectionnes sur le regime leurre, puis reutilises pour le regime aligne.
    let (k_pcr, k_pls, _, _) = cv_k(&pl.x_train_s, &pl.y_train_z, kmax, 5, graine);
    let plis_ridge = plis_contigus(&pl.x_train_s, 5);
    let (valeurs, u) = propres_sym(&gram(&pl.x_train_s));
    let mut comps = composantes(&pl.x_train_s, &valeurs, &u);
    comps.truncate(k_pcr);
    let (t_tr, t_te) = (
        scores_composantes(&pl.x_train_s, &comps),
        scores_composantes(&pl.x_test_s, &comps),
    );
    let bilan = |p: &Prep| -> Vec<f64> {
        let y: &[f64] = &p.y_train_z;
        let ols = rmse(&p.y_test_z, &predire(&p.x_train_s, &p.x_test_s, &ols_alpha(&valeurs, &u, y)).1);
        let lam = alphas[ridge_cv_sur_plis(&plis_ridge, y, alphas)];
        let ridge = rmse(
            &p.y_test_z,
            &predire(&p.x_train_s, &p.x_test_s, &ridge_alpha(&valeurs, &u, y, lam)).1,
        );
        let pcr = rmse(&p.y_test_z, &regression_scores(&t_tr, &t_te, y, k_pcr).1);
        let pls = rmse_pls_k(&p.x_train_s, y, &p.x_test_s, &p.y_test_z, k_pls);
        vec![ols, ridge, pcr, pls]
    };
    (bilan(&pl), bilan(&pa), k_pcr, k_pls)
}

// Une colonne d'un tableau de resultats (une ligne par graine).
#[allow(dead_code)]
fn colonne_resultats(tab: &[Vec<f64>], k: usize) -> Vec<f64> {
    tab.iter().map(|l| l[k]).collect()
}

// Nombre de lignes ou la premiere methode (OLS) est derniere ou ex aequo.
#[allow(dead_code)]
fn nb_derniere(tab: &[Vec<f64>]) -> usize {
    tab.iter()
        .filter(|l| l[0] >= l.iter().cloned().fold(f64::NEG_INFINITY, f64::max))
        .count()
}

// Plus petite et plus grande valeur d'une liste.
#[allow(dead_code)]
fn etendue(v: &[usize]) -> (usize, usize) {
    (v.iter().cloned().min().unwrap_or(0), v.iter().cloned().max().unwrap_or(0))
}

let graines = [0u64, 1, 2, 3, 7, 11, 21, 42, 99, 123];
let (alphas, kmax) = (logspace(-3.0, 3.0, 25), 45usize);
let mut leurre: Vec<Vec<f64>> = Vec::new();
let mut aligne: Vec<Vec<f64>> = Vec::new();
let mut k_pcr_tous: Vec<usize> = Vec::new();
let mut k_pls_tous: Vec<usize> = Vec::new();
for g in graines.iter() {
    let (l, a, kp, kl) = experience_complete(*g, &alphas, kmax);
    leurre.push(l);
    aligne.push(a);
    k_pcr_tous.push(kp);
    k_pls_tous.push(kl);
}

println!("{:18} {:>20} {:>20}", "methode", "regime leurre", "regime aligne");
for (k, m) in ["OLS min-norme", "Ridge (lambda CV)", "PCR", "PLS"].iter().enumerate() {
    let (lm, ls) = (
        moyenne(&colonne_resultats(&leurre, k)),
        ecart_type(&colonne_resultats(&leurre, k)),
    );
    let (am, ea) = (
        moyenne(&colonne_resultats(&aligne, k)),
        ecart_type(&colonne_resultats(&aligne, k)),
    );
    println!("{:18} {:12.3} +- {:5.3} {:12.3} +- {:5.3}", m, lm, ls, am, ea);
}
println!(
    "\nOLS min-norme derniere : {}/10 graines (leurre), {}/10 (aligne)",
    nb_derniere(&leurre),
    nb_derniere(&aligne)
);
let (p_min, p_max) = etendue(&k_pcr_tous);
let (q_min, q_max) = etendue(&k_pls_tous);
println!(
    "k retenus par graine   : PCR {}-{}, PLS {}-{}",
    p_min, p_max, q_min, q_max
);

methode                   regime leurre        regime aligne


OLS min-norme             0.327 +- 0.058        0.670 +- 0.159


Ridge (lambda CV)         0.300 +- 0.048        0.581 +- 0.115


PCR                       0.289 +- 0.044        0.611 +- 0.077


PLS                       0.275 +- 0.038        0.591 +- 0.110


OLS min-norme derniere : 7/10 graines (leurre), 6/10 (aligne)


k retenus par graine   : PCR 44-45, PLS 6-25


### Lecture du résultat

Sur ces dix graines, les sorties comptent les cas où OLS min-norme est dernière ou ex æquo : **7 fois sur 10 dans le régime leurre**, **6 fois sur 10 dans le régime aligné**. Les moyennes disent le reste :

- **Régime leurre** : la PLS mène ($0{,}275 \pm 0{,}038$), juste devant la PCR ($0{,}289 \pm 0{,}044$) — quand le signal vit hors des directions de grande variance, les composantes qui regardent la cible restent le bon geste. L'écart moyen avec OLS ($0{,}327 \pm 0{,}058$) est modeste mais systématique.
- **Régime aligné** : le ridge mène ($0{,}581 \pm 0{,}115$) — quand le signal s'aligne sur la variance dominante, la pénalité suffit, et payer des composantes (PCR $0{,}611 \pm 0{,}077$) n'apporte rien. C'est aussi le régime où l'écart-type d'OLS s'envole ($\pm 0{,}159$) : la min-norme présente ici une dispersion notable selon le tirage.
- **Les $k$ retenus varient avec la graine** : la PLS va de 6 à 25 composantes — elle n'est parcimonieuse que sur certains tirages ; la PCR CV ne trouve pas de coude net et reste au plafond de la grille (44 à 45 sur 45) — un $k$ « choisi » à la limite haute est un signal à lire, pas un nombre à citer.

Dix graines, deux régimes, hyperparamètres re-choisis à chaque tirage : les moyennes observées diffèrent selon le régime ; ces dix tirages ne garantissent pas un ordre identique pour chaque graine ni une loi universelle.

## 10. Exercices

Les trois exercices sont distribués après les sections ridge, PCR et comparaison des régimes. Les gardes contrôlent les quantités calculables ; les interprétations restent à discuter.

## À retenir

Trois réponses au même régime p > n : **ridge** rend le problème déterminé ($X^\top X + \lambda I$ inversible pour tout $\lambda > 0$) mais reste une régression sur *toutes* les directions ; **PCR** réduit sur des composantes choisies sans regarder Y — efficace quand la cible vit dans les variances dominantes, dispendieuse (45 composantes ici, au plafond de la grille) quand elle s'y cache ; **PLS** trie les directions par covariance avec la cible et trouve le signal en 7 composantes. La leçon transférable : *la variance d'une direction ne suffit pas à établir sa valeur prédictive* — Cov(Y, Xu) et Var(Xu) sont deux classements différents, et l'écart entre les deux est exploitable.

En pratique : commencer par la validation croisée ridge (un seul hyperparamètre, robuste) ; passer à la PLS quand p > n est extrême ou quand l'interprétation par composantes (loadings) a de la valeur ; la PCR reste pertinente quand les composantes principales ont un sens métier propre. C'est aussi le geste du pipeline QuantConnect du cours ([ML-Training-Pipeline](../../../QuantConnect/ML-Training-Pipeline/README.md)), où la régression ridge régularise les prédictions en présence de caractéristiques nombreuses et corrélées — sans re-développer ici de cas trading.

| Concept | Mesure dans ce carnet |
|---|---|
| Le régime, posé | `n_train = 120`, `n_test = 40`, `p = 205` (`p/n = 1.71`) ; `R²` du modèle vrai `0.973` ; plancher de bruit `≈ 0.169` |
| OLS n'est pas mauvais, il est indéterminé | `rang(X) = 119` sur `205` colonnes : `86` directions du noyau de X_train, donc coefficients non identifiables sur ce train ; RMSE train `0.0000`, test `0.2206` (1,3 fois le bruit) |
| Ridge rend le problème déterminé | `λ CV = 1.0000` : RMSE test `0.2094` ; le balayage va de `0.221` (λ=1e-3) au minimum `0.179` de cette grille test, puis `0.779` (λ=1e3) |
| PCR paie son aveuglement | meilleur `k = 45` pour `0.2019`, et c'est le plafond de la grille ; à `k=5` elle vaut encore `0.9673` : aucune composante ne porte plus de `10.4%` du bloc B (maximum à l'indice 43) |
| PLS regarde la cible | meilleur `k = 7` pour `0.1800` ; à budget égal (`k=5`), PCR est `5.1` fois moins bonne ; la part du bloc B dans `w₁` vaut `68.2%` (PCA : `2.44%`) |
| La CV choisit à côté, mais honnêtement | k retenus `PCR 45` / `PLS 9` contre optimaux de test `45` / `7` ; RMSE au k choisi : `0.2019` / `0.1895` |
| Le verdict dépend du régime | régime leurre : OLS `0.2206` ; régime aligné : OLS `0.5260` — et sur dix graines, `0.327 ± 0.058` contre `0.670 ± 0.159` |

**La leçon transférable.** La variance d'une direction ne suffit pas à établir sa valeur prédictive : `Cov(Y, Xu)` et `Var(Xu)` sont deux classements différents, et l'écart entre les deux est exploitable. En pratique : commencer par la validation croisée ridge (un hyperparamètre, robuste), passer à la PLS quand `p > n` est extrême ou quand les composantes ont valeur d'interprétation.

**Transition.** Le carnet suivant, [2.3d-Modele-Gaussien-LDA-QDA](2.3d-Modele-Gaussien-LDA-QDA.ipynb), revient à la classification et montre une autre hypothèse structurante : la forme des covariances de classes, qui décide de la frontière.

## Vérifiez votre compréhension

**1. `rang(X_train) = 119` sur 205 colonnes, et pourtant `RMSE train = 0.0000`. Pourquoi un ajustement parfait sur le train est-il ici la signature du problème, pas d'un bon modèle ?**

<details>
<summary>Réponse</summary>

Avec 120 observations centrées, l'espace des lignes n'a que 119 dimensions : le système est sous-déterminé, et la solution de norme minimale l'ajuste *exactement* — zéro résidu d'entraînement par construction. Les 86 directions du noyau ne modifient pas les prédictions sur X_train : les coefficients sont donc non identifiables sur ce train. Le rang établit cette indétermination ; le RMSE test (`0.2206`) en est une mesure distincte de généralisation, pas une preuve du rang.
</details>

**2. À `k=5` composantes, PCR affiche `0.9673` quand PLS affiche `0.1909`. D'où vient un écart aussi grand à budget égal ?**

<details>
<summary>Réponse</summary>

Les composantes PCA sont triées par variance expliquée, sans regarder `Y` : ici la 1re composante ne porte que `0.051` de la variance et `0.0244` seulement de la contribution du bloc signal ; la composante la plus chargée en signal est l'indice 43 (part `0.104`, variance cumulée `0.904`), et aucune n'atteint 15%. La PLS trie ses directions par covariance avec la cible : sa première direction place déjà `22%` de sa masse sur les 5 variables du signal. À `k=5`, l'une a presque tout le signal, l'autre presque rien.
</details>

**3. La validation croisée retient `k = 45` (PCR) et `k = 9` (PLS) alors que les optimaux de test sont `45` et `7`. La CV a-t-elle échoué ?**

<details>
<summary>Réponse</summary>

Non : elle a fait son travail, qui est de choisir *sans regarder le test*. Pour la PCR, la CV retient le plafond de la grille parce que sa courbe n'a pas de coude — les composantes de bruit ne coûtent presque plus une fois le signal inclus. Pour la PLS, elle se trompe de deux composantes, sur une courbe plate autour de l'optimum (estimation sur 120 exemples) : le RMSE obtenu aux k retenus (`0.2019` / `0.1895`) reste proche des meilleurs (`0.2019` / `0.1800`). Exiger de la CV qu'elle devine l'argmax exact du test, c'est lui demander de lire le test — précisément ce qu'elle existe pour éviter.
</details>

## Pour aller plus loin

- [README de la sous-série 02-ML-Cours](README.md) — la carte des carnets et de leurs concepts-phares.
- [README de la série DataScienceWithAgents](../README.md) — le rôle de ce socle ML dans la formation.
- [2.3d-Modele-Gaussien-LDA-QDA](2.3d-Modele-Gaussien-LDA-QDA.ipynb) — la suite immédiate : le modèle gaussien et ses frontières.
- [2.6-Clustering-KMeans-PCA](2.6-Clustering-KMeans-PCA.ipynb) — l'ACP comme méthode en soi (structure sans étiquettes), ici utilisée comme simple prétraitement.
- [2.4-Arbres-Forets-Ensembles](2.4-Arbres-Forets-Ensembles.ipynb) — la réponse non paramétrique au même régime : des modèles qui ne postulent pas de forme linéaire.

## Références

- **Hoerl, A. E. & Kennard, R. W. (1970).** *Ridge regression: biased estimation for nonorthogonal problems.* Technometrics 12(1). La pénalité $\ell_2$ qui rend un problème mal conditionné déterminé.
- **Wold, H. (1975).** *Path models with latent variables: the NIPALS approach.* Dans Quantitative Sociology. L'algorithme qui extrait des composantes corrélées à la cible — le point de départ de la PLS, et l'algorithme écrit à la main dans le préambule de ce carnet.
- **de Jong, S. (1993).** *SIMPLS: an alternative approach to partial least squares regression.* Chemometrics and Intelligent Laboratory Systems 18(3). Une autre formulation de PLS ; la variante retenue ici est celle de Wold.
- **Frank, I. E. & Friedman, J. H. (1993).** *A statistical view of some chemometrics regression tools.* Technometrics 35(2). Pourquoi ridge, PCR et PLS forment un continuum de pénalisations, et quand chacune gagne.
- **Hastie, T., Tibshirani, R. & Friedman, J. (2009).** *The Elements of Statistical Learning*, §3.4-3.5. Le traitement de référence du régime $p > n$ et des estimateurs à rétrécissement.
- **Jolliffe, I. T. (2002).** *Principal Component Analysis*, 2e éd. La hiérarchie de variance que la PCR consomme — et perd quand le signal vit ailleurs.
- **Pedregosa, F. et al. (2011).** *Scikit-learn: machine learning in Python.* JMLR 12. `RidgeCV`, `PCA`, `PLSRegression`, `cross_val_predict` — employés par la confrontation hors carnet qui a servi d'oracle à ce portage.

Les références bibliographiques sont reprises de la source versionnée du cours ; aucune lecture des publications n'est présumée pour la présente comparaison.

*Implémentation Rust* : aucune crate — algèbre linéaire, spectre par rotations de Jacobi, inversion, NIPALS et validation croisée écrits dans le préambule, confrontés hors carnet à scikit-learn 1.5.2 sur les **mêmes entrées** (59 quantités, écarts maximaux de l'ordre de $10^{-13}$).